#Constants and Underlying Functions

Game layout:

*   [...] is a game (a list of all regions)
*   [(...)] is a specific region (a tuple of loops)
*   [(...,###,...)]] is the enumeration of a loop within the region
*   (FREE) is a point not connected to anything.  Only one Loop has a FREE in it.  Current value = 0
*  (SING) is singleton-  a point connected to two edges, and which only appears once (i.e. one of the regions it's in is dead).  Current value = 2
*  (HANG) is a hanging point- a point connected to only one edge.  Current value = 1
*  (-3) is a connector point- a point that connects two loops, and is just a placeholder for connecting two loops
*  (BOUN) is a boundary point-  a point connected to two edges, which appears once in two different regions.  Current value = 3
*  (x>=SPLITSTART) is a point connected to two edges, which appears twice in the same loop.  Converted to a SPLITSTART the first time it's seen in the enumeration, and a SPLITEND the second time it's seen
*  (#####) is a loop of points, comprising of points connected to one or two edges

Other terminology:
*  Zero-loop:  A loop created by connecting an unconnected point (FREE) to itself
*  Free: A short version of "unconnected point"
*  Revisited: A point that is seen both sides on in a single loop
*  Enclosures: When connecting a loop to itself, two new are made, with at least one boundary point between them. Each side is an 'enclosure'
*  Deletions:  The loop you get when you delete one or two boundary points
*  Degenerations:  The loop you get when one or two boundary points end up having nothing to connect to on the other side (i.e. a boundary turns into a singleton)
*  Rotations:  The same loop but described from a different starting point
*  Pairings:  Whenever a new boundary is created, there are two sides to the boundary point.
*  Double-Pair Loop:  A loop that's just two boundaries, which both link to the same otherwise-empty region
*  Triple-Pair Loop:  A loop that's just three boundaries, which all link to the same otherwise-empty region
*  Disappearing point:  A boundary that links to a region whose only other point is a singleton (may or may not be in its own loop)

In [1]:
#@title Imports and drive mounting
import re
from random import randrange
from datetime import datetime

root_path = "/content/sample_data"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    root_path = "/content/drive/MyDrive/SproutsData"
    print(root_path)
except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SproutsData


In [2]:
#@title Define constants
HANG       =  1
SING       =  2
BOUN       =  3
SPLITSTART =  4
SPLITEND   =  SPLITSTART+1
FREE       =  0
EMPT       = -2
CONNECTOR  = -3
DOUPLEPAIRLOOP = -11
LONELYBOUNDARYPAIR = -12
LONELYBOUNDARYTRIPLE = -13
DISAPOINT  = (-2, -2, -2)
DEADPOINT  = (-1, -1, -1)
DISALOOP   = (-2, -2)
DISABOUNDCHECKLOOP   = (-3, -3)
DEADLOOP   = (-1, -1)
LEFTAPPEARINGLOOP = -4
RIGHTAPPEARINGLOOP = -5
DEADMAP    = -1
NOTBOUN    = -1
DISABOUNCHECK = -3
BINLEFT    = 0
BINRIGHT   = 1
BINCENTER  = 2
LEFT       = 0
RIGHT      = 1
EMPTYGAME  = 'NULL'
EMPTYGAMECOMPRESSED = "N"

ENCLOSUREENCODING      =  0
MERGEENCODING          =  1
NONMOVEENCODING        = -1
DISADROPENCODING       = -2
LONELYPAIRDROPENCODING = -3

SIDEENCODING = {}
SIDEENCODING['neither'] = 0
SIDEENCODING['left'   ] = 1
SIDEENCODING['right'  ] = 2
SIDEENCODING['both'   ] = 3

SIDEDECODING = {}
for key, val in SIDEENCODING.items():
    SIDEDECODING[val] = key

EMPTYENCLOSUREMULTIPLIER = len([ENCLOSUREENCODING, MERGEENCODING]) * len(SIDEENCODING) * (SPLITSTART + 1) * (SPLITSTART + 1)


STRHANG       = str(HANG)
STRSING       = str(SING)
STRBOUN       = str(BOUN)
STRSPLITSTART = str(SPLITSTART)
STRSPLITEND   = str(SPLITEND)
STREMPT       = str(EMPT)
STRFREE       = str(FREE)
STRANYPOINT   = {STRFREE:True, STRHANG:True, STRSING:True, STRBOUN:True, STRSPLITSTART:True, STRSPLITEND:True}
POINTENCODINGS = {FREE:'free', HANG:'hanging', SING:'singleton', BOUN:'boundary', SPLITSTART:'split'}

STRLONELYBOUNDARYPAIR = '6'
STRLONELYBOUNDARYTRIPLE = '9'

MINPOINT = min([FREE, HANG, SING, BOUN, SPLITSTART, SPLITEND])
SPECIALLOOPS = [DOUPLEPAIRLOOP, LONELYBOUNDARYPAIR, LONELYBOUNDARYTRIPLE]

In [3]:
#@title Verify constants meet necessary assumptions
allConsts = {'HANG':HANG, 'SING':SING, "BOUN":BOUN, "SPLITSTART":SPLITSTART, "SPLITEND":SPLITEND, "FREE":FREE, "EMPT":EMPT, "CONNECTOR":CONNECTOR}
for key1 in allConsts:
    for key2 in allConsts:
        if key1 != key2 and allConsts[key1] == allConsts[key2]:
            print("ERROR 0-0-0:  All point-type constants must be unique.", key1, "==", key2)
            exit()

if FREE < 0 or HANG < 0  or SING < 0 or BOUN < 0 or SPLITSTART < 0 or SPLITEND < 0:
    print("ERROR 0-0-1:  One of the point-type constants is negative.  Please ensure that the types are >= 0, as negatives are reserved for various other indicators", allConsts)
    exit()

if HANG < 1  or SING < 1 or BOUN < 1 or SPLITSTART < 1 or SPLITEND < 1:
    print("ERROR 0-0-2:  One of the point-type constants (other than FREE) is 0.  Since these are digits in numbers and can't be dropped due to being leading zeros, they need to be between 1 and 9.", allConsts)
    exit()

if FREE > 9 or HANG > 9 or SING > 9 or BOUN > 9 or SPLITSTART > 9 or SPLITEND > 9:
    print("ERROR 0-0-3:  One of the point-type constants is > 9.  Since these are digits in numbers, they need to be between 1 and 9.", allConsts)
    exit()

if SPLITSTART < FREE or SPLITSTART < HANG or SPLITSTART < SING or SPLITSTART < BOUN:
    print("ERROR 0-0-4:  SPLITSTART must be larger than the other point-type constants.", allConsts)
    exit()

if SPLITEND != SPLITSTART+1:
    print("ERROR 0-0-5:  Please keep SPLITEND = SPLITSTART+1 .  It's not technically crucial, but it helps keep things clearer", allConsts)
    exit()

if LEFT == RIGHT:
    print("ERROR 0-0-6:  LEFT == RIGHT.  Don't do that", allConsts)
    exit()

if LEFTAPPEARINGLOOP == RIGHTAPPEARINGLOOP:
    print("ERROR 0-0-7:  LEFTAPPEARINGLOOP == RIGHTAPPEARINGLOOP.  Don't do that", allConsts)
    exit()

if LEFT == LEFTAPPEARINGLOOP or LEFT == RIGHTAPPEARINGLOOP or RIGHT == LEFTAPPEARINGLOOP or RIGHT == RIGHTAPPEARINGLOOP:
    print("ERROR 0-0-8:  LEFT, RIGHT, LEFTAPPEARINGLOOP, and RIGHTAPPEARINGLOOP all need to be different numbers", allConsts)
    exit()

if ENCLOSUREENCODING == MERGEENCODING:
    print("ERROR 0-0-9:  ENCLOSUREENCODING and MERGEENCODING need to be different numbers")
    exit()

adjacentDensityCheck = []
for encoding in SIDEENCODING.values():
    adjacentDensityCheck.append(encoding)
adjacentDensityCheck.sort()
for index in range(len(adjacentDensityCheck)):
    if adjacentDensityCheck[index] != index:
        print("ERROR 0-0-A:  SIDEENCODING is not a set of numbers 0-n without gaps")
        exit()

In [4]:
#@title binaryTreeSet class (used for enclosures)
class binaryTreeSet:
    binaryList = None
    centerAdjusted = None

    def genbin(self, size, n = None, holdList = None):
        if n is None:
            n = size
        if holdList is None:
            holdList = ()
        remaining = n - 1
        if remaining >= 0:
            self.genbin(size, remaining, holdList + (0,))
            self.genbin(size, remaining, holdList + (1,))
        else:
            self.binaryList[size].append(holdList)
            self.centerAdjusted[holdList] = {}
            for i in range(size+1):
                self.centerAdjusted[holdList][i] = holdList[:i] + (BINCENTER,) + holdList[i:]

    def addSize(self, size):
        if size not in self.binaryList:
            self.binaryList[size] = []
            self.genbin(size)

    def __init__(self):
        self.binaryList = {}
        self.centerAdjusted = {}

In [5]:
#@title permutationMap class (used for canonization)
class permutationMap:
    permutationList = None

    def recursePermutes(self, size, remainingDests, permutationConstruction = None):
        if permutationConstruction is None:
            permutationConstruction = []

        if len(remainingDests) == 0:
            self.permutationList[size].append(permutationConstruction)
            return

        for destIndex in range(len(remainingDests)):
            newRemainingDests = remainingDests[:destIndex] + remainingDests[destIndex+1:]
            newConstruction = permutationConstruction[:] + [remainingDests[destIndex]]
            self.recursePermutes(size, newRemainingDests, permutationConstruction = newConstruction)


    def addSize(self, size):
        #We don't need to do this if the permutation is already built (though realistically, we shouldn't even be calling this in that case)
        if size in self.permutationList:
            return

        self.permutationList[size] = []
        remainingDests = []
        for index in range(size):
            remainingDests.append(index)

        self.recursePermutes(size, remainingDests)

    def __init__(self):
        self.permutationList = {}

In [6]:
#@title tuptupDict class (Used everywhere to save a ton of memory)
#Tuples of two-tuples and their corresponding dicts.  Made so that shorter dicts can be compressed to tuple-size instead of having a full dict-sized object each and every time
class tuptupDicts:
    tuptupToDict = None

    #Shift boundaries of a loop
    def tuptupFromBoundShifts(self, boundaryMapping, indexShift, loopLen):
        newTuptup = []
        for pointIndex in boundaryMapping:
            newTuptup.append((pointIndex, (boundaryMapping[pointIndex]-indexShift) % loopLen))
        return self.getDictFromUnsorted(newTuptup)


    #Link boundaries for when both sides are a pair (for enclosures)
    def tuptupFromPairedBounds(self, boundaryMapping, indexShift, loopLen, pairingShift, pairingLen):
        newTuptup = []
        for pointIndex in boundaryMapping:
            newTuptup.append(((pointIndex - indexShift) % loopLen, (boundaryMapping[pointIndex]-pairingShift) % pairingLen))
        return self.getDictFromUnsorted(newTuptup)


    #Shift boundaries for a deletion
    def deletetionBoundShift(self, unshiftedBoundaryMapping, newShift):
        adjustedBounds = unshiftedBoundaryMapping[newShift:] + unshiftedBoundaryMapping[:newShift]
        return self.boundShiftViaArray(adjustedBounds)


    #Shift boundaries in a generalized way via an array
    def boundShiftViaArray(self, shiftedBoundArray):
        newTuptup = []
        for boundIndex in range(len(shiftedBoundArray)):
            if shiftedBoundArray[boundIndex] != NOTBOUN:
                newTuptup.append((shiftedBoundArray[boundIndex], boundIndex))
        return self.getDictFromUnsorted(newTuptup)


    #Give it a sorted dict, get a map back.  The map is made here if one doesn't already exist.
    def getDict(self, tuptup):
        if tuptup not in self.tuptupToDict:
            newDict = {}
            for tup in tuptup:
                newDict[tup[0]] = tup[1]
            self.tuptupToDict[tuptup] = newDict
        return self.tuptupToDict[tuptup]


    #This can recieve a natural tuple-of-tuples OR a list of tuples
    def getDictFromUnsorted(self, tuptup):
        tuptup = list(tuptup)
        tuptup.sort()
        tuptup = tuple(tuptup)

        if tuptup not in self.tuptupToDict:
            newDict = {}
            for tup in tuptup:
                newDict[tup[0]] = tup[1]
            self.tuptupToDict[tuptup] = newDict
        return self.tuptupToDict[tuptup]


    #When making a tuptup natively is too messy, allow for a dict to be sent here, and return the perminantly-saved dict within this object
    def getDictFromDict(self, inputDict):
        tuptup = recreateTupTupFromDict(inputDict)

        if tuptup not in self.tuptupToDict:
            newDict = {}
            for tup in tuptup:
                newDict[tup[0]] = tup[1]
            self.tuptupToDict[tuptup] = newDict
        return self.tuptupToDict[tuptup]

    #Convert a dict saved as a string into a tuptupDict
    def dictLoadTuptupString(self, inputString, compressed = False, compressDeadSize = 1):
        tuptupArray = inputString.split("_")
        tuptupToConvert = []
        for tuptup in tuptupArray:
            tupSides = tuptup.split('>')
            srcString  = tuple(tupSides[0].split(','))
            destString = tuple(tupSides[1].split(','))

            src = strTupToInt(srcString)
            dest = strTupToInt(destString)

            if compressed and dest == (None,):
                dest = src
            if compressed and dest == (-1,):
                dest = (-1,)*compressDeadSize
            tuptupToConvert.append((src, dest))
        return(self.getDictFromUnsorted(tuptupToConvert))

    def __init__(self):
        self.tuptupToDict = {}


#Recreate the minimal tuptup for a given dictionary (used for storage purposes)
def recreateTupTupFromDict(inputDict):
    tuptup = []
    for key in inputDict:
        tuptup.append((key, inputDict[key]))
    tuptup.sort()
    return tuple(tuptup)

#Take a dict with tuple keys and values and make it a string for save files)
def dictSaveString(inputDict, compress = False):
    tuptup = recreateTupTupFromDict(inputDict)
    tupPairStrings = []
    for subTup in tuptup:
        src  = intTupToStr(subTup[0])
        dest = intTupToStr(subTup[1])
        if compress and (dest == '-1,-1' or dest == '-1,-1,-1'):
            tupPairStrings.append(src + '>-1')
        elif compress and src == dest:
            tupPairStrings.append(src + '>')
        else:
            tupPairStrings.append(src + '>' + dest)
    return '_'.join(tupPairStrings)

#Take integers in a tuple and convert them to a string (to allow .join to work)
def intTupToStr(tup):
    strTup = []
    for val in tup:
        strTup.append(str(val))
    return ','.join(strTup)

#Take strings in a tuple and convert them to an int (to load in stringified data)
def strTupToInt(tup):
    intTup = []
    for val in tup:
        if len(val) == 0:
            intTup.append(None)
        else:
            intTup.append(int(val))
    return tuple(intTup)

#Convert a dict saved as a string into a tuptupDict
def dictLoadString(self, inputString, compressed = False, compressDeadSize = 1):
    tuptupArray = inputString.split("_")
    outputDict = {}
    for tuptup in tuptupArray:
        tupSides = tuptup.split('>')
        srcString  = tuple(tupSides[0].split(','))
        destString = tuple(tupSides[1].split(','))

        src = []
        for srcVal in srcString:
            src.append(int(srcVal))
        src = tuple(src)

        dest = []
        for destVal in destString:
            dest.append(int(destVal))
        dest = tuple(dest)

        outputDict[src] = dest
    return(outputDict)

In [7]:
#@title boundaryCompressions and decToBase62 classes (used to define boundary strings)
class boundaryCompressions:
    compressionList = {}

    def addSize(self, size):
        #We don't need to do this if the permutation is already built (though realistically, we shouldn't even be calling this in that case)
        if size in self.compressionList:
            return

        compressableBounds = ''
        for boundIndex in range(int(size/2), size):
            compressableBounds += decToBase62(boundIndex)
        for boundIndex in range(int(size/2)):
            compressableBounds += decToBase62(boundIndex)
        self.compressionList[size] = compressableBounds

    def __init__(self):
        self.compressionList = {0:''}


#Convert a value in the range of 0-61 to its base-62 notation.  Used to create the boundary portion of game strings (the bit after the colon)
def decToBase62(digit):
    b62String = ''
    if digit < 0:
        print("ERROR: decToBase62 not intended to handle negative numbers")
        exit()
    elif digit <= 9:
        return str(digit)
    elif digit <= 35:
        return chr(digit+55)
    elif digit <= 61:
        return chr(digit+61)
    else:
        print("ERROR: decToBase62 not intended to handle digits over 61 (duh)")
        exit()

In [8]:
#@title UniversalData class (stores all data for a complete game tree)
class UniversalData:
    allLoops           = None  #A loop enum that points to actual loop objects
    allMerges          = None  #A pair of loop enums, ordered with the lesser enum first, that points to an actual merge object
    allRegions         = None  #A region in the form of a tuple-of-loop-enums, that points to an actual region object
    allGames           = None  #A complete game string that points to a game object
    allFullPointBounds = None  #allFullPointBounds[fullPointString] = {map of all seen boundary mappings : the game object that that boundary points to}
    allBinaryTrees     = None
    allPermutationMaps = None
    allTuptupDicts     = None
    allBoundCompress   = None
    useIsoRegions      = True #Convert isomorphic regions to a minimal form
    totalEdges         = None
    verbose            = False
    topDown            = None  #Set to 0 or more if you want to build the tree from the top down instead of building a full tree
                               #If >= 0, only levels with that many connections will be generated
                               #This can be ticked down to generate rows one at a time, or temporarily set to -1 to generate everything down a specific branch
    topDownRemaining   = None  #A dict with keys = to totalConnections, and values = a dict of all games at that level that need to be addressed
    unnimmedGames      = None  #Games we've calculated the descendants of, but haven't calculated a nimber for yet
    nextGeneration     = None  #Games that we've seen as children, but haven't properly created yet
    nonUnifiedBits     = None  #Games we've seen from nonunified games yet, but haven't properly created yet
    startTime          = None
    smartTree          = None

    def timeCheck(self):
        endTime = datetime.now()
        diff = endTime - self.startTime
        return diff.total_seconds()

    def __init__(self, useIsoRegions = True, topDown = -1):
        self.allLoops           = {}
        self.allMerges          = {}
        self.allRegions         = {}
        self.allFullPointBounds = {}
        self.allGames           = {}
        self.allBinaryTrees     = binaryTreeSet()
        self.allPermutationMaps = permutationMap()
        self.allTuptupDicts     = tuptupDicts()
        self.allBoundCompress   = boundaryCompressions()
        self.totalEdges         = 0
        self.useIsoRegions      = True
        self.verbose            = False
        self.topDown            = topDown
        self.topDownRemaining   = {}
        #self.unnimberedGames    = []
        #self.nextLayer          = []
        self.startTime          = datetime.now()
        self.smartTree          = None

        Loop(self, specialCase = LONELYBOUNDARYPAIR)
        Loop(self, specialCase = LONELYBOUNDARYTRIPLE)

# Functions

In [9]:
#@title General Functions
def errorOut(errorMessage, entryFunction = None):
    print(errorMessage)
    if entryFunction is not None:
        print(entryFunction)
    exit()
    return


#Either increment a value within a map (adding the value if it doesn't exist), or add a value to a list for the given key (creating a new list if the key doesn't exist yet)
def mapKeyAdd(countMap, key, listAddition = None, binaryMapAddition = None):
    if key not in countMap:
        if listAddition is None and binaryMapAddition is None:
            countMap[key] = 0
        elif binaryMapAddition is None:
            countMap[key] = []
        else:
            countMap[key] = {}
    if listAddition is None and binaryMapAddition is None:
        countMap[key] += 1
    elif binaryMapAddition is None:
        countMap[key].append(listAddition)
    else:
        countMap[key][binaryMapAddition] = True


def copyMap(originalMap):
    newMap = {}
    for key in originalMap:
        newMap[key] = originalMap[key]
    return newMap


def generateBaseGame(startingPoints, useIsoRegions = True, allData = None):
    if allData is None:
        allData = UniversalData()
    allData.useIsoRegions = useIsoRegions

    Loop(allData, definition = (0,))

    region = (0,)*startingPoints
    RegionClass(allData, region = region)

    regionBoundariesInput = {}
    newGame = GameClass(allData, game = [region], boundaries = regionBoundariesInput)

    allData.smartTree = createSmartTree(newGame.gameString)

    return allData


def saveData(allData, fileSuffix):
    sortedRegions = []
    for region, regionObject in allData.allRegions.items():
        regionObject = allData.allRegions[region]
        totalConnections = 0
        for loop in regionObject.region:
            for point in allData.allLoops[loop].points:
                if point == FREE or point == LONELYBOUNDARYTRIPLE:
                    totalConnections += 6
                elif point == HANG or point == LONELYBOUNDARYPAIR:
                    totalConnections += 4
                elif point == SING or point == BOUN:
                    totalConnections += 2
                elif point > SPLITSTART:
                    totalConnections += 1
            totalConnections += allData.allLoops[loop].connectionValue

        sortedRegions.append((totalConnections, len(region), region))
    sortedRegions.sort()

    sortedGames = []
    for game in allData.allGames:
        gameObject = allData.allGames[game]
        sortedGames.append((gameObject.totalConnections, len(gameObject.game), game))
    sortedGames.sort()

    with open(root_path + "/sproutsRegionData" + fileSuffix + ".txt", 'w') as outFile:
        print("Writing to " + root_path + "/sproutsRegionData" + fileSuffix + ".txt")
        for tup in sortedRegions:
            if allData.allRegions[tup[2]].merges is None:
                allData.allRegions[tup[2]].merges = []
            if allData.allRegions[tup[2]].enclosures is None:
                allData.allRegions[tup[2]].enclosures = []
            if allData.allRegions[tup[2]].disaDegens is None:
                allData.allRegions[tup[2]].disaDegens = {}
            if allData.allRegions[tup[2]].allPointReorders is None:
                allData.allRegions[tup[2]].allPointReorders = []
            print(allData.allRegions[tup[2]].saveRegionLine(compress = True), file=outFile)

    with open(root_path + "/sproutsGameData" + fileSuffix + ".txt", 'w') as outFile:
        print("Writing to " + root_path + "/sproutsGameData" + fileSuffix + ".txt")
        for tup in sortedGames:
            print(allData.allGames[tup[2]].printGameAsLine(compress = True), file=outFile)

In [10]:
#@title Spawning Functions
#If a loop doesn't exist in allLoops yet, create it.  Either way, return the loop's enumeration, as well as the shift to get to that loop, and the remapping of boundaries
def spawnNewLoop(allData, points, origin = None):
    enum = (int)(createEnumString(points))
    if enum not in allData.allLoops:
        Loop(allData, definition=points)
    newShift = allData.allLoops[enum].variantShifts[points]
    enum = allData.allLoops[enum].enum

    return (enum, newShift)


#If a merge doesn't exist in allMerges yet, create it.  Either way, return the merge object
def spawnNewMerge(allData, leftLoop, rightLoop, origin = None):
    if leftLoop.enum in allData.allMerges and rightLoop.enum in allData.allMerges[leftLoop.enum]:
        #The merge already exists, and we don't need to reinvent the wheel
        return allData.allMerges[leftLoop.enum][rightLoop.enum]
    return loopMerges(allData, leftLoop, rightLoop)


#If a loop doesn't exist in allLoops yet, create it.  Either way, return the loop's enumeration, as well as the shift to get to that loop, and the remapping of boundaries
def spawnNewRegion(allData, region, prevRemap = None, origin = None):
    reorderSetup = []
    for loopIndex in range(len(region)):
        reorderSetup.append((region[loopIndex], loopIndex))
    reorderSetup.sort()

    orderedRegion = []
    regionRemap = {}
    for index in range(len(reorderSetup)):
        (loopEnum, prevPos) = reorderSetup[index]
        orderedRegion.append(loopEnum)
        regionRemap[prevPos] = index
    orderedRegion = tuple(orderedRegion)

    if orderedRegion not in allData.allRegions:
        orderedRegionObject = RegionClass(allData, region = orderedRegion, origin = origin)
        allData.allRegions[orderedRegion] = orderedRegionObject

    if prevRemap is None:
        return (allData.allRegions[orderedRegion], regionRemap)

    finalRemap = {}
    for tup in prevRemap:
        (curLoopIndex, curPointIndex) = prevRemap[tup]
        if curLoopIndex in regionRemap:
            curLoopIndex = regionRemap[curLoopIndex]
        finalRemap[tup] = (curLoopIndex, curPointIndex)

    return (allData.allRegions[orderedRegion], allData.allTuptupDicts.getDictFromDict(finalRemap))


#If a loop doesn't exist in allLoops yet, create it.  Either way, return the loop's enumeration, as well as the shift to get to that loop, and the remapping of boundaries
def spawnNewGame(allData, game, boundaries, recurseDepth, connectionsDepth, creationMessage):
    (newGameString, fullPointString, boundaryString) = makeGameString(allData.allLoops, game, boundaries)

    if fullPointString not in allData.allFullPointBounds:
        GameClass(allData, game = game, boundaries = boundaries, preOrdered = True, creationMessage = "New fullPointString from " + creationMessage, recurseDepth = recurseDepth - 1, connectionsDepth = connectionsDepth)
    elif boundaryString not in allData.allFullPointBounds[fullPointString]:
        GameClass(allData, game = game, boundaries = boundaries, preOrdered = True, creationMessage = "New boundary mapping to pre-existing fullPointString from " + creationMessage, recurseDepth = recurseDepth - 1, connectionsDepth = connectionsDepth)


    trueNewGameString = allData.allFullPointBounds[fullPointString][boundaryString].gameString

    return trueNewGameString

In [11]:
#@title Loop Functions

#Shift boundaries
def boundTupToMap(boundaries):
    if boundaries is None:
        return {}

    boundaryMap = {}
    for oldBoundIndex in range(len(boundaries)):
        if boundaries[oldBoundIndex] is not None:
            boundaryMap[oldBoundIndex] = boundaries[oldBoundIndex]
    return boundaryMap


#Redefine a loop so that the split points are regularly named
def adjustSplits(loop):
    splitRemap = {}
    newLoop = ()
    for curPoint in loop:
        if curPoint in splitRemap:
            newLoop += (splitRemap[curPoint],)
        elif curPoint >= SPLITSTART:
            if curPoint not in splitRemap:
                splitRemap[curPoint] = len(splitRemap) + SPLITSTART
            newLoop += (splitRemap[curPoint],)
        else:
            newLoop += (curPoint,)

    return newLoop

#Redefine a loop so that the split points are regularly named, while also correcting boundaries
def adjustSplitsAndBoundaries(loop, boundaries, initialLoop, initialBoundaries, customIter, convertingSplits):
    splitRemap = {}
    newLoop = initialLoop
    newBoundaries = initialBoundaries

    for i in customIter:
        curPoint = loop[i]
        if curPoint in convertingSplits:
            newLoop += (BOUN,)
            newBoundaries += (convertingSplits[curPoint],)
        elif curPoint >= SPLITSTART:
            if curPoint not in splitRemap:
                splitRemap[curPoint] = len(splitRemap) + SPLITSTART
            newLoop += (splitRemap[curPoint],)
            newBoundaries += (0,)
        else:
            newLoop += (curPoint,)
            newBoundaries += (boundaries[i],)

    return (newLoop, newBoundaries)


#Create the string that's used to enumerate a loop.  This just creates a string based on what it's given, and doesn't find the unique (i.e. minimal) value of the loop.
def createEnumString(points):
    #The free point case is special, and needs to be handled by itself
    if points == ():
        return STREMPT
    elif points == (0,):
        return STRFREE

    enumString = ''  #A string unique to the loop, made of digits 0-4, where 0, 1, and 2 are hanging, singletons,
                     #  and boundary respectively, and 3/4 are ups/downs of the Dyck path of split points
    dyckPathCheck = [] #The Dyck Path is the ordering of the split points, with "3" being the first time a split point is seen,
                       #  and "4" being the second.  This allows for perfect reconstruction of the loop without storing
                       #  splits as arbitrarily higher values that could become multi-digit values that make short strings impossible

    for i in range(len(points)):
        point = points[i]
        if point < MINPOINT:
            errorOut("ERROR 1-1-1: Point with value of less than " + str(MINPOINT) + str(points))
        if i > 0 and point >= SPLITSTART and point == points[i-1]:
            errorOut("ERROR 1-1-2: Split occurs without anything between the split points: " + str(points))

        if point == HANG:
            enumString += STRHANG
        elif point == SING:
            enumString += STRSING
        elif point == BOUN:
            enumString += STRBOUN
        elif point >= SPLITSTART:
            if len(dyckPathCheck) > 0 and dyckPathCheck[-1] == point:
                dyckPathCheck = dyckPathCheck[:-1]
                enumString += STRSPLITEND
            elif point in dyckPathCheck:
                errorOut("ERROR 1-1-3: Splits are not in a possible order: " + str(points))
            else:
                dyckPathCheck.append(point)
                enumString += STRSPLITSTART
    if len(dyckPathCheck) > 0:
        errorOut("ERROR 1-1-4: Uneven number of splits: " + str(points))
    return enumString

#The same thing as createEnumString, but also calculates the total number of connections available in the loop.  It's easier to create them together,
def createEnumStringAndConnectVal(points):
    #The free point case is special, and needs to be handled by itself
    if points == ():
        return (EMPT, 0)
    elif points == (FREE,):
        return (STRFREE, 6)

    enumString = ''  #A string unique to the loop, made of digits 0-4, where 0, 1, and 2 are hanging, singletons,
                     #  and boundary respectively, and 3/4 are ups/downs of the Dyck path of split points
    connectionValue = 0  #Since boundaries are functionally half-points, we double this value to allow it to be an int, thus avoiding floating-point issues
    dyckPathCheck = [] #The Dyck Path is the ordering of the split points, with "SPLITSTART" being the first time a split point is seen,
                       #  and "SPLITEND" being the second.  This allows for perfect reconstruction of the loop without storing
                       #  splits as arbitrarily higher values that could become multi-digit values that make short strings impossible
    for i in range(len(points)):
        point = points[i]
        if point < MINPOINT and point not in SPECIALLOOPS:
            errorOut("ERROR 1-2-1: Point with value of less than " + str(MINPOINT) + ": ",  str(points))
        if i > 0 and point >= SPLITSTART and point == points[i-1]:
            errorOut("ERROR 1-2-2: Split occurs without anything between the split points:" + str(points))

        if point == HANG:
            enumString += STRHANG
            connectionValue += 4
        elif point == SING:
            enumString += STRSING
            connectionValue += 2
        elif point == BOUN:
            enumString += STRBOUN
            connectionValue += 1
        elif point >= SPLITSTART:
            if len(dyckPathCheck) > 0 and dyckPathCheck[-1] == point:
                dyckPathCheck = dyckPathCheck[:-1]
                enumString += STRSPLITEND
                connectionValue += 1
            elif point in dyckPathCheck:
                errorOut("ERROR 1-2-3: Splits are not in a possible order: " + str(points))
            else:
                dyckPathCheck.append(point)
                enumString += STRSPLITSTART
                connectionValue += 1
    if len(dyckPathCheck) > 0:
        errorOut("ERROR 1-2-4: Uneven number of splits: " + str(points))
    return (enumString, connectionValue)


def enumToTuple(enum):
    if int(enum) < 0:
        return (int(enum),)
    enum = str(enum)
    splitStack = []
    curSplit = SPLITSTART
    curTup = ()

    for curChar in enum:
        if int(curChar) < MINPOINT:
            print("ERROR: A-3-1:", enum, "has a value less than the minpoint", MINPOINT)
            exit()
        if int(curChar) < SPLITSTART:
            curTup += (int(curChar),)
        elif curChar == STRSPLITSTART:
            curTup += (curSplit,)
            splitStack.append(curSplit)
            curSplit += 1
        elif curChar == STRSPLITEND:
            curTup += (splitStack[-1],)
            splitStack = splitStack[:-1]
        else:
            print("ERROR: 1-3-2:", enum, "has a value not in the valid set of points")
            exit()

    return curTup

#Defines a map where each boundary enumeration is mapped to the point's index in that loop
#  Can be sent a custom "remap" dictionary that allows for the enumeration to be altered on the fly
#  Currently unused, but may be of value again later
def boundaryTupleToPositionMap(boundaries, reverse = False, remap = None):
    boundaryMap = {}
    for i in range(len(boundaries)):
        if boundaries[i] != 0:
            if remap is None:
                if reverse:
                    boundaryMap[i] = boundaries[i]
                else:
                    boundaryMap[boundaries[i]] = i
            else:
                if reverse:
                    boundaryMap[i] = remap[i]
                else:
                    boundaryMap[remap[i]] = i
    return boundaryMap


#Safely read strings when loading loops, so that
def safeReadParse(parseThisString):
    parsed = parseThisString.split('&')
    if len(parsed) == 1 and parsed[0] == "":
        parsed = []
    return parsed

In [12]:
#@title Game Functions
def redefineBoundariesForUnity(boundaries, boundaryMapping):
    newBoundaries = {}
    for boundary in boundaries:
        if boundary[0] in boundaryMapping:
            regionFrom = boundary[0]
            pointFrom = (boundaryMapping[regionFrom],) + boundary[1:]

            regionTo = boundaries[boundary][0]
            if regionTo == DISAPOINT[0]:
                newBoundaries[pointFrom] = DISAPOINT
            else:
                pointTo = (boundaryMapping[regionTo],) + boundaries[boundary][1:]
                newBoundaries[pointFrom] = pointTo
    return newBoundaries


def executeRemappingOfExactPoints(boundaries, boundaryMapping):
    newBoundaries = {}
    for (boundaryFrom, boundaryTo) in boundaries.items():
        if boundaryFrom in boundaryMapping and boundaryMapping[boundaryFrom] in [DEADPOINT, DISAPOINT]:
            continue
        if boundaryTo   in boundaryMapping and boundaryMapping[boundaryTo  ] == DEADPOINT:
            continue
        if boundaryFrom in boundaryMapping and boundaryTo in boundaryMapping:
            newBoundaries[boundaryMapping[boundaryFrom]] = boundaryMapping[boundaryTo]
        elif boundaryFrom in boundaryMapping:
            newBoundaries[boundaryMapping[boundaryFrom]] = boundaryTo
        elif boundaryTo in boundaryMapping:
            newBoundaries[boundaryFrom] = boundaryMapping[boundaryTo]
        else:
            newBoundaries[boundaryFrom] = boundaryTo
    return newBoundaries


def executeRemappingOfLoopsInRegion(regionNum, boundaries, boundaryMapping):
    #Skip for trivial cases
    if len(boundaries) == 0 or len(boundaryMapping) < 2:
        return boundaries

    remapCount = 0
    for (origin, dest) in boundaryMapping.items():
        if origin != dest:
            remapCount += 1

    #If we aren't actually remapping anything, just return the original locations
    if remapCount == 0:
        return boundaries

    newBoundaries = {}
    for (boundary, boundaryTo) in boundaries.items():
        if boundary[0] == regionNum and boundary[1] in boundaryMapping:
            loopFrom = boundary[1]
            pointFrom = (boundary[0],) + (boundaryMapping[loopFrom],) + (boundary[2],)
            newBoundaries[pointFrom] = boundaryTo
        elif boundaryTo[0] == regionNum and boundaryTo[1] in boundaryMapping:
            loopTo = boundaryTo[1]
            pointTo = (boundaryTo[0],) + (boundaryMapping[loopTo],) + (boundaryTo[2],)
            newBoundaries[boundary] = pointTo
        else:
            newBoundaries[boundary] = boundaryTo
    return newBoundaries


def executeRemappingOfAllRegions(boundaries, boundaryMapping):
    newBoundaries = {}
    for (boundaryFrom, boundaryTo) in boundaries.items():
        regionFrom = boundaryFrom[0]
        pointFrom = (boundaryMapping[regionFrom],) + boundaryFrom[1:]

        #If the boundary is actually a disappearing point, we don't want to remap it (the other side doesn't exist anyway)
        if boundaryTo == DISAPOINT:
            newBoundaries[pointFrom] = DISAPOINT
            continue

        regionTo = boundaryTo[0]
        pointTo = (boundaryMapping[regionTo],) + boundaryTo[1:]
        newBoundaries[pointFrom] = pointTo
    return newBoundaries



#Find the various parts of a game that are wholly unattached to
#"Zero" in this function refers to the region at index 0, which is arbitrarily chosen as the region we build off of
# I say "arbitrarily," but if we're honest, who in their right mind would use anything other than index 0?  Maybe index -1 if they're feeling crazy
def checkRegionUnity(game, boundaries):
    if len(game) == 0:
        print("Error 5-5-5: Game with no regions sent to CheckRegionUnity.")
        exit()

    if len(game) == 1 and len(boundaries) > 0:
        for boundaryTo in boundaries.values():
            if boundaryTo != DISAPOINT:
                print("Error 5-5-6: Game with only one region has boundary points.", game, boundaries)
                exit()

    #The code below assumes more than one region, so we should just bypass that and return the current game if there's only one region, since it's definitionally unified
    if len(game) == 1:
        return ([game], [boundaries])

    activeRegions = {}
    boundaryLinks = {}
    for key in boundaries:
        mapKeyAdd(boundaryLinks, key[0], binaryMapAddition = boundaries[key][0])

    disconnectedFromZero = {}
    for i in range(1, len(game)):
        disconnectedFromZero[i] = True

    #We start by looking at all connections from Region Zero.  (1)
    #  In the event that there are none, we can actually skip this section and carry on treating 0 as its own unified region, and cleaning up the rest below(2)
    #  We then look at all connections from there (3), and add every connected region we see to connectedToZero (4), and remove from disconnectedFromZero (5)
    #  If there are no loops in disconnectedFromZero at the end (6), then we know that the whole region is unified, and we can return the entire game as a single unified game (7)
    connectedToZero = {0:True}                                ##(1)
    linksRemaining = [0]
    if 0 in boundaryLinks:                                    ##(2)
        while len(linksRemaining) > 0:                        ##(3)
            newLinksRemaining = []
            for link in linksRemaining:
                for otherSide in boundaryLinks[link]:
                    if otherSide not in connectedToZero and otherSide != DISAPOINT[0]:
                        connectedToZero[otherSide] = True     ##(4)
                        del(disconnectedFromZero[otherSide])  ##(5)
                        newLinksRemaining.append(otherSide)
            linksRemaining = newLinksRemaining
    if len(disconnectedFromZero) == 0:                        ##(6)
        return ([game], [boundaries])               ##(7)

    #If we have disconnected regions, we need to start looking at disconnectedFromZero to see if it needs to be broken down further
    #  The regions connected to zero will be nearly stashed into unifiedRegions (1), since we know they're clean.  We'll keep track of the region relocations for boundary correction too (2)
    #  The others will get put into a "remaining" category(3) with their boundaries mapping tracked(4) so that we can later be checked for further unity
    unifiedRegions = []
    unifiedBoundaryMapping = {}
    remainingRegions = []
    remainingBoundaryMapping = {}
    for regionNum in range(len(game)):
        if regionNum in connectedToZero:                                ##(1)
            unifiedBoundaryMapping[regionNum] = len(unifiedRegions)     ##(2)
            unifiedRegions.append(game[regionNum])                      ##(1)
        else:
            remainingBoundaryMapping[regionNum] = len(remainingRegions) ##(4)
            remainingRegions.append(game[regionNum])                    ##(3)

    unifiedBoundaries   = redefineBoundariesForUnity(boundaries,   unifiedBoundaryMapping)
    remainingBoundaries = redefineBoundariesForUnity(boundaries, remainingBoundaryMapping)

    (recursedRemaining, recursedBoundaries) = checkRegionUnity(remainingRegions, remainingBoundaries)

    distinctGames = [unifiedRegions]
    for recursed in recursedRemaining:
        distinctGames.append(recursed)

    distinctGameBoundaries = [unifiedBoundaries]
    for recursed in recursedBoundaries:
        distinctGameBoundaries.append(recursed)

    return (distinctGames, distinctGameBoundaries)



def makeBoundaryString(allLoops, game, boundaries, boundaryPointPositions):
    #The boundary half of the string is comprised of just the boundary points in order, with a letter/number that points to the index of the other side of the boundary
    #  (indecies in this case are excluding the non-boundaries)
    boundaryString = ''
    for regionIndex in range(len(game)):
        for loopIndex in range(len(game[regionIndex])):
            for pointIndex in range(len(allLoops[game[regionIndex][loopIndex]].points)):
                pointVal = allLoops[game[regionIndex][loopIndex]].points[pointIndex]
                if pointVal == BOUN:
                    otherSide = -3 #otherSide will almost certainly be given a different value later, but these hardcoded lines are useful for testing
                    if (regionIndex, loopIndex, pointIndex) in boundaries:
                        otherSide = -2
                        if boundaries[(regionIndex, loopIndex, pointIndex)] == DISAPOINT:
                             otherSide = -1
                        if boundaries[(regionIndex, loopIndex, pointIndex)] in boundaryPointPositions:
                            otherSide = boundaryPointPositions[boundaries[(regionIndex, loopIndex, pointIndex)]]
                    if otherSide == -3:
                        boundaryString+='*'
                    elif otherSide == -2:
                        boundaryString+='^'
                    elif otherSide == -1:
                        boundaryString+='!'
                    else:
                        boundaryString+=decToBase62(otherSide)

    return boundaryString


def makeGameString(allLoops, game, boundaries):
    fullPointString = ''
    positionsVal = 0
    boundaryPointPositions = {}
    for region in range(len(game)):
        for loop in range(len(game[region])):
            fullPointString += str(game[region][loop]) + ','
            for point in range(len(allLoops[game[region][loop]].points)):
                pointVal = allLoops[game[region][loop]].points[point]
                if pointVal == BOUN:
                    boundaryPointPositions[(region, loop, point)] = positionsVal
                    positionsVal += 1
        fullPointString = fullPointString[:-1] + '|'
    fullPointString = fullPointString[:-1]

    boundaryString = makeBoundaryString(allLoops, game, boundaries, boundaryPointPositions)
    gameString = fullPointString + ":" + boundaryString

    return (gameString, fullPointString, boundaryString)


def convertGameStringToGame(allData, gameString, gameStringCompressed = False, preProcessed = False):
    if gameStringCompressed:
        gameString = decompressGameString(gameString, allData.allBoundCompress)

    if gameString == ":" or gameString == "NULL":
        return([], {})

    boundaryCoordinates = []
    pointsAndBounds = gameString.split(":")
    pointsString = pointsAndBounds[0]
    boundsString = pointsAndBounds[1]
    splitByRegions = pointsString.split("|")
    game = []
    for regionIndex in range(len(splitByRegions)):
        loops = splitByRegions[regionIndex].split(",")
        region = ()
        for loopIndex in range(len(loops)):
            loop = loops[loopIndex]
            enum = int(loop)
            region += (enum,)
            if enum in SPECIALLOOPS:
                if enum not in allData.allLoops:
                    Loop(allData, specialCase = LONELYBOUNDARYPAIR)
            else:
                if enum not in allData.allLoops:
                    loopTuple = enumToTuple(enum)
                    Loop(allData, definition=loopTuple)
                for pointIndex in range(len(loop)):
                    if loop[pointIndex] == STRBOUN:
                        boundaryCoordinates.append((regionIndex, loopIndex, pointIndex))
        game.append(region)
        if not(preProcessed) and region not in allData.allRegions:
            spawnNewRegion(allData, region, origin = "Generating region required for convertGameStringToGame")

    boundaries = {}
    for pointIndex in range(len(boundsString)):
        otherSide = boundsString[pointIndex]
        otherSideIndex = -1
        if otherSide >= '0' and otherSide <= '9':
            otherSideIndex = ord(otherSide) - ord('0')
        elif otherSide >= 'A' and otherSide <= 'Z':
            otherSideIndex = ord(otherSide) - ord('A') + 10
        elif otherSide >= 'a' and otherSide <= 'z':
            otherSideIndex = ord(otherSide) - ord('a') + 36
        elif otherSide == "!":
            otherSideIndex = -2

        if otherSideIndex == -2:
            boundaries[boundaryCoordinates[pointIndex]] = DISAPOINT
        else:
            boundaries[boundaryCoordinates[pointIndex]] = boundaryCoordinates[otherSideIndex]

    game, boundaries = fullGameCleanup(allData, game, boundaries)

    return (game, boundaries)


#Gets rid of excess characters while maintaining uniqueness of game strings
def compressGameString(gameString, allBoundCompress):
    if gameString == EMPTYGAME:
        return EMPTYGAMECOMPRESSED
    #If a string ends with a series of split points, we can remove them and still easily calculate how many such points are necessary and add them back later
    newString = re.sub(r'5+,', ',', gameString)
    newString = re.sub(r'5+\|', '|', newString)
    newString = re.sub(r'5+:', ':', newString)

    #If the last character in the gameString is a colon, it's because there's no boundaries.  We can remove the colon to save a character, and also skip the rest of the compression, as the rest is just compressing boundaries
    if newString[-1:] == ':':
        return newString[:-1]
    gameBounds = newString.split(':')[1]

    if len(gameBounds) not in allBoundCompress.compressionList:
        allBoundCompress.addSize(len(gameBounds))
    if gameBounds == allBoundCompress.compressionList[len(gameBounds)]:
        return newString.split(':')[0]
    return newString


#Gets rid of excess characters while maintaining uniqueness of game strings
def decompressGameString(compressedGameString, allBoundCompress):
    if compressedGameString == EMPTYGAMECOMPRESSED:
        return EMPTYGAME

    decompressed = ''
    boundCount = 0
    splitsRemaining = 0
    colonFound = False
    specialCaseFound = False
    for curChar in compressedGameString:
        #If we've seen a colon, than everything else is part of the boundary encoding, and doesn't abide by the same rules as the points
        if colonFound:
            decompressed += curChar
            continue

        #Automatically include any point we see
        if curChar in STRANYPOINT or curChar == '-':
            decompressed += curChar

        #Keep track if we've seen evidence of a special-case loop, and don't handle other values until we've moved past that special case
        if specialCaseFound or curChar == '-':
            specialCaseFound = True
        else:
            if curChar == STRBOUN:
                boundCount += 1
            elif curChar == STRSPLITSTART:
                splitsRemaining += 1
            elif curChar == STRSPLITEND:
                splitsRemaining -= 1

        #If we've seen the end of a loop, add any remaining SplitEnds, since those are removed in compression
        #  Also, reset the special char search
        if curChar in [',', '|', ':']:
            for addRemainingSplit in range(splitsRemaining):
                decompressed += STRSPLITEND
            decompressed += curChar
            splitsRemaining = 0
            specialCaseFound = False

        if curChar == ':':
            colonFound = True

    for addRemainingSplit in range(splitsRemaining):
        decompressed += STRSPLITEND
    splitsRemaining = 0

    if not colonFound:
        decompressed += ':'
        if boundCount not in allBoundCompress.compressionList:
            allBoundCompress.addSize(boundCount)
        decompressed += allBoundCompress.compressionList[boundCount]

    return decompressed

In [13]:
#@title Game Cleanup
def deletePointFromLoop(loopToDelete, game, deletionEnum, deletionBoundaries, deletePointIndices):
    (deleteRegionIndex, deleteLoopIndex) = loopToDelete

    newGame = game[:deleteRegionIndex] + [game[deleteRegionIndex][:deleteLoopIndex] + (deletionEnum,) + game[deleteRegionIndex][deleteLoopIndex+1:]] + game[deleteRegionIndex+1:]
    additionalBoundaryMappings = {}
    for (mapping, mapDest) in deletionBoundaries.items():
        additionalBoundaryMappings[(deleteRegionIndex, deleteLoopIndex, mapping)] = (deleteRegionIndex, deleteLoopIndex, mapDest)
    for pointIndex in deletePointIndices:
        additionalBoundaryMappings[(deleteRegionIndex, deleteLoopIndex, pointIndex)] = DEADPOINT

    return (newGame, additionalBoundaryMappings)


def removeEmptyLoops(game, boundaries):
    boundaryRemapping = {}
    replaceRegions = {}
    for regionIndex in range(len(game)):
        checkTheresOnlyOne = False
        for loopIndex in range(len(game[regionIndex])):
            if game[regionIndex][loopIndex] == EMPT:
                if checkTheresOnlyOne:
                    print("Two dead loops in a single region, which shouldn't be possible.  Verify logic.")
                checkTheresOnlyOne = True
                replaceRegions[regionIndex] = game[regionIndex][:loopIndex] + game[regionIndex][loopIndex+1:]
                for pointTup in boundaries:
                    if pointTup[0] == regionIndex and pointTup[1] > loopIndex:
                        boundaryRemapping[pointTup] = (pointTup[0], pointTup[1]-1, pointTup[2])

    completeNewGame = game[:]
    for regionIndex in replaceRegions:
        completeNewGame = completeNewGame[:regionIndex] + [replaceRegions[regionIndex]] + completeNewGame[regionIndex+1:]

    newBoundaries = executeRemappingOfExactPoints(boundaries, boundaryRemapping)

    return (completeNewGame, newBoundaries)


def removeEmptyRegions(allData, game, boundaries):
    boundaryRemapping = {}
    deleteRegions = []
    degenPoints = []
    regionsDeletedBeforeThisIndex = [0] #We get rid of the leading zero later so that indexes actually match up, but having a 0 to start makes things smoother

    for regionIndex in range(len(game)):
        #If the region is exclusively a boundary point, we degenerate it...  but NOT if that boundary point is actually a disappearing point
        if game[regionIndex] == (BOUN,) and boundaries[(regionIndex, 0, 0)] != DISAPOINT:
            deleteRegions.insert(0,regionIndex)  #We prepend instead of appending so that deletion later can be done quickly, and without reverse-sorting the list
            regionsDeletedBeforeThisIndex.append(regionsDeletedBeforeThisIndex[-1] + 1)
            degenPoints.append(boundaries[(regionIndex, 0, 0)])
            boundaryRemapping[(regionIndex, 0, 0)] = DEADPOINT
        elif game[regionIndex] == (SING,):
            deleteRegions.insert(0,regionIndex)  #Another prepend
            regionsDeletedBeforeThisIndex.append(regionsDeletedBeforeThisIndex[-1] + 1)
        elif game[regionIndex] == ():
            deleteRegions.insert(0,regionIndex)  #Another prepend
            regionsDeletedBeforeThisIndex.append(regionsDeletedBeforeThisIndex[-1] + 1)
        else:
            regionsDeletedBeforeThisIndex.append(regionsDeletedBeforeThisIndex[-1])

    regionsDeletedBeforeThisIndex = regionsDeletedBeforeThisIndex[1:]  #Delete the leading zero as described above

    completeNewGame = game[:]

    degenedLoops = {}
    for degenPoint in degenPoints:
        #When both sides are (BOUN,) they're going to get deleted anyway, so no need to deal with degens
        #  Also, we won't degenerate boundaries that are actually disappearing points
        if completeNewGame[degenPoint[0]] == (BOUN,) or degenPoint == DISAPOINT:
            continue
        degenLoop = (degenPoint[0], degenPoint[1])
        mapKeyAdd(degenedLoops, degenLoop, listAddition = degenPoint[2])

    loopRemappings = {}
    for (degenLoop, loopList) in degenedLoops.items():
        if len(degenLoop) > 2:
            print("ERROR E-E-E: More degenerated points than possible:", degenedLoops)
            exit()
        if len(degenLoop) == 2:
            degenLoopTarget = completeNewGame[degenLoop[0]][degenLoop[1]]
            loopList.sort()

            if degenLoopTarget not in allData.allLoops:
                print("Warning E-E-F: Loop " + str(degenLoopTarget) + " not in allData.allLoops.")
                print("  Origin Game:"      , game      )
                print("  Origin Boundaries:", boundaries)
                print("  Creating now, but double-check logic if there was no manual halt.")
                loopTuple = enumToTuple(degenLoopTarget)
                Loop(allData, definition=loopTuple)
            degenedLoopData = allData.allLoops[degenLoopTarget].degenerations[tuple(loopList)]

            degenEnum = degenedLoopData[0]
            degenMapping = degenedLoopData[1]

            degenRegion = completeNewGame[degenLoop[0]]
            degenRegion = degenRegion[:degenLoop[1]] + (degenEnum,) + degenRegion[degenLoop[1]+1:]
            completeNewGame = completeNewGame[:degenLoop[0]] + [degenRegion] + completeNewGame[degenLoop[0]+1:]
            loopRemappings[(degenLoop[0], degenLoop[1])] = degenMapping

    for deleteRegion in deleteRegions:  #The list is reverse-sorted, so we  don't need to worry about deleting regions after they've changed  places
        completeNewGame = completeNewGame[:deleteRegion] + completeNewGame[deleteRegion+1:]

    for boundary in boundaries:
        (bounRegion, bounLoop, bounPoint) = boundary
        if boundary not in boundaryRemapping:  #Boundaries that are degenerated are already taken care of, so don't overwrite them with the more general case
            remappedPoint = bounPoint
            if (bounRegion, bounLoop) in loopRemappings and bounPoint in loopRemappings[(bounRegion, bounLoop)]:
                remappedPoint = loopRemappings[(bounRegion, bounLoop)][bounPoint]
            boundaryRemapping[boundary] = (bounRegion - regionsDeletedBeforeThisIndex[bounRegion],) + (bounLoop,) + (remappedPoint,)

    newBoundaries = executeRemappingOfExactPoints(boundaries, boundaryRemapping)
    for regionIndex in range(len(completeNewGame)):
        if completeNewGame[regionIndex] not in allData.allRegions:
            #Region with degeneration is not in allRegions
            (completeNewGame, newBoundaries) = orderLoopsInRegion(regionIndex, completeNewGame, newBoundaries)
            doubleCheckOrder = completeNewGame[regionIndex]
            (fixedRegion, dumpThis) = spawnNewRegion(allData, completeNewGame[regionIndex])
            if doubleCheckOrder != fixedRegion.region:
                print ("New region was not correctly ordered after orderLoopsInRegion for some reason", completeNewGame, game)
                exit()

    return (completeNewGame, newBoundaries)


#Cycle through the regions of a game and, if a mirror flip has a lesser value, use the mirror flip
#Note that games are isomorphic to games with any number of their regions flipped)
def flipReversedRegions(allRegions, game, boundaries):
    newGame = []
    flippingRemap = {}
    flippableBoundRegions = ()
    for regionIndex in range(len(game)):
        region = game[regionIndex]
        if allRegions[region].lesserOrder:
            newGame.append(region)
        else:
            newGame.append(allRegions[region].reversedRegion.region)
            for (mapping, mapDest) in allRegions[region].reverseBoundMap.items():
                flippingRemap[(regionIndex,) + mapDest] = (regionIndex,) + mapping
    newBoundaries = executeRemappingOfExactPoints(boundaries, flippingRemap)
    return(newGame, newBoundaries)



#For the loops in any given region,
def orderLoopsInRegion(regionIndex, game, boundaries):
    region = game[regionIndex]
    orderedLoops = []
    for loopIndex in range(len(region)):
        enum = region[loopIndex]
        orderedLoops.append((enum, loopIndex))
    orderedLoops.sort()

    boundaryMappingInRegion = {}
    for loopIndex in range(len(orderedLoops)):
        boundaryMappingInRegion[orderedLoops[loopIndex][1]] = loopIndex
        orderedLoops[loopIndex] = orderedLoops[loopIndex][0]
    newBoundaries = executeRemappingOfLoopsInRegion(regionIndex, boundaries, boundaryMappingInRegion)

    boundaries = newBoundaries
    game = list(game[:regionIndex]) + [tuple(orderedLoops),] + list(game[regionIndex+1:])

    return (game, boundaries)


def orderRegionsInGame(game, boundaries):
    for regionIndex in range(len(game)):
        (game, boundaries) = orderLoopsInRegion(regionIndex, game, boundaries)

    orderedRegions = []
    for regionIndex in range(len(game)):
        region = game[regionIndex]
        orderedRegions.append((len(region), region, regionIndex))
    orderedRegions.sort()

    boundaryMappingInRegion = {}
    for regionIndex in range(len(orderedRegions)):
        regionTup = orderedRegions[regionIndex][1]
        boundaryMappingInRegion[orderedRegions[regionIndex][2]] = regionIndex
        orderedRegions[regionIndex] = orderedRegions[regionIndex][1]

    game = orderedRegions
    boundaries = executeRemappingOfAllRegions(boundaries, boundaryMappingInRegion)
    return (game, boundaries)


#Search through a game and determine if a Lonely Pair exists (i.e. a loop of two boundaries whose other side is just a region with only those two boundaries)
def findLonelyPair(allData, game, boundaries):
    lonelyPairCheckEnum = BOUN*10 + BOUN
    loopsToChangeToLonelyPairs = []
    lonelyPairInteriorsToDelete = []
    lonelyPairBoundMapping = {}
    for regionIndex in range(len(game)):
        region = game[regionIndex]
        if region == (lonelyPairCheckEnum,):
            leftBound  = boundaries[(regionIndex, 0, 0)]
            rightBound = boundaries[(regionIndex, 0, 1)]
            #Check:
            #    - That the other side of the boundary is a higher region (we do this to ensure we don't dump both sides of a lonely pair)
            #    - That the left bound and right bound are attachedto the same loop on the other side
            #    - That the other side of the loop is itself a lonely
            if leftBound[0] > regionIndex and leftBound[:2] == rightBound[:2] and game[leftBound[0]][leftBound[1]] == lonelyPairCheckEnum:
                loopsToChangeToLonelyPairs.append(leftBound[:2])
                lonelyPairInteriorsToDelete.append(regionIndex)
                lonelyPairBoundMapping[(regionIndex, 0, 0)] = DEADPOINT
                lonelyPairBoundMapping[(regionIndex, 0, 1)] = DEADPOINT
                lonelyPairBoundMapping[leftBound] = DEADPOINT
                lonelyPairBoundMapping[rightBound] = DEADPOINT

    if len(loopsToChangeToLonelyPairs) == 0:
        return (game, boundaries)

    newBoundaries = executeRemappingOfExactPoints(boundaries, lonelyPairBoundMapping)

    completeNewGame = []
    for regionIndex in range(len(game)):
        if regionIndex in lonelyPairInteriorsToDelete:
            completeNewGame.append(())
        else:
            region = game[regionIndex]
            curRegion = ()
            for loopIndex in range(len(region)):
                if (regionIndex, loopIndex) in loopsToChangeToLonelyPairs:
                    curRegion += (LONELYBOUNDARYPAIR,)
                else:
                    curRegion += (region[loopIndex],)
            completeNewGame.append(curRegion)

    (completeNewGame, newBoundaries) = fullGameCleanup(allData, completeNewGame, newBoundaries, searchForSpecialCases=False)

    return (completeNewGame, newBoundaries)


#Search through a game and determine if a Lonely Pair exists (i.e. a loop of two boundaries whose other side is just a region with only those two boundaries)
def findLonelyTriple(allData, game, boundaries):
    lonelyTripleCheckEnum = BOUN*100 + BOUN*10 + BOUN
    loopsToChangeToLonelyTriples = []
    lonelyTripleInteriorsToDelete = []
    lonelyTripleBoundMapping = {}
    for regionIndex in range(len(game)):
        region = game[regionIndex]
        if region == (lonelyTripleCheckEnum,):
            boundOne   = boundaries[(regionIndex, 0, 0)]
            boundTwo   = boundaries[(regionIndex, 0, 1)]
            boundThree = boundaries[(regionIndex, 0, 2)]
            #Check:
            #    - That the other side of the boundaries is a higher region (we do this to ensure we don't dump both sides of a lonely triple)
            #    - That the left bounds and right bounds are attached to the same loop on the other side
            #    - That the other side of the loop is itself a lonely triple
            if boundOne[0] > regionIndex and boundOne[:2] == boundTwo[:2] and boundTwo[:2] == boundThree[:2] and game[boundOne[0]][boundOne[1]] == lonelyTripleCheckEnum:
                loopsToChangeToLonelyTriples.append(boundOne[:2])
                lonelyTripleInteriorsToDelete.append(regionIndex)
                lonelyTripleBoundMapping[(regionIndex, 0, 0)] = DEADPOINT
                lonelyTripleBoundMapping[(regionIndex, 0, 1)] = DEADPOINT
                lonelyTripleBoundMapping[(regionIndex, 0, 2)] = DEADPOINT
                lonelyTripleBoundMapping[boundOne]   = DEADPOINT
                lonelyTripleBoundMapping[boundTwo]   = DEADPOINT
                lonelyTripleBoundMapping[boundThree] = DEADPOINT

    if len(loopsToChangeToLonelyTriples) == 0:
        return (game, boundaries)

    newBoundaries = executeRemappingOfExactPoints(boundaries, lonelyTripleBoundMapping)

    completeNewGame = []
    for regionIndex in range(len(game)):
        if regionIndex in lonelyTripleInteriorsToDelete:
            completeNewGame.append(())
        else:
            region = game[regionIndex]
            curRegion = ()
            for loopIndex in range(len(region)):
                if (regionIndex, loopIndex) in loopsToChangeToLonelyTriples:
                    curRegion += (LONELYBOUNDARYTRIPLE,)
                else:
                    curRegion += (region[loopIndex],)
            completeNewGame.append(curRegion)

    (completeNewGame, newBoundaries) = fullGameCleanup(allData, completeNewGame, newBoundaries, searchForSpecialCases=False)
    return (completeNewGame, newBoundaries)


#Search through a game and determine if a Disappearing Point exists (i.e. a boundary whose other side is just that boundary and singleton (doesn't matter if part of the same loop or seperate))
def findDisappearing(allData, game, boundaries):
    disappearingRegions = {}
    if allData.useIsoRegions:
        disappearingRegions[(SING*10 + BOUN,)] = (0,1)
    #We can treat (2,3) as (23) for strategic purposes, but only do that if we're reducing isomorphisms (otherwise, larger-game information is lost)
    if allData.useIsoRegions:
        disappearingRegions[(SING, BOUN)] = (1,0)

    newGame = game
    loopsToChangeToDisa = []
    disappearingRegionsInteriorsToDelete = []
    newBoundaries = copyMap(boundaries)
    for regionIndex in range(len(game)):
        region = newGame[regionIndex]
        #If the region is one of the disappearing regions *and* the boundary point isn't already a disappearing point
        if region in disappearingRegions and newBoundaries[(regionIndex,) + disappearingRegions[region]] != DISAPOINT:
            interiorBound = (regionIndex,0,1)
            if region == (SING, BOUN):  #We'll treat (SING*10 + BOUN) as the default and then just course-correct if it's (SING, BOUN)
                interiorBound = (regionIndex,1,0)
            exteriorBound = newBoundaries[interiorBound]
            exteriorRegionIndex = exteriorBound[0]
            exteriorRegion = newGame[exteriorRegionIndex]

            #Check:
            #    - That the other side of the boundary either a non-disappearing region or is a higher region (we do this to ensure we don't dump both sides of a disappearing region)
            if exteriorRegion not in disappearingRegions or exteriorRegionIndex > regionIndex:
                #For reasons I can't recall, the deletions take care of (SING,) but not a null loop, so be cheap and just use what works
                #  Not gonna lie, adding DISAPOINTs is one of the most disrupting thing I've done in the code
                newGame = newGame[:regionIndex] + [(SING,)] + newGame[regionIndex+1:]

                del(newBoundaries[interiorBound])
                newBoundaries[exteriorBound] = DISAPOINT

    return (newGame, newBoundaries)


def simpleCompressions(allData, game, boundaries):
    if not allData.useIsoRegions:
        print("ERROR:  Entered 'simpleCompressions' despite allData.useIsoRegions being false; check where it's called")
        exit()

    newGame = game[:]
    boundMapping = {}

    for regionIndex in range(len(game)):
        region = newGame[regionIndex]
        if   region == (2,2):   #   (2,2) = (1,)
            newGame[regionIndex] = (1,)
        elif region == (1,1):   #   (1,1) = (11,)
            newGame[regionIndex] = (11,)
        elif region == (2,2,2) or region == (2,22) or region == (1,2) or region == (2425,):
            newGame[regionIndex] = (222,)

        elif region == (2,3):   #   (2,3) = (23,)  This one is usually covered by disappearing points, but if the 3 in this region a disappearing point, this will still help a tiny bit
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            newGame[regionIndex] = (23,)
        elif region == (3,3):   #   (3,3) = (33,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            newGame[regionIndex] = (33,)
        elif region == (3,13):   #   (3,13) = (133,)
            boundMapping[(regionIndex, 0, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 1, 1)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (133,)

        #There's actually six regions that are ultimately equivalent to (13,)
        elif region == (2,2,3): # (2,2,3) = (13,)
            boundMapping[(regionIndex, 2, 0)] = (regionIndex, 0, 1)
            newGame[regionIndex] = (13,)
        elif region == (2,23):  # (2,23) = (13,)
            boundMapping[(regionIndex, 1, 1)] = (regionIndex, 0, 1)
            newGame[regionIndex] = (13,)
        elif region == (223,) or region == (2435,):  #This and the next segment have reboundings that overlap, so we can avoid two ifs
            boundMapping[(regionIndex, 0, 2)] = (regionIndex, 0, 1)
            newGame[regionIndex] = (13,)
        elif region == (1,3) or region == (22,3):   #   (1,3) = (13,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            newGame[regionIndex] = (13,)

        #There's three regions that are ultimately equivalent to (233,)
        elif region == (2,33):  #  (2,33) = (233,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 1, 1)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (233,)
        elif region == (3,23):  #  (3,23) = (233,)
            boundMapping[(regionIndex, 0, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 1, 1)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (233,)
        elif region == (2,3,3): # (2,3,3) = (233,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 2, 0)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (233,)

        #And now we take care of the two regions equivalent to (333,)
        elif region == (3,33):   # (3,33) = (333,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 1, 1)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (333,)
        elif region == (3,3,3):  # (3,3,3) = (333,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 2, 0)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (333,)

        for loopIndex in range(len(region)):
            loop = region[loopIndex]
            if loop == 22:  #22 = 1
                newRegion =  region[:  loopIndex] + (1,)     +  region[  loopIndex+1:]
                newGame[regionIndex] = newRegion

    newBoundaries = executeRemappingOfExactPoints(boundaries, boundMapping)

    return (newGame, newBoundaries)


#Determine if there are any parts of a game that match the special cases currently allowed (special cases can be turned on or off as desired, typically for readability/efficiency trade-offs)
def findSpecialCases(allData, game, boundaries):
    (game, boundaries) = simpleCompressions(allData, game, boundaries)
    (game, boundaries) = findLonelyPair(allData, game, boundaries)
    (game, boundaries) = findLonelyTriple(allData, game, boundaries)
    (game, boundaries) = findDisappearing(allData, game, boundaries)
    (game, boundaries) = fullGameCleanup(allData, game, boundaries, searchForSpecialCases=False)

    #FullGameCleanup is called again in findDisappearing if necessary, so no need to call it here too
    return (game, boundaries)


#Run all of the functions that are used to rearrange a game into a minimal, ordered state
def fullGameCleanup(allData, completeNewGame, newBoundaries, searchForSpecialCases=True):
    (completeNewGame, newBoundaries) = removeEmptyLoops(completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = removeEmptyRegions(allData, completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = flipReversedRegions(allData.allRegions, completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = orderRegionsInGame(completeNewGame, newBoundaries)

    if searchForSpecialCases:
        if allData.useIsoRegions:
            (completeNewGame, newBoundaries) = simpleCompressions(allData, completeNewGame, newBoundaries)
        (completeNewGame, newBoundaries) = findLonelyPair    (allData, completeNewGame, newBoundaries)
        (completeNewGame, newBoundaries) = findLonelyTriple  (allData, completeNewGame, newBoundaries)
        (completeNewGame, newBoundaries) = findDisappearing  (allData, completeNewGame, newBoundaries)
        (completeNewGame, newBoundaries) = fullGameCleanup   (allData, completeNewGame, newBoundaries, searchForSpecialCases=False)

    return (completeNewGame, newBoundaries)

In [14]:
#@title Min GameString Functions
def recursiveRegionOrLoopReordering(completeMapping, allDupes, remapping = None, depth = 0, regionIndex = None):
    if remapping is None:
        remapping = {}

    if len(allDupes) == 0:
        saveRemapping = {}
        for (src, dest) in remapping.items():
            saveRemapping[src] = dest
        completeMapping.append(saveRemapping)
        return

    for dupeVariant in allDupes[0]:
        for remap in dupeVariant:
            if regionIndex is None:
                remapping[remap[0]] = remap[1]
            else:
                remapping[(regionIndex, remap[0])] = (regionIndex, remap[1])
        recursiveRegionOrLoopReordering(completeMapping, allDupes[1:], remapping, depth + 1, regionIndex)
    return

def recursiveCrossProductOfPermuteLists(allCrossProducts, permuteList, crossProduct = None):
    if crossProduct is None:
        crossProduct = []

    if len(permuteList) == 0:
        makeAMap = {}
        for tup in crossProduct:
            makeAMap[tup[0]] = tup[1]
        allCrossProducts.append(makeAMap)
        return

    curArea = permuteList[0]
    curAreaIndex = curArea[0]
    curAreaPermutes = curArea[1]
    remainingList = permuteList[1:]
    for permute in curAreaPermutes:
        recursiveCrossProductOfPermuteLists(allCrossProducts, remainingList, crossProduct = crossProduct[:] + [[curAreaIndex, permute]])


def findLowestGameName(allData, game, regionBoundaries):
    verbose = True
    start = datetime.now()
    fullPointString = ''
    positionsVal = 0
    boundaryPointPositions = {}
    for region in range(len(game)):
        for loop in range(len(game[region])):
            fullPointString += str(game[region][loop]) + ','
            for point in range(len(allData.allLoops[game[region][loop]].points)):
                pointVal = allData.allLoops[game[region][loop]].points[point]
                if pointVal == BOUN:
                    boundaryPointPositions[(region, loop, point)] = positionsVal
                    positionsVal += 1
        fullPointString = fullPointString[:-1] + '|'
    fullPointString = fullPointString[:-1]

    if len(boundaryPointPositions) == 0:
        allOrdersQuickRef = {'':True}
        return fullPointString+":", fullPointString, '', allOrdersQuickRef


    completeRegionMapping = []
    uniqueLoopMappings    = []  #Format:  [[regionNum, {permutations of that region}]]
    uniqueRotationMapping = []
    loopRotations = {}

    #Find which regions are duplicates of others pre-boundary-mapping, and create a complete list of isomorphic mappings
    allDupeRegions = []
    startIndex = 0
    while startIndex < len(game):
        endIndex = startIndex
        while endIndex + 1 < len(game) and game[endIndex] == game[endIndex+1]:
            endIndex += 1
        if endIndex > startIndex:
            dupeRegionsCount = endIndex - startIndex + 1
            if dupeRegionsCount not in allData.allPermutationMaps.permutationList:
                allData.allPermutationMaps.addSize(dupeRegionsCount)
            curDupeRegions = []
            for permutation in allData.allPermutationMaps.permutationList[dupeRegionsCount]:
                newDupeRegion = []
                for permuteIndex in range(len(permutation)):
                    newDupeRegion.append((permuteIndex + startIndex, permutation[permuteIndex] + startIndex))
                curDupeRegions.append(newDupeRegion)
            allDupeRegions.append(curDupeRegions)
        startIndex = endIndex+1
    if len(allDupeRegions) > 0:
        recursiveRegionOrLoopReordering(completeRegionMapping, allDupeRegions)

    uniqueLoopsAndPointsMappings = []
    for regionIndex in range(len(game)):
        uniqueLoopsAndPointsMappings.append([regionIndex, allData.allRegions[game[regionIndex]].allPointReorders])
    completeLoopsAndPointsMapping = []
    recursiveCrossProductOfPermuteLists(completeLoopsAndPointsMapping, uniqueLoopsAndPointsMappings)

    if len(completeRegionMapping) == 0:
        completeRegionMapping.append({0:0})
    if len(completeLoopsAndPointsMapping) == 0:
        completeLoopsAndPointsMapping.append({0: {(0,0): (0,0)}})

    leastOrder = 'zz' #This is certain to be greater than any possible boundaryString, so will be overwritten immediately
    allOrdersQuickRef = {}

    totalRotations = 0
    goodSkips = 0
    badSkips = 0
    for loopAndPointsToRemap in completeLoopsAndPointsMapping:
        firstRotation = True  #If the first rotation's result matches a previous mapping, it's because the rotation is actually a repreat of one we already tried
                              #  Therefore, we can skip that entire "points to rotate" function which saves a *ton* of time on those really big isomorphism checks
                              #  (Note that I haven't rigorously proven this, but it checks out in testing, and the worst case is that we just end up with a few duplicates;
                              #    duplicated isomorphisms aren't a deal-breaker, just an inconvenience)
        for regionsToRemap in completeRegionMapping:
            completeRemap = {}
            for point in regionBoundaries:
                curRegion = point[0]
                curLoopAndPoint = point[1:]

                newRegion = curRegion
                if curRegion in regionsToRemap:
                    newRegion  = regionsToRemap[newRegion]

                newLoopAndPoint = curLoopAndPoint
                if curRegion in loopAndPointsToRemap and curLoopAndPoint in loopAndPointsToRemap[curRegion]:
                    if curLoopAndPoint not in loopAndPointsToRemap[curRegion]:
                        print("Error A-Y-Z: Point in findLowestGameNameTest's point rotation not in the apparent loop?")
                    newLoopAndPoint = loopAndPointsToRemap[curRegion][curLoopAndPoint]

                if newRegion != curRegion or newLoopAndPoint != curLoopAndPoint:
                    completeRemap[point] = (newRegion,) + newLoopAndPoint

            curOrder = makeBoundaryString(allData.allLoops, game, executeRemappingOfExactPoints(regionBoundaries, completeRemap), boundaryPointPositions)

            #Refer back to the "firstRotation" comment above to see why we're doing this
            totalRotations += 1
            if firstRotation and curOrder in allOrdersQuickRef:
                break
            firstRotation = False

            allOrdersQuickRef[curOrder] = True
            if curOrder < leastOrder:
                leastOrder = curOrder

    gameString = fullPointString + ":" + leastOrder

    end = datetime.now()
    diff = end - start
    timeCheck = str(diff.total_seconds())
    timeCheck = timeCheck[:timeCheck.find(" ")]
    if verbose and timeCheck.find("e") < 0 and timeCheck[0] > "1":
        print(len(allData.allGames), str(diff.total_seconds())[:4], gameString, len(allOrdersQuickRef))

    return gameString, fullPointString, leastOrder, allOrdersQuickRef

In [15]:
#@title Move encoding/decoding

def decodeMoveData(encoding):
    if encoding < 0:  #Special encodings put here
        if encoding == NONMOVEENCODING:
            return "Non-move"
        elif encoding == DISADROPENCODING:
            return "Disappearing point removed"
        elif encoding == LONELYPAIRDROPENCODING:
            return "Dropped lonely pair"

    encoding = int(encoding)
    enclosure = False
    unencoded = ''
    if encoding % 2 == ENCLOSUREENCODING:
        unencoded += 'Enclosure:  '
        enclosure = True
    else:
        unencoded += 'Merge:  '

    encoding = int(encoding / 2)
    rightPoint = encoding % (SPLITSTART + 1)
    encoding = int(encoding / (SPLITSTART + 1))
    leftPoint = encoding % (SPLITSTART + 1)
    encoding = int(encoding / (SPLITSTART + 1))
    adjacentPoints = encoding % len(SIDEENCODING)
    encoding = int(encoding / len(SIDEENCODING))
    emptyEnclosure = encoding % len(SIDEENCODING)

    if enclosure and leftPoint == FREE and rightPoint == FREE:
        unencoded += 'Free point to self'
    elif enclosure and rightPoint == HANG and leftPoint == FREE:
        unencoded += 'Hanging point to self'
    else:
        unencoded += POINTENCODINGS[leftPoint] + " point to " + POINTENCODINGS[rightPoint]

    if not enclosure:
        return unencoded

    if adjacentPoints == SIDEENCODING['neither']:
        if emptyEnclosure ==  SIDEENCODING['neither']:
            unencoded += ', non-adjacent points connected, enclosed loops on both sides'
        elif emptyEnclosure == SIDEENCODING['both']:
            unencoded += ', non-adjacent points connected, no other loops in region'
        elif emptyEnclosure == SIDEENCODING['left'] or emptyEnclosure == SIDEENCODING['right']:
            unencoded += ', non-adjacent points connected, one side empty'
        else:
            print("Error D-F-G:  emptyEnclosure not in range:", emptyEnclosure, 'not in', SIDEENCODING)
            exit()
    elif adjacentPoints == SIDEENCODING['both']:
        if emptyEnclosure ==  SIDEENCODING['neither']:
            unencoded += ', two-point loop, enclosed loops on both sides'
        elif emptyEnclosure == SIDEENCODING['both']:
            unencoded += ', two-point loop, no other loops in region'
        elif emptyEnclosure == SIDEENCODING['left'] or emptyEnclosure == SIDEENCODING['right']:
            unencoded += ', two-point loop, other loops all sent to one side'
        else:
            print("Error D-F-H:  emptyEnclosure not in range:", emptyEnclosure, 'not in', SIDEENCODING)
            exit()
    elif adjacentPoints == SIDEENCODING['left'] or adjacentPoints == SIDEENCODING['right']:
        if emptyEnclosure ==  SIDEENCODING['neither']:
            unencoded += ', adjacent points connected, enclosed loops on both sides'
        elif emptyEnclosure == SIDEENCODING['both']:
            unencoded += ', adjacent points connected, no other loops in region'
        elif emptyEnclosure == adjacentPoints:
            unencoded += ', adjacent points connected, other loops all sent to adjacent side'
        else:
            unencoded += ', adjacent points connected, other loops all sent to opposite side'

    return unencoded


def encodeLoopMoveData(moveType, leftPointType, rightPointType, hangToSelf = False, adjacentPoints = 'neither'):
    #Order the point types so that encodings are as normalized as possible
    if leftPointType < rightPointType:
        (leftPointType, rightPointType) = (rightPointType, leftPointType)

    encodedValue = SIDEENCODING[adjacentPoints]

    encodedValue *= (SPLITSTART + 1)
    if hangToSelf:
        encodedValue += HANG
    else:
        encodedValue += rightPointType
        encodedValue *= (SPLITSTART + 1)
        encodedValue += leftPointType

    encodedValue *= 2
    if moveType == 'enclosure':
        encodedValue += ENCLOSUREENCODING
    elif moveType == 'merge':
        encodedValue += MERGEENCODING

    return encodedValue


def compressEncoding(encoding):
    compression = {488:248, 438:238, 738:688, 538:288, 526:276, 726:676, 436:236, 736:686, 536:286, 486:336, 476:326, 426:226, 528:278, 228:428, 478:328, 488:338, 728:678, 116:66, 716:666, 718:668, 416:216, 518:268, 418:218, 516:266, 402:202, 466:316, 468:318, 112:62, 114:64, 118:68, 126:76, 136:86, 128:78, 138:88, 412:212, 414:214, 424:224, 464:314, 512:262, 514:264, 524:274, 462:312, 124:74, 474:324, 712:662, 714:664 , 724:674}
    if encoding in compression:
        return(compression[encoding])
    return(encoding)

def compressEncodingArray(moveArray):
    compressedArray = []
    for encoding in moveArray:
        compressed = compressEncoding(encoding)
        if compressed not in compressedArray:
            compressedArray.append(compressed)
    compressedArray.sort()
    return compressedArray


def decodeMoveArray(moveArray):
    decodedMoves = []
    for encodedMove in moveArray:
        decodedMoves.append(decodeMoveData(encodedMove))
    return ' ; '.join(decodedMoves)

def testEncode():
    for moveType in ['enclosure', 'merge']:
        for leftPointType in POINTENCODINGS:
            for rightPointType in POINTENCODINGS:
                encoded = moveType + ' ' + POINTENCODINGS[leftPointType] + ' ' + POINTENCODINGS[rightPointType]
                if moveType == 'enclosure':
                    for adjacentPoints in [True, False]:
                        print(encoded, adjacentPoints)
                        print(decodeMoveData(encodeLoopMoveData(moveType, leftPointType, rightPointType, adjacentPoints = adjacentPoints)))
                        print()
                else:
                    print(encoded)
                    print(decodeMoveData(encodeLoopMoveData(moveType, leftPointType, rightPointType)))
                    print()

#testEncode()
#print("\n".join(decodeMoveArray([322,186,586,336,686,786,37,286,236,636,676,276,27,288,576,688]).split(" ; ")))
pass


In [16]:
#@title Optimal Play
def recursivePlay(smartMoveTree, curGame):
    if curGame.nimber == 0:
        for childGame in curGame.childGames:
            childGameString = childGame.gameString
            smartMoveTree[curGame.gameString].append(childGameString)
            if childGameString not in smartMoveTree:
                smartMoveTree[childGameString] = []
                recursivePlay(smartMoveTree, childGame)
    else:
        for childGame in curGame.childGames:
            if childGame.nimber == 0:
                childGameString = childGame.gameString
                smartMoveTree[curGame.gameString].append(childGameString)
                if childGameString not in smartMoveTree:
                    smartMoveTree[childGameString] = []
                    recursivePlay(smartMoveTree, childGame)

#A smart tree from the initial game is automatically generated in the generateBaseGame function,
#  but it's also possible to generate off of lower trees if one is so inclined
def createSmartTree(baseGameString):
    smartMoveTree = {baseGameString:[]}
    recursivePlay(smartMoveTree, dataSummary.allGames[baseGameString])
    return smartMoveTree


def printSmartTree(smartTree = None, gameString = None, printAllLines = True):
    if smartTree is None:
        if gameString is None:
            print("Pre-built smart tree or a gameString to generate from one required for printSmartTree")
            return
        else:
            smartTree = createSmartTree(gameString)

    edges = 0
    for curGameString, childGameList in smartTree.items():
        keyGame = dataSummary.allGames[curGameString]
        for childGameString in childGameList:
            childGame = dataSummary.allGames[childGameString]
            keyDat   = (  keyGame.nimber,   keyGame.totalConnections,   keyGame.minMoves,   keyGame.maxMoves)
            childDat = (childGame.nimber, childGame.totalConnections, childGame.minMoves, childGame.maxMoves)
            difs     = (  keyGame.totalConnections - childGame.totalConnections,   keyGame.minMoves - childGame.minMoves,   keyGame.maxMoves - childGame.maxMoves)
            if printAllLines:
                print(str(keyGame.nimber) + "\t" + str(childGame.nimber) + "\t" + str(difs) + "\t" + str(keyDat) + " " + curGameString + "\t" + str(childDat) + " " + childGameString)
            edges += 1

        for childGameString in childGameList:
            print(len(smartTree), edges)

    return smartTree


#This is basically a recreation of the handleNonUnifiedGames function in the game class,
#  but used to create one-offs on the fly rather than save them to the full game array,
#  which would only serve to add massive data to the storage
def generateNonUnifiedMoves(allData, distinctGames):
    childGames = []
    for subGameIndex in range(len(distinctGames)):
        baseNewGame = distinctGames[:subGameIndex] + distinctGames[subGameIndex+1:]
        curSubGame = distinctGames[subGameIndex]
        for curChildGame, moveType in curSubGame.childGames.items():
            newGame = baseNewGame[:]
            for subGame in curChildGame.subGames:
                newGame.append(allData.allGames[subGame])

            subNimbers    = []
            maxSubNimLen  = 0
            minMoves = 0
            maxMoves = 0
            totalConnections = 0

            for subGame in newGame:
                minMoves         += subGame.minMoves
                maxMoves         += subGame.maxMoves
                totalConnections += subGame.totalConnections
                curNimberInBinary = bin(subGame.nimber)[2:]
                subNimbers.append(curNimberInBinary)
                if len(curNimberInBinary) > maxSubNimLen:
                    maxSubNimLen = len(curNimberInBinary)

            #There's a faster way to calculate a nimber, but there's not a ton of these and they're all tiny numbers, so I don't care
            digitSum = [0]*maxSubNimLen
            for nimberIndex in range(len(subNimbers)):
                curNimber = '0'*(maxSubNimLen-len(subNimbers[nimberIndex])) + subNimbers[nimberIndex]
                for digitIndex in range(maxSubNimLen):
                    digitSum[digitIndex] += int(curNimber[digitIndex])

            xoredBinary = ''
            for digit in digitSum:
                xoredBinary += str(digit%2)
            nimber = int(xoredBinary, 2)

            childGames.append((nimber, minMoves, maxMoves, totalConnections, moveType, newGame))

    return childGames


def generateWinPath(dataSummary, winPath, curGame = None, curSubGameList = None, nimber = None, minMoves = None, maxMoves = None, totalConnections = None, smartMoveType = 'random'):
    if curGame is None and curSubGameList is None:
        print("generateWinPath requires a curGame or a curSubGameList")
        return

    if curGame is not None and curSubGameList is not None:
        print("generateWinPath cannot have both a curGame or a curSubGameList")

    #If the input is just a normal game, split it out into subGames (presumably there's just one, but it works regardless)
    if curGame is not None:
        nimber           = curGame.nimber
        minMoves         = curGame.minMoves
        maxMoves         = curGame.maxMoves
        totalConnections = curGame.totalConnections
        curSubGameList = []
        for subGameString in curGame.subGames:
            curSubGameList.append(dataSummary.allGames[subGameString])

    #Get rid of any empty subgames, and if the entire game is composed of empty subgames, we can end the recursion
    allEmpty = True
    cleanSubGameList = []
    for gameObject in curSubGameList:
        if gameObject.gameString != 'NULL':
            cleanSubGameList.append(gameObject)
            allEmpty = False
    if allEmpty:
            return()

    #NonUnified games have nothing in their childGames dict, so we need to use a variant of game-building to work our way down instead
    childGames = generateNonUnifiedMoves(dataSummary, cleanSubGameList)

    #If the gamestate is winnable, keep only winning moves.  Otherwise, keep them all.
    smartMoves = []
    enumMap = {}
    for childData in childGames:
        (childNimber, childMinMoves, childMaxMoves, childTotalConnections, childMoveType, childSubgames) = childData
        if nimber == 0 or childNimber == 0:
            enum = randrange(100000000)
            smartMoves.append((childMaxMoves, childTotalConnections, childMinMoves, enum, childNimber, childMoveType))
            enumMap[enum] = childSubgames

    winning = 'Winning'
    if nimber == 0:
        winning = 'Losing'

    #Choose a random game from all smart moves
    nextIndex = randrange(len(smartMoves))
    if smartMoveType == "shortest":
        smartMoves.sort()
        nextIndex = 0
    if smartMoveType == "longest":
        smartMoves.sort(reverse=True)
        nextIndex = 0


    (childMaxMoves, childTotalConnections, childMinMoves, childEnum, childNimber, childMoveType) = smartMoves[nextIndex]
    childSubgames = enumMap[childEnum]
    subGameList = []
    for subGame in childSubgames:
        subGameList.append(subGame.gameString)
    #winPath.append(str((winning, childMinMoves, childMaxMoves, childTotalConnections, minMoves - childMinMoves, maxMoves - childMaxMoves, totalConnections - childTotalConnections)) + ' - ' + str(subGameList) + ' - ' + decodeMoveArray(childMoveType))
    winPath.append((winning, subGameList, childMinMoves, childMaxMoves, childTotalConnections, minMoves - childMinMoves, maxMoves - childMaxMoves, totalConnections - childTotalConnections, decodeMoveArray(childMoveType)))

    newGameStringArray = []
    for subGame in childSubgames:
        newGameStringArray.append(subGame.gameString)
    generateWinPath(dataSummary, winPath, curSubGameList = childSubgames, nimber = childNimber, minMoves = childMinMoves, maxMoves = childMaxMoves, totalConnections = childTotalConnections)

    return

#Instead of generating a winpath, we'll just generate an entire tree structure beneith the given game
def generateFullSubgame(dataSummary, childParentMoves, seenGames = None, curGame = None, curSubGameList = None, parentString = None):
    if curGame is None and curSubGameList is None:
        print("generateFullSubgame requires a curGame or a curSubGameList")
        return

    if curGame is not None and curSubGameList is not None:
        print("generateFullSubgame cannot have both a curGame or a curSubGameList")

    if seenGames is None:
        seenGames = {}

    #If the input is just a normal game, split it out into subGames (presumably there's just one, but it needs to be made an array regardless)
    if curGame is not None:
        parentString = curGame.gameString
        curSubGameList = []
        for subGameString in curGame.subGames:
            curSubGameList.append(dataSummary.allGames[subGameString])

    #Get rid of any empty subgames, and if the entire game is composed of empty subgames, we can end the recursion
    allEmpty = True
    cleanSubGameList = []
    for gameObject in curSubGameList:
        if gameObject.gameString != 'NULL':
            cleanSubGameList.append(gameObject.gameString)
            allEmpty = False
    if curSubGameList[0] == 'NULL':
        return()

    cleanSubObjectList = []
    for gameString in cleanSubGameList:
        cleanSubObjectList.append(dataSummary.allGames[gameString])

    #NonUnified games have nothing in their childGames dict, so we need to use a variant of game-building to work our way down instead
    childGames = generateNonUnifiedMoves(dataSummary, cleanSubObjectList)

    #If the gamestate is winnable, keep only winning moves.  Otherwise, keep them all.
    nextMoves = []
    for childData in childGames:
        (childNimber, childMinMoves, childMaxMoves, childTotalConnections, childMoveType, childObject) = childData
        subGameStrings = []
        for gameObject in childObject:
            if gameObject.gameString != 'NULL':
                subGameStrings.append(gameObject.gameString)
        if len(subGameStrings) == 0:
            subGameStrings.append('NULL')
        subGameStrings.sort()

        subGameObjects = []
        for gameString in subGameStrings:
            subGameObjects.append(dataSummary.allGames[gameString])
        singularString = ';'.join(subGameStrings)

        nextMoves.append((singularString, subGameObjects, childMoveType))

    for tup in nextMoves:
        (singularString, subGameObjects, moveType) = tup
        if singularString not in childParentMoves:
            childParentMoves[singularString] = []
        childParentMoves[singularString].append((parentString, moveType))
        if singularString not in seenGames:
            seenGames[singularString] = True
            generateFullSubgame(dataSummary, childParentMoves, seenGames = seenGames, curSubGameList = subGameObjects, parentString = singularString)

    return

In [17]:
#@title Region-swap functions (for testing)
#Compare with replacement region, or add two variants and compare
def addSegment(allData, gameObject, regionIndex, added, newBoundMap, spawnAsNecessary):
    region = gameObject.game[regionIndex]
    newGame = gameObject.game[:regionIndex] + [region + added[0]] + gameObject.game[regionIndex+1:] + added[1:]
    newBoundaries = copyMap(gameObject.boundaries)
    for boundary, otherSide in newBoundMap.items():
        newBoundary  = (len(gameObject.game) +  boundary[0] - 1,) +  boundary[1:]
        newOtherside = (len(gameObject.game) + otherSide[0] - 1,) + otherSide[1:]
        if boundary[0] == 0:
            newBoundary  = (regionIndex,) + (len(region) + boundary[1],) + boundary[2:]
        if otherSide[0] == 0:
            newOtherside  = (regionIndex,) + (len(region) + otherSide[1],) + otherSide[2:]
        newBoundaries[newBoundary] = newOtherside

    (newGame, newBoundaries) = fullGameCleanup(allData, newGame, newBoundaries)

    newGameString, A, B, C = findLowestGameName(allData, newGame, newBoundaries)
    if spawnAsNecessary:
        result = GameClass(dataSummary, game = newGame, boundaries = newBoundaries)
        newGameString = result.gameString
        if newGameString not in allData.allGames:
            allData.allGames[newGameString] = result

    if newGameString in allData.allGames:
        return allData.allGames[newGameString]
    return None

def addAndCompare(allData, regionAndMapPairings, fieldOrdering = None, maxConnections = 4, printSpecificPairs = None, excludeSplitGames = True, spawnAsNecessary = False):
    if printSpecificPairs is None:
        printSpecificPairs = []

    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max']

    allGamesTemp = []
    for game in dataSummary.allGames.values():
        allGamesTemp.append(game)

    metaCount = {}
    outputBlocks = []
    for gameObject in allGamesTemp:
        if gameObject.totalConnections > maxConnections:
            continue

        if excludeSplitGames and len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            outputBlock1 = ''
            outputBlock2 = ()
            for (added, newBoundMap) in regionAndMapPairings:
                newGame = addSegment(allData, gameObject, regionIndex, added, newBoundMap, spawnAsNecessary)

                if newGame is not None:
                    outputBlock2 += (newGame.cleanString + ' ',)
                    if len(outputBlock2) == 2 and outputBlock1.strip() + str(newGame.nimber) in printSpecificPairs:
                        print(outputBlock2, outputBlock1, str(newGame.nimber))
                    outputBlock1 += str(newGame.nimber) + ' '
                else:
                    break
            if len(outputBlock2) == len(regionAndMapPairings):
                mapKeyAdd(metaCount, outputBlock1)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0], str(tup[1]))


#########

def preparePrintLine(allData, fieldOrdering, orderedGameDat):
    metaDat = ''
    for fieldType in fieldOrdering:
        for gameDat in orderedGameDat:
            if fieldType == 'nimber':
                metaDat += str(gameDat.nimber).ljust(2) + '& '
            elif fieldType == 'totalC':
                metaDat += str(gameDat.totalConnections).ljust(2) + '& '
            elif fieldType == 'min':
                metaDat += str(gameDat.minMoves).ljust(2) + '& '
            elif fieldType == 'max':
                metaDat += str(gameDat.maxMoves).ljust(2) + '& '
            elif fieldType == 'regionCount':
                metaDat += str(len(gameDat.game)).ljust(2) + '& '
            elif fieldType == 'gameString':
                metaDat += str(gameDat.gameString).ljust(2) + '& '
            else:
                print("Field not recognized:"  )
        metaDat += '/ '
    metaDat = metaDat[:-3]

    orderedGameStrings = []
    for curGame in orderedGameDat:
        orderedGameStrings.append(curGame.gameString)
    outputBlock = metaDat + ' / ' + '\t'.join(orderedGameStrings)

    return (metaDat, outputBlock)


def compareWithReplacementRegion(allData, removeRegion, addRegion, changeBounds = None, fieldOrdering = None, maxConnections = 100, printSpecificPairs = None, excludeSplitGames = True, sameRegionOtherSide = False, sameLoopOtherSide = False):
    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max']
    if changeBounds is None:
        changeBounds = {}
    if printSpecificPairs is None:
        printSpecificPairs = []

    metaCount = {}
    outputBlocks = []
    allOutputs = {}
    for gameString, gameObject in dataSummary.allGames.items():
        if gameObject.totalConnections > maxConnections:
            continue
        if excludeSplitGames and len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            if region == removeRegion:
                regionFound = False
                alteredGame = gameObject.game[:regionIndex] + [addRegion] + gameObject.game[regionIndex+1:]
                #alteredBoundaries = copyMap(gameObject.boundaries)

                '''for removeBound, addBound in changeBounds.items():
                    removeBoundary = (regionIndex,) + removeBound
                    addBoundary    = (regionIndex,) +    addBound
                    otherside = gameObject.boundaries[removeBoundary]
                    del(alteredBoundaries[removeBoundary])
                    alteredBoundaries[addBoundary] = otherside
                    if otherside != DISAPOINT:
                        alteredBoundaries[otherside  ] = addBoundary'''

                otherRegionIndicies = {}
                otherLoopIndicies = {}
                completeRemapping = {}
                for removeBound, addBound in changeBounds.items():
                    otherRegionIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[0]] = True
                    otherLoopIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[1]] = True
                    completeRemapping[(regionIndex,) + removeBound] = (regionIndex,) + addBound

                alteredBoundaries = executeRemappingOfExactPoints(gameObject.boundaries, completeRemapping)
                (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries)

                alteredString, A, B, C = findLowestGameName(allData, alteredGame, alteredBoundaries)
                if sameLoopOtherSide   and (len(otherRegionIndicies) > 1 or len(otherLoopIndicies) > 1):
                    continue
                if sameRegionOtherSide and len(otherRegionIndicies) > 1:
                    #print(gameString, alteredString)
                    continue



                if alteredString in dataSummary.allGames:
                    allOutputs[gameString] = False
                    orderedGameDat = [gameObject, dataSummary.allGames[alteredString]]
                    (metaDat, outputBlock) = preparePrintLine(allData, fieldOrdering, orderedGameDat)
                        #baseDat    = dataSummary.allGames[   baseString]
                        #alteredDat = dataSummary.allGames[alteredString]
                        #metaDat = str(baseDat.nimber) + ' ' + str(alteredDat.nimber) + ' ' + str(gameObject.nimber)# + ' / ' + str(gameObject.totalConnections) + ' ' + str(alteredDat.totalConnections)# + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredDat.minMoves) + ' ' + str(alteredDat.maxMoves) + ')'
                        #outputBlocks.append(metaDat + ' / ' + baseString + '\t' + alteredString + '\t' + gameString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)
                    mapKeyAdd(metaCount, metaDat)

                    #if metaDat == "4  2 ":
                    if metaDat.strip().replace(" ", "") in printSpecificPairs:
                        print(gameString, alteredString)
                    #if not(metaDat == "0 & 0 &" or metaDat == "1 & 1 &" or metaDat == "2 & 2 &" or metaDat == "3 & 3 &" or metaDat == "4 & 4 &" or metaDat == "5 & 5 &" or metaDat == "6 & 6 &" or metaDat == "7 & 7 &" or metaDat == "8 & 8 &"):
                    #    print (metaDat, gameObject.cleanString.replace("A", "P").replace("B", "Q").replace("C", "R").replace("D", "S").replace("E", "T").replace("F", "V").replace("G", "W"))
                else:
                    pass
                    #print(gameString)
                    #if gameObject.nimber != dataSummary.allGames[alteredString].nimber:
                    #    print(gameObject.gameString)
                    #if metaDat.strip() in ["3  3"]:
                    #    print(metaDat, gameString, alteredString)
                #else:
                #    print("Unfound:", gameString, "-->", alteredString)

    print("!!!", len(allOutputs))

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0] + '\t' + str(tup[1]))

# Classes

In [18]:
#@title Loop Class
class Loop:
    allData          = None
    allLoops         = None
    allTuptupDicts   = None
    points           = None   #The points in the loop
    boundaries       = None   #An enumeration of the boundaries, where non-boundary points are valued 0
    hasBoundaries    = None
    enumString       = None   #A string unique to the loop, made of digits 1-5, where 1, 2, and 3 are hanging, singletons, and boundary respectively, and 4/5 are ups/downs of the Dyck path of split points
    enum             = None   #The enumString as an integer.  Preferable in the interest of efficiency, but beware integer overflows if the program gets really big
    isoRotations     = None   #The rotations that can be performed on the loop that are identical to the base loop (things like (1,1), which can be rotated for better ordering of a full game layout)
    isoMappings      = None
    reverseIsoMaps   = None
    maxSplit         = None
    selfReversable   = None
    reversedLoop     = None
    reversedMapping  = None
    connectionValue  = 0      #Twice the number of connections that can be made to this side of the loop.  Because there are half-points, we just avoid floating-point errors by doubling and treating as an int
    variantShifts    = None   #variantShifts[loop] = shift of that loop
    connectors       = None   #connectors[] = [((tuple of partial loop with -3 as the connector),(boundaries, with a 0 at the connector when it exists), the index of the connected point, bool that's true if the connected point is a boundary)]
                              #  All of the loop-fragments used to connect to other loops
    enclosures       = None   #enclosures[] = ((int)leftEnum, (map of int:int)leftBoundMapping, (int)rightEnum, (map of int:int)rightBoundMapping, (map of int:int)leftToRightPairing, (list of ints)boundaryRemoval)
                              #  The BoundMapping maps point from the old index number to the new one.  The BoundPairing maps point from the new loop's boundary indecies to the paired point's index
    boundToDisa      = None
    disappearances   = None   #The loops remaining after each of the DISA points have been removed (basically a boundary point getting removed, but the boundary is otherwise treated as a singleton)
    deletions        = None   #deletions[(i,) or (i,j)] = {}
    degenerations    = None   #degenerations[(i,) or (i,j) or (i,j,k)] = {}

    def errorOutLoop(self, errorMessage, forcePoints = None):
        if forcePoints is not None:
            points = forcePoints
        print(errorMessage)
        print("Loop: ", self.points)
        exit()


    #Before creating the loop, make sure that the loop doesn't break a few fundamental rules about how split points can be placed
    def verifyValidPart1(self, definition):
        if len(definition) >= 2 and definition[0] >= SPLITSTART and definition[0] == definition[-1]:
            self.errorOutLoop("ERROR 2-1-1: The same split exists at the start and end of the definition:", definition)
        splitStack = []
        for i in range(len(definition)):
            if i > 0 and definition[i] >= SPLITSTART and definition[i] == definition[i-1]:
                self.errorOutLoop("ERROR 2-1-2: Split occurs without anything between the split points:", definition)
            point = definition[i]
            if point >= SPLITSTART:
                if len(splitStack) > 0 and splitStack[-1] == point:
                    splitStack = splitStack[:-1]
                elif point in splitStack:
                    self.errorOutLoop("ERROR 2-1-3: Splits are not in a possible order:", definition)
                else:
                    splitStack.append(point)
        if len(splitStack) > 0:
            self.errorOutLoop("ERROR 2-1-4: Uneven number of splits:", definition)


    #After creating the loop, do a double-check to make sure that hasBoundaries is accurately labeled
    def verifyValidPart2(self):
        for point in self.points:
            if point == BOUN:
                if self.hasBoundaries:
                    return
                else:
                    self.errorOutLoop("ERROR 2-2-1: hasBoundaries = False, but self.points has a boundary: " + str(self.points))
        if self.hasBoundaries:
            self.errorOutLoop("ERROR 2-2-2: hasBoundaries = True, but self.points has no boundaries: " + str(self.points))



    #Consider all possible starting points of the loop, and choose the one that's the lowest numerically.
    #  Store other variants in allLoops for easy access via other loops, noting rotations necessary to return to the base loop
    #  Also construct the enumString for the loop, and count how many remaining edges can be drawn from it.
    def simplifyAndEnumerate(self, definition):
        points = definition[:]  #Temporarily hold points until we push it to self.points at the end

        allVariants = []  #allVarients[x] = (the variant of the loop, the number of shifts made to get to the given varient)
        allShifts = {}    #allShifts[loop] = [the list of shifts that can be made to get to that loop]
        allShiftsReverse = {} #
        revMappingList = tuple([*range(len(points))])[::-1]
        for shift in range(len(points)):
            newVariant = points[shift:] + points[:shift]
            splitMapping = {}
            revSplitMapping = {}

            for pointIndex in range(len(newVariant)):
                point = newVariant[pointIndex]
                if point >= SPLITSTART:
                    if point not in splitMapping:
                        splitMapping[point] = len(splitMapping)+SPLITSTART
                    newVariant = newVariant[:pointIndex] + (splitMapping[point],) + newVariant[pointIndex+1:]

            allVariants.append((newVariant,shift))
            if newVariant not in allShifts:
                allShifts[newVariant] = []
            allShifts[newVariant].append(shift)
            self.maxSplit = len(splitMapping)+SPLITSTART-1

        #If it turns out that the inversion of a loop leads to the same functional strategy as the uninverted loop, we can simplify the full map by almost half here
        #  This has not been shown, though, so we're leaving it as-is for now.
        #    allVariants.append((invertedSimplified[i:] + invertedSimplified[:i], inverted[i:] + inverted[:i]))
        allVariants.sort()

        self.points = allVariants[0][0]
        baseShift = allVariants[0][1]

        #If there is more than one variant that has a least value, keep track of which rotations are identical
        self.isoRotations = []
        for variant in allVariants:
            if variant[0] == self.points:
                self.isoRotations.append(variant[1]-baseShift)
            else:
                break

        #Calculate enumString and connectionValue below.  We could easily calculate connectionValue earlier if we need, but it fits most naturally here
        self.enumString = ''  #A string unique to the loop, made of digits 0-4, where 0, 1, and 2 are hanging, singletons,
                               #  and boundary respectively, and 3/4 are ups/downs of the Dyck path of split points
        self.connectionValue = 0  #Since boundaries are functionally half-points, we double this value to allow it to be an int, thus avoiding floating-point issues
        dyckPathCheck = [] #The Dyck Path is the ordering of the split points, with "3" being the first time a split point is seen,
                           #  and "4" being the second.  This allows for perfect reconstruction of the loop without storing
                           #  splits as arbitrarily higher values that could become multi-digit values that make short strings impossible

        (self.enumString, self.connectionValue) = createEnumStringAndConnectVal(self.points)
        self.enum = int(self.enumString)

        if self.enum in self.allLoops:
            self.errorOutLoop('ERROR 2-3-1: Loop' + str(self.points) + '(encoding ' + self.enumString + ') already in allLoops.  Please check catches for loops that already exist')

        #Create a list of all variants and the shift necessary to get to them, and then have all of the variants point to this object via allLoops
        for variant in allShifts:
            shiftList = []
            for shift in allShifts[variant]:
                shiftList.append( (baseShift-shift) % len(self.points) )
            shiftList.sort()
            self.variantShifts[variant] = shiftList[0]
            variantEncoding = (int)(createEnumString(variant))
            self.allLoops[variantEncoding] = self

        #Generate the boundary enumerations, which allow the boundaries to be matched to each other between regions
        self.boundaries = ()
        self.hasBoundaries = False
        for pointIndex in range(len(self.points)):
            point = self.points[pointIndex]
            if point == BOUN:
                self.hasBoundaries = True
                self.boundaries += (pointIndex,)
            else:
                self.boundaries += (NOTBOUN,)

        #Generate the enumString and connectionValue, as well as some basic safety measures
        if len(self.points) >= 2 and self.points[0] >= SPLITSTART and self.points[0] == self.points[-1]:
            self.errorOutLoop("ERROR 2-3-2: The same split exists at the start and end of the points:", self.points)


    def createReversedLoop(self):
        revLoop = adjustSplits(self.points[::-1])
        revMappingList = self.boundaries[::-1]
        (revEnum, revShift) = spawnNewLoop(self.allData, revLoop, origin='reversingLoop: ' + str(self.points))
        self.reversedLoop = revEnum

        revBoundaryTuptup = []
        for pointIndex in range(len(revMappingList)):
            if revMappingList[pointIndex] != NOTBOUN:
                revBoundaryTuptup.append((revMappingList[pointIndex], (pointIndex-revShift) % len(revMappingList)))
        revBoundaryTuptup.sort()

        self.reversedMapping = self.allTuptupDicts.getDict(tuple(revBoundaryTuptup))
        return


    def createISOMappings(self):
        self.isoMappings = []
        uniqueRotations = {}

        #If there is more than one variant that has a least value, keep track of which rotations are identical
        loopRotations = []
        for rotation in self.isoRotations:
            curRotation = ()
            for pointIndex in range(len(self.points)):
                if self.points[pointIndex] == BOUN:
                    curRotation = curRotation + ((pointIndex, (pointIndex + rotation) % len(self.points)),)
            uniqueRotations[curRotation] = True

        for rotation in uniqueRotations:
            curMapping = {}
            for mapping in rotation:
                curMapping[mapping[0]] = mapping[1]
            self.isoMappings.append(curMapping)

        if len(self.isoMappings) == 1:
            self.isoMappings = []
        return


    #We always move left from i, which will require that either the left or the right side will involve looping back to the start of the tuple of points
    #To speed up this process, we define the left and right ranges here, and so that we can just iterate them each time.
    def leftRightIterationArrays(self, i, j, points = None):
        #We sometimes want a list of points that's a cropped form of the standard self.points, but we'll default to self.points if nothing is provided
        if points == None:
            points = self.points

        leftIter = []
        rightIter = []
        if i < j:
            leftIter = list(range(i,j))
            rightIter = list(range(j,len(points))) + list(range(0,i))
        else:
            leftIter = list(range(i,len(points))) + list(range(0,j))
            rightIter = list(range(j,i))
        return (leftIter, rightIter)


    def enclosureSplitsToBoundaries(self, points, boundaries, splitsToBoundaries):
        if len(points) != len(boundaries):
            print("ERROR: A-4-1:  Points has a different length than boundaries:", points, boundaries)
            exit()

        newPoints     = points[:]
        newBoundaries = boundaries[:]
        for pointIndex in range(len(newPoints)):
            point = newPoints[pointIndex]
            if point >= SPLITSTART and point in splitsToBoundaries:
                newPoints     = newPoints[:pointIndex]     + (BOUN,) + newPoints[pointIndex+1:]
                newBoundaries = newBoundaries[:pointIndex] + (point+len(self.points)+3,) + newBoundaries[pointIndex+1:]
                #The 3 above is to automatically start after the maximum 3 new boundaries created in the enclosure (the new enclosure and each of the connected points if they're both hanging)
                #Technically it's not needed since we can be pretty sure SPLITSTART is already >= 3, but it doesn't hurt to have higher values in boundaries, so long as we know they match the other side
        return (newPoints, newBoundaries)


    #Every point can be connected to every other point, creating two seperate regions, which can carry any combination of other loops in the loop's current region
    #  The two seperate regions will be called the "left" and "right" regions
    #  Pre-existing boundaries will be "mapped" boundaries, denoting that the old position will be mapped to a new position
    #  New boundaries will be "paired," denoting that both sides are new, and there's not some pre-existing boundary to point to
    #Because boundaries are both being created by this function, as well as being pre-existing boundaries being split from one region to two, the code is particularly ugly just to keep track of which boundary is which
    #  This is probably the worst part of handling this entire problem with loop/region classes instead of unique strings that recalculate everything all the time.  But hopefully it pays off long-term?
    #Note:  It's a year later.  I'm still not sure if this is worth it.  But in for a penny, in for a pound!
    def enclosureConstruction(self, leftPoint, rightPoint, newBoundary, removeSplit, moveEncoding):
        if self.points[leftPoint] >= SPLITSTART and self.points[leftPoint] == self.points[rightPoint]:
            self.errorOutLoop('ERROR 2-4-1: Attempting to connect split point to its other end.  Check safety measures before calling enclosureConstruction:', self.points, leftPoint, rightPoint)
        if leftPoint == rightPoint:
            self.errorOutLoop('ERROR 2-4-2: Attempting to connect a point to itself.  Check safety measures before calling enclosureConstruction', self.points, leftPoint, rightPoint)

        origPoints = (leftPoint, rightPoint)
        boundaryRemoval = ()
        for point in [leftPoint, rightPoint]:
            if self.points[point] == BOUN:
                if self.boundaries == NOTBOUN:
                    self.errorOutLoop('ERROR 2-4-3: Incorrectly matching boundary point:', point, self.points, self.boundaries)
                boundaryRemoval = boundaryRemoval + (point,)

        #Setting a copy of points and boundaries is necessary for cases where we're removing split points
        tempPoints = self.points[:]
        boundaries = [NOTBOUN]*len(self.points)
        for pointIndex in range(len(self.points)):
            if self.points[pointIndex] == BOUN:
                boundaries[pointIndex] = pointIndex
        boundaries = tuple(boundaries)

        #If one of the endpoints is a split value, we need to exclude the split value in the next segment.
        if removeSplit != None and len(removeSplit) > 0:
            if len(removeSplit) > 2:
                self.errorOutLoop('ERROR 2-4-4: RemoveSplit is an invalid size:', removeSplit)

            #For each value in removeSplit, we iterate over the loop, find the split point that's not part of the new line, and remove it from both the points and boundary tuples
            #We also adjust the left and right points in the event that their position changes in the new tuple
            for removeThis in removeSplit:
                for i in range(len(tempPoints)):
                    if i != leftPoint and i != rightPoint and tempPoints[i] == removeThis:
                        tempPoints = tempPoints[:i] + tempPoints[i+1:]
                        boundaries = boundaries[:i] + boundaries[i+1:]
                        if leftPoint > i:
                            leftPoint -= 1
                        if rightPoint > i:
                            rightPoint -= 1
                        break

        leftLoop        = ()
        leftBoundaries  = ()
        rightLoop       = ()
        rightBoundaries = ()
        if leftPoint < rightPoint:
            leftLoop        = tempPoints[leftPoint+1:rightPoint]  #The +1s are because we don't want to include the point we're actually connecting to
            leftBoundaries  = boundaries[leftPoint+1:rightPoint]
            rightLoop       = tempPoints[rightPoint+1:] + tempPoints[:leftPoint]
            rightBoundaries = boundaries[rightPoint+1:] + boundaries[:leftPoint]
        else:
            leftLoop        = tempPoints[leftPoint+1:] + tempPoints[:rightPoint]
            leftBoundaries  = boundaries[leftPoint+1:] + boundaries[:rightPoint]
            rightLoop       = tempPoints[rightPoint+1:leftPoint]
            rightBoundaries = boundaries[rightPoint+1:leftPoint]

        splitCountInLeft = {}
        for point in leftLoop:
            if point >= SPLITSTART:
                mapKeyAdd(splitCountInLeft, point)

        splitsToBoundaries = {}
        for split in splitCountInLeft:
            if splitCountInLeft[split] == 1:
                splitsToBoundaries[split] = True

        ( leftLoop,  leftBoundaries) = self.enclosureSplitsToBoundaries( leftLoop,  leftBoundaries, splitsToBoundaries)
        (rightLoop, rightBoundaries) = self.enclosureSplitsToBoundaries(rightLoop, rightBoundaries, splitsToBoundaries)

        #Add the new BOUN points to both sides
        for i in range(len(newBoundary)):  #newBoundary has one point for each of the new boundary points we need- one by default, and an additional one for each connection that's a hanging point
            leftLoop        = adjustSplits(leftLoop + (BOUN,))
            leftBoundaries  = leftBoundaries + (len(self.points) + i,)
            rightLoop       = adjustSplits((BOUN,) + rightLoop)
            rightBoundaries = (len(self.points) + i,) + rightBoundaries

        leftOldBoundaries = {}
        leftPairedBoundaryLocation = {}  #Since we don't have a sense of where the right boundaries are located, we just store the left boundary's location for now, and actually do the mapping later
        for pointIndex in range(len(leftLoop)):
            if leftBoundaries[pointIndex] != NOTBOUN and leftBoundaries[pointIndex] < len(self.points):
                leftOldBoundaries[leftBoundaries[pointIndex]] = pointIndex
            elif leftBoundaries[pointIndex] != NOTBOUN:
                leftPairedBoundaryLocation[leftBoundaries[pointIndex]] = pointIndex

        rightOldBoundaries = {}
        leftToRightPairing = {}
        for pointIndex in range(len(rightLoop)):
            if rightBoundaries[pointIndex] != NOTBOUN and rightBoundaries[pointIndex] < len(self.points):
                rightOldBoundaries[rightBoundaries[pointIndex]] = pointIndex
            elif rightBoundaries[pointIndex] != NOTBOUN:
                leftToRightPairing[leftPairedBoundaryLocation[rightBoundaries[pointIndex]]] = pointIndex

        (leftEnum,  leftShift ) = spawnNewLoop(self.allData, leftLoop,  origin='leftEnclosure: '  + str(self.points) + ' from (left,right) indexes ' + str(origPoints))
        (rightEnum, rightShift) = spawnNewLoop(self.allData, rightLoop, origin='rightEnclosure: ' + str(self.points) + ' from (left,right) indexes ' + str(origPoints))

        shiftedLeftToRightPairing  = self.allTuptupDicts.tuptupFromPairedBounds(leftToRightPairing, leftShift, len( leftLoop), pairingShift = rightShift, pairingLen = len(rightLoop))
        shiftedLeftOldBounds       = self.allTuptupDicts.tuptupFromBoundShifts(  leftOldBoundaries,  leftShift, len( leftLoop))
        shiftedRightOldBounds      = self.allTuptupDicts.tuptupFromBoundShifts(  rightOldBoundaries, rightShift, len(rightLoop))


        #TODO: Avoid duplicate enclosures, which may happen often
        self.enclosures.append((leftEnum, shiftedLeftOldBounds, rightEnum, shiftedRightOldBounds, shiftedLeftToRightPairing, boundaryRemoval, moveEncoding))
        return


    #Connect a free point to itself
    def freeToSelf(self):
        if self.points != (FREE,):
            return

        (freeEnclosureEnum, dumpThis1) = spawnNewLoop(self.allData, (BOUN, BOUN), origin='Free Enclosure')
        emptyBoundMap = self.allTuptupDicts.getDict(())
        pairingMap = self.allTuptupDicts.getDict(((0,0),(1,1)))

        moveEncoding = encodeLoopMoveData('enclosure', FREE, FREE)
        self.enclosures.append((freeEnclosureEnum, emptyBoundMap, freeEnclosureEnum, emptyBoundMap, pairingMap, (), moveEncoding))



    #Connect any hanging points to themselves (this will create two regions, same as the standard enclosure function, but the underlying math is too different)
    def hangingToSelf(self, connectPoint):
        leftLoop = (BOUN,)
        leftBoundaries = (0,)
        (leftEnum, dumpThis1) = spawnNewLoop(self.allData, (BOUN,), origin='Hanging to self, left loop' + str(self.points))

        rightLoop = self.points[:connectPoint] + (BOUN,) + self.points[connectPoint+1:]
        (rightEnum, rightShift) = spawnNewLoop(self.allData, rightLoop, origin="Hanging to self, right loop: " + str(self.points))

        rightBoundaryTuptup = []
        rightBoundaryMap    = {}
        for pointIndex in range(len(self.boundaries)):
            if self.boundaries[pointIndex] != NOTBOUN:
                rightBoundaryTuptup.append((pointIndex, (pointIndex-rightShift) % len(self.boundaries)))
                rightBoundaryMap[pointIndex] = (pointIndex-rightShift) % len(self.boundaries)
        rightBoundaryTuptup.sort()
        rightBoundaryMap = self.allTuptupDicts.getDict(tuple(rightBoundaryTuptup))

        rightPairingIndex = (connectPoint-rightShift) % len(self.boundaries)
        pairingMap = self.allTuptupDicts.getDict(tuple(((0,rightPairingIndex),)))
        emptyBoundMap = self.allTuptupDicts.getDict(())

        moveEncoding = encodeLoopMoveData('enclosure', HANG, HANG, hangToSelf = True)
        self.enclosures.append((leftEnum, emptyBoundMap, rightEnum, rightBoundaryMap, pairingMap,(), moveEncoding))


    #When two points on a loop are connected, the result will be two seperate regions with other loops in the current split arbitrarily between the new regions.
    #  This is where we pull out every viable pair of points to connect, at which point it's handed off to the EnclosureConstruction function to do all the nitty-gritty
    def deriveEnclosures(self):
        self.freeToSelf()
        for i in range(len(self.points)):
            for j in range(i, len(self.points)):
                if i == j:
                    #The only time we connect a point to itself is if it's a hanging point.
                    if self.points[i] == HANG:
                        self.hangingToSelf(i)
                    continue #If i == j, we continue to the next iteration even if it's not a hanging point, since we don't want to connect a non-hanging to itself

                #We can't join a split point to itself, so continue past these
                if self.points[i] >= SPLITSTART and self.points[i] == self.points[j]:
                    continue

                removeSplit = []
                pointTypes  = []  #We're going to keep which type of points we're using so that that can be added to metadata later on
                newBoundary = (BOUN,)
                for k in [i,j]:
                    if self.points[k] >= SPLITSTART: #Add split points to a list that we need to remove later
                        removeSplit.append(self.points[k])
                    elif self.points[k] == HANG: #Add an extra boundary point for each split that we're joining
                        newBoundary += (BOUN,)

                    if self.points[k] >= SPLITSTART:
                        pointTypes.append(SPLITSTART)
                    else:
                        pointTypes.append(self.points[k])

                adjacentPoints = 'neither'
                if len(self.points) == 2:
                    adjacentPoints = 'both'
                elif i + 1 == j:
                    adjacentPoints = 'left'
                elif (i == 0 and j == len(self.points)-1):
                    adjacentPoints = 'right'
                moveEncoding = encodeLoopMoveData('enclosure', pointTypes[0], pointTypes[1], adjacentPoints = adjacentPoints)
                self.enclosureConstruction(i, j, newBoundary, removeSplit, moveEncoding)


    #Derive the the segment of a loop that will be added to the other loop during a merge
    def deriveConnectors(self):
        #Special Cases:
        if self.points == (FREE,):
            self.connectors.append(( (CONNECTOR, HANG) , {} , None, self.points[0]))
            return
        elif self.points == (HANG,):
            self.connectors.append(( (CONNECTOR, SING) , {} , None, self.points[0]))
            return
        elif self.points == (SING,):
            self.connectors.append(( (SING,)           , {} , None, self.points[0]))
            return
        elif self.points == (BOUN,):
            self.connectors.append(( (SING,)           , {} , 0   , self.points[0]))
            return

        for i in range(len(self.points)):
            connector            = None
            connectBoundaryArray = None
            deleteBoundary       = None
            pointType = self.points[i]

            #Create a connector off of a hanging point
            if self.points[i] == HANG:
                connector            = (CONNECTOR,) + (self.maxSplit+1,) + self.points[i+1:]     + self.points[:i]     + (self.maxSplit+1,)
                connectBoundaryArray = (NOTBOUN  ,) + (NOTBOUN        ,) + self.boundaries[i+1:] + self.boundaries[:i] + (NOTBOUN        ,)

            elif self.points[i] == SING or self.points[i] == BOUN:
                #Create a connector off of a singleton or boundary point
                connector            = (CONNECTOR,) + self.points[i+1:]     + self.points[:i]
                connectBoundaryArray = (NOTBOUN  ,) + self.boundaries[i+1:] + self.boundaries[:i]
                if self.points[i] == BOUN:
                    deleteBoundary = i

            elif self.points[i] >= SPLITSTART:
                #Adjust pointType to be a SPLITSTART as opposed to any potential higher values
                pointType = SPLITSTART

                #Create a connector off of a split point
                splitVal = self.points[i]
                connector            = self.points[i+1:]     + self.points[:i]
                connectBoundaryArray = self.boundaries[i+1:] + self.boundaries[:i]
                for j in range(len(connector)):
                    if connector[j] == splitVal:
                        connector            = connector[:j]            + connector[j+1:]
                        connectBoundaryArray = connectBoundaryArray[:j] + connectBoundaryArray[j+1:]
                        break
                connector            = (CONNECTOR,) + connector
                connectBoundaryArray = (NOTBOUN  ,) + connectBoundaryArray

            connectBoundaries    = self.allTuptupDicts.boundShiftViaArray(connectBoundaryArray)
            self.connectors.append((connector, connectBoundaries, deleteBoundary, pointType))


    #When the other side of a boundary is connected, we need to delete the boundary on the other side.  This sets up all of the possible deletions that can happen, pointing to the new loop for each combination of deletions
    #  This also keeps track of degenerations, i.e. when the other side of boundary becomes the only remaining point in that region.
    def deriveDeletions(self):
        if not self.hasBoundaries:
            return

        for i in range(len(self.points)):
            if self.points[i] == BOUN:
                singleDegeneratePoints     = self.points[:i]     + (SING,) + self.points[i+1:]
                (newEnum, newShift) = spawnNewLoop(self.allData, singleDegeneratePoints, origin='Single-boundary degeneracy ('+ str(i) + '): ' + str(self.points))

                postDegenBoundTuptup = []
                for boundIndex in range(len(self.points)):
                    if boundIndex != i and self.points[boundIndex] == BOUN:
                        postDegenBoundTuptup.append((boundIndex, (boundIndex-newShift) % len(self.points)))
                postDegenBoundMapping = self.allTuptupDicts.getDictFromUnsorted(postDegenBoundTuptup)
                self.degenerations[(i,)] = (newEnum, postDegenBoundMapping)

                singleRemovalPoints     = ()
                singleRemovalBoundaries = ()

                deletionMessage = 'Single-boundary deletion ('
                #Special case:  The boundary is the first point, and the points around it are splits.
                if i == 0 and len(self.points) > 2 and self.points[1] >= SPLITSTART and self.points[1] == self.points[-1]:
                    deletionMessage += 'a-'+str(i)+'): '
                    singleRemovalPoints     = (SING,)     + self.points[2:-1]
                    singleRemovalBoundaries = ( NOTBOUN,) + self.boundaries[2:-1]
                    singleRemovalPoints = adjustSplits(singleRemovalPoints)
                #Special case:  The boundary is the last point, and the points around it are splits
                #  In theory, this should never happen due to how loops are ordered, but this is here to ensure
                #  everything works if the ordering or notation change at some later point
                elif i == len(self.boundaries) - 1 and self.points[0] >= SPLITSTART and self.points[0] == self.points[-2]:
                    deletionMessage += 'b-'+str(i)+'): '
                    print("Warning:  Unexpected code execution in deriveDeltions - Please ensure loop ordering logic is active")
                    singleRemovalPoints     = (SING,)     + self.points[1:-2]
                    singleRemovalBoundaries = ( NOTBOUN,) + self.boundaries[1:-2]
                #Special Case:  The are splits on either side of the boundary being removed,
                #  which we will merge into a singleton.
                elif i != 0 and len(self.points) > i+1 and self.points[i-1] >= SPLITSTART and self.points[i-1] == self.points[i+1]:
                    deletionMessage += 'c-'+str(i)+'): '
                    singleRemovalPoints     = self.points[:i-1]     + (SING,   ) + self.points[i+2:]
                    singleRemovalBoundaries = self.boundaries[:i-1] + (NOTBOUN,) + self.boundaries[i+2:]
                else:
                    deletionMessage += 'd-'+str(i)+'): '
                    singleRemovalPoints     = self.points[:i]     + self.points[i+1:]
                    singleRemovalBoundaries = self.boundaries[:i] + self.boundaries[i+1:]

                singleRemovalPoints = adjustSplits(singleRemovalPoints)
                (newEnum, newShift) = spawnNewLoop(self.allData, singleRemovalPoints, origin=deletionMessage + str(self.points))

                postDeleteBoundMapping = self.allTuptupDicts.deletetionBoundShift(singleRemovalBoundaries, newShift)

                self.deletions[(i,)] = (newEnum, postDeleteBoundMapping)

                for j in range(i+1, len(self.boundaries)):
                    if self.points[j] == BOUN:
                        doubleDegeneratePoints     = self.points[:i]     + (SING,)    + self.points[i+1:j]     + (SING,)    + self.points[j+1:]
                        (newEnum, newShift) = spawnNewLoop(self.allData, doubleDegeneratePoints, origin='Double-boundary degeneracy ('+str(i)+'-'+str(j)+'): ' + str(self.points))

                        postDegenBoundTuptup = []
                        for boundIndex in range(len(self.points)):
                            if boundIndex != i and boundIndex != j and self.points[boundIndex] == BOUN:
                                postDegenBoundTuptup.append((boundIndex, (boundIndex-newShift) % len(self.points)))
                        postDegenBoundMapping = self.allTuptupDicts.getDictFromUnsorted(postDegenBoundTuptup)
                        self.degenerations[(i,j)] = (newEnum, postDegenBoundMapping)

                        doubleRemovalPoints     = ()
                        doubleRemovalBoundaries = ()

                        message = 'Double-boundary deletion ('
                        if self.points == (BOUN,SPLITSTART,BOUN,SPLITSTART) and i == 0 and j == 2:
                            message += 'e-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = (SING,   )
                            doubleRemovalBoundaries = (NOTBOUN,)
                        elif self.points == (BOUN,SPLITSTART,SPLITSTART+1,BOUN,SPLITSTART+1,SPLITSTART) and i == 0 and j == 3:
                            message += 'f-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = (SING,    SING   )
                            doubleRemovalBoundaries = (NOTBOUN, NOTBOUN)
                        #Special case:  The boundaries in question are the first two points of the loop, and are surrounded by a split point
                        #  Note that the loop will be no less than five points long
                        elif i == 0 and j == 1 and self.points[:3] == (BOUN,BOUN,SPLITSTART) and self.points[-1] == SPLITSTART:
                            message += 'g-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = (SING,   ) + self.points[3:-1]
                            doubleRemovalBoundaries = (NOTBOUN,) + self.boundaries[3:-1]
                        #Special case:  The first boundary is the start of the loop and surrounded by a split point, and the other point is elsewhere in the loop and also surrounded by a split point
                        #  Note that the loop will be no less than six points long
                        elif i == 0 and j >= 3 and len(self.points) > j+2 and self.points[1] == (SPLITSTART) and self.points[-1] == SPLITSTART and self.points[j-1] > SPLITSTART and self.points[j-1] == self.points[j+1]:
                            message += 'h-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = (SING,   ) + self.points[2:j-1]     + (SING,   ) + self.points[j+2:-1]
                            doubleRemovalBoundaries = (NOTBOUN,) + self.boundaries[2:j-1] + (NOTBOUN,) + self.boundaries[j+2:-1]
                        #Special case: The first boundary is at the start of the loop and between split points, and the other point is not immediately surrounded by splits
                        elif i == 0 and self.points[1] == (SPLITSTART) and self.points[-1] == SPLITSTART:
                            message += 'i-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = (SING,)    + self.points[2:j]     + self.points[j+1:-1]
                            doubleRemovalBoundaries = (NOTBOUN,) + self.boundaries[2:j] + self.boundaries[j+1:-1]
                        #Special case:  The boundaries are consecutive, and surrounded by a split point
                        elif i > 0 and i == j-1 and len(self.points) > j+1 and self.points[i-1] >= SPLITSTART and self.points[i-1] == self.points[j+1]:
                            message += 'j-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = self.points[:i-1]     + (SING,   ) + self.points[j+2:]
                            doubleRemovalBoundaries = self.boundaries[:i-1] + (NOTBOUN,) + self.boundaries[j+2:]
                        #Special case: The boundaries are not consecutive, and both are surrounded by their own split point
                        elif i != 0 and len(self.points) > i+1 and self.points[i-1] >= SPLITSTART and self.points[i-1] == self.points[i+1] and j != 0 and len(self.points) > j+1 and self.points[j-1] >= SPLITSTART and self.points[j-1] == self.points[j+1]:
                            message += 'k-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = self.points[:i]     + (SING,   ) + self.points[i+1:j]     + (SING,   ) + self.points[j+1:]
                            doubleRemovalBoundaries = self.boundaries[:i] + (NOTBOUN,) + self.boundaries[i+1:j] + (NOTBOUN,) + self.boundaries[j+1:]
                        #Special case: The boundaries are not consecutive, and i is surrounded by a split point, but j is not
                        elif i != 0 and len(self.points) > i+1 and self.points[i-1] >= SPLITSTART and self.points[i-1] == self.points[i+1]:
                            message += 'l-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = self.points[:i]     + (SING,   ) + self.points[i+1:j]     + self.points[j+1:]
                            doubleRemovalBoundaries = self.boundaries[:i] + (NOTBOUN,) + self.boundaries[i+1:j] + self.boundaries[j+1:]
                        #Special case: The boundaries are not consecutive, and j is surrounded by a split point, but i is not
                        elif j != 0 and len(self.points) > j+1 and self.points[j-1] >= SPLITSTART and self.points[j-1] == self.points[j+1]:
                            message += 'm-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = self.points[:i]     + self.points[i+1:j]     + (SING,   ) + self.points[j+1:]
                            doubleRemovalBoundaries = self.boundaries[:i] + self.boundaries[i+1:j] + (NOTBOUN,) + self.boundaries[j+1:]
                        else:
                            message += 'n-'+str(i)+'-'+str(j)+'): '
                            doubleRemovalPoints     = self.points[:i]     + self.points[i+1:j]     + self.points[j+1:]
                            doubleRemovalBoundaries = self.boundaries[:i] + self.boundaries[i+1:j] + self.boundaries[j+1:]

                        doubleRemovalPoints = adjustSplits(doubleRemovalPoints)

                        (newEnum, newShift) = spawnNewLoop(self.allData, doubleRemovalPoints, origin=message + str(self.points))
                        postDoubleDeleteBoundMapping = self.allTuptupDicts.deletetionBoundShift(doubleRemovalBoundaries, newShift)

                        self.deletions[(i,j)] = (newEnum, postDoubleDeleteBoundMapping)

                        for k in range(j+1, len(self.boundaries)):
                            if self.points[k] == BOUN:
                                tripleDegeneratePoints     = self.points[:i]     + (SING,)    + self.points[i+1:j]     + (SING,)    + self.points[j+1:k]    + (SING,)    + self.points[k+1:]

                                (newEnum, newShift) = spawnNewLoop(self.allData, tripleDegeneratePoints, origin='Triple-boundary degeneracy ('+str(i)+'-'+str(j)+'-'+str(k)+'): ' + str(self.points))

                                postDegenBoundTuptup = []
                                for boundIndex in range(len(self.points)):
                                    if boundIndex != i and boundIndex != j and boundIndex != k and self.points[boundIndex] == BOUN:
                                        postDegenBoundTuptup.append((boundIndex, (boundIndex-newShift) % len(self.points)))
                                postDegenBoundMapping = self.allTuptupDicts.getDictFromUnsorted(postDegenBoundTuptup)
                                self.degenerations[(i,j,k)] = (newEnum, postDegenBoundMapping)


    def generateSpecialCases(self, definition, specialCase):
        if definition == ():
            if EMPT in self.allLoops:
                self.errorOutLoop('ERROR 2-5-3: Empty loop already in allLoops.  Please check catches for loops that already exist')
            self.points            = ()
            self.boundaries        = ()
            self.enumString        = STREMPT
            self.enum              = EMPT
            self.allLoops[EMPT]    = self
            self.variantShifts     = {():0}
            return

        if specialCase == LONELYBOUNDARYPAIR or specialCase == LONELYBOUNDARYTRIPLE:
            if specialCase in self.allLoops:
                self.errorOutLoop('ERROR 2-5-4: ' + str(specialCase) + ' loop already in allLoops.  Please check catches for loops that already exist')

            emptyBoundMap = self.allTuptupDicts.getDict(())

            self.points                 = (specialCase,)
            self.boundaries             = (NOTBOUN,)
            self.enumString             = str(specialCase)
            self.enum                   = specialCase
            self.allLoops[specialCase]  = self
            self.hasBoundaries          = False
            self.maxSplit               = 0
            self.isoRotations           = [0]
            self.isoMappings            = []
            self.selfReversable         = True
            self.reversedLoop           = specialCase
            self.reversedMapping        = ()
            self.variantShifts          = {(specialCase): 0}
            self.deletions              = {}
            self.degenerations          = {}
            if specialCase == LONELYBOUNDARYPAIR:
                spawnNewLoop(self.allData, (BOUN,), origin='LONELYBOUNDARYPAIR')
                moveEncoding = encodeLoopMoveData('enclosure', BOUN, BOUN)
                self.enclosures         = [(BOUN, emptyBoundMap, BOUN, emptyBoundMap, self.allTuptupDicts.getDict(((0, 0),)), (), moveEncoding)]
                self.connectors         = [((-3, SING), self.allTuptupDicts.getDict(()), None, BOUN)]
                self.connectionValue    = 4
            elif specialCase == LONELYBOUNDARYTRIPLE:
                spawnNewLoop(self.allData, (BOUN,     ), origin='LONELYBOUNDARYTRIPLE')
                spawnNewLoop(self.allData, (SING, BOUN), origin='LONELYBOUNDARYTRIPLE')
                moveEncoding = encodeLoopMoveData('enclosure', BOUN, BOUN)
                self.enclosures         = [(SING*10+BOUN, emptyBoundMap, BOUN, emptyBoundMap, self.allTuptupDicts.getDict(((1, 0),)), (), moveEncoding)]
                self.connectors         = []  #Merges will exist, but must be handled in their own special way to reintroduce boundaries
                self.connectionValue    = 6
            return

        if specialCase is not None:
            print("Loop class was sent a special case that's not within predefined special case")


    #allLoops = a list of all loops, with the index being the enumeration of the loop
    #allLoopsRev = reverse lookup of an enumeration, given a loop
    def __init__(self, allData, definition = None, loadData = None, specialCase = None, safetyChecks = True):
        self.allData         = allData
        self.allTuptupDicts  = allData.allTuptupDicts
        self.allLoops        = allData.allLoops

        checkSum = 0
        if definition is not None:
            checkSum += 1
        if loadData is not None:
            checkSum += 1
        if specialCase is not None:
            checkSum += 1
        if checkSum != 1:
            self.errorOutLoop('ERROR 2-5-1: A new loop must have a definition xor a loadData value xor a specialCase', forcePoints = definition)

        self.points          = None
        self.boundaries      = None
        self.hasBoundaries   = None
        self.maxSplit        = 0
        self.enumString      = None
        self.enum            = None
        self.isoRotations    = []
        self.isoMappings     = []
        self.selfReversable  = False
        self.reversedLoop    = None
        self.reversedMapping = None
        self.connectionValue = 0
        self.variantShifts   = {}
        self.connectors      = []
        self.enclosures      = []
        self.disappearances  = []
        self.boundToDisa     = {}
        self.deletions       = {}
        self.degenerations   = {}

        if loadData is not None:
            self.readIn(loadData)
            return

        if definition == () or specialCase is not None:
            self.generateSpecialCases(definition, specialCase)
            return

        if safetyChecks:
            self.verifyValidPart1(definition)

        self.simplifyAndEnumerate(definition)
        self.createISOMappings()
        self.createReversedLoop()

        if safetyChecks:
            self.verifyValidPart2()

        self.deriveConnectors()
        self.deriveEnclosures()
        self.deriveDeletions()

'''
allData = UniversalData()
allData.useIsoRegions = True

a = Loop(allData, definition = (3,5,3,5,4,4))

print(a.boundToDisa)'''
pass

In [19]:
#@title Loop Merges Class
class loopMerges():
    allData        = None
    allLoops       = None
    allMerges      = None
    leftLoop       = None
    rightLoop      = None
    combinedLoops  = None

    def combineTwoLoops(self, leftConnection, rightConnection):
        ( leftConnector,  leftBoundaryMapping,  leftBoundaryToDelete,  leftPointType) =  leftConnection
        (rightConnector, rightBoundaryMapping, rightBoundaryToDelete, rightPointType) = rightConnection

        moveEncoding = encodeLoopMoveData('merge', leftPointType, rightPointType)

        #Take care of the situation when both connectors are singletons or boundaries, which results in just a single singleton instead of two
        if (leftConnector == (SING,) or leftConnector == (BOUN,)) and (rightConnector == (SING,) or rightConnector == (BOUN,)):
            spawnNewLoop(self.allData, (SING,), origin='Merging: ' + str(leftConnector) + ' and ' + str(rightConnector))

            boundMap = {}
            if leftBoundaryToDelete is not None:
                boundMap[(LEFT,leftBoundaryToDelete)] = NOTBOUN
            if rightBoundaryToDelete is not None:
                boundMap[(RIGHT,rightBoundaryToDelete)] = NOTBOUN
            self.combinedLoops.append((SING, boundMap, moveEncoding))

            return

        combinedLoop = ()
        shiftAdjustment = 0

        if leftConnector == (SING,) or leftConnector == (BOUN,):  #When the left connector is from a one-pointed loop that's a singleton or a boundary, there's no split point that's a connector
            combinedLoop = leftConnector
            shiftAdjustment = 1
        else:  #For all other cases, track which splits are on which side, so that we can adjust their values later on
            leftSplits = []
            for point in leftConnector:
                if point >= SPLITSTART and point not in leftSplits:
                    leftSplits.append(point)
            if rightConnector == (SING,):  #This should technically only happen when the other node is (HANG,) or (FREE,), but it's left broader just in case
                combinedLoop = leftConnector[1:] + rightConnector
            else:
                newSplit = len(leftConnector) + len(rightConnector) + SPLITSTART
                combinedLoop = (newSplit,) + leftConnector[1:] + (newSplit,)


        if rightConnector != (SING,):  #We've already taken care of the case where the right connector is a singleton above, so skip this whole section if that's the case
            rightSplitMapping = {}
            for point in rightConnector:
                if point >= SPLITSTART and point not in rightSplitMapping:
                    rightSplitMapping[point] = point+len(leftConnector)

            for pointIndex in range(1, len(rightConnector)):
                point = rightConnector[pointIndex]
                if point < SPLITSTART:
                    combinedLoop += (point,)
                else:
                    combinedLoop += (rightSplitMapping[point],)

        combinedLoop = adjustSplits(combinedLoop)

        (newEnum, newShift) = spawnNewLoop(self.allData, combinedLoop, origin='Merging: ' + str(self.leftLoop.points) + ' and ' + str(self.rightLoop.points))

        fullRemapping = {}
        for i in leftBoundaryMapping:
            fullRemapping[(LEFT, i)] = (leftBoundaryMapping[i]-newShift) % len(combinedLoop)
        if leftBoundaryToDelete is not None:
            fullRemapping[(LEFT, leftBoundaryToDelete)] = NOTBOUN

        for i in rightBoundaryMapping:
            fullRemapping[(RIGHT, i)] = (rightBoundaryMapping[i] + len(leftConnector) - newShift - shiftAdjustment) % len(combinedLoop)
        if rightBoundaryToDelete is not None:
            fullRemapping[(RIGHT, rightBoundaryToDelete)] = NOTBOUN

        self.combinedLoops.append((newEnum, fullRemapping, moveEncoding))


    def specialTripleMapping(self, rightLoop):
        rightLoopEnum = rightLoop.enum
        #We'll first deal with that super-special case of merging two LONELYBOUNDARYTRIPLEs
        if rightLoopEnum == LONELYBOUNDARYTRIPLE:
            newLoopPoints = (BOUN, BOUN, SPLITSTART, BOUN, BOUN, SPLITSTART)
            (newEnum, newShift) = spawnNewLoop(self.allData, newLoopPoints, origin='spawning merge of two LONELYBOUNDARYTRIPLEs')

            fullRemapping = {}
            #0,1,3,4 are based on the positions of the boundaries in newLoopPoints, and will need to be changed if that loop is somehow redefined
            #The equations below are for the very off-chance that the point constants are changed and cause a reordering of the loop (holds true for next two "ifs" too)
            fullRemapping[(LEFTAPPEARINGLOOP , 0)] = (0 - newShift) % len(newLoopPoints)
            fullRemapping[(LEFTAPPEARINGLOOP , 1)] = (1 - newShift) % len(newLoopPoints)
            fullRemapping[(RIGHTAPPEARINGLOOP, 0)] = (3 - newShift) % len(newLoopPoints)
            fullRemapping[(RIGHTAPPEARINGLOOP, 1)] = (4 - newShift) % len(newLoopPoints)
            moveEncoding = encodeLoopMoveData('merge', BOUN, BOUN)
            self.combinedLoops.append((newEnum, fullRemapping, moveEncoding))
        elif rightLoopEnum == SING or rightLoopEnum == BOUN:
            newLoopPoints = (SING, BOUN, BOUN)
            (newEnum, newShift) = spawnNewLoop(self.allData, newLoopPoints, origin='spawning merge of LONELYBOUNDARYTRIPLE and SING/BOUN')

            fullRemapping = {}
            #1 and 2 are based on the positions of the boundaries in newLoopPoints, and will need to be changed if that loop is somehow redefined
            fullRemapping[(LEFTAPPEARINGLOOP, 0)] = (1 - newShift) % len(newLoopPoints)
            fullRemapping[(LEFTAPPEARINGLOOP, 1)] = (2 - newShift) % len(newLoopPoints)
            if rightLoopEnum == BOUN:
                fullRemapping[(RIGHT,0)] = DISABOUNCHECK
            moveEncoding = encodeLoopMoveData('merge', BOUN, rightLoopEnum)
            self.combinedLoops.append((newEnum, fullRemapping, moveEncoding))
        else:
            for connection in rightLoop.connectors:
                (rightConnector, rightBoundaryMapping, rightBoundaryToDelete, rightPointType) = connection

                leftLoopMerge = (len(rightLoop.points) + SPLITSTART, BOUN, BOUN, len(rightLoop.points) + SPLITSTART)
                combinedLoop = adjustSplits(leftLoopMerge + rightConnector[1:])
                (newEnum, newShift) = spawnNewLoop(self.allData, combinedLoop, origin='Merging: ' + str(LONELYBOUNDARYTRIPLE) + ' and ' + str(rightLoop.enum))
                shiftAdjustment = 2

                fullRemapping = {}
                #The first two boundaries
                fullRemapping[(LEFTAPPEARINGLOOP, 0)] = (1-newShift) % len(combinedLoop)
                fullRemapping[(LEFTAPPEARINGLOOP, 1)] = (2-newShift) % len(combinedLoop)

                for i in rightBoundaryMapping:
                    fullRemapping[(RIGHT, i)] = (rightBoundaryMapping[i] + len(leftLoopMerge) - newShift - 1) % len(combinedLoop)
                if rightBoundaryToDelete is not None:
                    fullRemapping[(RIGHT, rightBoundaryToDelete)] = NOTBOUN

                moveEncoding = encodeLoopMoveData('merge', BOUN, rightPointType)
                self.combinedLoops.append((newEnum, fullRemapping, moveEncoding))


    def __init__(self, allData, leftLoop, rightLoop):
        if leftLoop.enum > rightLoop.enum:
            print("ERROR: Ensure that the left loop's enumeration is less than the right loop's.  Not strictly logically necessary, but trust me, it makes things easier.")
            exit()

        self.allData   = allData
        self.allLoops  = allData.allLoops
        self.allMerges = allData.allMerges
        self.leftLoop  = leftLoop
        self.rightLoop = rightLoop
        self.combinedLoops = []

        if leftLoop.enum not in self.allMerges:
            self.allMerges[self.leftLoop.enum] = {}

        if rightLoop.enum in self.allMerges[self.leftLoop.enum]:
            errorOut("Loops already in the merger:" + str(leftLoop.enum) + " and " + str(rightLoop.enum))
        self.allMerges[self.leftLoop.enum][rightLoop.enum] = self

        if leftLoop.enum == LONELYBOUNDARYTRIPLE:
            self.specialTripleMapping(rightLoop)
            return

        for leftConnection in leftLoop.connectors:
            for rightConnection in rightLoop.connectors:
                self.combineTwoLoops(leftConnection, rightConnection)

In [20]:
#@title Region Class
class RegionClass():
    allData          = None
    allLoops         = None
    region           = None
    reversedRegion   = None
    reverseBoundMap  = None
    lesserOrder      = None
    selfReversable   = None
    merges           = None
    enclosures       = None
    disaDegens       = None
    binaryList       = None
    allPointReorders = None
    boundaries       = None

    def buildBoundaries(self):
        for loopIndex in range(len(self.region)):
            loop = self.allLoops[self.region[loopIndex]]
            for pointIndex in range(len(loop.points)):
                if loop.points[pointIndex] == BOUN:
                    self.boundaries.append((loopIndex, pointIndex))


    def generateMerges(self, leftLoopIndex, rightLoopIndex, uniqueMappings):
        region = self.region
        leftLoop = self.allLoops[region[leftLoopIndex]]
        rightLoop = self.allLoops[region[rightLoopIndex]]
        curMerge = spawnNewMerge(self.allData, leftLoop, rightLoop)

        #We're going to recreate the region here, which keeps the untouched loops and adds the new merged one, and then generates a new boundary mapping
        untouchedLoops = () #A tuple of loops in the region that aren't affected by the merge
        baseRemapping = {}
        for loopIndex in range(len(region)):
            if loopIndex != leftLoopIndex and loopIndex != rightLoopIndex:
                points = self.allLoops[region[loopIndex]].points
                for pointIndex in range(len(points)):
                    point = points[pointIndex]
                    if points[pointIndex] == BOUN:
                        baseRemapping[(loopIndex, pointIndex)] = (len(untouchedLoops), pointIndex)
                untouchedLoops += (region[loopIndex],)

        for tup in curMerge.combinedLoops:
            (newLoopEnum, mergeRemapping, moveEncoding) = tup
            #We need this region and base map to be clean at the start of each loop, so best to just copy them each time we need them.
            newRegion = untouchedLoops + (newLoopEnum,)
            completeRemapping = copyMap(baseRemapping)

            #For the newly-formed loop, put it at the end of the region, and map the boundary point positions appropriately
            for mapping in mergeRemapping:
                loopSide = mapping[0]
                pointIndex = mapping[1]
                curLoopIndex = leftLoopIndex
                if loopSide ==  RIGHT:
                    curLoopIndex = rightLoopIndex
                elif loopSide in [LEFTAPPEARINGLOOP, RIGHTAPPEARINGLOOP]:
                    curLoopIndex = loopSide

                if mergeRemapping[mapping] == NOTBOUN:
                    completeRemapping[(curLoopIndex, pointIndex)] = DEADLOOP
                elif mergeRemapping[mapping] == DISABOUNCHECK:
                    completeRemapping[(curLoopIndex, pointIndex)] = DISABOUNDCHECKLOOP
                else:
                    completeRemapping[(curLoopIndex, pointIndex)] = (len(newRegion)-1, mergeRemapping[mapping])

            (orderedRegion, storedRemapping) = spawnNewRegion(self.allData, newRegion, prevRemap = completeRemapping)

            checkUnique = str(orderedRegion) + str(storedRemapping)
            if checkUnique not in uniqueMappings:
                uniqueMappings[checkUnique] = True
                #This is the data we need to store so that this particular enclosure in this region can be applied to all instances of this region in any game
                self.merges.append((orderedRegion, storedRemapping, moveEncoding))
        return


    def generateEnclosures(self, splitLoopIndex, uniqueMappings):
        #self.allData.allBinaryTrees.binaryList[len(self.region) - 1] is a list of tuples representing each binary number with a length equal to the number of loops we'll be seperating out
        #self.allData.allBinaryTrees.centerAdjusted[seperation] is a list of all ways BINCENTER could be inserted into the binary loops
        for seperation in self.allData.allBinaryTrees.binaryList[len(self.region) - 1]: #Iterate over all possible ways to seperate the enclosures, with loopIndex being the variant needed for our current loop's index
            centerAdjusted = self.allData.allBinaryTrees.centerAdjusted[seperation][splitLoopIndex]
            #We first iterate over the center-adjusted loop and shove the given loop into the left or right depending on the current binary list, and store that as a basis to then apply all enclosures of the loop index of choice

            leftRegionBase = ()
            leftBoundMap = {}
            rightRegionBase = ()
            rightBoundMap = {}

            for sepLoopIndex in range(len(centerAdjusted)):
                leftRightVal = centerAdjusted[sepLoopIndex]
                if sepLoopIndex == splitLoopIndex: #Don't include the loop we're enclosing on
                    continue
                elif leftRightVal == BINLEFT:
                    leftRegionBase = leftRegionBase + (self.region[sepLoopIndex],)
                    leftBoundMap[sepLoopIndex] = len(leftRegionBase)-1
                elif leftRightVal == BINRIGHT:
                    rightRegionBase = rightRegionBase + (self.region[sepLoopIndex],)
                    rightBoundMap[sepLoopIndex] = len(rightRegionBase)-1
                else:
                    print("You probably just put the BINCENTER value in the wrong spot, dummy")
                    exit()

            moveEncodingAdjustment        = SIDEENCODING['neither'] * EMPTYENCLOSUREMULTIPLIER
            moveEncodingAdjustmentRev     = SIDEENCODING['neither'] * EMPTYENCLOSUREMULTIPLIER
            if len(self.region) == 1:
                moveEncodingAdjustment    = SIDEENCODING['both']    * EMPTYENCLOSUREMULTIPLIER
                moveEncodingAdjustmentRev = SIDEENCODING['both']    * EMPTYENCLOSUREMULTIPLIER
            elif len(leftRegionBase) == 0:
                moveEncodingAdjustment    = SIDEENCODING['left']    * EMPTYENCLOSUREMULTIPLIER
                moveEncodingAdjustmentRev = SIDEENCODING['right']   * EMPTYENCLOSUREMULTIPLIER
            elif len(rightRegionBase) == 0:
                moveEncodingAdjustment    = SIDEENCODING['right']   * EMPTYENCLOSUREMULTIPLIER
                moveEncodingAdjustmentRev = SIDEENCODING['left']    * EMPTYENCLOSUREMULTIPLIER

            boundaryRemappingBase = {}
            for boundary in self.boundaries:
                (loopIndex, pointIndex) = boundary
                if loopIndex in leftBoundMap: #We can technically avoid remapping left boundaries before the first right boundary, but that's a tiny save compared to everything else
                    boundaryRemappingBase[boundary] = ( LEFT,  leftBoundMap[loopIndex], pointIndex)
                elif loopIndex in rightBoundMap:
                    boundaryRemappingBase[boundary] = (RIGHT, rightBoundMap[loopIndex], pointIndex)
                elif centerAdjusted[loopIndex] != BINCENTER:
                    print("ERROR R-E-1:  Something went wrong with the enclosures over a region, and there's a loop index that doesn't exist in leftBoundMap, rightBoundMap, or sepLoopIndex")
                    print(centerAdjusted, self.region, self.boundaries, '/', leftRegionBase, rightRegionBase, boundary, leftBoundMap, rightBoundMap, sepLoopIndex)
                    exit()

            for enclosureTup in self.allLoops[self.region[splitLoopIndex]].enclosures:
                (leftEnum, leftBoundMapping, rightEnum, rightBoundMapping, leftToRightPairing, boundaryRemoval, moveEncoding) = enclosureTup
                leftRegion  =  leftRegionBase + ( leftEnum,)
                rightRegion = rightRegionBase + (rightEnum,)

                #We'll go ahead and add all of the new sub-regions, since they'll all to exist as new games are created.
                # Because this is we're doing this n * 2^(n-1) times for loops of size n, and all subsequent loops, in theory it becomes a n^2 * 2^n problem
                # In reality it's faster, as the deeper levels collapse rather than expand.  But still.  This is the time intensive part.
                ( orderedLeftRegion,  leftLoopsReordering) = spawnNewRegion(self.allData,  leftRegion)
                (orderedRightRegion, rightLoopsReordering) = spawnNewRegion(self.allData, rightRegion)
                leftRegionNewLoop  =  leftLoopsReordering[len( leftRegionBase)]
                rightRegionNewLoop = rightLoopsReordering[len(rightRegionBase)]

                #The leftToRightPairing can be given two-tuples in a similar fashion, instead of having to re-infer the loop index every time
                leftToRightPairingWithLoopsTuptup = []
                rightToLeftPairingWithLoopsTuptup = []
                for leftPointIndex in leftToRightPairing:
                    rightPointIndex = leftToRightPairing[leftPointIndex]
                    leftToRightPairingWithLoopsTuptup.append((( leftRegionNewLoop,  leftPointIndex), (rightRegionNewLoop, rightPointIndex)))
                    rightToLeftPairingWithLoopsTuptup.append(((rightRegionNewLoop, rightPointIndex), ( leftRegionNewLoop,  leftPointIndex)))
                leftToRightPairingWithLoops = self.allData.allTuptupDicts.getDictFromUnsorted(leftToRightPairingWithLoopsTuptup)
                rightToLeftPairingWithLoops = self.allData.allTuptupDicts.getDictFromUnsorted(rightToLeftPairingWithLoopsTuptup)

                #Create a remapping of the boundaries, spawned from our original base but with left/right reorderings taken into consideration
                boundaryRemapping = {}
                reversedBoundRemapping = {}  #We want to keep a reversed version so that we don't duplicate enclosure.  We'll check duplication later
                for boundary in boundaryRemappingBase:
                    destBound = boundaryRemappingBase[boundary]
                    if destBound[0] == LEFT:
                        boundaryRemapping[     boundary] = ( LEFT, leftLoopsReordering[destBound[1]], destBound[2])
                        reversedBoundRemapping[boundary] = (RIGHT, leftLoopsReordering[destBound[1]], destBound[2])
                    elif destBound[0] == RIGHT:
                        boundaryRemapping[     boundary] = (RIGHT, rightLoopsReordering[destBound[1]], destBound[2])
                        reversedBoundRemapping[boundary] = ( LEFT, rightLoopsReordering[destBound[1]], destBound[2])

                #Remap the boundaries in loops that touch the loop that was enclosed on
                for boundary in leftBoundMapping:
                    boundaryRemapping[     (splitLoopIndex, boundary)] = ( LEFT,  leftLoopsReordering[len( leftRegion)-1], leftBoundMapping[boundary])
                    reversedBoundRemapping[(splitLoopIndex, boundary)] = (RIGHT, rightLoopsReordering[len(rightRegion)-1], leftBoundMapping[boundary])
                for boundary in rightBoundMapping:
                    boundaryRemapping[     (splitLoopIndex, boundary)] = (RIGHT, rightLoopsReordering[len(rightRegion)-1], rightBoundMapping[boundary])
                    reversedBoundRemapping[(splitLoopIndex, boundary)] = ( LEFT,  leftLoopsReordering[len( leftRegion)-1], rightBoundMapping[boundary])
                for boundary in boundaryRemoval:
                    boundaryRemapping[     (splitLoopIndex, boundary)] = DEADPOINT
                    reversedBoundRemapping[(splitLoopIndex, boundary)] = DEADPOINT

                storedBoundaryRemapping    = self.allData.allTuptupDicts.getDictFromDict(     boundaryRemapping)
                storedRevBoundaryRemapping = self.allData.allTuptupDicts.getDictFromDict(reversedBoundRemapping)

                moveEncoding = compressEncoding(moveEncoding + moveEncodingAdjustment) #Depending on left/right, sometimes encodings have two variants, so we clean that up here

                checkUnique    = str( orderedLeftRegion.region) + str(orderedRightRegion.region) + str(   storedBoundaryRemapping) + str(leftToRightPairingWithLoops) + str(moveEncoding)
                checkUniqueRev = str(orderedRightRegion.region) + str( orderedLeftRegion.region) + str(storedRevBoundaryRemapping) + str(rightToLeftPairingWithLoops) + str(moveEncoding)

                #We can choose the lesser of the two checks, to ensure that if we end up with the other one elsewhere, we're just saving off one version
                if checkUnique not in uniqueMappings and checkUniqueRev not in uniqueMappings:
                    uniqueMappings[checkUnique] = True
                    self.enclosures.append((orderedLeftRegion, orderedRightRegion, storedBoundaryRemapping, leftToRightPairingWithLoops, moveEncoding))
        return


    #Special cases
    def generateLonelyPairDeletions(self, uniqueMappings):
        for loopIndex in range(len(self.region)):
            #Look for LONELYBOUNDARYPAIRS, and if one exists, delete it as a possible move (i.e. merge the interior bounds)
            #  Since these special cases don't have boundaries and are thus interchangable, we only ever have to handle it once even if there's several
            #  Technically we only need to look at the first index since the negative value means it should default to being the first,
            #  but we'll stay on the safe side just in case we add a lower negative special-case later
            if self.region[loopIndex] == LONELYBOUNDARYPAIR:
                newRegion = self.region[:loopIndex] + self.region[loopIndex+1:]

                prevRemap = {}
                for boundary in self.boundaries:
                    if boundary[0] > loopIndex:
                        prevRemap[boundary] = (boundary[0] - 1, boundary[1])

                (orderedRegion, storedRemapping) = spawnNewRegion(self.allData, newRegion, prevRemap = prevRemap)

                checkUnique = str(newRegion) + str(storedRemapping)
                if checkUnique not in uniqueMappings:
                    uniqueMappings[checkUnique] = True
                    self.merges.append((orderedRegion, storedRemapping, LONELYPAIRDROPENCODING))
                break
        return


    #Special cases
    def generateDisappearingDegenerations(self, uniqueMappings):
        for boundary in self.boundaries:
            (loopIndex, pointIndex) = boundary
            if self.region[loopIndex] == BOUN:  #If the loop itself is just a boundary, we'll need to delete it outright
                newRegion = self.region[:loopIndex] + self.region[loopIndex+1:]

                prevRemap = {}
                for remap in self.boundaries:
                    if remap[0] > loopIndex:
                        prevRemap[remap] = (remap[0] - 1, remap[1])

                (orderedRegion, storedRemapping) = spawnNewRegion(self.allData, newRegion, prevRemap = prevRemap)
                self.disaDegens[(boundary)] = (orderedRegion, storedRemapping)

            else:  #For all cases where the loop isn't just a boundary
                (newLoopEnum, degenMapping) = self.allData.allLoops[self.region[loopIndex]].deletions[(pointIndex,)]
                newRegion = self.region[:loopIndex] + (newLoopEnum,) + self.region[loopIndex+1:]

                prevRemap = {}
                for remap in self.boundaries:
                    if remap[0] == loopIndex:
                        if remap[1] in degenMapping:
                            prevRemap[remap] = (loopIndex, degenMapping[remap[1]])
                    else:
                        prevRemap[remap] = remap

                (orderedRegion, storedRemapping) = spawnNewRegion(self.allData, newRegion, prevRemap = prevRemap)

                checkUnique = str(orderedRegion.region) + str(storedRemapping)
                if checkUnique not in uniqueMappings:
                    uniqueMappings[checkUnique] = True
                    self.disaDegens[(boundary)] = (orderedRegion, storedRemapping)
        return


    def findLesserOrder(self, region, origin):
        sortedRegion = list(region)
        sortedRegion.sort()
        sortedRegion = tuple(sortedRegion)
        if sortedRegion != region:
            print("Error R.L.1: region not sorted:", region)
            print("Please ensure that region is sorted prior to creating it, as mapping becomes much uglier if that has to be dealt with in RegionClass")
            exit()

        reverseRegion = []
        for loopIndex in range(len(region)):
            loopEnum = region[loopIndex]
            reverseRegion.append(self.allData.allLoops[loopEnum].reversedLoop)
        reverseRegion.sort()
        reverseRegion = tuple(reverseRegion)

        (self.reversedRegion, dumpThis) = spawnNewRegion(self.allData, reverseRegion)

        self.lesserOrder = True
        self.selfReversable = False
        if reverseRegion < sortedRegion:
            self.lesserOrder = False
        elif reverseRegion == sortedRegion:
            self.selfReversable = True

        if self.lesserOrder:
            return(sortedRegion, reverseRegion)
        else:
            return(reverseRegion, sortedRegion)


    def reverseToMainBoundaryMapping(self):
        reversedWithOriginPositions = []
        for loopIndex in range(len(self.region)):
            loopEnum = self.region[loopIndex]
            reversedWithOriginPositions.append((self.allData.allLoops[loopEnum].reversedLoop, loopIndex))
        reversedWithOriginPositions.sort()

        self.reverseBoundMap = {}
        for sourceLoopIndex in range(len(reversedWithOriginPositions)):
            (sourceLoopEnum, destLoopIndex) = reversedWithOriginPositions[sourceLoopIndex]
            loopRevMap = self.allLoops[sourceLoopEnum].reversedMapping
            for sourcePointIndex in loopRevMap:
                destPointIndex = loopRevMap[sourcePointIndex]
                self.reverseBoundMap[(sourceLoopIndex, sourcePointIndex)] = (destLoopIndex, destPointIndex)
        return


    def defineRemapVariants(self):
        uniqueLoopMappings = []
        #Find which loops are duplicates of others in that loop, and create a complete list of isomorphic mappings
        allDupeLoops = []
        allDupeLoopMaps = {}
        startIndex = 0
        while startIndex < len(self.region):
            endIndex = startIndex
            #Each time we see a new unique loop in the region, mark the "startIndex" there and then continue on until the last copy of that loop is seen, and set that as the "endIndex"
            while endIndex + 1 < len(self.region) and self.region[endIndex] == self.region[endIndex+1]:
                endIndex += 1
            #Once we know the start and end indicies, create a permutation map of all ways of ordering those loops within the region
            #We only need to worry about any of this if the loops have boundaries to begin with, so I've thrown a check for that in there too
            if endIndex > startIndex and self.allLoops[self.region[startIndex]].hasBoundaries:
                dupeRegionsCount = endIndex - startIndex + 1
                if dupeRegionsCount not in self.allData.allPermutationMaps.permutationList:
                    self.allData.allPermutationMaps.addSize(dupeRegionsCount)
                curDupeRegions = []
                for permutation in self.allData.allPermutationMaps.permutationList[dupeRegionsCount]:
                    newDupeRegion = ()
                    for permuteIndex in range(len(permutation)):
                        newDupeRegion += ((permuteIndex + startIndex, permutation[permuteIndex] + startIndex),)
                    curDupeRegions.append(newDupeRegion)
                allDupeLoops.append(curDupeRegions)
            startIndex = endIndex+1

        #completeLoopMapping is going to contain mappings for all possible reorderings of identical loops
        completeLoopMapping = []
        recursiveRegionOrLoopReordering(completeLoopMapping, allDupeLoops)

        uniqueRotationMapping = []
        for loopIndex in range(len(self.region)):
            curLoop = self.allLoops[self.region[loopIndex]]
            if len(curLoop.isoMappings) > 0:
                uniqueRotationMapping.append([loopIndex, curLoop.isoMappings])

        completeRotationMapping = []
        recursiveCrossProductOfPermuteLists(completeRotationMapping, uniqueRotationMapping)

        #The maps need to have a length of at least one to execute, but portions where no remappings are necessary may end up empty
        #  To solve this, we just give them a dummy value that won't cause any alteration in boundary orders, so as to execute what's in each for loop
        if len(completeLoopMapping) == 0:
            completeLoopMapping.append({0:0})
        if len(completeRotationMapping) == 0:
            completeRotationMapping.append({0: {0: 0}})

        self.allPointReorders = []
        deDupReversals = {}
        for loopsToRemap in completeLoopMapping:
            #Intuitively, it makes more sense to shove all the reorders together.  However, due to how loop rotations and region remappings interact, we can get rid of a ton of
            #  repeated remappings by merely checking if the first variant of a region has already been checked.
            curLoopVariantReorder = []
            for pointsToRotate in completeRotationMapping:
                completeRemapTuptupArray = []
                completeRemap = {}
                revSafetyRemap = {} #If we're reversing the loop, we want to keep points that remap to themselves so that our product later includes
                                    #  bounds that are remapped when reversing but not remapped in the standard forward direction
                for point in self.boundaries:
                    curLoop = point[0]
                    curPoint = point[1]

                    newLoop = curLoop
                    if curLoop in loopsToRemap:
                        newLoop = loopsToRemap[curLoop]

                    newPoint = curPoint
                    if curLoop in pointsToRotate:
                        if curPoint not in pointsToRotate[curLoop]:
                            print("Error R-V-1: Point in findLowestGameNameTest's point rotation not in the apparent loop?")
                        newPoint = pointsToRotate[curLoop][curPoint]

                    if newLoop != curLoop or newPoint != curPoint:
                        completeRemapTuptupArray.append((point, (newLoop, newPoint)))
                        completeRemap[point] = (newLoop, newPoint)
                    revSafetyRemap[point] = (newLoop, newPoint)
                completeRemapTuptupArray.sort()
                completeRemapTuptup = tuple(completeRemapTuptupArray)

                #Sometimes we create a mapping via a reversal before we create it in a forwards direction, so we need to make sure we don't dedupe even in the forward direction
                if completeRemapTuptup not in deDupReversals:
                    deDupReversals[completeRemapTuptup] = True
                    self.allPointReorders.append(self.allData.allTuptupDicts.getDict(completeRemapTuptup))

                #If the region is reversible, we implement that here
                if self.selfReversable:
                    completeRevRemapTuptupArray = []
                    completeRevRemap = {}
                    for sourcePoint in revSafetyRemap:
                        destPoint = revSafetyRemap[sourcePoint]
                        if sourcePoint != self.reverseBoundMap[destPoint]:
                            completeRevRemapTuptupArray.append((sourcePoint, self.reverseBoundMap[destPoint]))
                            completeRevRemap[sourcePoint] = self.reverseBoundMap[destPoint]
                    completeRevRemapTuptupArray.sort()
                    completeRevRemapTuptup = tuple(completeRevRemapTuptupArray)
                    if completeRevRemapTuptup not in deDupReversals:
                        deDupReversals[completeRevRemapTuptup] = True
                        self.allPointReorders.append(self.allData.allTuptupDicts.getDict(completeRevRemapTuptup))
        return


    def handleEmptyRegion(self):
        self.region           = ()
        self.allData.allRegions[()] = self
        self.reversedRegion   = self
        self.reverseBoundMap  = self.allData.allTuptupDicts.getDict(())
        self.lesserOrder      = True
        self.selfReversable   = True


    def loadRegionLine(self, rawSavedLine, isCompressed = False):
        rawSavedLine = rawSavedLine.strip()
        fieldVals = rawSavedLine.split('|')
        if len(fieldVals) != 9:
            print("ERROR R-L-0: Invalid number of fields in region input line:", line)
            exit()

        region              = fieldVals[0]
        reversedRegion      = fieldVals[1]
        reverseBoundMap     = fieldVals[2]
        lesserOrder         = fieldVals[3]
        selfReversable      = fieldVals[4]
        mergeStringList     = fieldVals[5]
        enclosureStringList = fieldVals[6]
        disaStringList      = fieldVals[7]
        reorderStringList   = fieldVals[8]

        if region == EMPTYGAME:
            self.handleEmptyRegion()
            return

        self.region          = strTupToInt(region.split(','))
        self.allData.allRegions[self.region] = self

        revRegion = strTupToInt(reversedRegion.split(','))
        if revRegion in self.allData.allRegions:
            self.reversedRegion  = self.allData.allRegions[revRegion]
            if revRegion != region:
                self.reversedRegion.reversedRegion = self
        else:  #We rely on the line above to fill in the reversed region later.
            self.reversedRegion = None

        if len(reverseBoundMap) == 0:
            self.reverseBoundMap = self.allData.allTuptupDicts.getDict(())
        else:
            self.reverseBoundMap = self.allData.allTuptupDicts.dictLoadTuptupString(reverseBoundMap, compressed = isCompressed)

        self.lesserOrder    = (lesserOrder    == 'T')
        self.selfReversable = (selfReversable == 'T')

        self.merges = []
        if len(mergeStringList) != 0:
            mergeStringList = mergeStringList.split("/")
            for mergeStringIndex in range(len(mergeStringList)):
                mergeString = mergeStringList[mergeStringIndex]
                merge = mergeString.split(';')
                if len(merge) != 3:
                    print("ERROR R-L-1:  Merge string list length expected to be 2, is actually", len(mergeStringList), ":")
                    print("   ", merge)
                    exit()
                mergeRegion = strTupToInt(merge[0].split(','))
                if mergeRegion[0] is None:
                    mergeRegion = () + mergeRegion[1:]

                mergeEncoding = int(merge[1])

                mergeRemap = self.allData.allTuptupDicts.getDict(())
                if len(merge[2]) != 0:
                    mergeRemap = self.allData.allTuptupDicts.dictLoadTuptupString(merge[2], compressed = isCompressed, compressDeadSize = 2)
                self.merges.append((self.allData.allRegions[mergeRegion], mergeRemap, mergeEncoding))

        self.enclosures = []
        if len(enclosureStringList) != 0:
            enclosureStringList = enclosureStringList.split("/")
            for enclosureStringIndex in range(len(enclosureStringList)):
                enclosureString = enclosureStringList[enclosureStringIndex]
                enclosure = enclosureString.split(';')

                if len(enclosure) != 5:
                    print("ERROR R-L-2:  EnclosureStringList length expected to be 4, is actually", len(enclosureStringList), ":")
                    print("   ", enclosure)
                    exit()
                leftRegion  = strTupToInt(enclosure[0].split(','))
                rightRegion = strTupToInt(enclosure[1].split(','))
                enclosureEncoding = int(enclosure[2])

                enclosureRemap = self.allData.allTuptupDicts.getDict(())
                if len(enclosure[3]) != 0:
                    enclosureRemap = self.allData.allTuptupDicts.dictLoadTuptupString(enclosure[3], compressed = isCompressed, compressDeadSize = 3)

                leftToRightPairing = self.allData.allTuptupDicts.getDict(())
                if len(enclosure[4]) != 0:
                    leftToRightPairing = self.allData.allTuptupDicts.dictLoadTuptupString(enclosure[4], compressed = isCompressed, compressDeadSize = 2)

                self.enclosures.append((self.allData.allRegions[leftRegion], self.allData.allRegions[rightRegion], enclosureRemap, leftToRightPairing, enclosureEncoding))

        self.disaDegens = {}
        self.boundaries = []
        if len(disaStringList) != 0:
            disaStringList = disaStringList.split("/")
            for disaString in disaStringList:
                disa = disaString.split(';')
                if len(disa) != 3:
                    print("ERROR R-L-3:  disaStringList length expected to be 3, is actually", len(disaStringList), ":")
                    print("   ", disaString)
                    exit()

                disaOrigin    = strTupToInt(disa[0].split(','))
                disaNewRegion = ()
                if len(disa[1]) != 0:
                    disaNewRegion = strTupToInt(disa[1].split(','))
                disaRemap = self.allData.allTuptupDicts.getDict(())
                if len(disa[2]) != 0:
                    disaRemap = self.allData.allTuptupDicts.dictLoadTuptupString(disa[2], compressed = isCompressed, compressDeadSize = 2)

                self.disaDegens[disaOrigin] = (self.allData.allRegions[disaNewRegion], disaRemap)
                self.boundaries.append(disaOrigin)


        self.allPointReorders = []
        reorderStringList = reorderStringList.split("/")
        for reorderDict in reorderStringList:
            remap = self.allData.allTuptupDicts.getDict(())
            if len(reorderDict) != 0:
                remap = self.allData.allTuptupDicts.dictLoadTuptupString(reorderDict, compressed = isCompressed, compressDeadSize = 2)
            self.allPointReorders.append(remap)

        if revRegion in self.allData.allRegions and self.saveRegionLine(isCompressed) != rawSavedLine:
            print("ERROR G-L-2:  Saved line ends up changed when re-saving; assumed error in reading or storing:")
            print("Saved Line:       ", rawSavedLine)
            print("Re-written Line:  ", self.saveRegionLine(isCompressed))
            exit()
        return


    def saveRegionLine(self, compress = False):
        saveLineList = []
        if self.region == ():
            saveLineList.append("NULL")
            saveLineList.append("NULL")
            saveLineList.append("")
        else:
            saveLineList.append(intTupToStr(self.region))
            saveLineList.append(intTupToStr(self.reversedRegion.region))
            saveLineList.append(dictSaveString(self.reverseBoundMap, compress = compress))

        if self.lesserOrder:
            saveLineList.append('T')
        else:
            saveLineList.append('F')

        if self.selfReversable:
            saveLineList.append('T')
        else:
            saveLineList.append('F')

        mergeStringList = []
        for merge in self.merges:
            (mergedRegionObject, remapDict, moveEncoding) = merge
            mergedRegion = mergedRegionObject.region
            mergeStringList.append((intTupToStr(mergedRegion) + ';' + str(moveEncoding) + ";" + dictSaveString(remapDict, compress = compress)))
        saveLineList.append('/'.join(mergeStringList))

        enclosureStringList = []
        for enclosure in self.enclosures:
            (leftRegionObject, rightRegionObject, remapDict, leftToRightPairing, moveEncoding) = enclosure
            leftRegion  =  leftRegionObject.region
            rightRegion = rightRegionObject.region
            enclosureStringList.append((intTupToStr(leftRegion) + ';' + intTupToStr(rightRegion) + ';' + str(moveEncoding) + ";" + dictSaveString(remapDict, compress = compress) + ';' + dictSaveString(leftToRightPairing, compress = compress)))
        saveLineList.append('/'.join(enclosureStringList))

        disaStringList = []
        for boundary in self.disaDegens:
            (disaRegionObject, newBoundaryRemap) = self.disaDegens[boundary]
            disaRegion = disaRegionObject.region
            disaStringList.append((intTupToStr(boundary) + ';' + intTupToStr(disaRegion) + ';' + dictSaveString(newBoundaryRemap, compress = compress)))
        saveLineList.append('/'.join(disaStringList))

        reorderStringList = []
        for reorderDict in self.allPointReorders:
            reorderStringList.append(dictSaveString(reorderDict, compress = compress))
        saveLineList.append('/'.join(reorderStringList))

        return '|'.join(saveLineList)


    def printReadableData(self):
        print("region:          ", self.region           )
        print("reversedRegion:  ", self.reversedRegion.region   )
        print("reverseBoundMap: ", self.reverseBoundMap  )
        print("lesserOrder:     ", self.lesserOrder      )
        print("selfReversable:  ", self.selfReversable   )
        print("boundaries:      ", self.boundaries       )
        print("disaDegens:      ",)
        for disaIndex in self.disaDegens:
            print('   ', disaIndex, ":", self.disaDegens[disaIndex][0].region, self.disaDegens[disaIndex][1])
        print("allPointReorders:", self.allPointReorders )
        print("merges:")
        for mergeTup in self.merges:
            print('   ', mergeTup[0].region, mergeTup[1:])
        print("enclosures:")
        for enclosureTup in self.enclosures:
            print('   ', enclosureTup[0].region, enclosureTup[1].region, enclosureTup[2:])
        print()
        return


    def __init__(self, allData, region = None, loadString = None, origin = 0, compressedLoad = False):
        self.allData  = allData
        self.allLoops = allData.allLoops

        if loadString is not None:
            self.loadRegionLine(loadString, isCompressed = compressedLoad)
            return

        if region is not None and loadString is not None:
            print("Error R.I.0: Region creation requires either a region (as a tuple) or a load string; was sent both")
            exit()
        elif region is None and loadString is None:
            print("Error R.I.1: Region creation requires either a region (as a tuple) or a load string; was sent neither")
            exit()

        if region in allData.allRegions:
            print("Error R.I.2: region already in regionEnclosures", region, "--- Origin:", origin)
            exit()

        self.region           = region
        self.reversedRegion   = None
        self.reverseBoundMap  = None
        self.lesserOrder      = True
        self.selfReversable   = False
        self.merges           = []
        self.enclosures       = []
        self.disaDegens       = {}
        self.allPointReorders = []
        self.boundaries       = []

        if self.region == ():
            self.handleEmptyRegion()
            return

        self.buildBoundaries()

        self.allData.allRegions[self.region] = self
        (lesserSymmetry, greaterSymmetry) = self.findLesserOrder(region, origin)
        self.reverseToMainBoundaryMapping()

        #Seperation Size:  The number of loops to be seperated during enclosures (i.e. the size of the region - 1)
        if (len(self.region) - 1) not in self.allData.allBinaryTrees.binaryList:
            self.allData.allBinaryTrees.addSize(len(self.region) - 1)

        self.defineRemapVariants()

        uniqueMappings = {} #We don't want to keep uniqueMappings, as it's large and only helps with building the region

        #Cycle over each loop, and enclose other loops in every possible way over the enclosure options
        for loopIndex in range(len(self.region)):
            self.generateEnclosures(loopIndex, uniqueMappings)

        for leftLoopIndex in range(len(self.region)):  #Merges require a left side and a right side.  Run through all possible loops as the left
            for rightLoopIndex in range(leftLoopIndex+1, len(self.region)):  #To avoid redundancy, we always have the left side as having the lesser or equal encoding.  Since loops are already sorted, we can just pull the loops to the right of the left loop
                self.generateMerges(leftLoopIndex, rightLoopIndex, uniqueMappings)

        self.generateLonelyPairDeletions(uniqueMappings)
        self.generateDisappearingDegenerations(uniqueMappings)


'''
smallData = UniversalData()

if -12 not in smallData.allLoops:
    Loop(smallData, specialCase = -12)

if -13 not in smallData.allLoops:
    Loop(smallData, specialCase = -13)

#spawnNewLoop(dataSummary, (1,))
#spawnNewLoop(dataSummary, (3,))
spawnNewLoop(smallData, (1,4,1,4))
#spawnNewLoop(dataSummary, (3,3))

region = (-13,2)
spawnNewRegion(smallData, region)

smallData.allRegions[region].printReadableData()

#for region in dataSummary.allRegions:
#    print(dataSummary.allRegions[region].saveRegionLine(compress = True))
#
'''
'''
for region in dataSummary.allRegions:
    print(region)
    print(dataSummary.allRegions[region].merges)
    print(dataSummary.allRegions[region].disaDegens)
    print()

#print(dataSummary.allRegions[(-13,)].enclosures)
#print("---", dataSummary.allRegions[ (-13, -13, -12, 23)].allPointReorders)
#print("---", dataSummary.allRegions[ (-13, -13, -12, 23)].regionIndexPointReorders)'''
pass

In [21]:
#@title Game Class

#allData : A UniversalData object
#game = An array of tuples, with each tuple being a region. Must be paired with a boundaries.
#boundaries = A map of three-tuples mapped to other three-tuples
#gameString = The gameString of the game to be created. Results in the gameString being converted to a new "game" and "boundaries,"
#savedFileLine = A complete line from a save file. Skips all of the calculations, and simply sets all the data to the values in the line.
#preOrdered = Defaults to False. When this function handles 'game' or 'gameString', setting this to 'True' tells the system that everything was ordered previously, and that the ordering functions can be skipped
#creationMessage = For testing purposes
#compressed = Defaults to False. For 'gameString' and 'savedFileLine', set to "True" if the value being sent is in compressed format
#preProcessed = Defaults to False. For 'gameString' inputs, set to True if you know that the string has already processed all deletions and degenerations
#recurseDepth = Defaults to -1. If > 0, will count down each for each child spawned, and stop spawning new children when recursion hits 0. If set to 0, the initial data will be processed and children calcualted, but no children will be spawned (i.e. don't try calling them from allRegions, because they won't be there)
#connectionsDepth = Defaults to -1. If > 0, will cut off recursion whenever the game has that many connections left.

class GameClass:
    game            	= None  #game[regionNum] = [(loopEnum, loopEnum2, loopEnum3...),...]  for any number of regions
    creationMessage   = None
    allData           = None
    allLoops      	  = None
    allMerges     	  = None
    allRegions     	  = None
    allGames          = None
    allBinaryTrees    = None
    boundaries        = None
    gameString      	= None
    compressedString  = None
    subGames          = None  #An array of the unified game strings that make up the sum total of this game (only has one value if the game is itself unified)
    nimber            = None
    totalConnections  = None
    minMoves          = None
    maxMoves          = None
    childGames        = None

    def boundarySafetyCheck(self):
        #Safety check to make sure boundary locations are in matching pairs
        pairingsSeen = {}
        for key in self.boundaries:
            if self.boundaries[key] == DISAPOINT:
                if self.allLoops[self.game[key[0]][key[1]]].points[key[2]] != BOUN:
                    print("ERROR G-S-1: Disappearing point doesn't match a boundary has no match:")
                    print('Game:', self.game)
                    print('Boundary Locations:', self.boundaries)
                    print('Creation Message:', self.creationMessage)
                    print('Boundary:', key)
                    print('Other side (does not exist):', self.boundaries[key])
                    exit()
                continue  #If the boundary is a disappearing point, it doesn't need be verified beyond knowing that the key is a boundary
            if self.boundaries[key] not in self.boundaries:
                print('ERROR G-S-2: Boundary has no match:')
                print('Boundary:', key)
                print('Other side (does not exist):', self.boundaries[key])
                exit()
            if key[0] == self.boundaries[key][0]:
                print('ERROR G-S-3: Boundary connects region to itself:')
                print('Boundary:',  key)
                print('Other side:', self.boundaries[key])
                print('Region of side 1:', key[0])
                print('Region of side 2:', self.boundaries[self.boundaries[key]][0])
                exit()
            if key != self.boundaries[self.boundaries[key]]:
                print('ERROR G-S-4: Mismatched boundary:')
                print('Creation Message:', self.creationMessage)
                print('Boundary:',  key)
                print('Other side:', self.boundaries[key])
                print('Other side of other side', self.boundaries[self.boundaries[key]])
                exit()
            if key[0] >= len(self.game):
                print('ERROR G-S-5: Boundary region is higher than number of regions')
                print('Boundary:', key)
                print('All regions:', self.game)
                exit()
            if key[1] >= len(self.game[key[0]]):
                print('ERROR G-S-6: Boundary loop is greater than the number of loops in the given region')
                print('Boundary:', key)
                print('All regions:', self.game)
                print('Specific region:', self.game[key[0]])
                exit()
            loop = self.allLoops[self.game[key[0]][key[1]]].points
            if key[2] >= len(loop):
                print('ERROR G-S-7: Boundary point is greater than the number of points in the given loop')
                print('Boundary:', key)
                print('All regions:', self.game)
                print('Specific region:', self.game[key[0]])
                print('Loop:', loop)
                exit()
            if loop[key[2]] != BOUN:
                print('ERROR G-S-8: Boundary point in boundary mapping is not a boundary point in the given loop')
                print('This can happen if you\'re manually building regions, make sure they\'re all in lowest form, or this error sometimes happens.')
                print('Origin:', self.creationMessage)
                print('All boundaries:', self.boundaries)
                print('Boundary:', key)
                print('All regions:', self.game)
                print('Specific region:', self.game[key[0]])
                print('Loop:', loop)
                exit()
            if self.boundaries[key] in pairingsSeen:
                print('ERROR G-S-9: Key is paired to the same point as another key')
                print('Origin:', self.creationMessage)
                print('Pairing:', self.boundaries[key])
                print('All boundaries:', self.boundaries)
                print('Boundaries:', key, pairingsSeen[self.boundaries[key]])
                exit()
            pairingsSeen[self.boundaries[key]] = key

        #Safety check to make sure that every boundary point is addressed in boundaries:
        for regionNum in range(len(self.game)):
            for loopNum in range(len(self.game[regionNum])):
                loop = self.allLoops[self.game[regionNum][loopNum]].points
                for pointIndex in range(len(loop)):
                    if loop[pointIndex] == BOUN:
                        key = (regionNum, loopNum, pointIndex)
                        if key not in self.boundaries:
                            print('GRROR G-S-A: Boundary value within a loop does not have value in boundaries')
                            print(self.creationMessage)
                            print('Boundaries:', self.boundaries)
                            print('Key:', key)
                            print('All regions:', self.game)
                            print('Specific region:', self.game[key[0]])
                            print('Loop:', loop)
                            print('Actual point:', loop[pointIndex])
                            exit()


    def addChildGame(self, moveEncoding, newGameString = None, newGameObject = None):
        if newGameString is None and newGameObject is None:
            print("addChildGame needs either newGameString or newGameObject")
            exit()
        if newGameString is not None and newGameObject is not None:
            print("addChildGame needs only one, newGameString or newGameObject.  Received both")
            exit()

        if newGameString is not None:
            newGameObject = self.allGames[newGameString]

        if newGameObject not in self.childGames:
            self.childGames[newGameObject] = []
        if moveEncoding not in self.childGames[newGameObject]:
            self.childGames[newGameObject].append(moveEncoding)


    def executeAllMerges(self, recurseDepth, connectionsDepth):
        #Cycle through every region, and merge all loops in all possible ways in those regions
        for regionIndex in range(len(self.game)):
            region = self.game[regionIndex]
            merges = self.allRegions[region].merges   #Format:  [(newRegion, completeRemapping)]

            for mergeTup in merges:
                (newRegionObject , loopRemapping, moveEncoding) = mergeTup
                newRegion = newRegionObject.region

                #newRegion, completeRemapping, leftBoundaryToDelete, rightBoundaryToDelete
                #Insert the new version of the region into the complete game
                completeNewGame = self.game[:regionIndex] + [newRegion[:],] + self.game[regionIndex+1:]

                #The region class doesn't contain the region's index, so we need to add the region index back to the remapping
                #  We also need to keep track of any loops that are getting deleted, so we can follow up with that further below
                completeRemapping = {}
                boundariesToDelete = []
                appearingLoops = {}  #We'll add the keys LEFTAPPEARINGLOOP and RIGHTAPPEARINGLOOP to this whenever we see them in merges, with the value being the new region's index
                boundariesToAdd = []

                #print()
                #print("1!! ", self.gameString, mergeTup)
                #print("1!!!",  completeNewGame, self.boundaries)

                for remap in loopRemapping:

                    if loopRemapping[remap] == DISABOUNDCHECKLOOP:
                        #print(" a", loopRemapping[remap], remap)
                        completeRemapping[(regionIndex,) + remap] = DEADPOINT
                        if self.boundaries[(regionIndex,) + remap] != DISAPOINT:
                            #print(" b", (regionIndex,) + remap, self.boundaries[(regionIndex,) + remap])
                            boundariesToDelete.append((regionIndex,) + remap)
                    elif loopRemapping[remap] == DEADLOOP:# and self.boundaries[(regionIndex,)+remap] != DISAPOINT:
                        #print(" c", loopRemapping[remap], remap, self.boundaries[(regionIndex,)+remap])
                        completeRemapping[(regionIndex,) + remap] = DEADPOINT
                        if self.boundaries[(regionIndex,)+remap] != DISAPOINT:
                            boundariesToDelete.append((regionIndex,) + remap)
                    elif loopRemapping[remap] == DISALOOP:
                        #print(" d", loopRemapping[remap], remap, self.boundaries[(regionIndex,)+remap])
                        completeRemapping[(regionIndex,) + remap] = DISAPOINT
                    elif remap[0] in [LEFTAPPEARINGLOOP, RIGHTAPPEARINGLOOP]:
                        #print(" e", loopRemapping[remap], remap)
                        side = remap[0]
                        if side not in appearingLoops:
                            appearingLoops[side] = len(completeNewGame)
                            completeNewGame.append((BOUN*10+BOUN,))
                        boundariesToAdd.append(((regionIndex,) + loopRemapping[remap], (appearingLoops[side], 0, remap[1])))
                    else:
                        #print(" f", loopRemapping[remap], remap)
                        completeRemapping[(regionIndex,) + remap] = (regionIndex,) + loopRemapping[remap]

                #print()
                #print("2!!!", completeRemapping)
                #print("2!!!", boundariesToDelete)
                #print("2!!!", boundariesToAdd)

                if len(boundariesToDelete)> 2:
                    print("ERROR: G-M-1: There appear to be more than two deleted points on a merge, even though the max is two:", boundariesToDelete)
                    exit()

                for deletedBoundary in boundariesToDelete:
                    if deletedBoundary == DISAPOINT:
                        continue
                    boundarySideToDelete = self.boundaries[deletedBoundary]
                    (deleteRegionIndex, deleteLoopIndex, deletePointIndex) = boundarySideToDelete
                    loopToDelete = (deleteRegionIndex, deleteLoopIndex)

                    (deletionEnum, deletionBoundaries) = self.allLoops[self.game[deleteRegionIndex][deleteLoopIndex]].deletions[( deletePointIndex,)] # = {'enum':newEnum, 'boundaryMapping':newBoundaryMapping}
                    (completeNewGame, additionalBoundaryMappings) = deletePointFromLoop(loopToDelete, completeNewGame, deletionEnum, deletionBoundaries, (deletePointIndex,))
                    for boundary in additionalBoundaryMappings:
                        completeRemapping[boundary] = additionalBoundaryMappings[boundary]

                #print()
                #print("3!!!", completeRemapping)

                newBoundaries = executeRemappingOfExactPoints(self.boundaries, completeRemapping)

                #print()
                #print("4!!!", newBoundaries)
                for tup in boundariesToAdd:
                    newBoundaries[tup[0]] = tup[1]
                    newBoundaries[tup[1]] = tup[0]

                #print()
                #print("5!!!", completeNewGame, newBoundaries)
                #print()
                (completeNewGame, newBoundaries) = fullGameCleanup(self.allData, completeNewGame, newBoundaries)


                self.allData.totalEdges += 1
                if self.allData.totalEdges % 100000 == 0:
                    print(self.allData.totalEdges, self.allData.timeCheck())

                #Continue generating new games
                newGameString = spawnNewGame(self.allData, completeNewGame, newBoundaries, recurseDepth, connectionsDepth, creationMessage = "Merge on " + self.gameString)
                self.addChildGame(moveEncoding, newGameString = newGameString)
        return


    def executeAllEnclosures(self, recurseDepth, connectionsDepth):
        #Cycle through every region, and merge all loops in all possible ways in those regions
        for regionIndex in range(len(self.game)):
            region = self.game[regionIndex]
            enclosures = self.allRegions[region].enclosures   #Format:  [(leftRegion, rightRegion, boundaryRemapping, leftToRightPairing)]

            for enclosureTup in enclosures:
                (leftRegionObject, rightRegionObject, loopRemapping, leftToRightPairing, moveEncoding) = enclosureTup
                leftRegion = leftRegionObject.region
                rightRegion = rightRegionObject.region
                #Remove the index we're doing the enclosure on, replace it with the left loop, and then shove the right loop onto the end
                completeNewGame = self.game[:regionIndex] + [leftRegion[:],] + self.game[regionIndex+1:] + [rightRegion[:],]

                #The left region will maintain the position of the current region, while the right region will be an additional one added to the tail end of the array
                #  boundaryRemapping contains boundaries from the old loop that need to be relocated in the new game
                #  leftToRightPairing will require that new boundaries be added.  We handle it after this
                completeRemapping = {}
                boundariesToDelete = []
                deletionCount = 0
                for remap in loopRemapping:
                    if loopRemapping[remap] == DEADPOINT:
                        boundariesToDelete.append(self.boundaries[(regionIndex,) + remap]) #We'll want to know which boundaries to delete on the other side
                        completeRemapping[(regionIndex,) + remap] = DEADPOINT
                        deletionCount += 1
                    elif loopRemapping[remap][0] == LEFT:
                        completeRemapping[(regionIndex,) + remap] = (regionIndex,) + loopRemapping[remap][1:]
                    elif loopRemapping[remap][0] == RIGHT:
                        completeRemapping[(regionIndex,) + remap] = (len(completeNewGame)-1,) + loopRemapping[remap][1:]
                    else:
                        print("ERROR: G-E-1: Loop Remapping has a value that's neither dead, nor has LEFT, RIGHT, LEFTAPPEARINGLOOP, or RIGHTAPPEARINGLOOP as its first value", self.regious, regionIndex, enclosureTup)
                        exit()

                if deletionCount > 3:
                    print("ERROR: G-E-2: There appear to be more than three deleted points on a merge, even though the max is three:", boundariesToDelete)
                #If there's two boundaries, we need to check to see if the other sides are in the same loop, in which case we only need to do a deletion on a single loop
                if len(boundariesToDelete) == 2 and boundariesToDelete[0][:2] == boundariesToDelete[1][:2] and boundariesToDelete[0] != DISAPOINT:
                    #DeletePointFromLoop deletes from a given loop, which we know because the two points we want to delete share a loop to enter this 'if' statement
                    #First we get the coordinates of what to delete, and also assign it to "loopToDelete" for later
                    loopToDelete = boundariesToDelete[0][:2]
                    (regionOfLoopToDelete, loopIndexToDelete) = loopToDelete

                    deletePointIndices = [boundariesToDelete[0][2], boundariesToDelete[1][2]]
                    deletePointIndices.sort()
                    deletePointIndices = tuple(deletePointIndices)

                    (deletionEnum, deletionBoundaries) = self.allLoops[self.game[regionOfLoopToDelete][loopIndexToDelete]].deletions[deletePointIndices] # = {'enum':newEnum, 'boundaryMapping':newBoundaryMapping, 'removedBoundary':(self.boundaries[i],)}
                    (completeNewGame, additionalBoundaryMappings) = deletePointFromLoop(loopToDelete, completeNewGame, deletionEnum, deletionBoundaries, deletePointIndices)
                    for boundary in additionalBoundaryMappings:
                        completeRemapping[boundary] = additionalBoundaryMappings[boundary]

                else: #For cases where the boundaries we're deleting (if any) aren't in same loop, we just take them one at a time
                    for deletedBoundary in boundariesToDelete:
                        if deletedBoundary == DISAPOINT:
                            continue

                        loopToDelete = deletedBoundary[:2]
                        (regionOfLoopToDelete, loopIndexToDelete) = loopToDelete
                        deletePointIndex = (deletedBoundary[2],)

                        (deletionEnum, deletionBoundaries) = self.allLoops[self.game[regionOfLoopToDelete][loopIndexToDelete]].deletions[deletePointIndex] # = {'enum':newEnum, 'boundaryMapping':newBoundaryMapping, 'removedBoundary':(self.boundaries[i],)}
                        (completeNewGame, additionalBoundaryMappings) = deletePointFromLoop(loopToDelete, completeNewGame, deletionEnum, deletionBoundaries, (deletePointIndex,))

                        for boundary in additionalBoundaryMappings:
                            completeRemapping[boundary] = additionalBoundaryMappings[boundary]

                newBoundaries = executeRemappingOfExactPoints(self.boundaries, completeRemapping)

                for leftPointIndex in leftToRightPairing:
                    rightPointIndex = leftToRightPairing[leftPointIndex]
                    newBoundaries[(regionIndex,) + leftPointIndex] = (len(completeNewGame)-1,) + rightPointIndex
                    newBoundaries[(len(completeNewGame)-1,) + rightPointIndex] = (regionIndex,) + leftPointIndex

                (completeNewGame, newBoundaries) = fullGameCleanup(self.allData, completeNewGame, newBoundaries)

                self.allData.totalEdges += 1
                if self.allData.totalEdges % 100000 == 0:
                    print(self.allData.totalEdges, self.allData.timeCheck())

                #Continue generating new games
                newGameString = spawnNewGame(self.allData, completeNewGame, newBoundaries, recurseDepth, connectionsDepth, creationMessage = "Enclosure on " + self.gameString)
                self.addChildGame(moveEncoding, newGameString = newGameString)
        return


    def executeDisappearances(self, recurseDepth, connectionsDepth):
        for boundary in self.boundaries:
            if self.boundaries[boundary] == DISAPOINT:
                (regionIndex, loopIndex, pointIndex) = boundary
                region = self.game[regionIndex]
                (disaRegionObject, disaRemap) = self.allRegions[region].disaDegens[(loopIndex, pointIndex)]
                disaRegion = disaRegionObject.region
                #Insert the new version of the region into the complete game
                completeNewGame = self.game[:regionIndex] + [disaRegion] + self.game[regionIndex+1:]

                completeRemapping = {boundary:DEADPOINT}
                for remap in disaRemap:
                    completeRemapping[(regionIndex,) + remap] = (regionIndex,) + disaRemap[remap]
                newBoundaries = executeRemappingOfExactPoints(self.boundaries, completeRemapping)

                (completeNewGame, newBoundaries) = fullGameCleanup(self.allData, completeNewGame, newBoundaries)


                self.allData.totalEdges += 1
                if self.allData.totalEdges % 100000 == 0:
                    print(self.allData.totalEdges, self.allData.timeCheck())

                #Continue generating new games
                newGameString = spawnNewGame(self.allData, completeNewGame, newBoundaries, recurseDepth, connectionsDepth, creationMessage = "Disappearing on " + self.gameString + " at boundary " + str(boundary))
                self.addChildGame(DISADROPENCODING, newGameString = newGameString)
        return


    #Look for LONELYBOUNDARYPAIRS, and if one exists, delete it as a possible move (i.e. merge the interior bounds)
    #  Since these special cases don't have boundaries and are thus interchangable, we only ever have to handle it once even if there's several
    def executeLonelyBoundaryRemovals(self, recurseDepth, connectionsDepth):
        for regionIndex in range(len(self.game)):
            region = self.game[regionIndex]
            for loopIndex in range(len(region)):
                if region[loopIndex] == LONELYBOUNDARYPAIR:
                    newRegion = region[:loopIndex] + region[loopIndex+1:]
                    if newRegion == ():  #If the entire region is just a LONELYBOUNDARYPAIR, the result will be an empty region, which will be handled in the cleanup
                        newRegion = (EMPT,)
                    completeNewGame = self.game[:regionIndex] + [newRegion[:],] + self.game[regionIndex+1:]

                    completeRemapping = {}
                    for boundary in self.boundaries:
                        if boundary[0] == regionIndex and boundary[1] > loopIndex:
                            completeRemapping[boundary] = (boundary[0], boundary[1] - 1, boundary[2])
                    newBoundaries = executeRemappingOfExactPoints(self.boundaries, completeRemapping)

                    (completeNewGame, newBoundaries) = fullGameCleanup(self.allData, completeNewGame, newBoundaries)

                    self.allData.totalEdges += 1
                    if self.allData.totalEdges % 100000 == 0:
                        print(self.allData.totalEdges, self.allData.timeCheck())

                    #Continue generating new games
                    newGameString = spawnNewGame(self.allData, completeNewGame, newBoundaries, recurseDepth, connectionsDepth, creationMessage = "LONELYBOUNDARYPAIR from " + self.gameString)
                    self.addChildGame(LONELYPAIRDROPENCODING, newGameString = newGameString)

                    break  #If we've handled one case within a region, we've handled them all


    def handleEmptyGame(self):
        self.gameString = EMPTYGAME
        self.compressedString = EMPTYGAMECOMPRESSED
        self.subGames = [self.gameString]
        self.allGames[self.gameString] = self
        self.nimber = 0
        self.minMoves = 0
        self.maxMoves = 0

        #While EMPTYGAME makes for a great string to look at, we need to add the empty string to the segment we use to search for games in a more automated fashion
        if '' not in self.allFullPointBounds:
            self.allFullPointBounds[''] = {'':self}
        return


    def handleSingleDisappearing(self):
        self.gameString = STRBOUN + ":!"
        self.compressedString = STRBOUN + ":!"
        self.subGames = [self.gameString]
        self.allGames[self.gameString] = self
        self.totalConnections = 1
        self.nimber = 1
        self.minMoves = 1
        self.maxMoves = 1
        spawnNewGame(self.allData, (), {}, -1, -1, creationMessage = "Generating EMPTYGAME from BOUN:!" + self.gameString)

        moveEncoding = encodeLoopMoveData('merge', SING, BOUN)
        self.addChildGame(moveEncoding, newGameString = self.allGames[EMPTYGAME].gameString)

        if STRBOUN not in self.allFullPointBounds:
            self.allFullPointBounds[STRBOUN] = {"!":self}
        return


    #If the game is made of two or more disconnected segments, seperate for individual analysis
    def handleNonUnifiedCase(self, distinctGames, distinctGameBoundaries, recurseDepth, connectionsDepth):
        subNimbers    = []
        maxSubNimLen  = 0
        self.minMoves = 0
        self.maxMoves = 0
        self.totalConnections = 0

        for i in range(len(distinctGames)):
            (curGame, curBoundaries) = orderRegionsInGame(distinctGames[i], distinctGameBoundaries[i])

            curGameString = spawnNewGame(self.allData, curGame, curBoundaries, recurseDepth, connectionsDepth, creationMessage = "New distinct game in nonunified case, index = " + str(i) + ":" + str(curGame) + "|||" + str(curBoundaries))

            self.subGames.append(curGameString)

            self.minMoves         += self.allGames[curGameString].minMoves
            self.maxMoves         += self.allGames[curGameString].maxMoves
            self.totalConnections += self.allGames[curGameString].totalConnections
            #DELETE?  #self.allGames[curGameString].nimber
            curNimberInBinary = bin(self.allGames[curGameString].nimber)[2:]
            subNimbers.append(curNimberInBinary)
            if len(curNimberInBinary) > maxSubNimLen:
                maxSubNimLen = len(curNimberInBinary)

        #There's a faster way to calculate a nimber, but there's not a ton of these and they're all tiny numbers, so I don't care
        digitSum = [0]*maxSubNimLen
        for nimberIndex in range(len(subNimbers)):
            curNimber = '0'*(maxSubNimLen-len(subNimbers[nimberIndex])) + subNimbers[nimberIndex]
            for digitIndex in range(maxSubNimLen):
                digitSum[digitIndex] += int(curNimber[digitIndex])

        xoredBinary = ''
        for digit in digitSum:
            xoredBinary += str(digit%2)
        self.nimber = int(xoredBinary, 2)

        return


    #Handle the case when there's only a single set of connected regions (i.e. none are disconnected from the group, where boundaries are the "connections" in this case)
    def handleUnifiedCase(self, fullPointString, allOrdersQuickRef, recurseDepth, connectionsDepth):
        self.subGames.append(self.gameString)

        #Calculate how many points are in the overall graph.  Divide by two to take care of how half-points are handled in loops.
        self.totalConnections = 0
        for region in self.game:
            for loop in region:
                self.totalConnections += self.allLoops[loop].connectionValue
        for boundary in self.boundaries:
            if self.boundaries[boundary] == DISAPOINT:
                self.totalConnections += 3
        self.totalConnections = int(self.totalConnections/2) - 1

        self.executeAllMerges(recurseDepth, connectionsDepth)
        self.executeAllEnclosures(recurseDepth, connectionsDepth)
        self.executeLonelyBoundaryRemovals(recurseDepth, connectionsDepth)
        self.executeDisappearances(recurseDepth, connectionsDepth)

        allChildNimbers = {}
        minChildMinMoves = 9999
        maxChildMaxMoves = 0

        for child in self.childGames:
            allChildNimbers[child.nimber] = True
            if child.minMoves < minChildMinMoves:
                minChildMinMoves = child.minMoves
            if child.maxMoves > maxChildMaxMoves:
                maxChildMaxMoves = child.maxMoves

        self.minMoves = minChildMinMoves + 1
        self.maxMoves = maxChildMaxMoves + 1

        for i in range(1000):
            if i not in allChildNimbers:
                self.nimber = i
                break
            if i == 100:
                print("Nimber calculation exceeded expected values")
                exit()
        return


    def handleGameStringInput(self, gameString, isCompressed = False, preProcessed = False):
        if isCompressed:
            self.gameStringCompressed = gameString
            gameString = decompressGameString(gameString, self.allData.allBoundCompress)
        else:
            self.gameStringCompressed = compressGameString(gameString, self.allData.allBoundCompress)
        if gameString == ":":
            gameString = EMPTYGAME
        self.gameString = gameString

        (self.game, self.boundaries) = convertGameStringToGame(self.allData, gameString, isCompressed, preProcessed = preProcessed)
        if not preProcessed:
            (self.game, self.boundaries) = fullGameCleanup(self.allData, self.game, self.boundaries)
        return


    def loadInSavedLine(self, rawSavedLine, isCompressed = False):
        rawSavedLine = rawSavedLine.strip()
        parsedSavedLine = rawSavedLine.split(";")
        if len(parsedSavedLine) != 6:
            print("ERROR G-L-1:  Saved line does not have exactly five semicolons:")
            print(rawSavedLine)
            exit()

        gameAndSubgames = parsedSavedLine[0]
        gameString = gameAndSubgames.split("X")[0]
        if isCompressed:
            self.compressedString = gameString
            self.gameString = decompressGameString(gameString, self.allData.allBoundCompress)
        if gameString in self.allGames:
            self = self.allGames[gameString]
            return

        self.handleGameStringInput(gameString, isCompressed = isCompressed, preProcessed = True)

        subStringArray = gameAndSubgames.split("X")[1:]
        if len(subStringArray) == 0:
            self.subGames = [self.gameString]
        elif isCompressed:
            self.subGames = []
            for subGame in subStringArray:
                self.subGames.append(decompressGameString(subGame, self.allData.allBoundCompress))
        else:
            self.subGames = subStringArray

        self.genCleanString()

        self.nimber = int(parsedSavedLine[1])
        self.totalConnections = int(parsedSavedLine[2])
        self.minMoves = int(parsedSavedLine[3])
        self.maxMoves = int(parsedSavedLine[4])
        if parsedSavedLine[5] != '':
            childGamesArray = parsedSavedLine[5].split("/")
            for childGameDat in childGamesArray:
                childGameDat = childGameDat.split("@")
                childGame = childGameDat[0]
                childMoveEncodings = list(strTupToInt(childGameDat[1].split(",")))
                if isCompressed:
                    childGame = decompressGameString(childGame, self.allData.allBoundCompress)
                self.childGames[self.allGames[childGame]] = childMoveEncodings
                self.allData.totalEdges += 1

        self.allGames[self.gameString] = self

        if self.printGameAsLine(isCompressed) != rawSavedLine:
            print("ERROR G-L-2:  Saved line ends up changed when re-saving; assumed error in reading or storing:")
            print("Saved Line:       ", rawSavedLine)
            print("Re-written Line:  ", self.printGameAsLine(isCompressed))
            exit()
        return


    def printGameAsLine(self, compress = False):
        fullLine = self.gameString
        if compress:
            fullLine = self.compressedString
        if len(self.subGames) > 1:
            if not compress:
                fullLine += 'X' + 'X'.join(self.subGames)
            else:
                #print("!!", [self.game], [self.subGames])
                for subGame in self.subGames:
                    fullLine += 'X' + self.allGames[subGame].compressedString
        fullLine += ';' + str(self.nimber) + ";" + str(self.totalConnections) + ";" + str(self.minMoves) + ";" + str(self.maxMoves) + ";"
        childrenSegment = []
        for childGame in self.childGames:
            moveEncodings = "@" + intTupToStr(self.childGames[childGame])
            if compress:
                childrenSegment.append(childGame.compressedString + moveEncodings)
            else:
                childrenSegment.append(childGame.gameString + moveEncodings)
        fullLine += "/".join(childrenSegment)
        return fullLine


    def printReadableData(self, useClean = True):
        print("      CleanString:", self.cleanString)
        print("       GameString:", self.gameString)
        print("Compressed String:", self.compressedString)
        print("         SubGames:", self.subGames)
        print("           Nimber:", self.nimber)
        print("       Game Array:", self.game)
        print("       Boundaries:", self.boundaries)
        print("Total Connections:", self.totalConnections)
        print("    Minimum Moves:", self.minMoves)
        print("    Maximum Moves:", self.maxMoves)
        print("         Children:", len(self.childGames))

        maxChildLen = 0
        childGameList = []
        for childDat in self.childGames:
            if len(childDat.gameString) > maxChildLen:
                if useClean:
                    maxChildLen = len(childDat.cleanString)
                else:
                    maxChildLen = len(childDat.gameString)
            decodedMoves = decodeMoveArray(self.childGames[childDat])
            if useClean:
                childGameList.append((childDat.nimber, self.minMoves - childDat.minMoves, self.maxMoves - childDat.maxMoves, self.totalConnections - childDat.totalConnections, childDat.minMoves, childDat.maxMoves, childDat.totalConnections, childDat.cleanString, decodedMoves))
            else:
                childGameList.append((childDat.nimber, self.minMoves - childDat.minMoves, self.maxMoves - childDat.maxMoves, self.totalConnections - childDat.totalConnections, childDat.minMoves, childDat.maxMoves, childDat.totalConnections, childDat.gameString , decodedMoves))
        childGameList.sort()
        for tup in childGameList:
            prefix = "    "
            if tup[0] == 0:
                prefix = "**  "
            print(prefix, tup[7].ljust(maxChildLen), '\t', str(tup[0]), '\t', str(tup[1]), '\t', str(tup[2]), '\t', str(tup[3]), "  |  ", str(tup[4]), '\t', str(tup[5]), '\t', str(tup[6]), '\t', str(tup[8]))
        return

    def genCleanString(self):
        (fixedGame, fixedBounds) = convertGameStringToGame(self.allData, self.gameString)
        boundaryNames = {}
        curLetter = ord("A")
        newString = ''
        for regionIndex in range(len(fixedGame)):
            region = fixedGame[regionIndex]
            for loopIndex in range(len(region)):
                loop = region[loopIndex]
                if loop == LONELYBOUNDARYPAIR:
                    newString += STRLONELYBOUNDARYPAIR + ','
                    continue
                if loop == LONELYBOUNDARYTRIPLE:
                    newString += STRLONELYBOUNDARYTRIPLE + ','
                    continue
                loop = str(loop)
                for pointIndex in range(len(loop)):
                    coords = (regionIndex, loopIndex, pointIndex)
                    curPoint = loop[pointIndex]
                    if curPoint != STRBOUN:
                        newString += curPoint
                    else:
                        if coords in boundaryNames:
                            newString += boundaryNames[coords]
                        elif fixedBounds[coords] == DISAPOINT:
                            newString += "7"
                        else:
                            newString += chr(curLetter)
                            boundaryNames[fixedBounds[coords]] = chr(curLetter)
                            curLetter += 1
                newString += ","
            newString = newString[:-1] + '|'
        self.cleanString = newString[:-1]


    def __init__(self, dataSummary, game = None, boundaries = None, gameString = None, savedFileLine = None, preOrdered = False, creationMessage = '', compressed = False, preProcessed = False, recurseDepth = -1, connectionsDepth = -1, recursionEndpoint = None):
        #print("!", game, boundaries, recurseDepth, connectionsDepth, creationMessage, gameString, savedFileLine, preOrdered, compressed, preProcessed, recursionEndpoint)
        self.creationMessage    = creationMessage
        self.allData            = dataSummary
        self.allLoops      	    = dataSummary.allLoops
        self.allMerges     	    = dataSummary.allMerges
        self.allRegions         = dataSummary.allRegions
        self.allFullPointBounds = dataSummary.allFullPointBounds
        self.allGames      	    = dataSummary.allGames
        self.allBinaryTrees     = dataSummary.allBinaryTrees
        self.allPermutationMaps = dataSummary.allPermutationMaps
        self.game       	      = game
        self.boundaries         = boundaries
        self.totalConnections   = 0
        self.minMoves           = 0
        self.maxMoves           = 9999
        self.childGames         = {}
        self.gameString         = ""
        self.compressedString   = ""
        self.cleanString        = ""
        self.subGames           = []
        self.nimber             = -1

        if savedFileLine is not None:
            self.loadInSavedLine(savedFileLine, isCompressed = compressed)
            return
        elif gameString is not None:
            if game is not None:
                print("Error G-I-0:  Region and gameString in the same call of 'Game' class.  The two are mutually exclusive; please pick one.")
                exit()
            if boundaries is not None:
                print("Error G-I-1:  boundaries and gameString in the same call of 'Game' class.  The two are mutually exclusive; please pick one.")
                exit()
            self.handleGameStringInput(gameString, isCompressed = compressed, preProcessed = preProcessed)

        self.boundarySafetyCheck()

        #The null game can bypass the standard routines, as they don't make any sense and may even break.
        #  The functio called here sets all the appropriate values, and then we just bail so as to avoid all the unnecessary messiness.
        if len(self.game) == 0:
            self.handleEmptyGame()
            return

        #The game '3:!' is its own weird thing, and needs to be handled seperately.  It only points to the null game.
        if len(self.game) == 1 and self.game[0] == (BOUN,) and self.boundaries[(0,0,0)] == DISAPOINT:
            self.handleSingleDisappearing()
            return

        #Make sure the regions in the game are properly ordered if they weren't when entering the function
        if not preOrdered:
            (self.game, self.boundaries) = orderRegionsInGame(self.game, self.boundaries)

        #Find the lowest game name, so as to avoid excessive duplications of isomorphisms
        self.gameString, fullPointString, boundaryOrder, allOrdersQuickRef = findLowestGameName(self.allData, self.game, self.boundaries)
        self.genCleanString()

        if self.gameString in self.allGames and self.allGames[self.gameString].nimber != -1:
            #Check to see if the game already exists, *and* doesn't have nimber==-1 (which indicates that it was a recursionEndpoint that can be expanded and overwritten)
            #print("WARNING:  Game already exists.  We're going to set it to the pre-established value, but try to avoid these situations, because setting an object to equal another within itself is shady af")
            self = self.allGames[self.gameString]
            if fullPointString not in self.allFullPointBounds:
                self.allFullPointBounds[fullPointString] = {}
            for curOrder in allOrdersQuickRef:
                self.allFullPointBounds[fullPointString][curOrder] = self
            return

        self.compressedString = compressGameString(self.gameString, self.allData.allBoundCompress)
        self.allGames[self.gameString] = self

        if fullPointString not in self.allFullPointBounds:
            self.allFullPointBounds[fullPointString] = {}
        for curOrder in allOrdersQuickRef:
            self.allFullPointBounds[fullPointString][curOrder] = self

        (distinctGames, distinctGameBoundaries) = checkRegionUnity(self.game, self.boundaries)
        if len(distinctGames) > 1:
            self.handleNonUnifiedCase(distinctGames, distinctGameBoundaries, recurseDepth, connectionsDepth)
        else:
            self.handleUnifiedCase(fullPointString, allOrdersQuickRef, recurseDepth, connectionsDepth)

        return

#Create/Load data

In [22]:
#@title Generate size-n game
number_of_free_points = 3 #@param {type:"integer"}
compressGame = False #@param {type:"boolean"}
saveDataToFile = False #@param {type:"boolean"}
fileSuffix = "-6-Compressed" #@param {type:"string"}
dataSummary = UniversalData()
generateBaseGame(number_of_free_points, useIsoRegions = compressGame, allData = dataSummary)

if saveDataToFile:
    saveData(dataSummary, fileSuffix)

unifiedCount = 0
for gameObject in dataSummary.allGames.values():
    if len(gameObject.subGames) == 1:
        unifiedCount += 1

#printSmartTree(gameString = "0,0,0:")

print("Total games:           ", len(dataSummary.allGames))
print("Total unified:         ", unifiedCount)
print("Total non-unified:     ", len(dataSummary.allGames)-unifiedCount)
print("Total edges calculated:", dataSummary.totalEdges)
uniqueEdges = 0
for gameObject in dataSummary.allGames.values():
    uniqueEdges += len(gameObject.childGames)
print("Total unique edges:    ", uniqueEdges)
print("Total pointBoundSets:  ", len(dataSummary.allFullPointBounds))
loopTotal = 0
for enum in dataSummary.allLoops:
    if enum == dataSummary.allLoops[enum].enum:
        loopTotal += 1
print("Total loops:           ", loopTotal)
mergeLength = 0
for key1 in dataSummary.allMerges:
    for key2 in dataSummary.allMerges[key1]:
        mergeLength += 1
print("Total merges:          ", mergeLength)
print("Total regions:         ", len(dataSummary.allRegions))

Total games:            175
Total unified:          151
Total non-unified:      24
Total edges calculated: 1200
Total unique edges:     704
Total pointBoundSets:   175
Total loops:            173
Total merges:           95
Total regions:          219


In [49]:
#@title Generate special-case game

saveDataToFile = False #@param {type:"boolean"}
fileSuffix = "-BottomUp" #@param {type:"string"}
generateThis = "23|23|22423523:2301" #@param {type:"string"}

'''#Determine if there are any parts of a game that match the special cases currently allowed (special cases can be turned on or off as desired, typically for readability/efficiency trade-offs)
def findSpecialCases(allData, game, boundaries):
    (game, boundaries) = findLonelyPair(allData, game, boundaries)
    print("6-  ", game, boundaries)
    (game, boundaries) = findLonelyTriple(allData, game, boundaries)
    print("7-  ", game, boundaries)
    (game, boundaries) = findDisappearing(allData, game, boundaries)
    return (game, boundaries)


#Run all of the functions that are used to rearrange a game into a minimal, ordered state
def fullGameCleanup(allData, completeNewGame, newBoundaries, searchForSpecialCases=True):
    a = ''
    if not searchForSpecialCases:
        a = "    "
    print(a + "1-  ", completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = removeEmptyLoops(completeNewGame, newBoundaries)
    print(a + "2-  ", completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = removeEmptyRegions(allData, completeNewGame, newBoundaries)
    print(a + "3-  ", completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = flipReversedRegions(allData.allRegions, completeNewGame, newBoundaries)
    print(a + "4-  ", completeNewGame, newBoundaries)
    (completeNewGame, newBoundaries) = orderRegionsInGame(completeNewGame, newBoundaries)
    print(a + "5-  ", completeNewGame, newBoundaries)
    if searchForSpecialCases and allData.useSpecialCases:
        (completeNewGame, newBoundaries) = findSpecialCases(allData, completeNewGame, newBoundaries)
    return (completeNewGame, newBoundaries)'''


'''
region = (0,0,2,2,2)
if region not in dataSummary.allRegions:
    RegionClass(dataSummary, region = region)

regionBoundariesInput = {}
GameClass(dataSummary, game = [region], boundaries = regionBoundariesInput)
'''

'''
region = (0,0,0,0,33)
if region not in dataSummary.allRegions:
    RegionClass(dataSummary, region = region)

regionBoundariesInput = {}
regionBoundariesInput[(0,1,0)] = (1,4,0)
regionBoundariesInput[(0,1,1)] = (1,4,1)
regionBoundariesInput[(1,4,0)] = (0,1,0)
regionBoundariesInput[(1,4,1)] = (0,1,1)
GameClass(dataSummary, game = [(0,33), (0,0,0,0,33)], boundaries = regionBoundariesInput)
'''

dataSummary.startTime = datetime.now()
print("Previous Games:", len(dataSummary.allGames))
(game, regionBoundariesInput) = convertGameStringToGame(dataSummary, generateThis)
print("Creating game: ", game)
print("Boundaries: ", regionBoundariesInput)
for region in game:
    if region not in dataSummary.allRegions:
        RegionClass(dataSummary, region = region)

checkGame = GameClass(dataSummary, game = game, boundaries = regionBoundariesInput)
print("Resulting gameString:", checkGame.gameString)

if saveDataToFile:
    saveData(dataSummary, fileSuffix)

unifiedCount = 0
for gameObject in dataSummary.allGames.values():
    if len(gameObject.subGames) == 1:
        unifiedCount += 1

#printSmartTree(gameString = "0,0,0:")

print("Total games:           ", len(dataSummary.allGames))
print("Total unified:         ", unifiedCount)
print("Total non-unified:     ", len(dataSummary.allGames)-unifiedCount)
print("Total edges calculated:", dataSummary.totalEdges)
uniqueEdges = 0
for gameObject in dataSummary.allGames.values():
    uniqueEdges += len(gameObject.childGames)
print("Total unique edges:    ", uniqueEdges)
loopTotal = 0
for enum in dataSummary.allLoops:
    if enum == dataSummary.allLoops[enum].enum:
        loopTotal += 1
print("Total loops:           ", loopTotal)
mergeLength = 0
for key1 in dataSummary.allMerges:
    for key2 in dataSummary.allMerges[key1]:
        mergeLength += 1
print("Total merges:          ", mergeLength)
print("Total regions:         ", len(dataSummary.allRegions))
print("Total pointBoundSets:  ", len(dataSummary.allFullPointBounds))

Previous Games: 55526
Creating game:  [(23,), (23,), (22324325,)]
Boundaries:  {(0, 0, 1): (2, 0, 5), (1, 0, 1): (2, 0, 2), (2, 0, 5): (0, 0, 1), (2, 0, 2): (1, 0, 1)}
Resulting gameString: 23|23|22324325:2301
Total games:            55526
Total unified:          32346
Total non-unified:      23180
Total edges calculated: 792802
Total unique edges:     631040
Total loops:            3590
Total merges:           3637
Total regions:          8343
Total pointBoundSets:   43426


In [ ]:
#@title Load from file
fileSuffix = "-6-Compressed" #@param {type:"string"}
smartTreeString = "0,0,0:" #@param {type:"string"}
compressedGame = True #@param {type:"boolean"}

dataSummary = UniversalData()
dataSummary.useIsoRegions = compressedGame

loadedCount = 0
with open(root_path + "/sproutsRegionData" + fileSuffix + ".txt", 'r') as inFile:
    for line in inFile:
        RegionClass(dataSummary, loadString = line, compressedLoad = True)
        loadedCount += 1
        if loadedCount % 20000 == 0:
            print("Regions loaded:  ", loadedCount, '\t', dataSummary.timeCheck())
print("Regions complete:", len(dataSummary.allRegions), '\t', dataSummary.timeCheck())

dataSummary.allGames = {}
loadedCount = 0
with open(root_path + "/sproutsGameData" + fileSuffix + ".txt", 'r') as inFile:
    for line in inFile:
        GameClass(dataSummary, savedFileLine = line, compressed = True)
        loadedCount += 1
        if loadedCount % 20000 == 0:
            print("Games loaded:    ", loadedCount, '\t', dataSummary.timeCheck())

print("Games complete:  ", len(dataSummary.allGames), '\t', dataSummary.timeCheck())
dataSummary.smartTree = createSmartTree(smartTreeString)
#for game in allData.allGames:
#    print(allData.allGames[game].gameString, allData.allGames[game].nimber)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SproutsData/sproutsRegionData-6-Compressed.txt'

In [ ]:
#@title Top-down run
number_of_free_points = 5 #@param {type:"integer"}
useSpecialCases = True #@param {type:"boolean"}
topDown = True #@param {type:"boolean"}
#dataSummary = generateBaseGame(number_of_free_points, useSpecialCases = useSpecialCases, topDown = topDown)

By p-spot game:  Base / DISAcompressed / Full Compression:

2-spot (DISA compression irrelevant):
Total games:            20      18
Total unified:          20      18
Total non-unified:      0       0
Total edges calculated: 63      60
Total unique edges:     39      35
Total pointBoundSets:   20      18
Total loops:            32
Total merges:           15
Total regions:          37


3-spot:
Total games:            176     169   147
Total unified:          152     145   129
Total non-unified:      24      24    18
Total edges calculated: 1172    1143  1077
Total unique edges:     707     685   628
Total pointBoundSets:   176     169   147
Total loops:            173
Total merges:           95
Total regions:          219


4-spot:
Total games:            1863    1757    1483
Total unified:          1477    1383    1198
Total non-unified:      386     374     285
Total edges calculated: 21463   20676   19187
Total unique edges:     13244   12677   11521
Total pointBoundSets:   1803    1700    1422
Total loops:            1437
Total merges:           566
Total regions:          1453


5-spot:
Total games:            22470   20972   17175
Total unified:          17253   16064   13506
Total non-unified:      5217    4908    3669
Total edges calculated: 392873  374741  340298
Total unique edges:     243039  227815  218393
Total pointBoundSets:   20128   18794   15086
Total loops:            15113
Total merges:           3903
Total regions:          10944


6-spot:
Total games:            301529  279500  223154
Total unified:          229523  212764  174839
Total non-unified:      72006   66736   48315
Total edges calculated: 7475242 6940274 6175491
Total unique edges:     5066211 4782154 4250675
Total pointBoundSets:   234300  217718
Total loops:            179114  179114
Total merges:           32344
Total regions:          93400
(58m 20s for a base run)



6 + 0,0,0,0,0,1
338124
260291
77833
5948288
179114
6
102891
1214

Total games:            289302
Total unified:          227410
Total non-unified:      61892
Total edges calculated: 5898446
Total unique edges:     5363129
Total loops:            179114
Total merges:           8139
Total regions:          102891
Total pointBoundSets:   138199


23|2,3:10 ; 2,3|2,3:10 ; 23|33|2,3:1032 ; 23|2,3|3,3:2301 ; 33|2,3|2,3:2301 ; 2,3|2,3|3,3:2301

23|23:10 ; 23|2,3:10 ; 23|23|33:2301 ; 23|23|3,3:2301 ; 23|33|2,3:1032 ; 23|2,3|3,3:2301

#Basic Testing Functions

In [55]:
#@title Game Lookup

games_to_view = '33|0,33:2301' #@param {type:"string"}
gameList = games_to_view.split(';')
for gameString in gameList:
    gameLookup = gameString.strip()
    if gameLookup not in dataSummary.allGames:
        (check, boundaries) = convertGameStringToGame(dataSummary, gameLookup)
        gameLookup, dumpThis2, dumpThis3 = makeGameString(dataSummary.allLoops, check, boundaries)
        if gameLookup not in dataSummary.allGames:
            print("Game not found:", gameLookup)
    dataSummary.allGames[gameLookup].printReadableData()
    print()
    print()

      CleanString: 6,0
       GameString: -12,0:
Compressed String: -12,0
         SubGames: ['-12,0:']
           Nimber: 2
       Game Array: [(-12, 0)]
       Boundaries: {}
Total Connections: 4
    Minimum Moves: 3
    Maximum Moves: 4
         Children: 4
**   0   	 0 	 1 	 2 	 2   |   2 	 2 	 2 	 Dropped lonely pair
     0,2 	 1 	 1 	 1 	 1   |   2 	 3 	 3 	 Enclosure:  boundary point to boundary, non-adjacent points connected, one side empty
     1425 	 1 	 1 	 1 	 1   |   2 	 3 	 3 	 Merge:  free point to boundary
     6,6 	 3 	 1 	 1 	 1   |   2 	 3 	 3 	 Enclosure:  Free point to self, non-adjacent points connected, one side empty ; Enclosure:  Free point to self, non-adjacent points connected, one side empty




In [ ]:
#@title Create and print a smart tree
smartTreeGamestring = '3,1415:!' #@param {type:"string"}
dataSummary.smartTree = createSmartTree(smartTreeGamestring)
printSmartTree(dataSummary.smartTree)

In [ ]:
#@title Lazy verification
print (dataSummary.allGames['0:'].nimber)
print (dataSummary.allGames['0,0:'].nimber)
print (dataSummary.allGames['0,0,0:'].nimber)
print (dataSummary.allGames['0,0,0,0:'].nimber)
print (dataSummary.allGames['0,0,0,0,0:'].nimber)
print (dataSummary.allGames['0,0,0,0,0,0:'].nimber)

print (dataSummary.allGames['0:'].nimber)
print (dataSummary.allGames['0,2:'].nimber)
print (dataSummary.allGames['0,0,2:'].nimber)
print (dataSummary.allGames['0,0,0,2:'].nimber)
print (dataSummary.allGames['0,0,0,0,2:'].nimber)
print (dataSummary.allGames['0,0,0,0,0,2:'].nimber)

0
0
1
1
1
0
0
1
1
1
0
0


In [ ]:
#@title Find fixed-move positions
for game in dataSummary.allGames.values():
    if game.minMoves == game.maxMoves and len(game.subGames) == 1:
        print(game.minMoves, game.gameString)

In [ ]:
for game in dataSummary.allGames.values():
    subSubGames = {}
    if len(game.childGames) == 1:
        continue
    for childGame in game.childGames:
        for subSub in childGame.childGames:
            subSubString = subSub.gameString
            if subSubString != 'NULL':
                mapKeyAdd(subSubGames, subSubString)
    for key, val in subSubGames.items():
        if val == len(game.childGames):
            print(game.gameString, key, val)

In [ ]:
disaNimList = {}
for gameObject in dataSummary.allGames.values():
    for childObject, moveTypeList in gameObject.childGames.items():
        for val in moveTypeList:
            if val == DISADROPENCODING:
                tup = (gameObject.nimber, childObject.nimber)
                mapKeyAdd(disaNimList, tup)
orderedList = []
for tup, count in disaNimList.items():
    orderedList.append((tup, count))
orderedList.sort()

for tup in orderedList:
    print(tup[0], tup[1])

In [ ]:
orderedGames = []

for gameObject in dataSummary.allGames.values():
    if len(gameObject.subGames) == 1 and gameObject.minMoves == gameObject.maxMoves:
        childEncodings = []
        for childGame in gameObject.childGames:
            childEncodings.append(childGame.cleanString)
        orderedGames.append((str(gameObject.minMoves), gameObject.gameString, gameObject.cleanString, str(childEncodings)))

orderedGames.sort()
for curGame in orderedGames:
    print("\t".join(curGame))

In [ ]:
for checkString in ['23|23:10', '23|2,3:10', '23|2,3:10', '2,3|2,3:10','23|23|33:2301', '23|33|2,3:1032','23|23|3,3:2301', '23|2,3|3,3:2301','23|33|2,3:1032', '33|2,3|2,3:2301','23|2,3|3,3:2301', '2,3|2,3|3,3:2301']:
    print(checkString)
    for gameString in dataSummary.allGames:
        gameObject = dataSummary.allGames[gameString]
        for child in gameObject.childGames:
            if child.gameString == checkString:
                print(gameObject.cleanString)
    print("-----")
    print()

#'23|23:10', '23|2,3:10', '23|2,3:10', '2,3|2,3:10','23|23|33:2301', '23|33|2,3:1032','23|23|3,3:2301', '23|2,3|3,3:2301','23|33|2,3:1032', '33|2,3|2,3:2301','23|2,3|3,3:2301', '2,3|2,3|3,3:2301'

#Compare Games

In [ ]:
print(len(dataSummary.allGames))

addAndCompare(dataSummary, [convertGameStringToGame(dataSummary, '0:'), \
   convertGameStringToGame(dataSummary, '-12:')], \
   #convertGameStringToGame(dataSummary, '13|33435|3435|0,3|0,3:10462735'), \
   #convertGameStringToGame(dataSummary, '13|33435|3435|0,3|0,3:10643725')] \
   #convertGameStringToGame(dataSummary, '13|3435|3435|0,3,3|0,3:53617024')] \
   #convertGameStringToGame(dataSummary, '13|3435|3435|0,33|0,3:53617024')] \
   maxConnections = 8, printSpecificPairs = ["10","20","30","40","50","60"])#, spawnAsNecessary = True)

print(len(dataSummary.allGames))

224386
('0,7 ', '6,7 ') 1  0 !! 3 !!
('2A425B|0,AB ', '2A425B|6,AB ') 4  0 !! 3 !!
('1A|ABC|0,BC ', '1A|ABC|6,BC ') 2  0 !! 3 !!
('1A|BC|0,ABC ', '1A|BC|6,ABC ') 2  0 !! 3 !!
('ABCD|ABCE|0,DE ', 'ABCD|ABCE|6,DE ') 2  0 !! 2 !!
('ABC|ADE|0,BCDE ', 'ABC|ADE|6,BCDE ') 1  0 !! 2 !!
('22A2B|0,AB ', '22A2B|6,AB ') 4  0 !! 3 !!
('AB|2A2C|0,BC ', 'AB|2A2C|6,BC ') 2  0 !! 3 !!
('AB|2A2C|0,BC ', 'AB|2A2C|6,BC ') 2  0 !! 3 !!
('2A2B|0,2AB ', '2A2B|6,2AB ') 2  0 !! 3 !!
('AB|CD|ABE|0,CDE ', 'AB|CD|ABE|6,CDE ') 2  0 !! 3 !!
('AB|CD|ABE|0,CDE ', 'AB|CD|ABE|6,CDE ') 2  0 !! 3 !!
('AB|ABCDEF|0,CDEF ', 'AB|ABCDEF|6,CDEF ') 3  0 !! 3 !!
('ABC|ABCD|0,1D ', 'ABC|ABCD|6,1D ') 3  0 !! 2 !!
('0,744755 ', '6,744755 ') 1  0 !! 3 !!
('ABC|A475|0,BC ', 'ABC|A475|6,BC ') 2  0 !! 3 !!
('AB|74C5|0,ABC ', 'AB|74C5|6,ABC ') 2  0 !! 3 !!
('AB|CDE|2ACD|0,BE ', 'AB|CDE|2ACD|6,BE ') 5  0 !! 2 !!
('AB|ACD|0,BCD7 ', 'AB|ACD|6,BCD7 ') 2  0 !! 3 !!
('AB|AC|2DE|0,BCDE ', 'AB|AC|2DE|6,BCDE ') 2  0 !! 3 !!
('AB|ACD|0,BC7D ', 'A

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-27-381560356>", line 3, in <cell line: 0>
    addAndCompare(dataSummary, [convertGameStringToGame(dataSummary, '0:'), \
  File "<ipython-input-25-2124419380>", line 53, in addAndCompare
    newGame = addSegment(allData, gameObject, regionIndex, added, newBoundMap, spawnAsNecessary)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-25-2124419380>", line 18, in addSegment
    newGameString, A, B, C = findLowestGameName(allData, newGame, newBoundaries)
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-14-3296813067>", line 131, in findLowestGameName
    curOrder = makeBoundaryString(allData.allLoops, game, executeRemappingOfExactPoints(regionBoundaries,

In [ ]:
#@title Region-swap functions (in main code, but copied here in case changes are needed)
#Compare with replacement region, or add two variants and compare
def addSegment(allData, gameObject, regionIndex, added, newBoundMap, spawnAsNecessary):
    region = gameObject.game[regionIndex]
    newGame = gameObject.game[:regionIndex] + [region + added[0]] + gameObject.game[regionIndex+1:] + added[1:]
    newBoundaries = copyMap(gameObject.boundaries)
    for boundary, otherSide in newBoundMap.items():
        newBoundary  = (len(gameObject.game) +  boundary[0] - 1,) +  boundary[1:]
        newOtherside = (len(gameObject.game) + otherSide[0] - 1,) + otherSide[1:]
        if boundary[0] == 0:
            newBoundary  = (regionIndex,) + (len(region) + boundary[1],) + boundary[2:]
        if otherSide[0] == 0:
            newOtherside  = (regionIndex,) + (len(region) + otherSide[1],) + otherSide[2:]
        newBoundaries[newBoundary] = newOtherside

    (newGame, newBoundaries) = fullGameCleanup(allData, newGame, newBoundaries)

    newGameString, A, B, C = findLowestGameName(allData, newGame, newBoundaries)
    if spawnAsNecessary:
        result = GameClass(dataSummary, game = newGame, boundaries = newBoundaries)
        newGameString = result.gameString
        if newGameString not in allData.allGames:
            allData.allGames[newGameString] = result

    if newGameString in allData.allGames:
        return allData.allGames[newGameString]
    return None

def addAndCompare(allData, regionAndMapPairings, fieldOrdering = None, maxConnections = 4, printSpecificPairs = None, excludeSplitGames = True, spawnAsNecessary = False):
    if printSpecificPairs is None:
        printSpecificPairs = []

    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max']

    allGamesTemp = []
    for game in dataSummary.allGames.values():
        allGamesTemp.append(game)

    metaCount = {}
    outputBlocks = []
    for gameObject in allGamesTemp:
        if gameObject.totalConnections > maxConnections:
            continue

        if excludeSplitGames and len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            outputBlock1 = ''
            outputBlock2 = ()
            numZeros = 0
            for (added, newBoundMap) in regionAndMapPairings:
                newGame = addSegment(allData, gameObject, regionIndex, added, newBoundMap, spawnAsNecessary)

                if newGame is not None:
                    outputBlock2 += (newGame.cleanString + ' ',)
                    if len(outputBlock2) == 1:
                        for child in newGame.childGames:
                            if child.nimber == 0:
                                numZeros += 1
                    if len(outputBlock2) == 2 and outputBlock1.strip() + str(newGame.nimber) in printSpecificPairs:
                        pass
                        if numZeros < 4:
                            print(outputBlock2, outputBlock1, str(newGame.nimber), '>>' + str(numZeros)+ '<<')
                        #print(newGame.printReadableData())
                        #print(firstInPair)
                    outputBlock1 += str(newGame.nimber) + ' '
                else:
                    break
            if len(outputBlock2) == len(regionAndMapPairings):
                mapKeyAdd(metaCount, outputBlock1)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0], str(tup[1]))


#########

def preparePrintLine(allData, fieldOrdering, orderedGameDat):
    metaDat = ''
    for fieldType in fieldOrdering:
        for gameDat in orderedGameDat:
            if fieldType == 'nimber':
                metaDat += str(gameDat.nimber).ljust(2) + '& '
            elif fieldType == 'totalC':
                metaDat += str(gameDat.totalConnections).ljust(2) + '& '
            elif fieldType == 'min':
                metaDat += str(gameDat.minMoves).ljust(2) + '& '
            elif fieldType == 'max':
                metaDat += str(gameDat.maxMoves).ljust(2) + '& '
            elif fieldType == 'regionCount':
                metaDat += str(len(gameDat.game)).ljust(2) + '& '
            elif fieldType == 'gameString':
                metaDat += str(gameDat.gameString).ljust(2) + '& '
            else:
                print("Field not recognized:"  )
        metaDat += '/ '
    metaDat = metaDat[:-3]

    orderedGameStrings = []
    for curGame in orderedGameDat:
        orderedGameStrings.append(curGame.gameString)
    outputBlock = metaDat + ' / ' + '\t'.join(orderedGameStrings)

    return (metaDat, outputBlock)


def compareWithReplacementRegion(allData, removeRegion, addRegion, changeBounds = None, fieldOrdering = None, maxConnections = 100, printSpecificPairs = None, excludeSplitGames = True, sameRegionOtherSide = False, sameLoopOtherSide = False):
    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max']
    if changeBounds is None:
        changeBounds = {}
    if printSpecificPairs is None:
        printSpecificPairs = []

    metaCount = {}
    outputBlocks = []
    allOutputs = {}
    for gameString, gameObject in dataSummary.allGames.items():
        if gameObject.totalConnections > maxConnections:
            continue
        if excludeSplitGames and len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            if region == removeRegion:
                regionFound = False
                alteredGame = gameObject.game[:regionIndex] + [addRegion] + gameObject.game[regionIndex+1:]
                #alteredBoundaries = copyMap(gameObject.boundaries)

                '''for removeBound, addBound in changeBounds.items():
                    removeBoundary = (regionIndex,) + removeBound
                    addBoundary    = (regionIndex,) +    addBound
                    otherside = gameObject.boundaries[removeBoundary]
                    del(alteredBoundaries[removeBoundary])
                    alteredBoundaries[addBoundary] = otherside
                    if otherside != DISAPOINT:
                        alteredBoundaries[otherside  ] = addBoundary'''

                otherRegionIndicies = {}
                otherLoopIndicies = {}
                completeRemapping = {}
                for removeBound, addBound in changeBounds.items():
                    otherRegionIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[0]] = True
                    otherLoopIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[1]] = True
                    completeRemapping[(regionIndex,) + removeBound] = (regionIndex,) + addBound

                alteredBoundaries = executeRemappingOfExactPoints(gameObject.boundaries, completeRemapping)
                (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries)

                alteredString, A, B, C = findLowestGameName(allData, alteredGame, alteredBoundaries)
                if sameLoopOtherSide   and (len(otherRegionIndicies) > 1 or len(otherLoopIndicies) > 1):
                    continue
                if sameRegionOtherSide and len(otherRegionIndicies) > 1:
                    #print(gameString, alteredString)
                    continue



                if alteredString in dataSummary.allGames:
                    allOutputs[gameString] = False
                    orderedGameDat = [gameObject, dataSummary.allGames[alteredString]]
                    (metaDat, outputBlock) = preparePrintLine(allData, fieldOrdering, orderedGameDat)
                        #baseDat    = dataSummary.allGames[   baseString]
                        #alteredDat = dataSummary.allGames[alteredString]
                        #metaDat = str(baseDat.nimber) + ' ' + str(alteredDat.nimber) + ' ' + str(gameObject.nimber)# + ' / ' + str(gameObject.totalConnections) + ' ' + str(alteredDat.totalConnections)# + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredDat.minMoves) + ' ' + str(alteredDat.maxMoves) + ')'
                        #outputBlocks.append(metaDat + ' / ' + baseString + '\t' + alteredString + '\t' + gameString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)
                    mapKeyAdd(metaCount, metaDat)

                    #if metaDat == "4  2 ":
                    if metaDat.strip().replace(" ", "") in printSpecificPairs:
                        pass
                        #print(gameString, alteredString)
                    #if not(metaDat == "0 & 0 &" or metaDat == "1 & 1 &" or metaDat == "2 & 2 &" or metaDat == "3 & 3 &" or metaDat == "4 & 4 &" or metaDat == "5 & 5 &" or metaDat == "6 & 6 &" or metaDat == "7 & 7 &" or metaDat == "8 & 8 &"):
                    #    print (metaDat, gameObject.cleanString.replace("A", "P").replace("B", "Q").replace("C", "R").replace("D", "S").replace("E", "T").replace("F", "V").replace("G", "W"))
                else:
                    pass
                    #print(gameString)
                    #if gameObject.nimber != dataSummary.allGames[alteredString].nimber:
                    #    print(gameObject.gameString)
                    #if metaDat.strip() in ["3  3"]:
                    #    print(metaDat, gameString, alteredString)
                #else:
                #    print("Unfound:", gameString, "-->", alteredString)

    print("!!!", len(allOutputs))

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(">>>>",tup[0] + '\t' + str(tup[1]))

In [ ]:
print('123 : 0,333')
compareWithReplacementRegion(dataSummary, (123,), (2,3), changeBounds = {(0,2):(1,0)}, fieldOrdering = ['nimber'], sameLoopOtherSide = False)#, 'totalC', 'min', 'max'])
print()

In [ ]:
a,b = convertGameStringToGame(dataSummary, '0,33|2,233|3,33|23:47560231')
makeGameString(dataSummary.allLoops, a,b)

In [ ]:
firstGame = [(3,),(23,)]
firstMap = {}
firstMap[(0,0,0)] = (1,0,1)
firstMap[(1,0,1)] = (0,0,0)

secondGame = [(3,),(123,)]
secondMap = {}
secondMap[(0,0,0)] = (1,0,2)
secondMap[(1,0,2)] = (0,0,0)

#0,3|2,233|3,33:345012
addAndCompare(dataSummary, [(firstGame, firstMap), (secondGame, secondMap)], printSpecificPairs = ["03"], maxConnections = 4, spawnAsNecessary = True)
'''
secondMap[(1,0,1)] = (4,1,2)
secondMap[(4,1,2)] = (1,0,1)
secondMap[(1,1,2)] = (3,0,1)
secondMap[(3,0,1)] = (1,1,2)'''

In [ ]:
#print(convertGameStringToGame(dataSummary, '23|3,3435|3435|0,3|0,3:10462735'))

'''firstGame = [(2435,),(23,)]
firstMap = {}
firstMap[(0,0,2)] = (1,0,1)
firstMap[(1,0,1)] = (0,0,2)'''

'''secondGame = [(23,), (23,), (23,), (3435,), (3, 3435)]
secondMap = {}
secondMap[(0,0,1)] = (4,0,0)
secondMap[(4,0,0)] = (0,0,1)

secondMap[(1,0,1)] = (4,1,2)
secondMap[(4,1,2)] = (1,0,1)
secondMap[(1,1,2)] = (3,0,1)
secondMap[(3,0,1)] = (1,1,2)'''

print(len(dataSummary.allGames))
addAndCompare(dataSummary, [convertGameStringToGame(dataSummary, '13|23:10'), \
   convertGameStringToGame(dataSummary, '13|133|23:1032'),
   convertGameStringToGame(dataSummary, '13|123:10')] , maxConnections = 4, spawnAsNecessary = True)

print(len(dataSummary.allGames))




'''
A4B5|A4C5|2,B|2,C
  0      1      2      3        4
[(23,), (23,), (23,), (3435,), (3, 3435)]
Boundaries:  {(0, 0, 1): (4, 0, 0), (1, 0, 1): (4, 1, 2), (2, 0, 1): (3, 0, 2), (3, 0, 0): (4, 1, 0), (3, 0, 2): (2, 0, 1), (4, 0, 0): (0, 0, 1), (4, 1, 0): (3, 0, 0), (4, 1, 2): (1, 0, 1)}
'''

#Validated:


'''
7 0,3|0,3|1,33:2301	0,A|0,B|1,AB	['12|0,A|1,A', '0|24A5|0,A', '6,A|0,B|1,AB', '0,A|0,B|2,AB', '0|0|1,2']
7	0,3|0,3|2,2,3,3:2301	0,A|0,B|2,2,A,B	['12|0,A|2,2,A', '0,A|0,B|2,A,B', '0|0,A|2,2,A', '0|0|2,2,2', '6,A|0,B|2,2,A,B']
7	0,3|0,3|22,33:2301	0,A|0,B|22,AB	['12|0,A|A,22', '0|24A5|0,A', '6,A|0,B|22,AB', '0,A|0,B|2,AB', '0|0|2,22']
7	0,3|0,3|23,23:2301	0,A|0,B|2A,2B	['12|0,A|2,2A', 'A4B5|0,A|0,B', '0|24A5|0,A', '0|0|2425', '6,A|0,B|2A,2B', '0|0,A|2,2A']
7	133|3435|0,3|23,33:67452301	1AB|C4D5|0,C|2D,AB	['12|2A|1BC|2A,BC', '1A|A4B5|B4C5|0,C', '1A|2B|24A5|0,B', '2AB|C4D5|0,C|2D,AB', 'AB|ABC|D4E5|0,D|C,2E', '12|2A|A4B5|0,B', '2A|1BC|0,A|2,BC', '6|0|1AB|2,AB', '0|2A|1BC|2A,BC', '1AB|C4D5|6,C|2D,AB', '1|A4B5|0,A|2,2B']
7	133|3435|33435|0,3:45670123	1AB|C4D5|AB4C5|0,D	['12|2A|1BC|BC4A5', '2AB|C4D5|AB4C5|0,D', 'AB|ABC|C4D5|D4E5|0,E', '12|2A|A4B5|0,B', '0|2A|1BC|BC4A5', '6|0|1AB|2AB', '2A|1BC|2BC|0,A', '1|24A5|A4B5|0,B', '1A|AB|BC|C4D5|0,D', '1A|2B|CD|ACD|0,B', '1A|2AB|B4C5|0,C', '1AB|C4D5|AB4C5|6,D']
7	13|133|3435|0,3:103254	1A|1AB|B4C5|0,C	['12|1A|2B|1AB', '2A|1AB|B4C5|0,C', '6|1A|A4B5|0,B', '1A|2AB|B4C5|0,C', '1A|2B|CD|ACD|0,B', '1|AB|ABC|C4D5|0,D', '1|12|2A|0,A', '0|1A|2B|1AB', '6|0|1A|1A', '1A|1A|2B|0,B', '1A|1AB|B4C5|6,C']
7	13|3435|0,3|3,13:435102	1A|B4C5|0,B|A,1C	['12|1A|2B|A,1B', '1|24A5|A4B5|0,B', '1|12|2A|0,A', '2A|B4C5|0,B|A,1C', '6|1A|A4B5|0,B', '0|1A|2B|A,1B', '6|0|1A|1,A', '1A|2B|0,B|1,A', '1A|B4C5|6,B|A,1C', '1A|BC|B4D5|0,D|A,C', '1A|2B|6,A|0,B', '1A|B4C5|0,B|A,2C']
7	13|3435|0,3|3,13:534120	1A|B4C5|0,B|C,1A	['12|1A|2B|B,1A', '1A|2B|24A5|0,B', '1|12|2A|0,A', '2A|B4C5|0,B|C,1A', '6|A4B5|0,A|1,B', '1A|1A|2B|0,B', '6|0|1A|1A', '0|1A|2B|B,1A', '1A|B4C5|6,B|C,1A', '1A|AB|C4D5|0,C|B,D', '1|A4B5|6,A|0,B', '1A|B4C5|0,B|C,2A']
7	13|3435|344355|0,3:345012	1A|B4C5|A44B55|0,C	['12|1A|2B|A44B55', '2A|B4C5|A44B55|0,C', '6|24A5|A4B5|0,B', '1A|2B|24A5|0,B', '6|0|1A|24A5', '0|1A|2B|A44B55', '1|24A5|A4B5|0,B', '1|AB|ABC|C4D5|0,D', '9|1|2A|0,A', '1A|2AB|B4C5|0,C', '1A|2B|CD|ACD|0,B', '1A|AB|BC|C4D5|0,D', '1A|B4C5|A44B55|6,C']
7	1435|0,3|1,33:2301	14A5|0,B|1,AB	['12|14A5|1,A', '0|14A5|24A5', '12|24A5|0,A', '24A5|0,B|1,AB', 'AB|ABC|0,D|1,CD', '9|0,A|1,A', '12|0,A|1,A', '14A5|6,B|1,AB', '14A5|0,B|2,AB', '0|12|1,2']
7	1435|0,3|2,2,3,3:2301	14A5|0,B|2,2,A,B	['12|14A5|2,2,A', '14A5|0,B|2,A,B', '0|14A5|2,2,A', '12|0,A|2,2,A', '0|12|2,2,2', '24A5|0,B|2,2,A,B', 'AB|ABC|0,D|2,2,C,D', '9|0,A|2,2,A', '14A5|6,B|2,2,A,B']
7	1435|0,3|22,33:2301	14A5|0,B|22,AB	['12|14A5|A,22', '0|14A5|24A5', '12|24A5|0,A', '24A5|0,B|22,AB', 'AB|ABC|0,D|22,CD', '9|0,A|A,22', '12|0,A|A,22', '14A5|6,B|22,AB', '14A5|0,B|2,AB', '0|12|2,22']
7	1435|0,3|23,23:2301	14A5|0,B|2A,2B	['12|14A5|2,2A', '14A5|A4B5|0,B', '12|24A5|0,A', '0|14A5|24A5', '0|12|2425', '24A5|0,B|2A,2B', 'AB|ABC|0,D|2C,2D', '9|0,A|2,2A', '12|0,A|2,2A', '14A5|6,B|2A,2B', '0|14A5|2,2A']
7	1435|1435|2,2,3,3:2301	14A5|14B5|2,2,A,B	['14A5|14B5|2,A,B', '12|14A5|2,2,A', '12|12|2,2,2', '14A5|24B5|2,2,A,B', 'AB|ABC|14D5|2,2,C,D', '9|14A5|2,2,A']
7	1435|1435|23,23:2301	14A5|14B5|2A,2B	['14A5|14B5|A4B5', '12|14A5|24A5', '12|12|2425', '14A5|24B5|2A,2B', 'AB|ABC|14D5|2C,2D', '9|14A5|2,2A', '12|14A5|2,2A']
7	1435|3435|0,3|2,3,3:435102	14A5|B4C5|0,B|2,A,C	['12|2A|14B5|2,A,B', '12|A4B5|0,A|2,B', '2A|14B5|0,A|2,B', '12|2A|0,A|2,2', '24A5|B4C5|0,B|2,A,C', 'AB|ABC|D4E5|0,D|2,C,E', '9|A4B5|0,A|2,B', '6|0|14A5|2,A', '0|2A|14B5|2,A,B', '14A5|B4C5|6,B|2,A,C']
7	1435|3435|0,3|3,23:435102	14A5|B4C5|0,B|A,2C	['12|2A|14B5|B,2A', '12|2A|A4B5|0,B', '12|22|2A|0,A', '24A5|B4C5|0,B|A,2C', 'AB|ABC|D4E5|0,D|C,2E', '9|2A|A4B5|0,B', '2A|14B5|0,A|2,B', '6|0|14A5|2,A', '0|2A|14B5|B,2A', '14A5|B4C5|6,B|A,2C']
7	1435|3435|0,3|3,23:534120	14A5|B4C5|0,B|C,2A	['12|2A|14B5|A,2B', '2A|2B|14A5|0,B', '12|22|2A|0,A', '24A5|B4C5|0,B|C,2A', 'AB|ABC|D4E5|0,D|E,2C', '9|A4B5|0,A|2,B', '12|A4B5|0,A|2,B', '6|0|2A|14A5', '0|2A|14B5|A,2B', '14A5|B4C5|6,B|C,2A']
7	1435|3435|3435|0,3:103254	14A5|A4B5|B4C5|0,C	['12|2A|14B5|A4B5', '24A5|A4B5|B4C5|0,C', 'AB|ABC|C4D5|D4E5|0,E', '9|2A|A4B5|0,B', '12|2A|A4B5|0,B', '6|12|2A|0,A', '2A|2B|14A5|0,B', '6|0|2A|14A5', '0|2A|14B5|A4B5', '14A5|A4B5|B4C5|6,C']
7	233|3435|0,3|0,3:240513	2AB|A4C5|0,B|0,C	['12|2A|2AB|0,B', '12|2A|A4B5|0,B', '2A|2B|0,A|0,B', '0|2A|A4B5|0,B', '0|22|2A|0,A', '6|0|2A|0,A', '0|2A|2AB|0,B', '2AB|A4C5|6,C|0,B', '2AB|A4C5|6,B|0,C']
7	2435|3435|0,3|3,13:435102	24A5|B4C5|0,B|A,1C	['12|2A|24B5|B,1A', '22|24A5|A4B5|0,B', '12|22|2A|0,A', '2A|B4C5|0,B|A,1C', '6|1A|A4B5|0,B', '1A|22|A4B5|0,B', '0|2A|24B5|B,1A', '6|0|24A5|1,A', '2A|24B5|0,A|1,B', '24A5|B4C5|6,B|A,1C', 'AB|24C5|A4D5|0,D|B,C', '2A|24B5|6,B|0,A', '24A5|B4C5|0,B|A,2C']
7	2435|3435|0,3|3,13:534120	24A5|B4C5|0,B|C,1A	['12|2A|24B5|A,1B', '2A|24B5|24B5|0,A', '12|22|2A|0,A', '2A|B4C5|0,B|C,1A', '6|A4B5|0,A|1,B', '22|A4B5|0,A|1,B', '1A|2B|24A5|0,B', '6|0|1A|24A5', '0|2A|24B5|A,1B', '24A5|B4C5|6,B|C,1A', 'AB|24A5|C4D5|0,C|B,D', '22|A4B5|6,A|0,B', '24A5|B4C5|0,B|C,2A']
7	33|33|0,3|0,3|33,33:2607891345	AB|AC|0,D|0,E|BC,DE	['12|AB|AC|0,D|D,BC', '0|2A|A4B5|0,B', '0,A|0,B|2,AB', 'AB|AC|6,D|0,E|BC,DE', '0|0|AB|AC|2,BC']
7	33|33|1435|0,3|33,33:2607891345	AB|AC|14D5|0,E|BC,DE	['12|AB|AC|14D5|D,BC', '0|2A|14B5|A4B5', '12|2A|A4B5|0,B', '14A5|0,B|2,AB', 'AB|AC|24D5|0,E|BC,DE', 'AB|AC|DE|DEF|0,G|BC,FG', '9|AB|AC|0,D|D,BC', '12|AB|AC|0,D|D,BC', 'AB|AC|14D5|6,E|BC,DE', '0|12|AB|AC|2,BC']
7	3435|0,3|0,3|2,3,3:240513	A4B5|0,A|0,C|2,B,C	['12|A4B5|0,A|2,B', '12|2A|0,B|2,A,B', '0|A4B5|0,A|2,B', '2A|0,A|0,B|2,B', '0|2A|0,A|2,2', '6|0|0,A|2,A', '0|2A|0,B|2,A,B', 'A4B5|6,C|0,A|2,B,C', 'A4B5|6,A|0,C|2,B,C']
7	3435|0,3|0,3|2,33:240513	A4B5|0,A|0,C|2,BC	['12|2A|0,B|2,AB', '12|A4B5|0,A|2,B', '0|2A|A4B5|0,B', '2A|2B|0,A|0,B', '2A|0,A|0,B|2,B', '6|0|0,A|2,A', '0|2A|0,B|2,AB', 'A4B5|6,A|0,C|2,BC', 'A4B5|6,C|0,A|2,BC', '0|2A|0,A|2,2']
7	3435|0,3|0,3|3,23:240513	A4B5|0,A|0,C|B,2C	['12|2A|0,B|A,2B', '12|A4B5|0,A|2,B', '2A|2B|0,A|0,B', '0|22|2A|0,A', '0|2A|0,B|A,2B', '6|0|2A|0,A', 'A4B5|6,A|0,C|B,2C', 'A4B5|6,C|0,A|B,2C', '0|A4B5|0,A|2,B']
7	3435|0,3|0,3|3,23:250431	A4B5|0,A|0,C|C,2B	['12|2A|A4B5|0,B', '12|2A|0,B|B,2A', '0|2A|A4B5|0,B', '0|22|2A|0,A', '2A|0,A|0,B|2,B', '6|0|0,A|2,A', '0|2A|0,B|B,2A', 'A4B5|6,C|0,A|C,2B', 'A4B5|6,A|0,C|C,2B']
7	3435|3435|0,3|0,3:2405137
'''

pass

In [ ]:
specialChildren = ['23|113:10', '113|2,3:10', \
                   '13|113:10', '113|1,3:10', \
                   '113|3,12:10', '113|1,23:10', \
                   '113|3,22:10', '113|223:10', '113|2,23:10', \
                   '113|-12,3:10', '33|113|333:345012', \
                   '113|2435:10', \
                   '23|113|3435:2301', '113|3435|2,3:1032', \
                   '13|113|3435:2301', '113|3435|1,3:1032', '113|3435|0,3:1032', \
                   '113|0,3:10', \
                   '113|0,23:10', \
                   '113|1435:10', '113|-12,23:10', \
                   '33|33|113|23,33:26075413', \
                   '33|33|113|333:25067134', '33|33|113|3,33:26075413', \
                   '23|113|23,23:2301', '113|2,3|23,23:2301', \
                   '113|2,2,3:10', \
                   '23|113|3,23:2301', '113|2,3|3,23:2301', \
                   '23|113|3,23:3210','113|2,3|3,23:3210', \
                   '23|113|2,3,3:2301', \
                   '113|2,3|2,3,3:2301', \
                   '23|113|233:2301', '113|233|2,3:1032', \
                   '23|113|2,33:2301', '113|2,3|2,33:2301', \

                   '23|113|133:2301', '113|133|2,3:1032', \
                   '23|113|1,33:2301', '113|1,33|2,3:1032', \

                   '23|113|22,33:2301', '113|2,3|22,33:2301', \

                   '113|22,23:10', \
                   '23|33|33|113|33,33:6381970524', \
                   '33|33|113|2,3|33,33:2607891345', \
                   '113|2,2,2,3:10', \
                   '1133|2,33:2301', '233|1133:2301',\
                   '1133|1,33:2301', '133|1133:2301',\
                   '1133|0,33:2301',
                   '33|33|1133|33,33:2607891345']


# 113|2,2,23:10 is difinitively not the same as 113|1,3:10
# same for 113|2,3|2,2,33:2301 and 23|113|2,3,23:2301 and 113|2,2,2,23:10 and 13|2,3|2,3,23:3210

# 13|23|133:2301 is in g_1 (where (23,) is the standard)

invalids = ['113|2,3|3,3|3,3:240513', '23|113|3,3|3,3:240513', \
              '33|113|2,3|3,3:345012', '23|33|113|3,3:104523', \
              '33|113|2,3|3,3:240513', '33|33|113|2,3:240513', \
              '23|33|33|113:103254', '23|33|113|3,3:435102', \
              '23|113|2,3,23:2301', '113|2,3|2,2,33:2301', \
              '23|113|2,2,33:2301', '113|2,3|2,3,23:3210', \
              '113|2,2,2,23:10', '113|2,2,23:10', \
              '33|33|33|1133:24061735', \
              '113|123:10']


for checkThis in specialChildren:
    if checkThis not in dataSummary.allGames:
        print(checkThis)

for gameString, gameObject in dataSummary.allGames.items():
    if len(gameObject.subGames) != 1 or '113' not in gameString or \
      '11333' in gameString or \
      '11323' in gameString or \
      '3,113' in gameString or \
      '2,1133' in gameString or \
      gameString in invalids:
        continue
    childCount = 0
    for child in gameObject.childGames:
        if "113" in child.gameString:
            childCount += 1
    if childCount == 0:
        continue
    outputString = ''
    for child in gameObject.childGames:
        if child.gameString in specialChildren:
            childCount -= 1
            #print(gameString, child.gameString)
            outputString += child.cleanString + "    " + child.gameString + "\n"
    if childCount == 0:
        if gameString in specialChildren:
            print ("!!!", gameObject.cleanString + "    " + gameString, "\n" + outputString + "\n")
        else:
            print (">>>", gameObject.cleanString + "    " + gameString, "\n" + outputString + "\n")

'''
!!! 11AB|2,2,AB    1133|2,2,33:2301
11AB|2,AB    1133|2,33:2301
11A|2,2A    113|2,23:10


!!! 11AB|22,AB    1133|22,33:2301
11A|24A5    113|2435:10
11AB|2,AB    1133|2,33:2301


!!! 2A|11BC|2A,BC    23|1133|23,33:345012
2A|11B|A4B5    23|113|3435:2301
11A|24A5    113|2435:10
11AB|2,AB    1133|2,33:2301
'''



'''
!!! 2AB
!!! 2,AB
!!! 1,AB
!!! 1AB

>>>27,AB
--11B|74B5
--24A5
--11AB|2,AB


>>> 2,2,AB
>>> 22,AB



2,A,7
---2,A


2,A7
--2,A
--2A


A,27
--2,A


7,2A
--2A


2A7
--2A

2,7,A
---2,A


2,7A
--2A
--2,A


27A
--2A|11A


7,2B
--2A


A,27
--2,A


A8
---2A



1,A7   (in G_1)
--24A5
--1,A
--2,A7


1A7
--1A
--2A7
--6A


22,A7
--A,22
--24A5
--2,A7


2A,27   (In G_1)
--2,2A
--A475
--24A5


1,7A
--24A5
--1,A
--2,7A


17B
--1A
--27A
--6A


22,7B
--24A5
--A,22
--2,7A


27,2A
--A475
--24A5
--2,2A


2A,8
--A475
--2,2A
'''

pass

In [ ]:
def removeThreeZeros(allData):
    allResults = []
    resultCounts = {}
    for gameString, gameObject in dataSummary.allGames.items():
        if "0,0,0," in gameString:
            gameStringAltered = gameString.replace("0,0,0,", "")
            newGame = convertGameStringToGame(allData, gameStringAltered)
            newGameString = findLowestGameName(allData, newGame[0], newGame[1])[0]
            if newGameString in dataSummary.allGames:
                newGameObject = dataSummary.allGames[newGameString]
                allResults.append((newGameObject.totalConnections, gameObject.nimber, newGameObject.nimber, gameString, newGameString))
                mapKeyAdd(resultCounts, str(gameObject.nimber) + ' ' + str(newGameObject.nimber))
    allResults.sort()
    for tup in allResults:
        print(tup)
    print(resultCounts)

removeThreeZeros(dataSummary)

In [ ]:
#@title Compare with replacement region

def preparePrintLine(allData, fieldOrdering, orderedGameDat):
    metaDat = ''
    for fieldType in fieldOrdering:
        for gameDat in orderedGameDat:
            if fieldType == 'nimber':
                metaDat += str(gameDat.nimber).ljust(2) + ' '
            elif fieldType == 'totalC':
                metaDat += str(gameDat.totalConnections).ljust(2) + ' '
            elif fieldType == 'min':
                metaDat += str(gameDat.minMoves).ljust(2) + ' '
            elif fieldType == 'max':
                metaDat += str(gameDat.maxMoves).ljust(2) + ' '
            elif fieldType == 'regionCount':
                metaDat += str(len(gameDat.game)).ljust(2) + ' '
            elif fieldType == 'gameString':
                metaDat += str(gameDat.gameString).ljust(2) + ' '
            else:
                print("Field not recognized:"  )
        metaDat += '/ '
    metaDat = metaDat[:-3]

    orderedGameStrings = []
    for curGame in orderedGameDat:
        orderedGameStrings.append(curGame.gameString)
    outputBlock = metaDat + ' / ' + '\t'.join(orderedGameStrings)

    return (metaDat, outputBlock)


def compareWithReplacementRegion(allData, removeRegion, addRegion, changeBounds = None, fieldOrdering = None, maxConnections = 100, excludeSplitGames = True, sameRegionOtherside = False):
    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max']
    if changeBounds is None:
        changeBounds = {}

    metaCount = {}
    outputBlocks = []
    for gameString, gameObject in dataSummary.allGames.items():
        if gameObject.totalConnections > maxConnections:
            continue
        if excludeSplitGames and len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            if region == removeRegion:
                regionFound = False
                alteredGame = gameObject.game[:regionIndex] + [addRegion] + gameObject.game[regionIndex+1:]
                #alteredBoundaries = copyMap(gameObject.boundaries)

                '''for removeBound, addBound in changeBounds.items():
                    removeBoundary = (regionIndex,) + removeBound
                    addBoundary    = (regionIndex,) +    addBound
                    otherside = gameObject.boundaries[removeBoundary]
                    del(alteredBoundaries[removeBoundary])
                    alteredBoundaries[addBoundary] = otherside
                    if otherside != DISAPOINT:
                        alteredBoundaries[otherside  ] = addBoundary'''

                otherRegionIndicies = {}
                completeRemapping = {}
                for removeBound, addBound in changeBounds.items():
                    otherRegionIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[0]] = True
                    completeRemapping[(regionIndex,) + removeBound] = (regionIndex,) + addBound

                alteredBoundaries = executeRemappingOfExactPoints(gameObject.boundaries, completeRemapping)
                (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries)

                alteredString, A, B, C = findLowestGameName(allData, alteredGame, alteredBoundaries)
                if sameRegionOtherside and len(otherRegionIndicies) > 1:
                    #print(gameString, alteredString)
                    continue

                if alteredString in dataSummary.allGames:
                    orderedGameDat = [gameObject, dataSummary.allGames[alteredString]]
                    (metaDat, outputBlock) = preparePrintLine(allData, fieldOrdering, orderedGameDat)
                        #baseDat    = dataSummary.allGames[   baseString]
                        #alteredDat = dataSummary.allGames[alteredString]
                        #metaDat = str(baseDat.nimber) + ' ' + str(alteredDat.nimber) + ' ' + str(gameObject.nimber)# + ' / ' + str(gameObject.totalConnections) + ' ' + str(alteredDat.totalConnections)# + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredDat.minMoves) + ' ' + str(alteredDat.maxMoves) + ')'
                        #outputBlocks.append(metaDat + ' / ' + baseString + '\t' + alteredString + '\t' + gameString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)
                    mapKeyAdd(metaCount, metaDat)

                    if metaDat == "0  2 ":
                        print(gameString, alteredString)
                else:
                    print(gameString)
                    #if gameObject.nimber != dataSummary.allGames[alteredString].nimber:
                    #    print(gameObject.gameString)
                    #if metaDat.strip() in ["3  3"]:
                    #    print(metaDat, gameString, alteredString)
                #else:
                #    print("Unfound:", gameString, "-->", alteredString)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0] + '\t' + str(tup[1]))

compareWithReplacementRegion(dataSummary, (1233,), (0,33), changeBounds = {(0,2):(1,0), (0,3):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

In [ ]:
def convertGameStringToGame(allData, gameString, gameStringCompressed = False, preProcessed = False):
    if gameStringCompressed:
        gameString = decompressGameString(gameString, allData.allBoundCompress)

    if gameString == ":" or gameString == "NULL":
        return([], {})

    boundaryCoordinates = []
    pointsAndBounds = gameString.split(":")
    pointsString = pointsAndBounds[0]
    boundsString = pointsAndBounds[1]
    splitByRegions = pointsString.split("|")
    game = []
    for regionIndex in range(len(splitByRegions)):
        loops = splitByRegions[regionIndex].split(",")
        region = ()
        for loopIndex in range(len(loops)):
            loop = loops[loopIndex]
            enum = int(loop)
            region += (enum,)
            if enum in SPECIALLOOPS:
                if enum not in allData.allLoops:
                    Loop(allData, specialCase = LONELYBOUNDARYPAIR)
            else:
                if enum not in allData.allLoops:
                    loopTuple = enumToTuple(enum)
                    Loop(allData, definition=loopTuple)
                for pointIndex in range(len(loop)):
                    if loop[pointIndex] == STRBOUN:
                        boundaryCoordinates.append((regionIndex, loopIndex, pointIndex))
        game.append(region)
        if not(preProcessed) and region not in allData.allRegions:
            spawnNewRegion(allData, region, origin = "Generating region required for convertGameStringToGame")

    boundaries = {}
    for pointIndex in range(len(boundsString)):
        otherSide = boundsString[pointIndex]
        otherSideIndex = -1
        if otherSide >= '0' and otherSide <= '9':
            otherSideIndex = ord(otherSide) - ord('0')
        elif otherSide >= 'A' and otherSide <= 'Z':
            otherSideIndex = ord(otherSide) - ord('A') + 10
        elif otherSide >= 'a' and otherSide <= 'z':
            otherSideIndex = ord(otherSide) - ord('a') + 36
        elif otherSide == "!":
            otherSideIndex = -2

        if otherSideIndex == -2:
            boundaries[boundaryCoordinates[pointIndex]] = DISAPOINT
        else:
            boundaries[boundaryCoordinates[pointIndex]] = boundaryCoordinates[otherSideIndex]

    game, boundaries = fullGameCleanup(allData, game, boundaries)

    return (game, boundaries)



'''
firstGame = [(33,), (33,33), (2,33)]
firstMap = {}
firstMap[(0,0,0)] = (1,0,0)
firstMap[(0,0,1)] = (1,0,1)
firstMap[(1,0,0)] = (0,0,0)
firstMap[(1,0,1)] = (0,0,1)
firstMap[(1,1,0)] = (2,1,0)
firstMap[(1,1,1)] = (2,1,1)
firstMap[(2,1,0)] = (1,1,0)
firstMap[(2,1,1)] = (1,1,1)

secondGame = [(33,), (0,33)]
secondMap = {}
secondMap[(0,0,0)] = (1,1,0)
secondMap[(0,0,1)] = (1,1,1)
secondMap[(1,1,0)] = (0,0,0)
secondMap[(1,1,1)] = (0,0,1)


addAndCompare(dataSummary, [(firstGame, firstMap), (secondGame, secondMap)], maxConnections = 2)


firstGame = [(3,),(33,),(23,)]
firstMap = {}
firstMap[(0,0,0)] = (1,0,0)
firstMap[(1,0,0)] = (0,0,0)
firstMap[(1,0,1)] = (2,0,1)
firstMap[(2,0,1)] = (1,0,1)

secondGame = [(3,),(23,)]
secondMap = {}
secondMap[(0,0,0)] = (1,0,1)
secondMap[(1,0,1)] = (0,0,0)
'''

firstGame = [(2,)]
firstMap = {}

secondGame = [(3,),(23,)]
secondMap = {}
secondMap[(0,0,0)] = (1,0,1)
secondMap[(1,0,1)] = (0,0,0)

print(len(dataSummary.allGames))
addAndCompare(dataSummary, [(firstGame, firstMap), (secondGame, secondMap)], maxConnections = 3, spawnAsNecessary = True)
print(len(dataSummary.allGames))

Outliers:
('33|233:2301 ', '233|0,0,0,33:2301 ') 2  2
('-12,0: ', '-12,0,0,0,0: ') 2  2
('-12,2: ', '-12,0,0,0,2: ') 2  2
('33|2323:2301 ', '2323|0,0,0,33:2301 ') 2  2
('33|33|2323:240513 ', '33|2323|0,0,0,33:240513 ') 2  2
('33|33|2323:240513 ', '33|2323|0,0,0,33:240513 ') 2  2
('33|2,3:!21 ', '33|0,0,0,2,3:!21 ') 2  2
('33|2435|2,3:2301 ', '33|2435|0,0,0,2,3:2301 ') 3  3
('13|33|2,3:1032 ', '13|33|0,0,0,2,3:1032 ') 3  3

In [ ]:
#@title Compare with added loop
def compareWithAddedLoop(allData, addLoop):
    outputBlocks = []
    for gameString, gameObject in dataSummary.allGames.items():
        if len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):

            alteredBoundaries = copyMap(gameObject.boundaries)
            #alteredBoundaries[(regionIndex, len(gameObject.game[regionIndex]), 0)] = DISAPOINT

            newRegion = gameObject.game[regionIndex] + addLoop
            alteredGame = gameObject.game[:regionIndex] + [newRegion] + gameObject.game[regionIndex+1:]
            (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries, searchForSpecialCases=allData.useIsoRegions)
            alteredString, fullPointString, boundaryOrder, allOrdersQuickRef = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps, alteredGame, alteredBoundaries)

            if alteredString in dataSummary.allGames:
                alteredThing = dataSummary.allGames[alteredString]
                outputBlocks.append(str(gameObject.nimber) + ' ' + str(alteredThing.nimber) + ' / ' + str(gameObject.totalConnections) + ' ' + str(dataSummary.allGames[alteredString].totalConnections) + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredThing.minMoves) + ' ' + str(alteredThing.maxMoves) + ') / ' + gameString + '\t' + alteredString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)

    outputBlocks.sort()
    for block in outputBlocks:
        print(block)


def compareWithRemovedLoop(allData, removeLoop):
    metaCount = {}
    outputBlocks = []
    for gameString, gameObject in dataSummary.allGames.items():
        if len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            for loopIndex in range(len(region)):
                if region[loopIndex] == removeLoop:
                    newRegion = region[:loopIndex] + (EMPT,) + region[loopIndex+1:]
                    alteredGame = gameObject.game[:regionIndex] + [newRegion] + gameObject.game[regionIndex+1:]

                    alteredBoundaries = copyMap(gameObject.boundaries)
                    #alteredBoundaries[(regionIndex, len(gameObject.game[regionIndex]), 0)] = DISAPOINT
                    (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries, searchForSpecialCases=allData.useIsoRegions)
                    alteredString, fullPointString, boundaryOrder, allOrdersQuickRef = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps, alteredGame, alteredBoundaries)

                    if alteredString in dataSummary.allGames:
                        alteredThing = dataSummary.allGames[alteredString]
                        metaDat = str(alteredThing.nimber) + ' ' + str(gameObject.nimber)# + ' / ' + str(gameObject.totalConnections) + ' ' + str(dataSummary.allGames[alteredString].totalConnections)# + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredThing.minMoves) + ' ' + str(alteredThing.maxMoves) + ')'
                        outputBlocks.append(metaDat + ' / ' + gameString + '\t' + alteredString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)
                        mapKeyAdd(metaCount, metaDat)

    outputBlocks.sort()
    for block in outputBlocks:
        print(block)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0] + '\t' + str(tup[1]))


def compareWithReplacementRegion(allData, removeRegion, addRegion, changeBounds = None, fieldOrdering = None, maxConnections = 100, excludeSplitGames = True, sameRegionOtherside = False, sameLoopOtherSide = False):
    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max']
    if changeBounds is None:
        changeBounds = {}

    metaCount = {}
    outputBlocks = []
    for gameString, gameObject in dataSummary.allGames.items():
        if gameObject.totalConnections > maxConnections:
            continue
        if excludeSplitGames and len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            if region == removeRegion:
                alteredGame = gameObject.game[:regionIndex] + [addRegion] + gameObject.game[regionIndex+1:]
                #alteredBoundaries = copyMap(gameObject.boundaries)

                '''for removeBound, addBound in changeBounds.items():
                    removeBoundary = (regionIndex,) + removeBound
                    addBoundary    = (regionIndex,) +    addBound
                    otherside = gameObject.boundaries[removeBoundary]
                    del(alteredBoundaries[removeBoundary])
                    alteredBoundaries[addBoundary] = otherside
                    if otherside != DISAPOINT:
                        alteredBoundaries[otherside  ] = addBoundary'''

                otherRegionIndicies = {}
                otherLoopIndicies = {}
                completeRemapping = {}
                for removeBound, addBound in changeBounds.items():
                    otherRegionIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[0]] = True
                    otherLoopIndicies[(gameObject.boundaries[(regionIndex,) + removeBound])[1]] = True
                    completeRemapping[(regionIndex,) + removeBound] = (regionIndex,) + addBound

                alteredBoundaries = executeRemappingOfExactPoints(gameObject.boundaries, completeRemapping)
                (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries)

                alteredString, A, B, C = findLowestGameName(dataSummary, alteredGame, alteredBoundaries)
                if sameLoopOtherSide and len(otherRegionIndicies) > 1 and len(otherLoopIndicies) > 1:
                    continue
                if sameRegionOtherside and len(otherRegionIndicies) > 1:
                    #print(gameString, alteredString)
                    continue


                if alteredString in dataSummary.allGames:
                    orderedGameDat = [gameObject, dataSummary.allGames[alteredString]]
                    (metaDat, outputBlock) = preparePrintLine(allData, fieldOrdering, orderedGameDat)
                        #baseDat    = dataSummary.allGames[   baseString]
                        #alteredDat = dataSummary.allGames[alteredString]
                        #metaDat = str(baseDat.nimber) + ' ' + str(alteredDat.nimber) + ' ' + str(gameObject.nimber)# + ' / ' + str(gameObject.totalConnections) + ' ' + str(alteredDat.totalConnections)# + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredDat.minMoves) + ' ' + str(alteredDat.maxMoves) + ')'
                        #outputBlocks.append(metaDat + ' / ' + baseString + '\t' + alteredString + '\t' + gameString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)
                    mapKeyAdd(metaCount, metaDat)

                    if metaDat == "1  0 ":
                        print(gameString, alteredString)
                    #if gameObject.nimber != dataSummary.allGames[alteredString].nimber:
                    #    print(gameObject.gameString)
                    #if metaDat.strip() in ["3  3"]:
                    #    print(metaDat, gameString, alteredString)
                #else:
                #    print("Unfound:", gameString, "-->", alteredString)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0] + '\t' + str(tup[1]))


def compareWithReplacementLoop(allData, removeLoop, addLoop, reverseRelation = False, fieldOrdering = None):
    if fieldOrdering is None:
        fieldOrdering = ['nimber', 'totalC', 'min', 'max', 'gameString']

    metaCount = {}
    outputBlocks = []
    for gameString in dataSummary.allGames:
        gameObject = dataSummary.allGames[gameString]
        if gameObject.totalConnections > 12:
            continue
        if len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            for loopIndex in range(len(region)):
                if region[loopIndex] == removeLoop:
                    baseRegion = region[:loopIndex] + (   EMPT,) + region[loopIndex+1:]
                    newRegion  = region[:loopIndex] + (addLoop,) + region[loopIndex+1:]
                    baseGame    = gameObject.game[:regionIndex] + [baseRegion] + gameObject.game[regionIndex+1:]
                    alteredGame = gameObject.game[:regionIndex] + [ newRegion] + gameObject.game[regionIndex+1:]

                    baseBoundaries = copyMap(gameObject.boundaries)
                    alteredBoundaries = copyMap(gameObject.boundaries)
                    #alteredBoundaries[(regionIndex, len(gameObject.game[regionIndex]), 0)] = DISAPOINT
                    (   baseGame,    baseBoundaries) = fullGameCleanup(allData,    baseGame,    baseBoundaries, searchForSpecialCases=allData.useIsoRegions)
                    (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries, searchForSpecialCases=allData.useIsoRegions)

                    baseString   , X, Y, Z = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps,    baseGame,    baseBoundaries)
                    alteredString, A, B, C = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps, alteredGame, alteredBoundaries)

                    if alteredString in dataSummary.allGames and baseString in dataSummary.allGames:
                        orderedGameDat = [gameObject, dataSummary.allGames[baseString], dataSummary.allGames[alteredString]]
                        if reverseRelation:
                            orderedGameDat = [dataSummary.allGames[alteredString], dataSummary.allGames[baseString], gameObject]

                        (metaDat, outputBlock) = preparePrintLine(allData, fieldOrdering, orderedGameDat)

                        baseDat    = dataSummary.allGames[   baseString]
                        alteredDat = dataSummary.allGames[alteredString]
                        metaDat = str(baseDat.nimber) + ' ' + str(alteredDat.nimber) + ' ' + str(gameObject.nimber)# + ' / ' + str(gameObject.totalConnections) + ' ' + str(alteredDat.totalConnections)# + ' / (' + str(gameObject.minMoves) + ' ' + str(gameObject.maxMoves) + ') (' + str(alteredDat.minMoves) + ' ' + str(alteredDat.maxMoves) + ')'
                        outputBlocks.append(metaDat + ' / ' + baseString + '\t' + alteredString + '\t' + gameString)# + '\n    '+childAlso000 + childMaybe000 + partialChildNims + childNot000)
                        mapKeyAdd(metaCount, metaDat)
                    #else:
                        #print("Unfound:", gameString, "-->", alteredString, "-->", baseString)

    outputBlocks.sort()
    for block in outputBlocks:
        print(block)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0] + '\t' + str(tup[1]))


def reduceByAZero(allData, gameObject, regionIndex, loopIndex):
    region = gameObject.game[regionIndex]
    newRegion  = region[:loopIndex]   + (EMPT,) + region[loopIndex+1:]
    alteredGame = gameObject.game[:regionIndex] + [ newRegion] + gameObject.game[regionIndex+1:]

    alteredBoundaries = copyMap(gameObject.boundaries)
    (alteredGame, alteredBoundaries) = fullGameCleanup(allData, alteredGame, alteredBoundaries, searchForSpecialCases=True)

    alteredString, A, B, C = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps, alteredGame, alteredBoundaries)
    return (alteredGame, alteredBoundaries, alteredString)


def preparePrintLine(allData, fieldOrdering, orderedGameDat):
    metaDat = ''
    for fieldType in fieldOrdering:
        for gameDat in orderedGameDat:
            if fieldType == 'nimber':
                metaDat += str(gameDat.nimber).ljust(2) + ' '
            elif fieldType == 'totalC':
                metaDat += str(gameDat.totalConnections).ljust(2) + ' '
            elif fieldType == 'min':
                metaDat += str(gameDat.minMoves).ljust(2) + ' '
            elif fieldType == 'max':
                metaDat += str(gameDat.maxMoves).ljust(2) + ' '
            elif fieldType == 'regionCount':
                metaDat += str(len(gameDat.game)).ljust(2) + ' '
            elif fieldType == 'gameString':
                metaDat += str(gameDat.gameString).ljust(2) + ' '
            else:
                print("Field not recognized:"  )
        metaDat += '/ '
    metaDat = metaDat[:-3]

    orderedGameStrings = []
    for curGame in orderedGameDat:
        orderedGameStrings.append(curGame.gameString)
    outputBlock = metaDat + ' / ' + '\t'.join(orderedGameStrings)

    return (metaDat, outputBlock)



def compareWithSpecial(allData, reverseRelation = False, fieldOrdering = None):
    if fieldOrdering is None:
        fieldOrdering = ['regionCount', 'nimber', 'min', 'max']
    metaCount = {}
    outputBlocks = []
    for gameString in dataSummary.allGames:
        gameObject = dataSummary.allGames[gameString]
        #if gameObject.totalConnections > 15:
        #    continue
        if len(gameObject.subGames) > 1:
            continue
        for regionIndex in range(len(gameObject.game)):
            region = gameObject.game[regionIndex]
            for loopIndex in range(1, len(region)):
                if region[loopIndex] == 0 and region[loopIndex-1] == 0:
                    oneLessRegion = region[:loopIndex] + region[loopIndex+1:]
                    twoLessRegion = region[:loopIndex-1] + region[loopIndex+1:]
                    oneLessGame   = gameObject.game[:regionIndex] + [oneLessRegion] + gameObject.game[regionIndex+1:]
                    twoLessGame   = gameObject.game[:regionIndex] + [twoLessRegion] + gameObject.game[regionIndex+1:]

                    oneLessRemap = {}
                    twoLessRemap = {}
                    for boundary in gameObject.boundaries:
                        if boundary[0] == regionIndex and boundary[1] > loopIndex:
                            oneLessRemap[boundary] = (regionIndex, boundary[1]-1, boundary[2])
                            twoLessRemap[boundary] = (regionIndex, boundary[1]-2, boundary[2])

                    oneLessBoundaries = executeRemappingOfExactPoints(gameObject.boundaries, oneLessRemap)
                    twoLessBoundaries = executeRemappingOfExactPoints(gameObject.boundaries, twoLessRemap)

                    (oneLessGame, oneLessBoundaries) = fullGameCleanup(allData, oneLessGame, oneLessBoundaries, searchForSpecialCases=False)
                    (twoLessGame, twoLessBoundaries) = fullGameCleanup(allData, twoLessGame, twoLessBoundaries, searchForSpecialCases=False)

                    oneLessString, dump1, dump2, dump3 = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps, oneLessGame, oneLessBoundaries)
                    twoLessString, dump1, dump2, dump3 = findLowestGameName(dataSummary, dataSummary.allLoops, dataSummary.allPermutationMaps, twoLessGame, twoLessBoundaries)

                    if oneLessString in dataSummary.allGames and twoLessString in dataSummary.allGames:
                        orderedGameDat = [gameObject, dataSummary.allGames[oneLessString], dataSummary.allGames[twoLessString]]
                        if reverseRelation:
                            orderedGameDat = [dataSummary.allGames[twoLessString], dataSummary.allGames[oneLessString], gameObject]

                        (metaDat, outputBlock) = preparePrintLine(allData, fieldOrdering, orderedGameDat)
                        outputBlocks.append(outputBlock)
                        mapKeyAdd(metaCount, metaDat)
                    else:
                        print("Unfound:", twoLessString, "-->", oneLessString, "-->", gameString )

    #outputBlocks.sort()
    #for block in outputBlocks:
    #    print(block)

    orderMeta = []
    for metaDat in metaCount:
        orderMeta.append((metaDat, metaCount[metaDat]))
    orderMeta.sort()
    for tup in orderMeta:
        print(tup[0] + '\t' + str(tup[1]))

#compareWithReplacementRegion(dataSummary, (2,2,3,3), (23,23), changeBounds = {(2,0):(0,1), (3,0):(1,1)}, fieldOrdering = ['nimber', 'totalC', 'min', 'max'])

#
#compareWithReplacementRegion(dataSummary, (1233,), (1233,), changeBounds = {(0,2):(0,3), (0,3):(0,2)}, fieldOrdering = ['nimber', 'totalC', 'min', 'max'], maxConnections = 6, excludeSplitGames = True)

#No 00, but has some rare cases that suggest larger numbers may resolve differently
#compareWithReplacementRegion(dataSummary, (1233,), (133,), changeBounds = {(0,2):(0,1), (0,3):(0,2)}, fieldOrdering = ['nimber'], maxConnections = 20)

#No 01 <---> 10
#compareWithReplacementRegion(dataSummary, (2233,), (133,), changeBounds = {(0,2):(0,1), (0,3):(0,2)}, fieldOrdering = ['nimber'], maxConnections = 20)

#No A0 for A >= 2, and no AA
#compareWithReplacementRegion(dataSummary, (233,), (133,), fieldOrdering = ['nimber'], maxConnections = 20)

#No 01 <---> 10
#compareWithReplacementRegion(dataSummary, (1435,), (3,2425), changeBounds = {(0,2):(0,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

# 01, 10, 23, 32
#compareWithReplacementRegion(dataSummary, (0,23), (1,23), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

# 01, 10, 23, 32, 45, 54
#compareWithReplacementRegion(dataSummary, (0,3), (1,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

#print('0:2')
# All =
#compareWithReplacementRegion(dataSummary, (0,3), (2,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
#print()



#compareWithReplacementRegion(dataSummary, (1,33), (133,), changeBounds = {(1,0):(0,1), (1,1):(0,2)}, sameRegionOtherside = True, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
#print()
'''
print('0:1')
# ?
compareWithReplacementRegion(dataSummary, (0,3), (1,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('1:-12')
# ?
compareWithReplacementRegion(dataSummary, (1,3), (-12,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('2:-12')
# ?
compareWithReplacementRegion(dataSummary, (2,3), (-12,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('0:-12')
# ?
compareWithReplacementRegion(dataSummary, (0,3), (-12,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
'''


# 01, 10, 23, 32, 45, 54
#compareWithReplacementRegion(dataSummary, (2,33), (233,), changeBounds = {(1,0):(0,1),(1,1):(0,2)}, fieldOrdering = ['nimber'], excludeSplitGames = False)#, 'totalC', 'min', 'max'])
#compareWithReplacementRegion(dataSummary, (3,23), (233,), changeBounds = {(0,0):(0,1),(1,1):(0,2)}, fieldOrdering = ['nimber'], excludeSplitGames = False)#, 'totalC', 'min', 'max'])
#compareWithReplacementRegion(dataSummary, (2,3,3), (233,), changeBounds = {(1,0):(0,1),(2,0):(0,2)}, fieldOrdering = ['nimber'], excludeSplitGames = False)#, 'totalC', 'min', 'max'])


#compareWithReplacementRegion(dataSummary, (0,3), (2,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

'''
        #And now we take care of the two regions equivalent to (333,)
        elif region == (3,33):   # (3,33) = (333,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 1, 1)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (333,)
        elif region == (3,3,3):  # (3,3,3) = (333,)
            boundMapping[(regionIndex, 1, 0)] = (regionIndex, 0, 1)
            boundMapping[(regionIndex, 2, 0)] = (regionIndex, 0, 2)
            newGame[regionIndex] = (333,)'''

'''
0  1 	288
1  0 	260
2  3 	100
3  2 	135
4  5 	37
5  4 	34
6  7 	6
7  6 	1
'''

#compareWithReplacementLoop(dataSummary, 1, 2, reverseRelation = True)
#Either 01 <---> 10 and 23 <---> 32
#compareWithReplacementRegion(dataSummary, (3435,), (133,), changeBounds = {(0,0):(0,1)}, fieldOrdering = ['nimber', 'totalC', 'min', 'max'])

#compareWithReplacementRegion(dataSummary, (2,2,3,3), (23,23), changeBounds = {(2,0):(0,1), (3,0):(1,1)})
#compareWithAddedLoop(dataSummary, (LONELYBOUNDARYTRIPLE,))
#compareWithRemovedLoop(dataSummary, LONELYBOUNDARYPAIR)
#compareWithRemovedLoop(dataSummary, 1415)
#compareWithReplacementLoop(dataSummary, 1415, 2)
#compareWithReplacementLoop(dataSummary, 1415, 2)
#compareWithSpecial(dataSummary, fieldOrdering = ['nimber'])


print('0,33 : 233, ')
compareWithReplacementRegion(dataSummary, (0,2,33), (2,33), changeBounds = {(2,0):(1,0), (2,1):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
'''
print('1435 : 1,23')
compareWithReplacementRegion(dataSummary, (1435,), (1,23), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('1435 : 2,13')
compareWithReplacementRegion(dataSummary, (1435,), (2,13), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

print()
print('123 : 3,12')
compareWithReplacementRegion(dataSummary, (123,), (3,12), changeBounds = {(0,2):(0,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('123 : 1,23')
compareWithReplacementRegion(dataSummary, (123,), (1,23), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('123 : 2,13')
compareWithReplacementRegion(dataSummary, (123,), (2,13), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

print()
print('3,12 : 1,23')
compareWithReplacementRegion(dataSummary, (3,12), (1,23), changeBounds = {(0,0):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('3,12 : 2,13')
compareWithReplacementRegion(dataSummary, (3,12), (2,13,), changeBounds = {(0,0):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

print()
print('1,23 : 2,13')
compareWithReplacementRegion(dataSummary, (1,23), (2,13), changeBounds = {(1,1):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])'''

pass

In [ ]:
nimCount = {}
for game in dataSummary.allGames.values():
    mapKeyAdd(nimCount, len(game.childGames))
    if len(game.childGames) == 145:
        game.printReadableData()


print(nimCount)

In [ ]:
'''print('0,23:0,3')
compareWithReplacementRegion(dataSummary, (0,23), (0,3), changeBounds = {(1,1):(1,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()

print('1435:0,3')
compareWithReplacementRegion(dataSummary, (1435,), (0,3), changeBounds = {(0,2):(1,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()

print('2,23:0,3')
compareWithReplacementRegion(dataSummary, (2,23), (0,3), changeBounds = {(1,1):(1,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()

print('0,3:2,3')
compareWithReplacementRegion(dataSummary, (0,3), (2,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()

print('0,3:1,3')
compareWithReplacementRegion(dataSummary, (0,3), (1,3), fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()


0,23:0,3
0  1 	502
1  0 	511
2  3 	168
3  2 	153
4  5 	18
5  4 	47

1435:0,3
0  0 	2371
1  1 	2328
2  2 	2458
3  3 	2735
4  4 	251
5  5 	153
6  6 	18
7  7 	3

0,3:2,3
0  0 	4445
1  1 	4470
2  2 	5108
3  3 	5533
4  4 	487
5  5 	357
6  6 	48
7  7 	13
8  8 	1

0,3:1,3
0  1 	3818
1  0 	3771
2  3 	4186
3  2 	4701
4  5 	454
5  4 	269
6  7 	44
7  6 	13
8  9 	1








print('1435 : 123')
compareWithReplacementRegion(dataSummary, (1435,), (123,), changeBounds = {(0,2):(0,2)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('1435 : 3,12')
compareWithReplacementRegion(dataSummary, (1435,), (3,12), changeBounds = {(0,2):(0,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('1435 : 1,23')
compareWithReplacementRegion(dataSummary, (1435,), (1,23), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('1435 : 2,13')
compareWithReplacementRegion(dataSummary, (1435,), (2,13), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

print()
print('123 : 3,12')
compareWithReplacementRegion(dataSummary, (123,), (3,12), changeBounds = {(0,2):(0,0)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('123 : 1,23')
compareWithReplacementRegion(dataSummary, (123,), (1,23), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('123 : 2,13')
compareWithReplacementRegion(dataSummary, (123,), (2,13), changeBounds = {(0,2):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

print()
print('3,12 : 1,23')
compareWithReplacementRegion(dataSummary, (3,12), (1,23), changeBounds = {(0,0):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])
print()
print('3,12 : 2,13')
compareWithReplacementRegion(dataSummary, (3,12), (2,13,), changeBounds = {(0,0):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

print()
print('1,23 : 2,13')
compareWithReplacementRegion(dataSummary, (1,23), (2,13), changeBounds = {(1,1):(1,1)}, fieldOrdering = ['nimber'])#, 'totalC', 'min', 'max'])

'''


In [ ]:
#@title min/max/connections/nimber compared to child min/maxes
datTable = {}

for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    if gameObject.nimber == gameObject.maxMoves:
        continue
    if gameObject.totalConnections > 10:
        continue
    datString = ''
    datString += str(gameObject.totalConnections) + ' '
    datString += str(gameObject.maxMoves) + ' '
    datString += str(gameObject.minMoves) + ' '
    if gameObject.nimber == 0:
        datString += str(gameObject.nimber) + ' '
    else:
        datString += '1 '

    #if datString == '8 7 4 0 ':
    #    gameObject.printReadableData()
    #    print()
    #    print()
    #datString += str(gameObject.nimber) + ' '
    '''regionBreakdown = []
    for region in gameObject.game:
        regionBreakdown.append(str(len(region)))
    regionBreakdown = ','.join(regionBreakdown)
    datString += regionBreakdown'''

    mapKeyAdd(datTable, datString)

datOrder = []
for dat in datTable:
    datOrder.append((dat, datTable[dat]))
datOrder.sort()

for tup in datOrder:
    #print(tup[1], '\t', tup[0])
    pass

In [ ]:
#@title min/max/connections/nimber compared to child min/maxes
datTable = {}

for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    if gameObject.nimber == 0 or gameObject.minMoves == gameObject.maxMoves or not (gameObject.minMoves == 6 and gameObject.maxMoves == 10):
        continue
    prefixString = ''
    prefixString += str(gameObject.minMoves) + ' '
    prefixString += str(gameObject.maxMoves) + ' '
    prefixString += str(gameObject.totalConnections) + ' >>> '
    for childGame in gameObject.childGames:
        datString = prefixString
        datString += str(gameObject.minMoves - childGame.minMoves) + ' '
        datString += str(gameObject.maxMoves - childGame.maxMoves) + ' '
        datString += str(gameObject.totalConnections - childGame.totalConnections) + ' '
        if childGame.nimber == 0:
            datString += 'good'
        else:
            datString += 'bad'
            continue

        if datString == "6 10 12 >>> 1 2 2 good":
            gameObject.printReadableData()
            print()
            print()

        #datString += str(gameObject.nimber) + ' '
        '''regionBreakdown = []
        for region in gameObject.game:
            regionBreakdown.append(str(len(region)))
        regionBreakdown = ','.join(regionBreakdown)
        datString += regionBreakdown'''

        mapKeyAdd(datTable, datString)

datOrder = []
for dat in datTable:
    datOrder.append((dat, datTable[dat]))
datOrder.sort()

for tup in datOrder:
    print(tup[1], '\t', tup[0])

In [ ]:
#@title min/max/connections/nimber compared to child min/maxes
datTable = {}
containsGood = {}
containsBad = {}

for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    if gameObject.nimber == 0 or gameObject.minMoves == gameObject.maxMoves or not (gameObject.minMoves == 6 and gameObject.maxMoves == 10):
        continue
    prefixString = ''
    prefixString += str(gameObject.minMoves) + ' '
    prefixString += str(gameObject.maxMoves) + ' '
    prefixString += str(gameObject.totalConnections) + ' >>> '
    for childGame in gameObject.childGames:
        if len(childGame.subGames) > 1:
            continue
        secondString = prefixString
        '''secondString += str( gameObject.minMoves         - childGame.minMoves         ) + ' '
        secondString += str( gameObject.maxMoves         - childGame.maxMoves         ) + ' '
        secondString += str( gameObject.totalConnections - childGame.totalConnections ) + ' = '
        secondString += str(childGame.minMoves         ) + ' '
        secondString += str(childGame.maxMoves         ) + ' '
        secondString += str(childGame.totalConnections ) + ' >>>>> ' '''

        for subChildGame in childGame.childGames:
            datString = secondString
            datString += str( subChildGame.minMoves         ) + ' '
            datString += str( subChildGame.maxMoves         ) + ' '
            datString += str( subChildGame.totalConnections ) + '('
            datString += str( gameObject.minMoves         - subChildGame.minMoves         ) + ' '
            datString += str( gameObject.maxMoves         - subChildGame.maxMoves         ) + ' '
            datString += str( gameObject.totalConnections - subChildGame.totalConnections ) + ')'


            if childGame.nimber == 0:
                containsGood[datString] = True
                #datString += str("good")
            else:
                containsBad[datString] = True
                #datString += str("bad")


            #if datString == "6 10 11 1 >>> 0 2 1 >>>>> 0 1 1 0":
            #    gameObject.printReadableData()
            #    print()
            #    print()

            #datString += str(gameObject.nimber) + ' '
            '''regionBreakdown = []
            for region in gameObject.game:
                regionBreakdown.append(str(len(region)))
            regionBreakdown = ','.join(regionBreakdown)
            datString += regionBreakdown'''

            mapKeyAdd(datTable, datString)

datOrder = []
for dat in datTable:
    datOrder.append((dat, datTable[dat]))
datOrder.sort()

for tup in datOrder:
    if tup[0] in containsBad and tup[0] not in containsGood:
        print(tup[1], '\t', tup[0])

In [ ]:
#@title min/max/connections/nimber compared to child min/maxes
datTable = {}
containsGood = {}
containsBad = {}

for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    if gameObject.nimber == 0 or gameObject.minMoves == gameObject.maxMoves:
        continue
    prefixTup = (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections)
    for childGame in gameObject.childGames:
        if len(childGame.subGames) > 1:
            continue

        for subChildGame in childGame.childGames:
            datTup = prefixTup + (subChildGame.minMoves, subChildGame.maxMoves, subChildGame.totalConnections)
            datTable[datTup] = True
            if childGame.nimber == 0:
                containsGood[datTup] = True
            else:
                containsBad[datTup] = True

dontUseTwoStep = {}
for dat in datTable:
    if dat in containsBad and dat not in containsGood:
        dontUseTwoStep[dat] = True

print("Dropping:")
for val in dontUseTwoStep:
    print(val)
print()


dontUse = {}
for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    if gameObject.nimber == 0 or gameObject.minMoves == gameObject.maxMoves:
        continue
    prefixTup = (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections)
    for childGame in gameObject.childGames:
        if len(childGame.subGames) > 1:
            continue

        for subChildGame in childGame.childGames:
            datTup = prefixTup + (subChildGame.minMoves, subChildGame.maxMoves, subChildGame.totalConnections)
            if datTup in dontUseTwoStep:
                dontUse[(gameString, childGame.gameString)] = True



cleanDatTable = {}
for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    if gameObject.nimber == 0 or gameObject.minMoves == gameObject.maxMoves:
        continue
    prefixTup = (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections)
    for childGame in gameObject.childGames:
        #if (gameString, childGame.gameString) in dontUse:
        #    continue
        if len(childGame.subGames) > 1:
            continue

        knownBad = False
        for subChildGame in childGame.childGames:
            datTup = prefixTup + (subChildGame.minMoves, subChildGame.maxMoves, subChildGame.totalConnections)
            if datTup in dontUseTwoStep:
                knownBad = True
        outTup = prefixTup
        if knownBad:
            outTup = prefixTup + (childGame.minMoves, childGame.maxMoves, childGame.totalConnections, 'knownBad')
        else:
            if childGame.nimber == 0:
                datTup = datTup + ('good',)
            else:
                datTup = datTup + ('bad',)
            if (gameString, childGame.gameString) in dontUse:
                datTup = datTup + ('removed',)
        mapKeyAdd(cleanDatTable, datTup)

datOrder = []
for dat in cleanDatTable:
    datOrder.append((dat, cleanDatTable[dat]))
datOrder.sort()

for tup in datOrder:
    print(tup[1], '\t', tup[0])

153 	 10 14 15 >>> 9 12 13(1 2 2)
6 	 10 14 16 >>> 8 10 14(2 4 2)
229 	 10 14 16 >>> 8 11 14(2 3 2)
220 	 10 14 16 >>> 8 12 14(2 2 2)
3 	 10 14 16 >>> 9 10 14(1 4 2)
4 	 11 15 16 >>> 9 12 13(2 3 3)
13 	 2 4 5 >>> 1 2 3(1 2 2)
5 	 2 4 6 >>> 1 2 3(1 2 3)
9 	 2 4 6 >>> 1 2 4(1 2 2)
12 	 3 4 8 >>> 1 2 4(2 2 4)
6 	 3 5 6 >>> 1 2 4(2 3 2)
97 	 3 5 7 >>> 1 1 2(2 4 5)
4 	 3 5 8 >>> 1 1 2(2 4 6)
55 	 3 5 8 >>> 1 1 3(2 4 5)
36 	 3 5 8 >>> 3 3 4(0 2 4)
30 	 3 6 7 >>> 1 2 3(2 4 4)
7526 	 4 6 9 >>> 2 4 7(2 2 2)
94233 	 4 7 8 >>> 3 5 6(1 2 2)
2755 	 4 7 8 >>> 4 5 6(0 2 2)
3 	 4 7 9 >>> 2 3 7(2 4 2)
3288 	 4 7 9 >>> 2 4 7(2 3 2)
28568 	 4 7 9 >>> 3 5 6(1 2 3)
1019 	 4 7 9 >>> 4 5 6(0 2 3)
22 	 4 8 9 >>> 2 4 5(2 4 4)
205 	 4 8 9 >>> 3 5 6(1 3 3)
37 	 4 8 9 >>> 3 6 7(1 2 2)
324 	 4 8 9 >>> 4 6 7(0 2 2)
15 	 5 10 11 >>> 4 8 9(1 2 2)
116 	 5 10 11 >>> 5 8 9(0 2 2)
3 	 5 6 10 >>> 3 3 5(2 3 5)
914 	 5 6 10 >>> 3 4 8(2 2 2)
11 	 5 7 10 >>> 3 3 5(2 4 5)
41 	 5 7 10 >>> 3 3 8(2 4 2)
357 	 5 7 10 >>> 3 4 5(2 3 5)
5 	 5 7 11 >>> 3 4 7(2 3 4)
3 	 5 7 11 >>> 3 4 8(2 3 3)
16 	 5 7 11 >>> 3 5 7(2 2 4)
11 	 5 7 11 >>> 3 5 8(2 2 3)
25644 	 5 7 8 >>> 3 5 6(2 2 2)
3784 	 5 7 8 >>> 4 5 6(1 2 2)
89 	 5 8 10 >>> 3 4 5(2 4 5)
11 	 5 8 11 >>> 3 4 7(2 4 4)
5 	 5 8 11 >>> 3 4 8(2 4 3)
919 	 5 8 9 >>> 3 4 5(2 4 4)
2868 	 6 10 11 >>> 4 8 9(2 2 2)
82 	 6 10 12 >>> 4 7 8(2 3 4)
134 	 6 10 12 >>> 4 8 10(2 2 2)
40 	 6 10 12 >>> 4 8 9(2 2 3)
13 	 6 11 12 >>> 4 7 10(2 4 2)
16 	 6 11 12 >>> 4 7 8(2 4 4)
19 	 6 11 12 >>> 4 8 10(2 3 2)
39 	 6 11 12 >>> 4 8 9(2 3 3)
85 	 6 11 12 >>> 6 7 10(0 4 2)
13 	 6 8 12 >>> 4 6 10(2 2 2)
4 	 6 8 12 >>> 5 5 10(1 3 2)
288 	 6 8 12 >>> 5 6 10(1 2 2)
25 	 6 9 12 >>> 4 7 8(2 2 4)
55 	 7 10 11 >>> 5 6 7(2 4 4)
1884 	 7 10 12 >>> 5 7 8(2 3 4)
3 	 7 10 13 >>> 5 6 10(2 4 3)
477 	 7 11 12 >>> 5 7 8(2 4 4)
151 	 7 12 13 >>> 7 10 11(0 2 2)
2 	 7 8 12 >>> 5 5 10(2 3 2)
121 	 7 8 12 >>> 5 6 10(2 2 2)
18 	 7 9 10 >>> 5 6 7(2 3 3)
13 	 7 9 12 >>> 5 5 10(2 4 2)
90 	 7 9 12 >>> 5 7 8(2 2 4)
10 	 8 11 13 >>> 8 9 11(0 2 2)
2 	 8 11 14 >>> 6 9 10(2 2 4)
19 	 8 12 13 >>> 8 9 11(0 3 2)
58 	 8 12 14 >>> 6 10 11(2 2 3)
57 	 8 13 14 >>> 6 10 11(2 3 3)
26 	 8 13 14 >>> 6 11 12(2 2 2)
8 	 8 13 14 >>> 6 9 10(2 4 4)
51 	 8 13 14 >>> 8 10 12(0 3 2)
21 	 8 13 14 >>> 8 11 12(0 2 2)
6 	 8 13 14 >>> 8 9 12(0 4 2)
2 	 9 10 13 >>> 7 7 10(2 3 3)
3 	 9 10 13 >>> 7 8 10(2 2 3)
2 	 9 11 13 >>> 7 9 10(2 2 3)
1582 	 9 12 13 >>> 7 10 11(2 2 2)
29 	 9 12 13 >>> 7 9 10(2 3 3)
4 	 9 12 14 >>> 7 9 10(2 3 4)
2 	 9 12 15 >>> 7 9 11(2 3 4)
7 	 9 13 14 >>> 7 9 10(2 4 4)
6 	 9 13 15 >>> 7 10 11(2 3 4)
4 	 9 13 15 >>> 7 9 11(2 4 4)
4 	 9 14 15 >>> 7 10 11(2 4 4)
6 	 9 14 15 >>> 9 12 13(0 2 2)

In [ ]:
count = 0
for gameString in dataSummary.allGames:
    if count > 10:
        break
    gameObject = dataSummary.allGames[gameString]
    if gameObject.minMoves == 6 and gameObject.maxMoves == 6:
        count += 1
        gameObject.printReadableData()
        print()

In [ ]:
#Compare non-special games to special games

number_of_free_points = 4
useSpecialCases = False
dataSummary = generateBaseGame(number_of_free_points, useSpecialCases = useSpecialCases)


dataSummary.useSpecialCases = True
Loop(dataSummary, specialCase = LONELYBOUNDARYPAIR)
Loop(dataSummary, specialCase = LONELYBOUNDARYTRIPLE)

gameMap = {}
gameSubMap = {}
for game in dataSummary.allGames:
    cleanNewGame, cleanBoundaries = findSpecialCases(dataSummary, dataSummary.allGames[game].game, dataSummary.allGames[game].boundaries)
    cleaner, fullPointString, boundaryOrder, allOrdersQuickRef = findLowestGameName(dataSummary, cleanNewGame, cleanBoundaries)

    gameObject = dataSummary.allGames[game]
    gameTup = (str(gameObject.totalConnections), str(gameObject.nimber),  str(gameObject.minMoves), str(gameObject.maxMoves), cleaner)
    gameMap[gameTup] = False
    gameSubMap[cleaner] = gameObject.gameString


useSpecialCases = True
dataSummary2 = generateBaseGame(number_of_free_points, useSpecialCases = useSpecialCases)

specialCases = []
for game in dataSummary2.allGames:
    gameObject = dataSummary2.allGames[game]
    gameTup = (str(gameObject.totalConnections), str(gameObject.nimber), str(gameObject.minMoves), str(gameObject.maxMoves), gameObject.gameString)
    if gameTup in gameMap:
        gameMap[gameTup] = True
    else:
        specialCases.append(gameTup)

sharedGames = []
specializable = []
for game in gameMap:
    if gameMap[game]:
        sharedGames.append(game)
    else:
        specializable.append(game)

sharedGames.sort(reverse=True)
specializable.sort(reverse=True)
specialCases.sort(reverse=True)

mergedSpecials = {}
for game in specializable:
    metaTup = game[:4]
    gameName = str(game[4:])
    if metaTup not in mergedSpecials:
        mergedSpecials[metaTup] = [[],[]]
    mergedSpecials[metaTup][0].append(gameName)

for game in specialCases:
    metaTup = game[:4]
    gameName = str(game[4:])
    if metaTup not in mergedSpecials:
        mergedSpecials[metaTup] = [[],[]]
    mergedSpecials[metaTup][1].append(gameName)

mergedSpecialsPrint = []
for metaTup in mergedSpecials:
    curLine = "\t".join(metaTup)
    for game in mergedSpecials[metaTup][0]:
        curLine += "\n" + game
    curLine += "\n-----"
    for game in mergedSpecials[metaTup][1]:
        curLine += "\n" + game
    curLine += '\n\n'
    print(curLine)
    print("!!!!!!!!!!!!")

for gameTup in sharedGames:
    print('\t'.join(gameTup))
print()
print()

'''
for gameTup in specializable:
    print('\t'.join(gameTup))
print()
print()
for gameTup in specialCases:
    print('\t'.join(gameTup))'''



5	3	2	3	33|33|33|3333:2607891345
5	3	2	3	33|33|33|233:24061735
5	3	2	3	33|33|33|2,33:24061735
5	3	2	3	33|33|333|333:4758029136
5	3	2	3	33|33|333:!456123
5	3	2	3	33|33|2,233:240513
5	3	2	3	33|33|-12,33:240513
5	3	2	3	33|33435:2301!
5	3	2	3	33|3,233:34!01
5	3	2	3	33|2435:!21
5	3	2	3	33|233|233:240513
5	3	2	3	33|233|2,33:240513
5	3	2	3	33|2333:240!1
5	3	2	3	33|2333:2301!
5	3	2	3	33|2,33|2,33:240513
5	3	2	3	33|2,333:2301!
5	3	2	3	33|-12,3:!21
5	3	2	3	333|3333:345012!
5	3	2	3	333|2,33:!3412
5	3	2	3	23425:!
5	3	2	3	233|333:2301!
5	3	2	3	233|3,33:34!01
5	3	2	3	233|2,233:2301
5	3	2	3	22|33|233:2301
5	3	2	3	22|-12,2:
5	3	2	3	2,33|3,33:34!01
5	3	2	3	2,33|2,233:2301
5	3	2	3	2,2435:!
5	3	2	3	1|33|233:2301
5	3	2	3	1|-12,2:
5	3	2	3	13|33:10!
5	3	2	3	-12|33|233:2301
5	3	2	3	-12|-12,2:
5	3	2	3	-12,23:!
5	3	2	3	-12,2,3:!
3|2,3|-12,2

#Custom Functions

In [ ]:
#@title Generate a winning game
winTupFreq = {}
for smartMoveType in ["random", "shortest", "longest"]:
    for i in range(100):
        winPath = []
        generateWinPath(dataSummary, winPath, dataSummary.allGames['0,0,0,0,0,0:'], smartMoveType=smartMoveType)
        if len(winPath) != 14:
            print(len(winPath))
            for pathStats in winPath:
                print(pathStats)
            print()
'''
        curStats = str(len(winPath)) + " - "
        for pathStats in winPath:
            curStats +=  str(pathStats[2]) + ' ' + str(pathStats[3]) + ' ; '
        mapKeyAdd(winTupFreq, curStats)

winTupList = []
for winTup, freq in winTupFreq.items():
    winTupList.append((winTup, freq))
winTupList.sort(reverse=True)

for tup in winTupList:
    print(tup)'''

In [ ]:
#@title Generate full subgame
def generateFullSubgame(dataSummary, childParentMoves, seenGames = None, curGame = None, curSubGameList = None, parentString = None):
    if curGame is None and curSubGameList is None:
        print("generateFullSubgame requires a curGame or a curSubGameList")
        return

    if curGame is not None and curSubGameList is not None:
        print("generateFullSubgame cannot have both a curGame or a curSubGameList")

    if seenGames is None:
        seenGames = {}

    #If the input is just a normal game, split it out into subGames (presumably there's just one, but it needs to be made an array regardless)
    if curGame is not None:
        parentString = curGame.gameString
        curSubGameList = []
        for subGameString in curGame.subGames:
            curSubGameList.append(dataSummary.allGames[subGameString])

    #Get rid of any empty subgames, and if the entire game is composed of empty subgames, we can end the recursion
    allEmpty = True
    cleanSubGameList = []
    for gameObject in curSubGameList:
        if gameObject.gameString != 'NULL':
            cleanSubGameList.append(gameObject.gameString)
            allEmpty = False
    if curSubGameList[0] == 'NULL':
        return()

    cleanSubObjectList = []
    for gameString in cleanSubGameList:
        cleanSubObjectList.append(dataSummary.allGames[gameString])

    #NonUnified games have nothing in their childGames dict, so we need to use a variant of game-building to work our way down instead
    childGames = generateNonUnifiedMoves(dataSummary, cleanSubObjectList)

    #If the gamestate is winnable, keep only winning moves.  Otherwise, keep them all.
    nextMoves = []
    for childData in childGames:
        (childNimber, childMinMoves, childMaxMoves, childTotalConnections, childMoveType, childObject) = childData
        subGameStrings = []
        for gameObject in childObject:
            if gameObject.gameString != 'NULL':
                subGameStrings.append(gameObject.gameString)
        if len(subGameStrings) == 0:
            subGameStrings.append('NULL')
        subGameStrings.sort()

        subGameObjects = []
        for gameString in subGameStrings:
            subGameObjects.append(dataSummary.allGames[gameString])
        singularString = ';'.join(subGameStrings)

        nextMoves.append((singularString, subGameObjects, childMoveType))

    for tup in nextMoves:
        (singularString, subGameObjects, moveType) = tup
        if singularString not in childParentMoves:
            childParentMoves[singularString] = []
        childParentMoves[singularString].append((parentString, moveType))
        if singularString not in seenGames:
            seenGames[singularString] = True
            generateFullSubgame(dataSummary, childParentMoves, seenGames = seenGames, curSubGameList = subGameObjects, parentString = singularString)

    return

'''
generateFullSubgame(dataSummary, gameGraph, curGame = dataSummary.allGames[gameString])
for childString, parentArray in gameGraph.items():
    for parentTup in parentArray:
        (parentString, moveType) = parentTup
        #moveType = compressEncodingArray(moveType)
        print(prefix[count] + parentString + '\t' + prefix[count] + childString + '\t' + str(moveType))

gameGraph = {}

prefix = ['A-', 'B-', 'C-', 'D-', 'E-']
count = 0
#for gameString in ['2223:!', '3,2425:!', '1435:!', '2223|3,2425:10', '1435|2223:10']:
for gameString in [ '-12,3|0,3:10', '-12,3|1,3:10', '-12,3|2,3:10']:
    gameGraph = {}
    generateFullSubgame(dataSummary, gameGraph, curGame = dataSummary.allGames[gameString])
    shorts  = {}
    for childString, parentArray in gameGraph.items():
        for parentTup in parentArray:
            (parentString, moveType) = parentTup
            #moveType = compressEncodingArray(moveType)
            if parentString not in shorts:
                addZero = ''
                if len(shorts) < 10:
                    addZero = '0'
                shorts[parentString] = prefix[count][:-1] + addZero + str(len(shorts))
            if childString not in shorts:
                addZero = ''
                if len(shorts) < 10:
                    addZero = '0'
                shorts[childString] = prefix[count][:-1] + addZero + str(len(shorts))
            print(shorts[parentString]+'-' + parentString + '\t' + \
                  shorts[childString]+'-' + childString + '\t' + \
                  '\t' + parentString + '\t' + childString + '\t' + str(moveType))
    count += 1
'''

In [ ]:
> : cut off from other side
< : cut off from this side
* : degen from this side
$ : degen from other side
= : other side intact
m : move on other side





[400, 200] - standard move


Intact:
A00--12,3|0,3:10	A04--12,3|-12,3:10		    4/0,3=	3/-12,3=	standard move
    A04--12,3|-12,3:10	A08--12,3|2,3:10		3/-12,3=	2/2,3=	-12 connect to bound
        A08--12,3|2,3:10	A05-22:		        2/2,3=	  NULL>
            X
        A08--12,3|2,3:10	A10--12:		      2/2,3=	  NULL<
            X
        A08--12,3|2,3:10	A11-2,2:		      2/2,3=	  1/2,2$
            A11-2,2:	A09-NULL		          1/2,2$	  NULL$
                X
    A04--12,3|-12,3:10	A07--12,2:		      3/-12,3=	2/*
        *
    A04--12,3|-12,3:10	A07--12,2:		      3/-12,3=	2/-12,2$
        A07--12,2:	A05-22:		              2/-12,2$	1/22$
            A05-22:	A09-NULL		            1/22$     NULL$
                X
        A07--12,2:	A11-2,2:		            2/-12,2$	1/2,2$
            A11-2,2:	A09-NULL		          1/2,2$	  NULL$
                X
        A07--12,2:	A09-NULL		            2/-12,2$	NULL$
            X
    A04--12,3|-12,3:10	A02--12:;22:		    3/-12,3=	1/-12<
        A02--12:;22:	A05-22:		            1/-12<	  NULL<
            X
    A04--12,3|-12,3:10	A02--12:;22:		    3/-12,3=	1/22>
        A02--12:;22:	A05-22:		            1/22>	    NULL>
            X

A00--12,3|0,3:10	A01-0:;22:		            4/0,3=	  2/0>
    A01-0:;22:	A02--12:;22:		            2/0>	    1/-12>
        A02--12:;22:	A05-22:		            1/-12>	  NULL>
            X

A00--12,3|0,3:10	A03--12:;12:		          4/0,3=	  2/12<
    A03--12:;12:	            		          2/12<	    1/22<
        A14--12:;-12:	A10--12:		          1/-12:<	  NULL<
            X
    A03--12:;12:	A14--12:;-12:		          2/12<	    1/-12:<
        A14--12:;-12:	        		          1/-12:<	  NULL<
            X

A00--12,3|0,3:10	A16-0,2:		              4/0,3=	  3/0,2$
    A16-0,2:	A06-12:		                    3/0,2$	  2/12:$
        A06-12:	A05-22:		                  2/12:$	  1/22:$
            A05-22:	A09-NULL		            1/22:$  	NULL$
                X
        A06-12:	A10--12:		                2/12:$	  1/-12:$
            A10--12:	A09-NULL		          1/-12:$ 	NULL$
                X
    A16-0,2:	A07--12,2:		                3/0,2$	  2/-12,2$
        A07--12,2:	A05-22:		              2/-12,2$	1/22$
            A05-22:	A09-NULL		            1/22$     NULL$
                X
        A07--12,2:	A11-2,2:		            2/-12,2$	1/2,2$
            A11-2,2:	A09-NULL		          1/2,2$	  NULL$
                X
        A07--12,2:	A09-NULL		            2/-12,2$	NULL$
            X

In [ ]:
for game in dataSummary.allGames.values():
    for moveType in game.childGames.values():
        if 236 in moveType and 338 in moveType:
            print(game.printReadableData())
print("!!")

In [ ]:
coincidenceMatrix = {}
orderedMoves = []
for game in dataSummary.allGames.values():
    for moveType in game.childGames.values():
        for i in range(len(moveType)):
            moveI = compressEncoding(moveType[i])
            if moveI not in coincidenceMatrix:
                coincidenceMatrix[moveI] = {}
                orderedMoves.append(moveI)
            for j in range(i, len(moveType)):
                moveJ = compressEncoding(moveType[j])
                if moveJ not in coincidenceMatrix:
                    coincidenceMatrix[moveJ] = {}
                    orderedMoves.append(moveJ)
                mapKeyAdd(coincidenceMatrix[moveI], moveJ)
                if moveI != moveJ:
                    mapKeyAdd(coincidenceMatrix[moveJ], moveI)
#orderedMoves.sort()
orderedMoves = [-3,-2,278,328,428,338,326,226,186,586,336,686,786,37,286,236,636,676,276,27,288,576,688,626,648,638,678,628,698,39,298,238,448,248,498,566,766,602,666,618,668,616,216,266,316,202,218,268,318,2,66,68,16,36,26,78,76,86,212,214,224,262,0,1,3,5,7,9,12,13,14,15,17,400,25,29,612,614,624,664,38,48,49,64,264,274,312,314,324,564,574,662,28,18,19,24,674,88,98,62,74,164,166,174,176,562,162,600,762,764,774,776]


with open(root_path + "/coincidenceMatrix3.csv", 'w') as outFile:
    newLine = ''
    for moveI in orderedMoves:
        newLine += ',' + str(moveI)
    print(newLine, file=outFile)

    for moveI in orderedMoves:
        newLine = str(moveI)
        for moveJ in orderedMoves:
            if moveJ in coincidenceMatrix[moveI]:
                newLine += ',' + str(coincidenceMatrix[moveI][moveJ])
            else:
                newLine += ',0'
        print(newLine, file=outFile)

In [ ]:
#@title Win/Lose Matrix
winCoincidenceMatrix = {}
losCoincidenceMatrix = {}
nulCoincidenceMatrix = {}
orderedMoves = []
for game in dataSummary.allGames.values():
    for childGame, moveType in game.childGames.items():
        for i in range(len(moveType)):
            moveI = compressEncoding(moveType[i])
            if moveI not in winCoincidenceMatrix:
                winCoincidenceMatrix[moveI] = {}
                losCoincidenceMatrix[moveI] = {}
                nulCoincidenceMatrix[moveI] = {}
                orderedMoves.append(moveI)
            for j in range(i, len(moveType)):
                moveJ = compressEncoding(moveType[j])
                if moveJ not in winCoincidenceMatrix:
                    winCoincidenceMatrix[moveJ] = {}
                    losCoincidenceMatrix[moveJ] = {}
                    nulCoincidenceMatrix[moveJ] = {}
                    orderedMoves.append(moveJ)
                if moveJ not in winCoincidenceMatrix[moveI]:
                    winCoincidenceMatrix[moveI][moveJ] = 0
                    losCoincidenceMatrix[moveI][moveJ] = 0
                    nulCoincidenceMatrix[moveI][moveJ] = 0
                if moveI not in winCoincidenceMatrix[moveJ]:
                    winCoincidenceMatrix[moveJ][moveI] = 0
                    losCoincidenceMatrix[moveJ][moveI] = 0
                    nulCoincidenceMatrix[moveJ][moveI] = 0
                if game.nimber == 0:
                    mapKeyAdd(nulCoincidenceMatrix[moveI], moveJ)
                elif childGame.nimber == 0:
                    mapKeyAdd(winCoincidenceMatrix[moveI], moveJ)
                else:
                    mapKeyAdd(losCoincidenceMatrix[moveI], moveJ)
                if moveI != moveJ:
                    if game.nimber == 0:
                        mapKeyAdd(nulCoincidenceMatrix[moveJ], moveI)
                    elif childGame.nimber == 0:
                        mapKeyAdd(winCoincidenceMatrix[moveJ], moveI)
                    else:
                        mapKeyAdd(losCoincidenceMatrix[moveJ], moveI)
#orderedMoves.sort()
orderedMoves = [-3,-2,278,328,428,338,326,226,186,586,336,686,786,37,286,236,636,676,276,27,288,576,688,626,648,638,678,628,698,39,298,238,448,248,498,566,766,602,666,618,668,616,216,266,316,202,218,268,318,2,66,68,16,36,26,78,76,86,212,214,224,262,0,1,3,5,7,9,12,13,14,15,17,400,25,29,612,614,624,664,38,48,49,64,264,274,312,314,324,564,574,662,28,18,19,24,674,88,98,62,74,164,166,174,176,562,162,600,762,764,774,776]


with open(root_path + "/coincidenceMatrix3.csv", 'w') as outFile:
    newLine = ''
    for moveI in orderedMoves:
        newLine += ',' + str(moveI)
    print(newLine, file=outFile)

    for moveI in orderedMoves:
        newLine = str(moveI)
        for moveJ in orderedMoves:
            if moveJ in coincidenceMatrix[moveI]:
                newLine += ",'" + str(winCoincidenceMatrix[moveI][moveJ]) + ":" + str(losCoincidenceMatrix[moveI][moveJ]) + ":" + str(nulCoincidenceMatrix[moveI][moveJ])
            else:
                newLine += ",'0"
        print(newLine, file=outFile)

In [ ]:
for game in dataSummary.allGames.values():
    for childObject, moveType in game.childGames.items():
        game.childGames[childObject] = compressEncodingArray(moveType)

#Various Functions

In [ ]:
#@title Regions in game states with fixed remaining moves
regionCount = {}
for gameObject in dataSummary.allGames.values():
    if gameObject.minMoves == gameObject.maxMoves:
        for region in gameObject.game:
            mapKeyAdd(regionCount, (region, gameObject.nimber))

for region, count in regionCount.items():
    print(region, count)
    regionCount[region] = 0

print(regionCount)

In [ ]:
endingRegions = {(2425,): 0, (1,): 0, (-12,): 0, (-13,): 0, (222,): 0, (23,): 0, (33,): 0, (233,): 0, (2435,): 0, (333,): 0, (3,): 0, (223,): 0, (3333,): 0, (3435,): 0, (33435,): 0, (344355,): 0, (12,): 0, (13,): 0, (133,): 0, (1435,): 0, (11,): 0, (1, 3): 0, (3, 333): 0, (3, 13): 0, (3, 3435): 0, (2, 23): 0, (1, 2): 0, (-12, 3): 0, (33, 33): 0, (23, 33): 0, (1, 33): 0, (1, 23): 0, (1, 1): 0, (3, 3, 33): 0, (1, 3, 3): 0, (23, 23): 0, (0, 3): 0, (0,): 0, (2, 2, 3, 3): 0, (2, 2, 2, 3): 0, (2, 2, 2, 2): 0, (2, 2, 2, 2, 3): 0, (2, 2, 2, 2, 2): 0, (2, 2, 2, 2, 2, 2): 0}
trueEndingRegions = {(2425,): 0, (1,): 0, (-12,): 0, (-13,): 0, (222,): 0, (23,): 0, (33,): 0, (233,): 0, (2435,): 0, (333,): 0, (3,): 0, (223,): 0, (3333,): 0, (3435,): 0, (33435,): 0, (344355,): 0, (12,): 0, (13,): 0, (133,): 0, (1435,): 0, (11,): 0, (1, 3): 0, (3, 333): 0, (3, 13): 0, (3, 3435): 0, (2, 23): 0, (1, 2): 0, (-12, 3): 0, (33, 33): 0, (23, 33): 0, (1, 33): 0, (1, 23): 0, (1, 1): 0, (3, 3, 33): 0, (1, 3, 3): 0, (23, 23): 0, (0, 3): 0, (0,): 0, (2, 2, 3, 3): 0, (2, 2, 2, 3): 0, (2, 2, 2, 2): 0, (2, 2, 2, 2, 3): 0, (2, 2, 2, 2, 2): 0, (2, 2, 2, 2, 2, 2): 0}
otherVals = {}

for gameObject in dataSummary.allGames.values():
    if len(gameObject.subGames) > 1:
        continue
    possible = True
    for region in gameObject.game:
        if region not in endingRegions:
            possible = False
            break

    if possible and gameObject.minMoves != gameObject.maxMoves:
        mapKeyAdd(otherVals, (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections, gameObject.nimber))
        for region in gameObject.game:
            mapKeyAdd(endingRegions, region)
    elif possible:
        for region in gameObject.game:
            mapKeyAdd(trueEndingRegions, region)
            if region == (3, 3435):
                print(gameObject.minMoves, gameObject.gameString)
            if region == (23,23):
                print("<<", gameObject.minMoves, gameObject.gameString)
                break

for region, count in endingRegions.items():
    print(region, count, trueEndingRegions[region])

print()

for gameDat, count in otherVals.items():
    print(gameDat, count)

In [ ]:
endingRegions = {((2425,), 0): 0, ((1,), 1): 0, ((-12,), 1): 0, ((-13,), 0): 0, ((222,), 0): 0, ((23,), 1): 0, ((33,), 1): 0, ((233,), 0): 0, ((2435,), 0): 0, ((33,), 0): 0, ((333,), 0): 0, ((3,), 1): 0, ((223,), 0): 0, ((1,), 0): 0, ((2435,), 1): 0, ((-12,), 0): 0, ((333,), 1): 0, ((3333,), 1): 0, ((223,), 1): 0, ((3,), 0): 0, ((2425,), 1): 0, ((3435,), 0): 0, ((-13,), 1): 0, ((222,), 1): 0, ((23,), 0): 0, ((233,), 1): 0, ((33435,), 1): 0, ((344355,), 1): 0, ((3435,), 1): 0, ((33435,), 0): 0, ((344355,), 0): 0, ((3333,), 0): 0, ((12,), 0): 0, ((13,), 0): 0, ((133,), 1): 0, ((13,), 1): 0, ((12,), 1): 0, ((1435,), 1): 0, ((133,), 0): 0, ((1435,), 0): 0, ((11,), 1): 0, ((11,), 0): 0, ((1, 3), 0): 0, ((3, 333), 1): 0, ((1, 3), 1): 0, ((3, 333), 0): 0, ((3, 13), 1): 0, ((3, 13), 0): 0, ((3, 3435), 1): 0, ((3, 3435), 0): 0, ((2, 23), 0): 0, ((2, 23), 1): 0, ((1, 2), 0): 0, ((1, 2), 1): 0, ((-12, 3), 0): 0, ((33, 33), 1): 0, ((23, 33), 1): 0, ((23, 33), 0): 0, ((1, 33), 1): 0, ((1, 33), 0): 0, ((33, 33), 0): 0, ((1, 23), 1): 0, ((1, 23), 0): 0, ((1, 1), 1): 0, ((1, 1), 0): 0, ((-12, 3), 1): 0, ((3, 3, 33), 1): 0, ((1, 3, 3), 1): 0, ((3, 3, 33), 0): 0, ((1, 3, 3), 0): 0, ((23, 23), 1): 0, ((23, 23), 0): 0, ((0, 3), 0): 0, ((0, 3), 1): 0, ((0,), 0): 0, ((0,), 1): 0, ((2, 2, 3, 3), 1): 0, ((2, 2, 3, 3), 0): 0, ((2, 2, 2, 3), 1): 0, ((2, 2, 2, 3), 0): 0, ((2, 2, 2, 2), 1): 0, ((2, 2, 2, 2), 0): 0, ((2, 2, 2, 2, 3), 0): 0, ((2, 2, 2, 2, 3), 1): 0, ((2, 2, 2, 2, 2), 0): 0, ((2, 2, 2, 2, 2), 1): 0, ((2, 2, 2, 2, 2, 2), 1): 0}
trueEndingRegions = {((2425,), 0): 0, ((1,), 1): 0, ((-12,), 1): 0, ((-13,), 0): 0, ((222,), 0): 0, ((23,), 1): 0, ((33,), 1): 0, ((233,), 0): 0, ((2435,), 0): 0, ((33,), 0): 0, ((333,), 0): 0, ((3,), 1): 0, ((223,), 0): 0, ((1,), 0): 0, ((2435,), 1): 0, ((-12,), 0): 0, ((333,), 1): 0, ((3333,), 1): 0, ((223,), 1): 0, ((3,), 0): 0, ((2425,), 1): 0, ((3435,), 0): 0, ((-13,), 1): 0, ((222,), 1): 0, ((23,), 0): 0, ((233,), 1): 0, ((33435,), 1): 0, ((344355,), 1): 0, ((3435,), 1): 0, ((33435,), 0): 0, ((344355,), 0): 0, ((3333,), 0): 0, ((12,), 0): 0, ((13,), 0): 0, ((133,), 1): 0, ((13,), 1): 0, ((12,), 1): 0, ((1435,), 1): 0, ((133,), 0): 0, ((1435,), 0): 0, ((11,), 1): 0, ((11,), 0): 0, ((1, 3), 0): 0, ((3, 333), 1): 0, ((1, 3), 1): 0, ((3, 333), 0): 0, ((3, 13), 1): 0, ((3, 13), 0): 0, ((3, 3435), 1): 0, ((3, 3435), 0): 0, ((2, 23), 0): 0, ((2, 23), 1): 0, ((1, 2), 0): 0, ((1, 2), 1): 0, ((-12, 3), 0): 0, ((33, 33), 1): 0, ((23, 33), 1): 0, ((23, 33), 0): 0, ((1, 33), 1): 0, ((1, 33), 0): 0, ((33, 33), 0): 0, ((1, 23), 1): 0, ((1, 23), 0): 0, ((1, 1), 1): 0, ((1, 1), 0): 0, ((-12, 3), 1): 0, ((3, 3, 33), 1): 0, ((1, 3, 3), 1): 0, ((3, 3, 33), 0): 0, ((1, 3, 3), 0): 0, ((23, 23), 1): 0, ((23, 23), 0): 0, ((0, 3), 0): 0, ((0, 3), 1): 0, ((0,), 0): 0, ((0,), 1): 0, ((2, 2, 3, 3), 1): 0, ((2, 2, 3, 3), 0): 0, ((2, 2, 2, 3), 1): 0, ((2, 2, 2, 3), 0): 0, ((2, 2, 2, 2), 1): 0, ((2, 2, 2, 2), 0): 0, ((2, 2, 2, 2, 3), 0): 0, ((2, 2, 2, 2, 3), 1): 0, ((2, 2, 2, 2, 2), 0): 0, ((2, 2, 2, 2, 2), 1): 0, ((2, 2, 2, 2, 2, 2), 1): 0}
otherVals = {}

for gameObject in dataSummary.allGames.values():
    if len(gameObject.subGames) > 1:
        continue
    possible = True
    for region in gameObject.game:
        if (region, gameObject.nimber) not in endingRegions:
            possible = False
            break

    if possible and gameObject.minMoves != gameObject.maxMoves:
        mapKeyAdd(otherVals, (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections, gameObject.nimber))
        for region in gameObject.game:
            mapKeyAdd(endingRegions, (region, gameObject.nimber))
    elif possible:
        for region in gameObject.game:
            mapKeyAdd(trueEndingRegions, (region, gameObject.nimber))
            #if region == (3, 3435):
            #    print(gameObject.minMoves, gameObject.gameString)
            #    break

orderedPrint = []
for region, count in endingRegions.items():
    orderedPrint.append((region, count, trueEndingRegions[region]))
orderedPrint.sort()

for tup in orderedPrint:
    print(tup)

print()

for gameDat, count in otherVals.items():
    print(gameDat, count)

In [ ]:
Single-region fixed-move games:
0:
1:
3:!
23:!
12:
11:
-12:
-13:
222:
1,2:
1,1:
2425:
2,2,2,2:
2,2,2,2,2:
2,2,2,2,2,2:


(33,) 4707 610
(233,) 737 142
(2435,) 550 161
(333,) 3555 312
(223,) 99 58
(3333,) 1726 44
(3435,) 457 266
(33435,) 464 56
(344355,) 130 28
(13,) 802 245
(133,) 802 67
(1435,) 636 75
(1, 3) 99 56
(3, 333) 567 21
(3, 13) 266 63
(3, 3435) 96 9
(2, 23) 99 56
(-12, 3) 931 114
(33, 33) 497 45
(23, 33) 138 62
(1, 33) 85 45
(1, 23) 11 16
(3, 3, 33) 78 6
(1, 3, 3) 22 6
(23, 23) 0 28
(0, 3) 931 114
(2, 2, 3, 3) 0 28
(2, 2, 2, 3) 11 16
(2, 2, 2, 2, 3) 0 7

In [ ]:
<< 3 23,23   :!!
>> 3 2,2,3,3 :!!

<< 4 2435          | 23,23   :10!
>> 4 2435          | 2,2,3,3 :10!
<< 4 33|333        | 23,23   :230154!
>> 4 33|333        | 2,2,3,3 :230154!
<< 4 13            | 23,23   :10!
>> 4 13            | 2,2,3,3 :10!
<< 4 -12,3         | 23,23   :10!
>> 4 -12,3         | 2,2,3,3 :10!

<< 5 1435          | 23,23    :10!
>> 5 1435          | 2,2,3,3  :10!
<< 5 2435|2435     | 23,23    :2301
>> 5 2435|2435     | 2,2,3,3  :2301
<< 5 33|333|2435   | 23,23    :23016745
>> 5 33|333|2435   | 2,2,3,3  :23016745
<< 5 13|2435       | 23,23    :2301
>> 5 13|2435       | 2,2,3,3  :2301
<< 5 33|33|333|333 | 23,23    :457801A23B69
>> 5 33|33|333|333 | 2,2,3,3  :457801A23B69
<< 5 13|33|333     | 23,23    :63412705
>> 5 13|33|333     | 2,2,3,3  :63412705
<< 5 13|13         | 23,23    :2301
>> 5 13|13         | 2,2,3,3  :2301
<< 5 2435|-12,3    | 23,23    :2301
>> 5 2435|-12,3    | 2,2,3,3  :2301
<< 5 33|333|-12,3  | 23,23    :23016745
>> 5 33|333|-12,3  | 2,2,3,3  :23016745
<< 5 13|-12,3      | 23,23    :2301
>> 5 13|-12,3      | 2,2,3,3  :2301
<< 5 0,3           | 23,23    :10!
>> 5 0,3           | 2,2,3,3  :10!
<< 5 -12,3|-12,3   | 23,23    :2301
>> 5 -12,3|-12,3   | 2,2,3,3  :2301


<< 6 1435|2435     | 23,23   :2301
>> 6 1435|2435     | 2,2,3,3 :2301
<< 6 33|333|1435   | 23,23   :23016745
>> 6 33|333|1435   | 2,2,3,3 :23016745
<< 6 13|1435       | 23,23   :2301
>> 6 13|1435       | 2,2,3,3 :2301
<< 6 1435|-12,3    | 23,23   :2301
>> 6 1435|-12,3    | 2,2,3,3 :2301
<< 6 2435|0,3      | 23,23   :2301
>> 6 2435|0,3      | 2,2,3,3 :2301
<< 6 33|333|0,3    | 23,23   :23016745
>> 6 33|333|0,3    | 2,2,3,3 :23016745
<< 6 13|0,3        | 23,23   :2301
>> 6 13|0,3        | 2,2,3,3 :2301
<< 6 -12,3|0,3     | 23,23   :2301
>> 6 -12,3|0,3     | 2,2,3,3 :2301

<< 7 1435|1435     | 23,23    :2301
>> 7 1435|1435     | 2,2,3,3  :2301
<< 7 1435|0,3      | 23,23    :2301
>> 7 1435|0,3      | 2,2,3,3  :2301
<< 7 0,3|0,3       | 23,23    :2301
>> 7 0,3|0,3       | 2,2,3,3  :2301

In [ ]:
#@title Generate example winning games
winTupFreq = {}
for smartMoveType in ["random", "shortest", "longest"]:
    for i in range(1000):
        winPath = []
        generateWinPath(dataSummary, winPath, dataSummary.allGames['0,0,0,0,0,0:'], smartMoveType=smartMoveType)
        for pathTup in winPath:
            #pathTup = tuple(winPath)
            mapKeyAdd(winTupFreq, pathTup[:4] + (smartMoveType,))

winTupList = []
for winTup, freq in winTupFreq.items():
    winTupList.append((winTup,freq))
winTupList.sort(reverse=True)

for tup in winTupList:
    print(tup)

In [ ]:
#@title Generate example winning games
winTupFreq = {}
for i in range(10000):
    winPath = []
    generateWinPath(dataSummary, winPath, dataSummary.allGames['0,0,0,0,0,0:'], smartMoveType="long")
    pathTup = tuple(winPath)
    mapKeyAdd(winTupFreq, len(pathTup))

winTupList = []
for winTup, freq in winTupFreq.items():
    winTupList.append((winTup,freq))
winTupList.sort(reverse=True)

for tup in winTupList:
    print(tup)

'''
Short:
(16, 64)
(14, 9853)
(12, 83)

Long:
(16, 145)
(14, 9834)
(12, 21)

Random:
(16, 86)
(14, 9867)
(12, 47)'''

In [ ]:
initialSubGames = []
for subGame in dataSummary.allGames['1425|0,2:'].subGames:
    initialSubGames.append(dataSummary.allGames[subGame])

childGames = generateNonUnifiedMoves(dataSummary, initialSubGames)

for childData in childGames:
    (nimber, minMoves, maxMoves, totalConnections, newGame) = childData
    newGameStringArray = []
    for subGame in newGame:
        newGameStringArray.append(subGame.gameString)
    print(childData[:-1], newGameStringArray)



In [ ]:
for game in dataSummary.allGames.values():
    if len(game.subGames) > 1:
        print(game.gameString, game.subGames)

#Move Pattern checks

In [ ]:
for gameString in dataSummary.allGames:
    if dataSummary.allGames[gameString].minMoves == 7 and dataSummary.allGames[gameString].maxMoves == 9 and dataSummary.allGames[gameString].totalConnections == 13:
        dataSummary.allGames[gameString].printReadableData()
        print()

In [ ]:
dat = {}
ordered = []

for gameString, gameObject in dataSummary.allGames.items():
    objects = 1
    lonelyCount = 0
    tripleCount = 0
    disaCount = 0
    for region in gameObject.game:
        objects += len(region) - 1
        for loop in region:
            if loop == -12:
                lonelyCount += 1
            elif loop == -13:
                tripleCount += 1
    for boundary in gameObject.boundaries:
        if gameObject.boundaries[boundary] == DISAPOINT:
            disaCount += 1
    held = (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections, objects, len(gameObject.game), lonelyCount, tripleCount, disaCount)
    if held not in dat:
        dat[held] = 0
        ordered.append(held)
    mapKeyAdd(dat, held)
    if held == (10, 14, 15, 3, 1, 0, 0, 0):
        gameObject.printReadableData()

ordered.sort()
for val in ordered:
    #print(val, dat[val])
    pass


In [ ]:
heldColumn = {}
heldColumn[(-1, 1, 1)] = 2
heldColumn[(-1, 1, 2)] = 3
heldColumn[(-1, 1, 3)] = 4
heldColumn[(-1, 2, 1)] = 5
heldColumn[(-1, 2, 2)] = 6
heldColumn[(-1, 2, 3)] = 7
heldColumn[(0, 1, 1)] = 8
heldColumn[(0, 1, 2)] = 9
heldColumn[(0, 1, 3)] = 10
heldColumn[(0, 2, 1)] = 11
heldColumn[(0, 2, 2)] = 12
heldColumn[(0, 2, 3)] = 13
heldColumn[(1, 1, 1)] = 14
heldColumn[(1, 1, 2)] = 15
heldColumn[(1, 1, 3)] = 16
heldColumn[(1, 2, 1)] = 17
heldColumn[(1, 2, 2)] = 18
heldColumn[(1, 2, 3)] = 19

dat = {}
ordered = []
allHeld = {}

for gameString, gameObject in dataSummary.allGames.items():
    for childGame in gameObject.childGames:
        if childGame.gameString == EMPTYGAME:
            continue
        decodedMoves = decodeMoveArray(gameObject.childGames[childGame]).split(' ; ')
        for move in decodedMoves:
            held = (gameObject.minMoves - childGame.minMoves, gameObject.maxMoves - childGame.maxMoves, gameObject.totalConnections - childGame.totalConnections)
            allHeld[held] = True
            winningness = 'Good'
            if gameObject.nimber!=0 and childGame.nimber!=0:
                winningness = 'Bad'
            elif gameObject.nimber==0:
                winningness = 'Losing'

            move = (move, winningness)
            if move not in dat:
                dat[move] = {}
                ordered.append(move)
            #if held == (-1, 2, 2) and move == "Enclosure:  boundary point to boundary":
            #    gameObject.printReadableData()
            mapKeyAdd(dat[move], held)
            #if held == (0, 1, 1) and move == 'Enclosure:  Free point to self, empty enclosure         ':
            #    print(gameObject.gameString)
            #    print(childGame.gameString)
            #    gameObject.printReadableData()
            #    print()

ordered.sort()

with open(root_path + "/sproutsMoveMetadata1.csv", 'w') as outFile:
    headerLine = [''] * 20
    headerLine[0] = 'MoveType'
    headerLine[1] = 'Severings'
    for key, val in heldColumn.items():
        headerLine[val] = str(key).replace(',', ';')
    print(','.join(headerLine), file=outFile)

    for move in ordered:
        outString = ['0']*20
        outString[0] = move[0].replace(',', ';')
        outString[1] = str(move[1]).replace(',', ';')
        for key, val in dat[move].items():
            outString[heldColumn[key]] = str(val)
        print(','.join(outString), file=outFile)





In [ ]:
heldColumn = {}

heldColumn[(-1, 1, 1)] = 1
heldColumn[(-1, 1, 2)] = 2
heldColumn[(-1, 1, 3)] = 3
heldColumn[(-1, 2, 1)] = 4
heldColumn[(-1, 2, 2)] = 5
heldColumn[(0, 1, 1)] = 6
heldColumn[(0, 1, 2)] = 7
heldColumn[(0, 1, 3)] = 8
heldColumn[(0, 2, 1)] = 9
heldColumn[(0, 2, 2)] = 10
heldColumn[(0, 2, 3)] = 11
heldColumn[(1, 1, 1)] = 12
heldColumn[(1, 1, 2)] = 13
heldColumn[(1, 1, 3)] = 14
heldColumn[(1, 2, 1)] = 15
heldColumn[(1, 2, 2)] = 16
heldColumn[(1, 2, 3)] = 17

dat = {}
ordered = []
allHeld = {}

for gameString, gameObject in dataSummary.allGames.items():
    for childGame in gameObject.childGames:
        if childGame.gameString == EMPTYGAME:
            continue
        decodedMoves = decodeMoveArray(gameObject.childGames[childGame]).split(' ; ')
        for move in decodedMoves:
            held = (gameObject.minMoves - childGame.minMoves, gameObject.maxMoves - childGame.maxMoves, gameObject.totalConnections - childGame.totalConnections)
            allHeld[held] = True
            if move not in dat:
                dat[move] = {}
                ordered.append(move)
            #if held == (-1, 2, 2) and move == "Enclosure:  boundary point to boundary":
            #    gameObject.printReadableData()
            mapKeyAdd(dat[move], held)
            #if held == (0, 1, 1) and move == 'Enclosure:  Free point to self, empty enclosure         ':
            #    print(gameObject.gameString)
            #    print(childGame.gameString)
            #    gameObject.printReadableData()
            #    print()

ordered.sort()

countCount = {}

with open(root_path + "/sproutsMoveMetadata2.csv", 'w') as outFile:
    headerLine = [''] * 18
    headerLine[0] = 'MoveType'
    #headerLine[1] = 'Severings'
    for key, val in heldColumn.items():
        headerLine[val] = str(key).replace(',', ';')
    print(','.join(headerLine), file=outFile)

    for move in ordered:
        outString = ['0']*18
        outString[0] = move.replace(',', ';')
        #outString[1] = str(move[1])
        for key, val in dat[move].items():
            outString[heldColumn[key]] = str(val)
            mapKeyAdd(countCount, val)
        print(','.join(outString), file=outFile)

'''
for move in ordered:
    for key, val in dat[move].items():
        if val in [4031, 88, 556, 350, 60, 108, 180, 58, 76, 2001, 3042]:
            print(move, key, val)
'''

#for key, val in countCount.items():
#    if val > 1:
#        print(key, val)

In [ ]:
count = 0

for gameObject in dataSummary.allGames.values():
    for region in gameObject.game:
        if region == (2,33):
            count += 1

print(count)

In [ ]:
orderLinesA = []
orderLinesB = []

stringA = 'Merge:  singleton point to singleton'
tupA = (-1, 1, 1)

stringB = ''
tupB = (1, 2, 2)

chiPar = {}
for gameString, gameObject in dataSummary.allGames.items():
    #if len(orderLinesA) > 20 and len(orderLinesB) > 20:
    #    break
    gsnim = gameString + "=" + str(gameObject.nimber)
    for childGame in gameObject.childGames:
        if childGame.gameString == EMPTYGAME:
            continue
        decodedMoves = decodeMoveArray(gameObject.childGames[childGame]).split(' ; ')
        for move in decodedMoves:
            held = (gameObject.minMoves - childGame.minMoves, gameObject.maxMoves - childGame.maxMoves, gameObject.totalConnections - childGame.totalConnections)
            dat = (str(gameObject.nimber), str(childGame.nimber), str(gameObject.minMoves), str(gameObject.maxMoves), gameObject.gameString, "-->", childGame.gameString)
            if held == tupA and move == stringA: # and len(orderLinesA) <= 20:
                if gsnim not in chiPar:
                    chiPar[gsnim] = {}
                if "A" not in chiPar[gsnim]:
                    chiPar[gsnim]["A"] = []
                chiPar[gsnim]["A"].append(childGame.gameString + "=" + str(childGame.nimber))
                orderLinesA.append(dat)
            if held == tupB and move == stringB: # and len(orderLinesB) <= 20:
                if gsnim not in chiPar:
                    chiPar[gsnim] = {}
                if "B" not in chiPar[gsnim]:
                    chiPar[gsnim]["B"] = []
                chiPar[gsnim]["B"].append(childGame.gameString + "=" + str(childGame.nimber))
                orderLinesB.append(dat)

orderLinesA.sort()
print(tupA, stringA)
for line in orderLinesA:
    print(" ".join(line))

orderLinesB.sort()
print(tupB, stringB)
for line in orderLinesB:
    print(" ".join(line))

#for val in chiPar:
#    if len(chiPar[val]) > 1:
#        for chi in chiPar[val]:
#            print(chi, val, chiPar[val][chi])

#for val in chiPar:
#    for chi in chiPar[val]:
#        if len(chiPar[val][chi]) > 1:
#            print(chi, val, chiPar[val][chi])


In [ ]:
orderLinesA = []
orderLinesB = []

for gameString, gameObject in dataSummary.allGames.items():
    if len(orderLinesA) > 18 and len(orderLinesB) > 18:
        break
    for childGame in gameObject.childGames:
        if len(orderLinesA) > 18 and len(orderLinesB) > 18:
            break
        if childGame.gameString == EMPTYGAME:
            continue
        decodedMoves = decodeMoveArray(gameObject.childGames[childGame]).split(' ; ')
        for move in decodedMoves:
            held = (gameObject.minMoves - childGame.minMoves, gameObject.maxMoves - childGame.maxMoves, gameObject.totalConnections - childGame.totalConnections)
            dat = (gameObject.nimber, childGame.nimber, gameObject.minMoves, gameObject.maxMoves, gameObject.gameString, "-->", childGame.gameString)
            if held == (0, 1, 2) and move == 'Enclosure:  hanging point to boundary, two-point loop, no other loops in region' and len(orderLinesA) < 20:
                orderLinesA.append(dat+('  ;  ', "0,1,2,A  ;  "))
            if held == (0, 1, 2) and move == 'Disappearing point removed' and len(orderLinesB) < 20:
                orderLinesB.append(dat+('  ;  ', "0,1,2,B  ;  "))
            #if held == (-1, 1, 2) and move == 'Merge:  hanging point to boundary':
            #    orderLines.append(dat+('  ;  ', "-1,1,2,B  ;  "))

orderLinesA.sort()
orderLinesB.sort()
for line in orderLinesA:
    print(line[-1], line[:-1])
for line in orderLinesB:
    print(line[-1], line[:-1])

In [ ]:
-1,1,2,A  ;   (1, 0, 5, 8, '3,12     | 1435,2425:10', '-->', '12|1442255:', '  ;  ')
-1,1,2,A  ;   (1, 0, 5, 8, '3,2425   | 1435,2425:10', '-->', '2425|1442255:', '  ;  ')
-1,1,2,A  ;   (2, 1, 4, 7, '33       | 1435,2425:!21', '-->', '3|1442255:!', '  ;  ')
-1,1,2,A  ;   (3, 0, 5, 8, '13|33    | 1435,2425:1032', '-->', '12|1442255:', '  ;  ')
-1,1,2,A  ;   (3, 0, 5, 8, '333|3333 | 1435,2425:34501276', '-->', '-13|1442255:', '  ;  ')
-1,1,2,A  ;   (3, 0, 5, 8, '33|2435  | 1435,2425:2301', '-->', '2425|1442255:', '  ;  ')
-1,1,2,A  ;   (0, 1, 6, 9, '113      | 1435,2425:10', '-->', '11|1442255:', '  ;  ')
-1,1,2,A  ;   (1, 0, 5, 8, '-13,3    | 1435,2425:10', '-->', '-13|1442255:', '  ;  ')
-1,1,2,A  ;   (1, 0, 5, 8, '123      | 1435,2425:10', '-->', '12|1442255:', '  ;  ')
-1,1,2,A  ;   (1, 0, 5, 8, '2,13     | 1435,2425:10', '-->', '1442255|1,2:', '  ;  ')
-1,1,2,A  ;   (1, 0, 5, 8, '244355   | 1435,2425:10', '-->', '2425|1442255:', '  ;  ')

-1,1,2,B  ;   (1, 0, 5, 8, '123      | 12,1435:10', '-->', '12|14442555:', '  ;  ')
-1,1,2,B  ;   (1, 0, 5, 8, '2,13     | 12,1435:10', '-->', '14442555|1,2:', '  ;  ')
-1,1,2,B  ;   (1, 0, 5, 8, '244355   | 12,1435:10', '-->', '2425|14442555:', '  ;  ')
-1,1,2,B  ;   (1, 0, 5, 8, '-13,3    | 12,1435:10', '-->', '-13|14442555:', '  ;  ')
-1,1,2,B  ;   (1, 0, 5, 8, '3,12     | 12,1435:10', '-->', '12|14442555:', '  ;  ')
-1,1,2,B  ;   (0, 1, 6, 9, '113      | 12,1435:10', '-->', '11|14442555:', '  ;  ')
-1,1,2,B  ;   (1, 0, 5, 8, '3,2425   | 12,1435:10', '-->', '2425|14442555:', '  ;  ')
-1,1,2,B  ;   (5, 0, 5, 8, '333|3333 | 12,1435:34501276', '-->', '-13|14442555:', '  ;  ')
-1,1,2,B  ;   (6, 1, 4, 7, '33       | 12,1435:!21', '-->', '3|14442555:!', '  ;  ')
-1,1,2,B  ;   (7, 0, 5, 8, '13|33    | 12,1435:1032', '-->', '12|14442555:', '  ;  ')
-1,1,2,B  ;   (7, 0, 5, 8, '33|2435  | 12,1435:2301', '-->', '2425|14442555:', '  ;  ')







0,1,2,A  ;   (0, 1, 3, 4, '13|22435:10', '-->', '-12|222:', '  ;  ')
0,1,2,A  ;   (2, 1, 3, 4, '13|2223:10', '-->', '-12|222:', '  ;  ')
0,1,2,A  ;   (2, 1, 3, 4, '13|33|223:1032', '-->', '-12|222:', '  ;  ')


0,1,2,B  ;   (1, 0, 2, 3, '22435:!', '-->', '222:', '  ;  ')
0,1,2,B  ;   (3, 0, 2, 3, '2223:!', '-->', '222:', '  ;  ')
0,1,2,B  ;   (3, 0, 2, 3, '33|223:!21', '-->', '222:', '  ;  ')



0,1,2,A  ;   (0, 1, 3, 4, '13|233|2333:345012', '-->', '-12|233|233:2301', '  ;  ')
0,1,2,A  ;   (0, 1, 3, 4, '13|233|2333:435102', '-->', '-12|233|233:2301', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '233|2333:2301!', '-->', '233|233:2301', '  ;  ')


0,1,2,A  ;   (0, 1, 3, 4, '13|23435:10!', '-->', '-12|2435:!', '  ;  ')
0,1,2,A  ;   (0, 1, 3, 4, '13|24335:10!', '-->', '-12|2435:!', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '23435:!!', '-->', '2435:!', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '24335:!!', '-->', '2435:!', '  ;  ')


0,1,2,A  ;   (0, 1, 3, 4, '13|244355:10', '-->', '-12|2425:', '  ;  ')
0,1,2,A  ;   (2, 1, 3, 4, '13|23425:10', '-->', '-12|2425:', '  ;  ')


0,1,2,A  ;   (0, 1, 3, 4, '13|333|333:104523!', '-->', '-12|33|333:2301!', '  ;  ')
0,1,2,A  ;   (0, 1, 3, 4, '13|33|233|333:10567234', '-->', '-12|233|233:2301', '  ;  ')
0,1,2,A  ;   (0, 1, 3, 4, '13|33|3333:345012!', '-->', '-12|33|333:2301!', '  ;  ')
0,1,2,A  ;   (1, 0, 4, 6, '13|234442555:10', '-->', '-12|24442555:', '  ;  ')
0,1,2,A  ;   (1, 0, 4, 6, '13|33|24433355:345012', '-->', '-12|33|2443355:2301', '  ;  ')
0,1,2,A  ;   (2, 1, 3, 4, '13|333|3333:45670123', '-->', '-13|-12:', '  ;  ')
0,1,2,A  ;   (2, 1, 3, 4, '13|33|2435:1032', '-->', '-12|2425:', '  ;  ')
0,1,2,A  ;   (3, 0, 2, 3, '13|33:10!', '-->', '-12|3:!', '  ;  ')
0,1,2,A  ;   (5, 2, 3, 5, '13|33|33|2333:10567234', '-->', '-12|33|2233:2301', '  ;  ')
0,1,2,A  ;   (5, 3, 3, 5, '13|33|33|2333:10657324', '-->', '-12|33|2323:2301', '  ;  ')
0,1,2,B  ;   (0, 1, 3, 5, '234442555:!', '-->', '24442555:', '  ;  ')
0,1,2,B  ;   (0, 1, 3, 5, '33|23443553:240!1', '-->', '33|234253:2301', '  ;  ')
0,1,2,B  ;   (0, 1, 3, 5, '33|334443555:2301!', '-->', '33|2443355:2301', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '23435:!!', '-->', '223:!', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '244355:!', '-->', '2425:', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '33|233|333:240513!', '-->', '33|33|233:240513', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '33|33|2333:240513!', '-->', '33|33|233:240513', '  ;  ')
0,1,2,B  ;   (1, 0, 2, 3, '33|33|2333:24061!3', '-->', '33|33|233:240513', '  ;  ')
0,1,2,B  ;   (1, 3, 2, 4, '33|33|33435:!645231', '-->', '33|24335:2301', '  ;  ')
0,1,2,B  ;   (2, 1, 1, 2, '33:!!', '-->', '3:!', '  ;  ')
0,1,2,B  ;   (3, 0, 2, 3, '23425:!', '-->', '2425:', '  ;  ')
0,1,2,B  ;   (3, 0, 2, 3, '333|3333:345012!', '-->', '-13:', '  ;  ')
0,1,2,B  ;   (3, 0, 2, 3, '33|2435:!21', '-->', '2425:', '  ;  ')
0,1,2,B  ;   (4, 2, 2, 4, '33|33|2333:!546213', '-->', '33|2323:2301', '  ;  ')

In [ ]:
(0, 1, 1) Enclosure:  singleton point to singleton, non-adjacent points connected, no other loops in region
0 1 3 5 2222425: --> 33|22233:2301
0 1 3 5 2224225: --> 33|22323:2301
0 1 3 5 22244255: --> 333|22333:345012
0 1 3 5 22424255: --> 333|23233:345012
0 1 3 5 22424255: --> 33|224335:2301
0 1 3 5 22424255: --> 33|234253:2301
0 1 3 5 22425425: --> 333|22333:345012
0 1 3 5 22425425: --> 33|233425:2301
0 1 3 5 22442255: --> 333|23233:345012
0 1 3 5 224442555: --> 3333|23333:45670123
0 2 3 5 122425: --> 33|1233:2301
0 2 3 5 124252: --> 33|1233:2301
0 2 3 5 1244255: --> 333|1333:345012
0 2 3 5 1424255: --> 33|14335:2301
0 2 3 5 1425425: --> 333|1333:345012
0 4 3 5 124225: --> 33|1323:2301
2 0 2 3 2323:!! --> 33|33:!21!
2 0 2 3 233|2323:2301 --> 33|33|233:240513
2 0 2 3 33|33|2323:240513 --> 33|33|33|33:24061735
3 0 2 3 23425:! --> 33|333:2301!
3 0 2 3 244255: --> -13:
(1, 2, 1) Enclosure:  hanging point to split, non-adjacent points connected, enclosed loops on both sides
0 3 5 7 2,2,144155: --> 2,333|2,1333:345012
1 0 4 7 33|2,2,133425:2301 --> 33|2,233|2,3333:45670123
1 2 4 6 2,2,12435:! --> 2,233|2,333:2301!
1 2 4 6 2,2,13425:! --> 2,233|2,333:2301!
1 2 4 6 2,2,144355:! --> 2,333|2,3333:345012!
1 2 4 6 2,3,12425:! --> 2,233|3,233:34!01
1 4 4 6 2,2,144255: --> 2,333|2,2333:345012
1 4 4 6 2,3,144255:! --> 2,333|3,2333:456!012
2 0 4 7 2,2,1244255: --> 2,333|2,23233:345012
2 0 4 7 33|2,2,124335:2301 --> 33|2,233|2,3333:45670123
2 3 4 7 -12,2,144255: --> -12,2333|2,333:345012
2 3 5 7 1,2,144255: --> 1,2333|2,333:345012
2 5 4 7 33|2,2,134253:2301 --> 33|2,333|2,2333:25067134
2 5 5 7 1,2,144255: --> 1,333|2,2333:345012
4 0 4 7 2,2,2,144255: --> 2,333|2,2,2333:345012
4 3 4 7 2,2,1442255: --> 2,333|2,22333:345012
4 3 4 7 2,2,14442555: --> 2,3333|2,23333:45670123
4 3 4 7 2,2,14442555: --> 2,333|2,243335:345012
5 2 4 6 2,2,12425: --> 2,233|2,233:2301
6 3 4 7 2,2,1424255: --> 2,333|2,22333:345012
6 4 4 7 -12,2,12425: --> -12,233|2,233:2301

#Smart Game tests

In [ ]:
metDat = {}
allDests = {}

for line in dataSummary.smartTree:
    nimber = dataSummary.allGames[line].nimber
    state = "Winning"
    if nimber == 0:
        state = " Losing"

    curGame = dataSummary.allGames[line]
    if len(curGame.subGames) > 1:
        print(state, "choice: ", line, ": Split = ", curGame.subGames)
        mapKeyAdd(metDat, srcDat, binaryMapAddition = "split")
    else:
        srcDat = (curGame.minMoves, curGame.maxMoves, state)
        #srcDat = (curGame.totalConnections, curGame.minMoves, curGame.maxMoves, curGame.nimber)
        print(state, "choice: ", line, curGame.totalConnections, curGame.minMoves, curGame.maxMoves, curGame.nimber)
        for goodChoice in curSmart[line]:
            goodGame = dataSummary.allGames[goodChoice]
            dstDat = (goodGame.minMoves, goodGame.maxMoves)
            #dstDat = (goodGame.maxMoves, goodGame.minMoves, goodGame.totalConnections, goodGame.nimber)
            #dstDat = (goodGame.maxMoves, goodGame.minMoves, goodGame.totalConnections, goodGame.nimber)
            print("                    ", goodChoice, goodGame.totalConnections, goodGame.minMoves, goodGame.maxMoves, goodGame.nimber)
            mapKeyAdd(metDat, srcDat, binaryMapAddition = dstDat)
            allDests[dstDat] = True

orderedDests = []
for dstDat in allDests:
    orderedDests.append(dstDat)
orderedDests.sort(reverse=True)

metDatOrdered = []
for curKey in metDat:
    metDatOrdered.append(curKey)
metDatOrdered.sort()

for srcDat in metDatOrdered:
    dests = []
    for dstDat in metDat[srcDat]:
        dests.append(dstDat)
    print(srcDat, dests)

for dst in orderedDests:
    print(" ", dst)


'''
cleverData = [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
for curGame in curSmart:
    curObject = dataSummary.allGames[curGame]
    cleverData[curObject.totalConnections].append((curObject.nimber, len(curSmart[curGame]), curGame))

gameSizes = {}
for i in range(len(cleverData)):
    for j in range(len(cleverData[i])):
        (nimber, viableChilds, curGame) = cleverData[i][j]
        gameObject = dataSummary.allGames[curGame]
        if nimber != 0:
            sizeOfSubGameChoices = []
            for childGame in gameObject.childGames:
                if childGame.nimber == 0:
                    sizeOfSubGameChoices.append(gameSizes[curGame]])
                    subGames[childGame.gameString] = gameSizes[curGame]

                    secondHop = childGame.childGames

                    for subGame in secondHop:
                        if subGame.nimber != 0:
                            subGames[subGame.gameString] = True
                            print('!!!', curGame, childGame.gameString, subGame.gameString)
            #print(curGame, viableChilds, len(subGames))
            gameSizes[curGame] = len(subGames)
        else:
            gameSizes[curGame] = viableChilds
            #print(curGame, viableChilds)'''
pass

In [ ]:
#@title Min/max connections with good/bad/null links and winning-game-tree options
curSmart = dataSummary.smartTree

srcOrdered = []
inDest = {}
destOrdered = {}
minMaxFromTo = {}
for gameString in dataSummary.allGames:
    gameObject = dataSummary.allGames[gameString]
    src = (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections)
    srcNim = gameObject.nimber
    if src not in minMaxFromTo:
        minMaxFromTo[src] = {}
        srcOrdered.append(src)
        destOrdered[src] = []

    for childGame in gameObject.childGames:
        dest = (childGame.minMoves, childGame.maxMoves,  childGame.totalConnections)
        destNim = childGame.nimber

        moveType = ((srcNim != 0 and destNim != 0)*1) + ((srcNim == 0)*2)

        if gameString in curSmart:
            moveType += 3

        moveType = chr(moveType+97)
        if dest not in minMaxFromTo[src]:
            minMaxFromTo[src][dest] = {}
            destOrdered[src].append(dest)
        mapKeyAdd(minMaxFromTo[src][dest], moveType)

srcOrdered.sort()
for src in destOrdered:
    destOrdered[src].sort()

for src in srcOrdered:
    for dest in destOrdered[src]:
        line = str(src) + '\t' + str(dest) + '\t('
        size = 0
        allMoves = []
        for moveType in minMaxFromTo[src][dest]:
            allMoves.append(moveType)
            size +=  minMaxFromTo[src][dest][moveType]
        allMoves.sort()
        print(str(src) + '\t' + str(dest) + '\t' + ''.join(allMoves) + '\t' + str(size))

In [ ]:
def quickSmartTree(smartTree = None, gameString = None, printAllLines = True):
    if smartTree is None:
        if gameString is None:
            print("Pre-built smart tree or a gameString to generate from one required for printSmartTree")
            return
        else:
            smartTree = createSmartTree(gameString)

    edges = 0
    for curGameString, childGameList in smartTree.items():
        keyGame = dataSummary.allGames[curGameString]
        for childGameString in childGameList:
            childGame = dataSummary.allGames[childGameString]
            keyDat   = (  keyGame.nimber,   keyGame.totalConnections,   keyGame.minMoves,   keyGame.maxMoves)
            childDat = (childGame.nimber, childGame.totalConnections, childGame.minMoves, childGame.maxMoves)
            difs     = (  keyGame.totalConnections - childGame.totalConnections,   keyGame.minMoves - childGame.minMoves,   keyGame.maxMoves - childGame.maxMoves)
            if printAllLines:
                print(str(keyGame.nimber) + "\t" + str(childGame.nimber) + "\t" + str(difs) + "\t" + str(keyDat) + " " + curGameString + "\t" + str(childDat) + " " + childGameString)
            edges += 1

        for childGameString in childGameList:
            print(len(smartTree), edges)

    return smartTree

printSmartTree(dataSummary.smartTree, "0,0,14225:")

In [ ]:
def preparePrintLine(allData, gameDat, fieldOrdering):
    metaDat = ''
    for fieldType in fieldOrdering:
        if fieldType == 'nimber':
            metaDat += str(gameDat.nimber) + ' '
        elif fieldType == 'totalC':
            metaDat += str(gameDat.totalConnections) + ' '
        elif fieldType == 'min':
            metaDat += str(gameDat.minMoves) + ' '
        elif fieldType == 'max':
            metaDat += str(gameDat.maxMoves) + ' '
        elif fieldType == 'gameString':
            metaDat += str(gameDat.gameString) + ' '
        else:
            print("Field not recognized:"  )
    metaDat = metaDat[:-1]

    return (metaDat)


def createGraphFromGame(gameMapTuptup, curGame):
    if len(curGame.subGames) > 1:
        gameMapTuptup[(curGame.gameString, "split")] = True
        return

    for childGame in curGame.childGames:
        if (curGame.gameString, childGame.gameString) not in gameMapTuptup:
            gameMapTuptup[(curGame.gameString, childGame.gameString)] = True
            createGraphFromGame(gameMapTuptup, childGame)


optimizeGame = "0,0,14225:"#@param {type:"string"}

fieldOrder = ["totalC", "min", "max", "nimber", "gameString"]

smartTree = printSmartTree(gameString = optimizeGame, printAllLines = False)

startGame = dataSummary.allGames[optimizeGame]
gameSubgraphTuptup = {}
createGraphFromGame(gameSubgraphTuptup, startGame)

generalCounts = {}
generalCounts['None'] = 0
generalCounts['Good'] = 0
generalCounts['Bad'] = 0
generalCounts['Lose'] = 0
generalCounts['SplitW'] = 0
generalCounts['SplitL'] = 0

orderedLines = []

#with open(root_path + "/networkExample.txt", 'w') as outFile:
if True:
    for tup in gameSubgraphTuptup:
        (src, dst) = tup
        srcGame = dataSummary.allGames[src]
        srcDat  = (srcGame.maxMoves, srcGame.minMoves, srcGame.totalConnections, srcGame.nimber)
        srcOut = preparePrintLine(dataSummary, srcGame, fieldOrder)

        dstOut = 'split'
        if dst != 'split':
            dstGame = dataSummary.allGames[dst]
            dstDat  = (dstGame.maxMoves, dstGame.minMoves, dstGame.totalConnections, dstGame.nimber)
            dstOut = preparePrintLine(dataSummary, dstGame, fieldOrder)
            dstOut += '\t' + decodeMoveArray(dataSummary.allGames[src].childGames[dataSummary.allGames[dst]])

        lineType = 'None'
        if src in smartTree:
            if srcGame.nimber == 0:
                lineType = 'Lose'
                if len(srcGame.subGames) > 1:
                    lineType = 'SplitL'
            else:
                if len(srcGame.subGames) > 1:
                    lineType = 'SplitW'
                elif dstGame.nimber == 0:
                    lineType = 'Good'
                else:
                    lineType = 'Bad'
        if lineType == 'Good' or lineType == 'Bad':
            print(lineType + '\t' + srcOut + '\t' + dstOut)#, file = outFile)
        generalCounts[lineType] += 1

print(len(gameSubgraphTuptup))
print(generalCounts)

pass

In [ ]:
loses = 0
wins = 0
totalConnectionSet = {}
for curGame in dataSummary.allGames:
    gameObject = dataSummary.allGames[curGame]
    if gameObject.minMoves == 2 and gameObject.maxMoves == 3:
        if gameObject.nimber == 0:
            dataSummary.allGames[curGame].printReadableData()
            mapKeyAdd(totalConnectionSet, gameObject.totalConnections)
            #print()
            loses += 1
        else:
            wins += 1
        #dataSummary.allGames[curGame].printReadableData()
        #print()
        #print()
print(loses, wins, totalConnectionSet)

In [ ]:
allConnects = {}
connectPairingMap = {}
for curGame in dataSummary.allGames:
    gameObject = dataSummary.allGames[curGame]
    if len(gameObject.subGames) > 1:
        continue
    minMax = (gameObject.minMoves, gameObject.maxMoves)

    curConnects = {}
    for nextGameObject in gameObject.childGames:
        curConnects[(nextGameObject.minMoves, nextGameObject.maxMoves, gameObject.minMoves-nextGameObject.minMoves, gameObject.maxMoves-nextGameObject.maxMoves)] = True
    curArray = []
    moveSpecialVal = 0
    for connect in curConnects:
        curArray.append(connect)
        if connect[-2:] == (1,1):
            moveSpecialVal += 1
        elif connect[-2:] == (1,2):
            moveSpecialVal += 2
        elif connect[-2:] == (0,1):
            moveSpecialVal += 4
        elif connect[-2:] == (0,2):
            moveSpecialVal += 8
        elif connect[-2:] == (-1,1):
            moveSpecialVal += 16

    curArray.sort()
    specialTup = (minMax, moveSpecialVal)
    mapKeyAdd(allConnects, specialTup)
    #if specialTup == 9:
    #    gameObject.printReadableData()
    if specialTup[0] == (4, 8):
        gameObject.printReadableData()

#for connect in allConnects:
#    connectPairingMap

orderThis = []
for connect in allConnects:
    orderThis.append((connect, allConnects[connect]))
orderThis.sort()

for connect in orderThis:
    print(connect)

In [ ]:
#@title close
allConnects = {}
sumConnects = {}
for curGame in dataSummary.allGames:
    gameObject = dataSummary.allGames[curGame]
    minMax = (gameObject.minMoves, gameObject.maxMoves)

    disaCount = 0
    lonely2Count = 0
    lonely3Count = 0

    for charPos in range(len(gameObject.gameString)):
        if gameObject.gameString[charPos] == '!':
            disaCount += 1

    for regionIndex in range(len(gameObject.game)):
        for loopIndex in range(len(gameObject.game[regionIndex])):
            loopVal = gameObject.game[regionIndex][loopIndex]
            if loopVal == LONELYBOUNDARYPAIR:
                lonely2Count += 1
            elif loopVal == LONELYBOUNDARYTRIPLE:
                lonely3Count += 1
        if gameObject.gameString[charPos] == '!':
            disaCount += 1

    regionCount = disaCount + lonely2Count + lonely3Count + len(gameObject.game)

    mapKeyAdd(allConnects, (regionCount, gameObject.minMoves, gameObject.maxMoves, gameObject.nimber))
    mapKeyAdd(sumConnects, (regionCount, gameObject.minMoves, gameObject.maxMoves))

    if (regionCount, gameObject.minMoves, gameObject.maxMoves, gameObject.nimber) == (3, 6, 7, 2):
        dataSummary.allGames[curGame].printReadableData()
        print()


    #for nextGameObject in gameObject.childGames:
    #    allConnects[(gameObject.minMoves, gameObject.maxMoves, nextGameObject.minMoves, nextGameObject.maxMoves, disaCount)] = True

orderThis = []
for connect in allConnects:
    orderThis.append((connect, allConnects[connect], sumConnects[connect[:3]]))
orderThis.sort()

for connect in orderThis:
    print(connect)

In [ ]:
allConnects = {}
allGood = {}
allBad  = {}
winnableDisas = {}
losingDisa = {}
for curGame in dataSummary.allGames:
    gameObject = dataSummary.allGames[curGame]
    minMax = (gameObject.minMoves, gameObject.maxMoves)

    regionCount = len(gameObject.game)
    for charPos in range(len(gameObject.gameString)):
        if gameObject.gameString[charPos] == '!':
            regionCount += 1
    if gameObject.minMoves == 6 and gameObject.maxMoves == 10:
        if gameObject.nimber == 0:
            mapKeyAdd(losingDisa, regionCount)
        else:
            mapKeyAdd(winnableDisas, regionCount)
        #print(curGame, regionCount)

    curConnects = {}
    for nextGameObject in gameObject.childGames:
        for connectionType in gameObject.childGames[nextGameObject]:
            if gameObject.minMoves == gameObject.maxMoves or nextGameObject.minMoves == nextGameObject.maxMoves:
                continue
            #print(gameObject.gameString, nextGameObject.gameString)
            #a = (decodeMoveData(connectionType), gameObject.minMoves - nextGameObject.minMoves,  gameObject.maxMoves - nextGameObject.maxMoves)
            #if not(decodeMoveData(connectionType).split(":")[0] == 'Merge' and gameObject.minMoves == 6 and gameObject.maxMoves == 10):
            if not(decodeMoveData(connectionType) == 'Enclosure:  Free point to self'):
                continue
            a = (decodeMoveData(connectionType), gameObject.minMoves, gameObject.maxMoves, regionCount)
            mapKeyAdd(allConnects, a)
            if gameObject.nimber != 0:
                if nextGameObject.nimber == 0:
                    mapKeyAdd(allGood, a)
                else:
                    mapKeyAdd(allBad , a)
            #if a == ('singleton point to split', -1, 2):
            #    dataSummary.allGames[curGame].printReadableData()

orderThis = []
for connect in allConnects:
    orderThis.append((connect, allConnects[connect]))
orderThis.sort()

for connect in orderThis:
    if True:
    #if (connect[0] in allGood and connect[0] not in allBad) or (connect[0] not in allGood and connect[0] in allBad):
        print(connect)
        if connect[0] in allGood:
            print('    allGood:', allGood[connect[0]])
        else:
            print('    allGood: 0')
        if connect[0] in allBad :
            print('    allBad :', allBad [connect[0]])
        else:
            print('    allBad : 0')

print(winnableDisas)
print(losingDisa)

#Other Tests

In [ ]:
#@title List games where min=max where totalConnections is a different value
tupCount = {}
for gameObject in dataSummary.allGames.values():
    '''for childObject in gameObject.childGames:
        print(childObject.gameString)
        if childObject.gameString == '23:!':
            print(gameObject.gameString)'''
    if gameObject.minMoves == gameObject.maxMoves:
        if gameObject.maxMoves != gameObject.totalConnections:
            tup = (gameObject.minMoves, gameObject.maxMoves, gameObject.totalConnections)
            if tup == (4,4,8):
                print(gameObject.gameString)
            mapKeyAdd(tupCount, tup)

print(tupCount)

In [ ]:
#

holdLine = {}
with open(root_path + "/sproutsGameDataX1.txt", 'r') as inFile:
    for line in inFile:
        holdLine[line.strip()] = False

added = 0
removed = 0
with open(root_path + "/sproutsGameData.txt", 'r') as inFile:
    for line in inFile:
        line = line.strip()
        if line not in holdLine:
            print("+++", line)
            added += 1
        else:
            holdLine[line] = True

for line in holdLine:
    if not holdLine[line]:
        print("---", line)
        removed += 1

print(added, removed)


In [ ]:
import gc
print(gc.get_threshold())
print(gc.get_count())
print(gc.collect())
print(gc.get_count())

In [ ]:
enumEnum = {}

sortedGames = []
for game in dataSummary.allGames:
    gameObject = dataSummary.allGames[game]
    if gameObject.totalConnections < 5:
        sortedGames.append((gameObject.totalConnections, len(gameObject.game), gameObject.gameString))
sortedGames.sort(reverse=True)

for gameTup in sortedGames:
    enumEnum[gameTup[2]] = len(enumEnum)

for gameTup in sortedGames:
    for child in dataSummary.allGames[gameTup[2]].childGames:
        print(str(enumEnum[gameTup[2]]) + '\t' + str(enumEnum[child.gameString]))
print(enumEnum)

#Mex calcs

In [ ]:
# prompt: Calculate mex(A,B,C,D) for all combinations of A,B,C,D in (0,1,2,3,4)

def mex(inputs):
  """Return the mex of a, b, c, and d."""
  for i in range(6):
    if i not in inputs:
      return i
  return 500

def bitflips(num):
  bf1 = num - 1
  if num%2 == 0:
    bf1 = num + 1
  bf2 = num + 2
  if num == 2 or num == 3:
    bf2 = num - 2
  return (bf1, bf2)

#a = <<
#b = >>
#c = <>
#d = ><

placeholder = 0
for a in range(5):
  (n1a, n2a) = bitflips(a)

  n3a = 3
  if a == 1:
    n3a = 2
  elif a == 2:
    n3a = 1
  elif a == 3:
    n3a = 0
  elif a == 4:
    n3a = 7

  for b in range(5):
    (n1b, n2b) = bitflips(b)
    for c in range(5):
      (n1c, n2c) = bitflips(c)
      mexAC = mex([a,c])
      mexBC = mex([b,c])
      for d in range(5):
        (n1d, n2d) = bitflips(d)
        mexAD = mex([a,d])
        mexBD = mex([b,d])

        e63 = mex([n1a,n1c,placeholder,mexAC])
        f63 = mex([n1a,n1d,placeholder,mexAD])

        g6X3 = mex([n1b,n1d,placeholder,mexBD])
        h6X3 = mex([n1b,n1c,placeholder,mexBC])

        e2435 = mex([n1a,n1c,mexAC])
        f2435 = mex([n1a,n1d,mexAD])

        g0X3 = mex([d,g6X3])
        h0X3 = mex([c,h6X3])

        e1435 = mex([a,c,e2435,e63])
        f1435 = mex([a,d,f2435,f63])

        iX33 = mex([b,n1a,placeholder,placeholder])
        i233 = mex([a,b,mexAC,mexAD,mexBD,mexBC])
        i633 = mex([n3a,n1b,i233,iX33,e2435,f2435,g6X3,h6X3])
        i033 = mex([a,b,i633,e1435,e63,g0X3,h0X3])

        print(a,b,c,d,'/',n1a,n1b,n1c,n1d,'/',n2a,n2b,n2c,n2d,'/',n3a,'|',mexAC,mexBC,mexAD,mexBD,'|', e63, f63, g6X3, h6X3, '|', e2435, f2435, '|', g0X3, h0X3, '|', e1435, f1435, '|',i033,i633,i233,iX33)

'''
i033 = mex([A,B,i633,e1435,e63,g0X3,h0X3])
i633 = mex([n3A,n1B,i233,iX33,e2435,f2435,g6X3,h6X3])
i233 = mex([a,b,mexAC,mexAD,mexBD,mexBC])
iX33 = mex([b,n1a,placeholder,placeholder])
e1435 = mex([A,C,e2435,e63])
f1435 = mex([A,D,f2435,f63])
g0X3 = mex([D,g6X3])
h0X3 = mex([C,h6X3])
e1435 "mex(A,C,mex(n1A,n1C,mex(A,C)),mex(n1A,n1C,<*,mex(A,C)))"
f1435 "mex(A,D,mex(n1A,n1D,mex(A,D)),mex(n1A,n1D,<*mex(A,D)))"
g0X3  "mex(D,mex(n1B,n1D,>*,mex(B,D)))"
h0X3  "mex(C,mex(n1B,n1C,*>,mex(B,C)))"
e2435 "mex(n1A,n1C,mex(A,C))"
f2435 "mex(n1A,n1D,mex(A,D))"
e63 "mex(n1A,n1C,<*,mex(A,C))"
f63 "mex(n1A,n1D,*<,mex(A,D))"
g6X3 "mex(n1B,n1D,>*,mex(B,D))"
h6X3 "mex(n1B,n1C,*>,mex(B,C))"
"mex(A, C)"
"mex(B, D)"
"mex(A, D)"
"mex(B, C)"
'''
'''
e1435 = mex([A,C,e2435,e63])
f1435 = mex([A,D,f2435,f63])
g0X3 = mex([D,g6X3])
h0X3 = mex([C,h6X3])
'''



# Various math

In [ ]:
maxVal = 23

def createRotations(newInt, allSeen):
    testTup = enumToTuple(newInt)
    tempRotations = []
    for j in range(len(testTup)):
        tempTup = testTup[j:] + testTup[:j]
        tempTup = adjustSplits(tempTup)
        enumAndConnect = createEnumString(tempTup)
        tempRotations.append(int(enumAndConnect))
        allSeen.add(int(enumAndConnect))

        '''revEnum = ""
        for k in enumAndConnect:
            revEnum = k + revEnum
        revEnum = revEnum.replace('4', '6').replace('5', '4').replace('6', '5')
        tempRotations.append(int(revEnum))
        allSeen.add(int(revEnum))'''

    tempRotations.sort()
    return tempRotations[0]

def plus1(curVal, allSeen):
    strVal = str(curVal)
    bounVals = []
    for i in range(len(strVal)+1):
        newVal = strVal[:i] + STRBOUN + strVal[i:]
        newInt = int(newVal)
        if newInt in allSeen:
            continue
        bounVals.append(createRotations(newInt, allSeen))

    return bounVals

def plus2(curVal, allSeen):
    strVal = str(curVal)
    singVals = []
    splitVals = []
    for i in range(len(strVal)+1):
        singVal = strVal[:i] + STRSING + strVal[i:]
        singInt = int(singVal)
        if singInt in allSeen:
            continue
        singVals.append(createRotations(singInt, allSeen))

    for i in range(len(strVal)+1):
        for j in range(i+1, len(strVal)+1):
            splitVal = strVal[:i] + STRSPLITSTART + strVal[i:j] + STRSPLITEND + strVal[j:]
            if (STRSPLITSTART + STRSPLITEND) in splitVal or (splitVal[0] == STRSPLITSTART and splitVal[-1] == STRSPLITEND):
                continue
            splitInt = int(splitVal)
            if splitInt in allSeen:
                continue
            splitVals.append(createRotations(splitInt, allSeen))
    return (singVals, splitVals)


def plus4(curVal, allSeen):
    strVal = str(curVal)
    hangVals = []
    for i in range(len(strVal)+1):
        newVal = strVal[:i] + STRHANG + strVal[i:]
        newInt = int(newVal)
        if newInt in allSeen:
            continue
        hangVals.append(createRotations(newVal, allSeen))
    return hangVals

borderSizes = {}

borderSizes[2] = [2, 33]
borderSizes[3] = [23, 333]
borderSizes[4] = [1, 22, 233, 3333, 3435]

allSeen = {3}
for val in borderSizes.values():
    for num in val:
        allSeen.add(num)

borderSizes[1] = [3]


for size in range(5,maxVal+1):
    nextPlus1 = []
    nextPlus2 = []
    nextPlus4 = []
    borderSizes[size] = set()
    for val in borderSizes[size-1]:
        nextPlus1 = plus1(str(val), allSeen)
        for val in nextPlus1:
            borderSizes[size].add(val)
    for val in borderSizes[size-2]:
        singVals, splitVals = plus2(str(val), allSeen)
        for val in singVals:
            borderSizes[size].add(val)
        for val in splitVals:
            borderSizes[size].add(val)
    for val in borderSizes[size-4]:
        nextPlus4 = plus4(str(val), allSeen)
        for val in nextPlus4:
            borderSizes[size].add(val)

    print(size, len(borderSizes[size]))


In [ ]:
size = 21

nextPlus1 = []
nextPlus2 = []
nextPlus4 = []
borderSizes[size] = set()
for val in borderSizes[size-1]:
    nextPlus1 = plus1(str(val), allSeen)
    for val in nextPlus1:
        borderSizes[size].add(val)
for val in borderSizes[size-2]:
    singVals, splitVals = plus2(str(val), allSeen)
    for val in singVals:
        borderSizes[size].add(val)
    for val in splitVals:
        borderSizes[size].add(val)
for val in borderSizes[size-4]:
    nextPlus4 = plus4(str(val), allSeen)
    for val in nextPlus4:
        borderSizes[size].add(val)

print(size, len(borderSizes[size]))

1 1
2 2
3 2
4 5
5 6
6 13
7 19
8 44
9 79
10 186
11 388
12 955
13 2218
14 5613
15 13968
16 36214
17 93687
18 247593
19 656096
20 1759332

In [ ]:
def iterateVal(curString, curPos=0):
    truePos = len(curString)-curPos-1
    if curString == "3" or curString[0:2] == "35":
        return '1'*(len(curString)+1)
    if curString[0:2] == "15":
        return '2' + ('1'*(len(curString)-1))
    if curString[0:2] == "25":
        return '3' + ('1'*(len(curString)-1))
    if curString[truePos] == '5':
        return(iterateVal(curString[:truePos] + '1' + curString[truePos+1:], curPos+1))
    return(curString[:truePos] + chr(ord(curString[truePos])+1) + curString[truePos+1:])

minEnumMapping = {}
allRotations = {}
allSeen = {}
validEnums = [(EMPT, 0), (FREE, 6)]
testEnum = '0'
maxEnum = '0'


maxLength = 7
reverseAllowed = True


for i in range(1000000000):
    testEnum = iterateVal(testEnum)
    if testEnum[-1] == '4':
        testEnum = iterateVal(testEnum)
    if len(testEnum) > maxLength:
        break

    maxEnum = testEnum
    if testEnum in allSeen:
        continue
    if testEnum[0] == STRSPLITSTART and testEnum[-1] == STRSPLITEND:
        continue

    checkConnects = 0
    goToNext = False
    for point in testEnum:
        if point < testEnum[0]:
            goToNext = True
            break
        if point == STRHANG:
            checkConnects += 4
        elif point == STRSING:
            checkConnects += 2
        elif point == STRBOUN or point == STRSPLITSTART or point == STRSPLITEND:
            checkConnects += 1
        if checkConnects > maxLength:
            goToNext = True
            break
    if goToNext:
        continue


    valid = True
    dyckPathCheck = 0  #The Dyck Path is the ordering of the split points, with "3" being the first time a split point is seen,
                       #  and "4" being the second.  This allows for perfect reconstruction of the loop without storing
                       #  splits as arbitrarily higher values that could become multi-digit values that make short strings impossible
    for j in range(len(testEnum)):
        point = testEnum[j]
        if j > 0 and point == STRSPLITEND and testEnum[j-1] == STRSPLITSTART:
            valid = False
            break

        if point == STRSPLITSTART:
            dyckPathCheck += 1
        elif point == STRSPLITEND:
            dyckPathCheck -= 1
        if dyckPathCheck < 0:
            valid = False
            break

    if not(valid and dyckPathCheck == 0):
        continue  #Don't bother doing further calculations if the string isn't actually a valid loop

    testTup = enumToTuple(testEnum)
    tempRotations = []
    for j in range(len(testTup)):
        tempTup = testTup[j:] + testTup[:j]
        tempTup = adjustSplits(tempTup)
        enumAndConnect = createEnumString(tempTup)
        tempRotations.append((int(enumAndConnect), checkConnects))

        if reverseAllowed:
            revEnum = ""
            for k in enumAndConnect:
                revEnum = k + revEnum
            revEnum = revEnum.replace('4', '6').replace('5', '4').replace('6', '5')
            tempRotations.append((int(revEnum), checkConnects))

        allSeen[enumAndConnect] = True
        allSeen[revEnum] = True

    tempRotations.sort()

    allRotations[tempRotations[0]] = tempRotations
    for rotation in tempRotations:
        minEnumMapping[rotation[0]] = tempRotations

allRotationsArray = [(EMPT, 0), (FREE, 6)]
for rotation in allRotations:
    allRotationsArray.append(rotation)
allRotationsArray.sort()

sizeLoops = {}
for rotation in allRotationsArray:
    #print(rotation)
    enum = rotation[0]
    size = rotation[1]
    mapKeyAdd(sizeLoops, size, listAddition = enum)

sizeLoopsOrdered = []
for key in sizeLoops:
    sizeLoopsOrdered.append(key)
sizeLoopsOrdered.sort()

print("Max:", maxEnum)
for key in sizeLoopsOrdered:
    if key <= maxLength:
        print(key, ":", len(sizeLoops[key]), ":", sizeLoops[key])
    else:
        print(key, ":", len(sizeLoops[key]))


'''
0 : 1
1 : 1
2 : 2
3 : 2
4 : 5
5 : 6
6 : 14
7 : 19
8 : 44
9 : 79
10 : 186
11 : 337
12 : 621
13 : 909
14 : 1555
15 : 2702
16 : 4562


Max: 324555555425
0 : 1
1 : 1
2 : 2
3 : 2
4 : 5
5 : 6
6 : 14
7 : 19
8 : 44
9 : 79
10 : 186
11 : 388
12 : 847
13 : 2013
14 : 3993
15 : 6702
16 : 10984
'''



Max: 351111111111
0 : 1 : [-2]
1 : 1 : [3]
2 : 2 : [2, 33]
3 : 2 : [23, 333]
4 : 5 : [1, 22, 233, 3333, 3435]
5 : 6 : [13, 223, 2333, 2435, 33333, 33435]
6 : 15 : [0, 12, 133, 222, 2233, 2323, 2425, 23333, 23435, 24335, 24353, 333333, 333435, 334335, 344355]
7 : 23 : [123, 132, 1333, 1435, 2223, 22333, 22435, 23233, 23425, 24253, 233333, 233435, 234335, 234353, 243335, 243353, 243533, 244355, 3333333, 3333435, 3334335, 3344355, 3434355]
8 : 55 : [122, 1233, 1323, 1332, 1425, 2222, 13333, 13435, 14335, 14353, 22233, 22323, 22425, 223333, 223435, 224335, 224353, 232333, 232435, 233233, 233425, 234235, 234253, 234325, 242533, 243253, 244255, 2333333, 2333435, 2334335, 2334353, 2343335, 2343353, 2343533, 2344355, 2433335, 2433353, 2433533, 2434355, 2435333, 2435435, 2443355, 2443535, 2443553, 33333333, 33333435, 33334335, 33343335, 33344355, 33434355, 33435435, 33443355, 33443535, 34343535, 34443555]
9 : 117 : [113, 1223, 1232, 1322, 12333, 12435, 13233, 13323, 13332, 13425, 14235, 14253, 14325, 14352, 22223, 133333, 133435, 134335, 134353, 143335, 143353, 143533, 144355, 222333, 222435, 223233, 223323, 223425, 224235, 224253, 224325, 232323, 232425, 2233333, 2233435, 2234335, 2234353, 2243335, 2243353, 2243533, 2244355, 2323333, 2323435, 2324335, 2324353, 2332333, 2332435, 2333425, 2334235, 2334253, 2334325, 2342353, 2342533, 2343253, 2343325, 2344255, 2424355, 2425333, 2425435, 2432533, 2434255, 2442553, 23333333, 23333435, 23334335, 23334353, 23343335, 23343353, 23343533, 23344355, 23433335, 23433353, 23433533, 23434355, 23435333, 23435435, 23443355, 23443535, 23443553, 24333335, 24333353, 24333533, 24334355, 24335333, 24335435, 24343355, 24343535, 24343553, 24353333, 24353435, 24354335, 24354353, 24433355, 24433535, 24433553, 24435335, 24435353, 24435533, 24443555, 333333333, 333333435, 333334335, 333343335, 333344355, 333434355, 333435435, 333443355, 333443535, 334334355, 334335435, 334343355, 334343535, 334353435, 334443555, 343443555, 343544355, 344354355]
10 : 285 : [112, 1133, 1222, 1313, 1415, 12233, 12323, 12332, 12425, 13223, 13232, 13322, 14225, 14252, 22222, 123333, 123435, 124335, 124353, 132333, 132435, 133233, 133323, 133332, 133425, 134235, 134253, 134325, 134352, 142335, 142353, 142533, 143235, 143253, 143325, 143352, 143523, 143532, 144255, 222233, 222323, 222425, 223223, 224225, 1333333, 1333435, 1334335, 1334353, 1343335, 1343353, 1343533, 1344355, 1433335, 1433353, 1433533, 1434355, 1435333, 1435435, 1443355, 1443535, 1443553, 2223333, 2223435, 2224335, 2224353, 2232333, 2232435, 2233233, 2233323, 2233425, 2234235, 2234253, 2234325, 2242335, 2242353, 2242533, 2243235, 2243253, 2243325, 2243523, 2244255, 2323233, 2323425, 2324235, 2324253, 2324325, 2332425, 2424255, 22333333, 22333435, 22334335, 22334353, 22343335, 22343353, 22343533, 22344355, 22433335, 22433353, 22433533, 22434355, 22435333, 22435435, 22443355, 22443535, 22443553, 23233333, 23233435, 23234335, 23234353, 23243335, 23243353, 23243533, 23244355, 23323333, 23323435, 23324335, 23324353, 23332333, 23332435, 23333425, 23334235, 23334253, 23334325, 23342335, 23342353, 23342533, 23343235, 23343253, 23343325, 23344255, 23423533, 23424355, 23425333, 23425435, 23432353, 23432533, 23433253, 23433325, 23434255, 23435425, 23442355, 23442535, 23442553, 23443255, 23443525, 24243355, 24243535, 24243553, 24253333, 24253435, 24254335, 24254353, 24324355, 24325333, 24325435, 24332533, 24334255, 24342535, 24342553, 24343255, 24352435, 24425533, 24432553, 24442555, 233333333, 233333435, 233334335, 233334353, 233343335, 233343353, 233343533, 233344355, 233433335, 233433353, 233433533, 233434355, 233435333, 233435435, 233443355, 233443535, 233443553, 234333335, 234333353, 234333533, 234334355, 234335333, 234335435, 234343355, 234343535, 234343553, 234353333, 234353435, 234354335, 234354353, 234433355, 234433535, 234433553, 234435335, 234435353, 234435533, 234443555, 243333335, 243333353, 243333533, 243334355, 243335333, 243335435, 243343355, 243343535, 243343553, 243353333, 243353435, 243354335, 243354353, 243433355, 243433535, 243433553, 243435335, 243435353, 243435533, 243443555, 243533333, 243533435, 243534335, 243534353, 243543335, 243543353, 243543533, 243544355, 244333355, 244333535, 244333553, 244335335, 244335353, 244335533, 244343555, 244353335, 244353353, 244353533, 244354355, 244355333, 244355435, 244433555, 244435355, 244435535, 244435553, 3333333333, 3333333435, 3333334335, 3333343335, 3333344355, 3333433335, 3333434355, 3333435435, 3333443355, 3333443535, 3334334355, 3334335435, 3334343355, 3334343535, 3334353435, 3334354335, 3334433355, 3334433535, 3334435335, 3334443555, 3343343355, 3343343535, 3343353435, 3343433535, 3343443555, 3343533435, 3343544355, 3344343555, 3344354355, 3344355435, 3344433555, 3344435355, 3344435535, 3434343555, 3434354355, 3434435355, 3434435535, 3435434355, 3444435555]
11 : 663 : [1123, 1132, 1213, 11333, 11435, 12223, 12232, 12322, 13133, 13222, 13415, 14153, 122333, 122435, 123233, 123323, 123332, 123425, 124235, 124253, 124325, 124352, 132233, 132323, 132332, 132425, 133223, 133232, 133322, 134225, 134252, 142235, 142253, 142325, 142352, 142523, 142532, 143225, 143252, 143522, 222223, 1233333, 1233435, 1234335, 1234353, 1243335, 1243353, 1243533, 1244355, 1323333, 1323435, 1324335, 1324353, 1332333, 1332435, 1333233, 1333323, 1333332, 1333425, 1334235, 1334253, 1334325, 1334352, 1342335, 1342353, 1342533, 1343235, 1343253, 1343325, 1343352, 1343523, 1343532, 1344255, 1423335, 1423353, 1423533, 1424355, 1425333, 1425435, 1432335, 1432353, 1432533, 1433235, 1433253, 1433325, 1433352, 1433523, 1433532, 1434255, 1435233, 1435323, 1435332, 1435425, 1442355, 1442535, 1442553, 1443255, 1443525, 1443552, 2222333, 2222435, 2223233, 2223323, 2223425, 2224235, 2224253, 2224325, 2232233, 2232323, 2232425, 2234225, 2242253, 2242325, 2242523, 13333333, 13333435, 13334335, 13334353, 13343335, 13343353, 13343533, 13344355, 13433335, 13433353, 13433533, 13434355, 13435333, 13435435, 13443355, 13443535, 13443553, 14333335, 14333353, 14333533, 14334355, 14335333, 14335435, 14343355, 14343535, 14343553, 14353333, 14353435, 14354335, 14354353, 14433355, 14433535, 14433553, 14435335, 14435353, 14435533, 14443555, 22233333, 22233435, 22234335, 22234353, 22243335, 22243353, 22243533, 22244355, 22323333, 22323435, 22324335, 22324353, 22332333, 22332435, 22333233, 22333323, 22333425, 22334235, 22334253, 22334325, 22342335, 22342353, 22342533, 22343235, 22343253, 22343325, 22343523, 22344255, 22423335, 22423353, 22423533, 22424355, 22425333, 22425435, 22432335, 22432353, 22432533, 22433235, 22433253, 22433325, 22433523, 22434255, 22435233, 22435323, 22435425, 22442355, 22442535, 22442553, 22443255, 22443525, 23232333, 23232435, 23233233, 23233425, 23234235, 23234253, 23234325, 23242335, 23242353, 23242533, 23243235, 23243253, 23243325, 23244255, 23323425, 23324235, 23324253, 23324325, 23332425, 23424255, 23425425, 23442525, 24242535, 24242553, 24243255, 24243525, 24254253, 223333333, 223333435, 223334335, 223334353, 223343335, 223343353, 223343533, 223344355, 223433335, 223433353, 223433533, 223434355, 223435333, 223435435, 223443355, 223443535, 223443553, 224333335, 224333353, 224333533, 224334355, 224335333, 224335435, 224343355, 224343535, 224343553, 224353333, 224353435, 224354335, 224354353, 224433355, 224433535, 224433553, 224435335, 224435353, 224435533, 224443555, 232333333, 232333435, 232334335, 232334353, 232343335, 232343353, 232343533, 232344355, 232433335, 232433353, 232433533, 232434355, 232435333, 232435435, 232443355, 232443535, 232443553, 233233333, 233233435, 233234335, 233234353, 233243335, 233243353, 233243533, 233244355, 233323333, 233323435, 233324335, 233324353, 233332435, 233333425, 233334235, 233334253, 233334325, 233342335, 233342353, 233342533, 233343235, 233343253, 233343325, 233344255, 233423353, 233423533, 233424355, 233425333, 233425435, 233432353, 233432533, 233433235, 233433253, 233433325, 233434255, 233435425, 233442355, 233442535, 233442553, 233443255, 233443525, 234234355, 234235333, 234235435, 234243355, 234243535, 234243553, 234253333, 234253435, 234254335, 234254353, 234323533, 234324355, 234325333, 234325435, 234332533, 234333253, 234333325, 234334255, 234335425, 234342355, 234342535, 234342553, 234343255, 234343525, 234352435, 234353425, 234354253, 234354325, 234423553, 234425335, 234425353, 234425533, 234432535, 234432553, 234433255, 234433525, 234435253, 234435325, 234442555, 242433355, 242433535, 242433553, 242435335, 242435353, 242435533, 242443555, 242533333, 242533435, 242534335, 242534353, 242543335, 242543353, 242543533, 242544355, 243243355, 243243535, 243243553, 243253333, 243253435, 243254335, 243254353, 243324355, 243325333, 243325435, 243334255, 243342535, 243342553, 243343255, 243352435, 243425353, 243425533, 243432553, 243433255, 243442555, 243524353, 243544255, 244254355, 244255333, 244255435, 244325533, 244342555, 244425553, 2333333333, 2333333435, 2333334335, 2333334353, 2333343335, 2333343353, 2333343533, 2333344355, 2333433335, 2333433353, 2333433533, 2333434355, 2333435333, 2333435435, 2333443355, 2333443535, 2333443553, 2334333335, 2334333353, 2334333533, 2334334355, 2334335333, 2334335435, 2334343355, 2334343535, 2334343553, 2334353333, 2334353435, 2334354335, 2334354353, 2334433355, 2334433535, 2334433553, 2334435335, 2334435353, 2334435533, 2334443555, 2343333335, 2343333353, 2343333533, 2343334355, 2343335333, 2343335435, 2343343355, 2343343535, 2343343553, 2343353333, 2343353435, 2343354335, 2343354353, 2343433355, 2343433535, 2343433553, 2343435335, 2343435353, 2343435533, 2343443555, 2343533333, 2343533435, 2343534335, 2343534353, 2343543335, 2343543353, 2343543533, 2343544355, 2344333355, 2344333535, 2344333553, 2344335335, 2344335353, 2344335533, 2344343555, 2344353335, 2344353353, 2344353533, 2344354355, 2344355333, 2344355435, 2344433555, 2344435355, 2344435535, 2344435553, 2433333335, 2433333353, 2433333533, 2433334355, 2433335333, 2433335435, 2433343355, 2433343535, 2433343553, 2433353333, 2433353435, 2433354335, 2433354353, 2433433355, 2433433535, 2433433553, 2433435335, 2433435353, 2433435533, 2433443555, 2433533333, 2433533435, 2433534335, 2433534353, 2433543335, 2433543353, 2433543533, 2433544355, 2434333355, 2434333535, 2434333553, 2434335335, 2434335353, 2434335533, 2434343555, 2434353335, 2434353353, 2434353533, 2434354355, 2434355333, 2434355435, 2434433555, 2434435355, 2434435535, 2434435553, 2435333333, 2435333435, 2435334335, 2435334353, 2435343335, 2435343353, 2435343533, 2435344355, 2435433335, 2435433353, 2435433533, 2435434355, 2435435333, 2435435435, 2435443355, 2435443535, 2435443553, 2443333355, 2443333535, 2443333553, 2443335335, 2443335353, 2443335533, 2443343555, 2443353335, 2443353353, 2443353533, 2443354355, 2443355333, 2443355435, 2443433555, 2443435355, 2443435535, 2443435553, 2443533335, 2443533353, 2443533533, 2443534355, 2443535333, 2443535435, 2443543355, 2443543535, 2443543553, 2443553333, 2443553435, 2443554335, 2443554353, 2444333555, 2444335355, 2444335535, 2444335553, 2444353355, 2444353535, 2444353553, 2444355335, 2444355353, 2444355533, 2444435555, 33333333333, 33333333435, 33333334335, 33333343335, 33333344355, 33333433335, 33333434355, 33333435435, 33333443355, 33333443535, 33334334355, 33334335435, 33334343355, 33334343535, 33334353435, 33334354335, 33334433355, 33334433535, 33334435335, 33334443555, 33343334355, 33343335435, 33343343355, 33343343535, 33343353435, 33343354335, 33343433355, 33343433535, 33343435335, 33343443555, 33343533435, 33343534335, 33343544355, 33344335335, 33344343555, 33344354355, 33344355435, 33344433555, 33344435355, 33344435535, 33433433535, 33433435335, 33433443555, 33433544355, 33434343555, 33434354355, 33434355435, 33434433555, 33434435355, 33434435535, 33435344355, 33435434355, 33435435435, 33435443355, 33435443535, 33443354355, 33443355435, 33443433555, 33443435355, 33443435535, 33443534355, 33443535435, 33443543535, 33443553435, 33444353535, 33444435555, 34343435355, 34343435535, 34343534355, 34344435555, 34354443555, 34434435555, 34435443555]
12 : 1668 : [1122, 1212, 11233, 11323, 11332, 11425, 12133, 12222, 12313, 12415, 13132, 14152, 113333, 113435, 114335, 114353, 122233, 122323, 122332, 122425, 123223, 123232, 123322, 124225, 124252, 131333, 131435, 132223, 132232, 132322, 133133, 133222, 133415, 134135, 134153, 134315, 141533, 142225, 142252, 142522, 143153, 144155, 222222, 1223333, 1223435, 1224335, 1224353, 1232333, 1232435, 1233233, 1233323, 1233332, 1233425, 1234235, 1234253, 1234325, 1234352, 1242335, 1242353, 1242533, 1243235, 1243253, 1243325, 1243352, 1243523, 1243532, 1244255, 1322333, 1322435, 1323233, 1323323, 1323332, 1323425, 1324235, 1324253, 1324325, 1324352, 1332233, 1332323, 1332332, 1332425, 1333223, 1333232, 1333322, 1334225, 1334252, 1342235, 1342253, 1342325, 1342352, 1342523, 1342532, 1343225, 1343252, 1343522, 1422335, 1422353, 1422533, 1423235, 1423253, 1423325, 1423352, 1423523, 1423532, 1424255, 1425233, 1425323, 1425332, 1425425, 1432235, 1432253, 1432325, 1432352, 1432523, 1432532, 1433225, 1433252, 1433522, 1435223, 1435232, 1435322, 1442255, 1442525, 1442552, 2222233, 2222323, 2222425, 2223223, 2224225, 12333333, 12333435, 12334335, 12334353, 12343335, 12343353, 12343533, 12344355, 12433335, 12433353, 12433533, 12434355, 12435333, 12435435, 12443355, 12443535, 12443553, 13233333, 13233435, 13234335, 13234353, 13243335, 13243353, 13243533, 13244355, 13323333, 13323435, 13324335, 13324353, 13332333, 13332435, 13333233, 13333323, 13333332, 13333425, 13334235, 13334253, 13334325, 13334352, 13342335, 13342353, 13342533, 13343235, 13343253, 13343325, 13343352, 13343523, 13343532, 13344255, 13423335, 13423353, 13423533, 13424355, 13425333, 13425435, 13432335, 13432353, 13432533, 13433235, 13433253, 13433325, 13433352, 13433523, 13433532, 13434255, 13435233, 13435323, 13435332, 13435425, 13442355, 13442535, 13442553, 13443255, 13443525, 13443552, 14233335, 14233353, 14233533, 14234355, 14235333, 14235435, 14243355, 14243535, 14243553, 14253333, 14253435, 14254335, 14254353, 14323335, 14323353, 14323533, 14324355, 14325333, 14325435, 14332335, 14332353, 14332533, 14333235, 14333253, 14333325, 14333352, 14333523, 14333532, 14334255, 14335233, 14335323, 14335332, 14335425, 14342355, 14342535, 14342553, 14343255, 14343525, 14343552, 14352333, 14352435, 14353233, 14353323, 14353332, 14353425, 14354235, 14354253, 14354325, 14354352, 14423355, 14423535, 14423553, 14425335, 14425353, 14425533, 14432355, 14432535, 14432553, 14433255, 14433525, 14433552, 14435235, 14435253, 14435325, 14435352, 14435523, 14435532, 14442555, 22223333, 22223435, 22224335, 22224353, 22232333, 22232435, 22233233, 22233323, 22233425, 22234235, 22234253, 22234325, 22242335, 22242353, 22242533, 22243235, 22243253, 22243325, 22243523, 22244255, 22322333, 22322435, 22323233, 22323323, 22323425, 22324235, 22324253, 22324325, 22332233, 22332323, 22332425, 22334225, 22342235, 22342253, 22342325, 22342523, 22343225, 22422533, 22423235, 22423253, 22423325, 22423523, 22424255, 22425233, 22425323, 22425425, 22432253, 22432325, 22432523, 22442255, 22442525, 23232323, 23232425, 23242325, 24242525, 133333333, 133333435, 133334335, 133334353, 133343335, 133343353, 133343533, 133344355, 133433335, 133433353, 133433533, 133434355, 133435333, 133435435, 133443355, 133443535, 133443553, 134333335, 134333353, 134333533, 134334355, 134335333, 134335435, 134343355, 134343535, 134343553, 134353333, 134353435, 134354335, 134354353, 134433355, 134433535, 134433553, 134435335, 134435353, 134435533, 134443555, 143333335, 143333353, 143333533, 143334355, 143335333, 143335435, 143343355, 143343535, 143343553, 143353333, 143353435, 143354335, 143354353, 143433355, 143433535, 143433553, 143435335, 143435353, 143435533, 143443555, 143533333, 143533435, 143534335, 143534353, 143543335, 143543353, 143543533, 143544355, 144333355, 144333535, 144333553, 144335335, 144335353, 144335533, 144343555, 144353335, 144353353, 144353533, 144354355, 144355333, 144355435, 144433555, 144435355, 144435535, 144435553, 222333333, 222333435, 222334335, 222334353, 222343335, 222343353, 222343533, 222344355, 222433335, 222433353, 222433533, 222434355, 222435333, 222435435, 222443355, 222443535, 222443553, 223233333, 223233435, 223234335, 223234353, 223243335, 223243353, 223243533, 223244355, 223323333, 223323435, 223324335, 223324353, 223332333, 223332435, 223333233, 223333323, 223333425, 223334235, 223334253, 223334325, 223342335, 223342353, 223342533, 223343235, 223343253, 223343325, 223343523, 223344255, 223423335, 223423353, 223423533, 223424355, 223425333, 223425435, 223432335, 223432353, 223432533, 223433235, 223433253, 223433325, 223433523, 223434255, 223435233, 223435323, 223435425, 223442355, 223442535, 223442553, 223443255, 223443525, 224233335, 224233353, 224233533, 224234355, 224235333, 224235435, 224243355, 224243535, 224243553, 224253333, 224253435, 224254335, 224254353, 224323335, 224323353, 224323533, 224324355, 224325333, 224325435, 224332335, 224332353, 224332533, 224333235, 224333253, 224333325, 224333523, 224334255, 224335233, 224335323, 224335425, 224342355, 224342535, 224342553, 224343255, 224343525, 224352333, 224352435, 224353233, 224353323, 224353425, 224354235, 224354253, 224354325, 224423355, 224423535, 224423553, 224425335, 224425353, 224425533, 224432355, 224432535, 224432553, 224433255, 224433525, 224435235, 224435253, 224435325, 224435523, 224442555, 232323333, 232323435, 232324335, 232324353, 232332333, 232332435, 232333233, 232333425, 232334235, 232334253, 232334325, 232342335, 232342353, 232342533, 232343235, 232343253, 232343325, 232344255, 232423335, 232423353, 232423533, 232424355, 232425333, 232425435, 232432335, 232432353, 232432533, 232433235, 232433253, 232433325, 232434255, 232435233, 232435425, 232442355, 232442535, 232442553, 232443255, 232443525, 233233233, 233233425, 233234235, 233234253, 233234325, 233242335, 233242353, 233242533, 233243235, 233243253, 233243325, 233244255, 233323425, 233324235, 233324253, 233324325, 233332425, 233424255, 233425425, 233442525, 234234255, 234235425, 234242355, 234242535, 234242553, 234243255, 234243525, 234252435, 234253425, 234254253, 234254325, 234324255, 234325425, 234342525, 234352425, 234425253, 234425325, 234432525, 242425335, 242425353, 242425533, 242432535, 242432553, 242433255, 242433525, 242435253, 242435325, 242442555, 242532435, 242534253, 242542533, 242543253, 242544255, 243243255, 244254255, 2233333333, 2233333435, 2233334335, 2233334353, 2233343335, 2233343353, 2233343533, 2233344355, 2233433335, 2233433353, 2233433533, 2233434355, 2233435333, 2233435435, 2233443355, 2233443535, 2233443553, 2234333335, 2234333353, 2234333533, 2234334355, 2234335333, 2234335435, 2234343355, 2234343535, 2234343553, 2234353333, 2234353435, 2234354335, 2234354353, 2234433355, 2234433535, 2234433553, 2234435335, 2234435353, 2234435533, 2234443555, 2243333335, 2243333353, 2243333533, 2243334355, 2243335333, 2243335435, 2243343355, 2243343535, 2243343553, 2243353333, 2243353435, 2243354335, 2243354353, 2243433355, 2243433535, 2243433553, 2243435335, 2243435353, 2243435533, 2243443555, 2243533333, 2243533435, 2243534335, 2243534353, 2243543335, 2243543353, 2243543533, 2243544355, 2244333355, 2244333535, 2244333553, 2244335335, 2244335353, 2244335533, 2244343555, 2244353335, 2244353353, 2244353533, 2244354355, 2244355333, 2244355435, 2244433555, 2244435355, 2244435535, 2244435553, 2323333333, 2323333435, 2323334335, 2323334353, 2323343335, 2323343353, 2323343533, 2323344355, 2323433335, 2323433353, 2323433533, 2323434355, 2323435333, 2323435435, 2323443355, 2323443535, 2323443553, 2324333335, 2324333353, 2324333533, 2324334355, 2324335333, 2324335435, 2324343355, 2324343535, 2324343553, 2324353333, 2324353435, 2324354335, 2324354353, 2324433355, 2324433535, 2324433553, 2324435335, 2324435353, 2324435533, 2324443555, 2332333333, 2332333435, 2332334335, 2332334353, 2332343335, 2332343353, 2332343533, 2332344355, 2332433335, 2332433353, 2332433533, 2332434355, 2332435333, 2332435435, 2332443355, 2332443535, 2332443553, 2333233333, 2333233435, 2333234335, 2333234353, 2333243335, 2333243353, 2333243533, 2333244355, 2333323333, 2333323435, 2333324335, 2333324353, 2333332435, 2333333425, 2333334235, 2333334253, 2333334325, 2333342335, 2333342353, 2333342533, 2333343235, 2333343253, 2333343325, 2333344255, 2333423335, 2333423353, 2333423533, 2333424355, 2333425333, 2333425435, 2333432335, 2333432353, 2333432533, 2333433235, 2333433253, 2333433325, 2333434255, 2333435425, 2333442355, 2333442535, 2333442553, 2333443255, 2333443525, 2334233533, 2334234355, 2334235333, 2334235435, 2334243355, 2334243535, 2334243553, 2334253333, 2334253435, 2334254335, 2334254353, 2334323353, 2334323533, 2334324355, 2334325333, 2334325435, 2334332353, 2334332533, 2334333235, 2334333253, 2334333325, 2334334255, 2334335425, 2334342355, 2334342535, 2334342553, 2334343255, 2334343525, 2334352435, 2334353425, 2334354235, 2334354253, 2334354325, 2334423355, 2334423535, 2334423553, 2334425335, 2334425353, 2334425533, 2334432355, 2334432535, 2334432553, 2334433255, 2334433525, 2334435235, 2334435253, 2334435325, 2334442555, 2342343355, 2342343535, 2342343553, 2342353333, 2342353435, 2342354335, 2342354353, 2342433355, 2342433535, 2342433553, 2342435335, 2342435353, 2342435533, 2342443555, 2342533333, 2342533435, 2342534335, 2342534353, 2342543335, 2342543353, 2342543533, 2342544355, 2343234355, 2343235333, 2343235435, 2343243355, 2343243535, 2343243553, 2343253333, 2343253435, 2343254335, 2343254353, 2343323533, 2343324355, 2343325333, 2343325435, 2343332533, 2343333253, 2343333325, 2343334255, 2343335425, 2343342355, 2343342535, 2343342553, 2343343255, 2343343525, 2343352435, 2343353425, 2343354253, 2343354325, 2343423535, 2343423553, 2343425335, 2343425353, 2343425533, 2343432355, 2343432535, 2343432553, 2343433255, 2343433525, 2343435253, 2343435325, 2343442555, 2343523435, 2343524335, 2343524353, 2343532435, 2343533425, 2343534253, 2343534325, 2343542533, 2343543253, 2343543325, 2343544255, 2344235533, 2344243555, 2344253335, 2344253353, 2344253533, 2344254355, 2344255333, 2344255435, 2344323553, 2344325335, 2344325353, 2344325533, 2344332535, 2344332553, 2344333255, 2344333525, 2344335253, 2344335325, 2344342555, 2344352533, 2344353253, 2344353325, 2344354255, 2344355425, 2344423555, 2344425355, 2344425535, 2344425553, 2344432555, 2344435255, 2344435525, 2424333355, 2424333535, 2424333553, 2424335335, 2424335353, 2424335533, 2424343555, 2424353335, 2424353353, 2424353533, 2424354355, 2424355333, 2424355435, 2424433555, 2424435355, 2424435535, 2424435553, 2425333333, 2425333435, 2425334335, 2425334353, 2425343335, 2425343353, 2425343533, 2425344355, 2425433335, 2425433353, 2425433533, 2425434355, 2425435333, 2425435435, 2425443355, 2425443535, 2425443553, 2432433355, 2432433535, 2432433553, 2432435335, 2432435353, 2432435533, 2432443555, 2432533333, 2432533435, 2432534335, 2432534353, 2432543335, 2432543353, 2432543533, 2432544355, 2433243355, 2433243535, 2433243553, 2433253333, 2433253435, 2433254335, 2433254353, 2433324355, 2433325333, 2433325435, 2433334255, 2433342535, 2433342553, 2433343255, 2433352435, 2433425335, 2433425353, 2433425533, 2433432535, 2433432553, 2433433255, 2433442555, 2433524335, 2433524353, 2433532435, 2433544255, 2434243555, 2434253533, 2434254355, 2434255333, 2434255435, 2434325353, 2434325533, 2434332553, 2434333255, 2434342555, 2434354255, 2434425355, 2434425535, 2434425553, 2434432555, 2434435255, 2435243533, 2435244355, 2435324353, 2435344255, 2435424355, 2435425435, 2435434255, 2435442553, 2435443255, 2442543355, 2442543535, 2442543553, 2442553333, 2442553435, 2442554335, 2442554353, 2443254355, 2443255333, 2443255435, 2443325533, 2443342555, 2443425535, 2443425553, 2443432555, 2443525435, 2444255533, 2444325553, 2444425555, 23333333333, 23333333435, 23333334335, 23333334353, 23333343335, 23333343353, 23333343533, 23333344355, 23333433335, 23333433353, 23333433533, 23333434355, 23333435333, 23333435435, 23333443355, 23333443535, 23333443553, 23334333335, 23334333353, 23334333533, 23334334355, 23334335333, 23334335435, 23334343355, 23334343535, 23334343553, 23334353333, 23334353435, 23334354335, 23334354353, 23334433355, 23334433535, 23334433553, 23334435335, 23334435353, 23334435533, 23334443555, 23343333335, 23343333353, 23343333533, 23343334355, 23343335333, 23343335435, 23343343355, 23343343535, 23343343553, 23343353333, 23343353435, 23343354335, 23343354353, 23343433355, 23343433535, 23343433553, 23343435335, 23343435353, 23343435533, 23343443555, 23343533333, 23343533435, 23343534335, 23343534353, 23343543335, 23343543353, 23343543533, 23343544355, 23344333355, 23344333535, 23344333553, 23344335335, 23344335353, 23344335533, 23344343555, 23344353335, 23344353353, 23344353533, 23344354355, 23344355333, 23344355435, 23344433555, 23344435355, 23344435535, 23344435553, 23433333335, 23433333353, 23433333533, 23433334355, 23433335333, 23433335435, 23433343355, 23433343535, 23433343553, 23433353333, 23433353435, 23433354335, 23433354353, 23433433355, 23433433535, 23433433553, 23433435335, 23433435353, 23433435533, 23433443555, 23433533333, 23433533435, 23433534335, 23433534353, 23433543335, 23433543353, 23433543533, 23433544355, 23434333355, 23434333535, 23434333553, 23434335335, 23434335353, 23434335533, 23434343555, 23434353335, 23434353353, 23434353533, 23434354355, 23434355333, 23434355435, 23434433555, 23434435355, 23434435535, 23434435553, 23435333333, 23435333435, 23435334335, 23435334353, 23435343335, 23435343353, 23435343533, 23435344355, 23435433335, 23435433353, 23435433533, 23435434355, 23435435333, 23435435435, 23435443355, 23435443535, 23435443553, 23443333355, 23443333535, 23443333553, 23443335335, 23443335353, 23443335533, 23443343555, 23443353335, 23443353353, 23443353533, 23443354355, 23443355333, 23443355435, 23443433555, 23443435355, 23443435535, 23443435553, 23443533335, 23443533353, 23443533533, 23443534355, 23443535333, 23443535435, 23443543355, 23443543535, 23443543553, 23443553333, 23443553435, 23443554335, 23443554353, 23444333555, 23444335355, 23444335535, 23444335553, 23444353355, 23444353535, 23444353553, 23444355335, 23444355353, 23444355533, 23444435555, 24333333335, 24333333353, 24333333533, 24333334355, 24333335333, 24333335435, 24333343355, 24333343535, 24333343553, 24333353333, 24333353435, 24333354335, 24333354353, 24333433355, 24333433535, 24333433553, 24333435335, 24333435353, 24333435533, 24333443555, 24333533333, 24333533435, 24333534335, 24333534353, 24333543335, 24333543353, 24333543533, 24333544355, 24334333355, 24334333535, 24334333553, 24334335335, 24334335353, 24334335533, 24334343555, 24334353335, 24334353353, 24334353533, 24334354355, 24334355333, 24334355435, 24334433555, 24334435355, 24334435535, 24334435553, 24335333333, 24335333435, 24335334335, 24335334353, 24335343335, 24335343353, 24335343533, 24335344355, 24335433335, 24335433353, 24335433533, 24335434355, 24335435333, 24335435435, 24335443355, 24335443535, 24335443553, 24343333355, 24343333535, 24343333553, 24343335335, 24343335353, 24343335533, 24343343555, 24343353335, 24343353353, 24343353533, 24343354355, 24343355333, 24343355435, 24343433555, 24343435355, 24343435535, 24343435553, 24343533335, 24343533353, 24343533533, 24343534355, 24343535333, 24343535435, 24343543355, 24343543535, 24343543553, 24343553333, 24343553435, 24343554335, 24343554353, 24344333555, 24344335355, 24344335535, 24344335553, 24344353355, 24344353535, 24344353553, 24344355335, 24344355353, 24344355533, 24344435555, 24353333333, 24353333435, 24353334335, 24353334353, 24353343335, 24353343353, 24353343533, 24353344355, 24353433335, 24353433353, 24353433533, 24353434355, 24353435333, 24353435435, 24353443355, 24353443535, 24353443553, 24354333335, 24354333353, 24354333533, 24354334355, 24354335333, 24354335435, 24354343355, 24354343535, 24354343553, 24354353333, 24354353435, 24354354335, 24354354353, 24354433355, 24354433535, 24354433553, 24354435335, 24354435353, 24354435533, 24354443555, 24433333355, 24433333535, 24433333553, 24433335335, 24433335353, 24433335533, 24433343555, 24433353335, 24433353353, 24433353533, 24433354355, 24433355333, 24433355435, 24433433555, 24433435355, 24433435535, 24433435553, 24433533335, 24433533353, 24433533533, 24433534355, 24433535333, 24433535435, 24433543355, 24433543535, 24433543553, 24433553333, 24433553435, 24433554335, 24433554353, 24434333555, 24434335355, 24434335535, 24434335553, 24434353355, 24434353535, 24434353553, 24434355335, 24434355353, 24434355533, 24434435555, 24435333335, 24435333353, 24435333533, 24435334355, 24435335333, 24435335435, 24435343355, 24435343535, 24435343553, 24435353333, 24435353435, 24435354335, 24435354353, 24435433355, 24435433535, 24435433553, 24435435335, 24435435353, 24435435533, 24435443555, 24435533333, 24435533435, 24435534335, 24435534353, 24435543335, 24435543353, 24435543533, 24435544355, 24443333555, 24443335355, 24443335535, 24443335553, 24443353355, 24443353535, 24443353553, 24443355335, 24443355353, 24443355533, 24443435555, 24443533355, 24443533535, 24443533553, 24443535335, 24443535353, 24443535533, 24443543555, 24443553335, 24443553353, 24443553533, 24443554355, 24443555333, 24443555435, 24444335555, 24444353555, 24444355355, 24444355535, 24444355553, 333333333333, 333333333435, 333333334335, 333333343335, 333333344355, 333333433335, 333333434355, 333333435435, 333333443355, 333333443535, 333334333335, 333334334355, 333334335435, 333334343355, 333334343535, 333334353435, 333334354335, 333334433355, 333334433535, 333334435335, 333334443555, 333343334355, 333343335435, 333343343355, 333343343535, 333343353435, 333343354335, 333343433355, 333343433535, 333343435335, 333343443555, 333343533435, 333343534335, 333343543335, 333343544355, 333344333355, 333344333535, 333344335335, 333344343555, 333344353335, 333344354355, 333344355435, 333344433555, 333344435355, 333344435535, 333433343355, 333433343535, 333433353435, 333433354335, 333433433355, 333433433535, 333433435335, 333433443555, 333433533435, 333433534335, 333433544355, 333434333535, 333434335335, 333434343555, 333434354355, 333434355435, 333434433555, 333434435355, 333434435535, 333435333435, 333435334335, 333435344355, 333435434355, 333435435435, 333435443355, 333435443535, 333443343555, 333443354355, 333443355435, 333443433555, 333443435355, 333443435535, 333443534355, 333443535435, 333443543355, 333443543535, 333443553435, 333443554335, 333444333555, 333444335355, 333444335535, 333444353355, 333444353535, 333444355335, 333444435555, 334334335335, 334334343555, 334334354355, 334334355435, 334334433555, 334334435355, 334334435535, 334335344355, 334335434355, 334335435435, 334335443355, 334335443535, 334343343555, 334343354355, 334343355435, 334343433555, 334343435355, 334343435535, 334343534355, 334343535435, 334343543355, 334343543535, 334343553435, 334344335355, 334344335535, 334344353355, 334344353535, 334344435555, 334353344355, 334353434355, 334353435435, 334353443355, 334353443535, 334354334355, 334354335435, 334354343355, 334354343535, 334354353435, 334354443555, 334433543355, 334433543535, 334433553435, 334434335535, 334434353535, 334434435555, 334435335435, 334435343535, 334435353435, 334435443555, 334435544355, 334443435555, 334443543555, 334443554355, 334443555435, 334444335555, 334444353555, 334444355355, 334444355535, 343434353535, 343434435555, 343435343535, 343435443555, 343435544355, 343443435555, 343443543555, 343443554355, 343444353555, 343444355355, 343444355535, 343543443555, 343543544355, 343544343555, 343544354355, 344344355355, 344354354355, 344444355555]
13 : 3779
14 : 7501
15 : 13020
16 : 21270
17 : 31502
18 : 42502
19 : 54298
20 : 65413
21 : 72582
22 : 77574
23 : 79021
24 : 75076
25 : 69067
26 : 61279
27 : 50583
28 : 40991
29 : 32002
30 : 23281
31 : 16716
32 : 11677
33 : 7381
34 : 4850
35 : 2922
36 : 1688
37 : 946
38 : 546
39 : 239
40 : 148
41 : 56
42 : 31
43 : 11
44 : 6
45 : 1
46 : 1


Max: 3511111111111
0 : 1 : [-2]
1 : 1 : [3]
2 : 2 : [2, 33]
3 : 2 : [23, 333]
4 : 5 : [1, 22, 233, 3333, 3435]
5 : 6 : [13, 223, 2333, 2435, 33333, 33435]
6 : 14 : [0, 12, 133, 222, 2233, 2323, 2425, 23333, 23435, 24335, 333333, 333435, 334335, 344355]
7 : 19 : [123, 1333, 1435, 2223, 22333, 22435, 23233, 23425, 233333, 233435, 234335, 234353, 243335, 244355, 3333333, 3333435, 3334335, 3344355, 3434355]
8 : 44 : [11, 122, 1233, 1323, 1425, 2222, 13333, 13435, 14335, 22233, 22323, 22425, 223333, 223435, 224335, 232333, 232435, 233233, 233425, 234235, 234253, 234325, 244255, 2333333, 2333435, 2334335, 2334353, 2343335, 2343353, 2344355, 2433335, 2434355, 2435435, 2443355, 33333333, 33333435, 33334335, 33343335, 33344355, 33434355, 33435435, 33443355, 34343535, 34443555]
9 : 79 : [113, 1223, 1232, 12333, 12435, 13233, 13425, 14235, 22223, 133333, 133435, 134335, 134353, 143335, 144355, 222333, 222435, 223233, 223425, 224235, 232323, 232425, 2233333, 2233435, 2234335, 2234353, 2243335, 2244355, 2323333, 2323435, 2324335, 2332333, 2332435, 2333425, 2334235, 2334253, 2334325, 2342353, 2344255, 2424355, 2434255, 23333333, 23333435, 23334335, 23334353, 23343335, 23343353, 23343533, 23344355, 23433335, 23433353, 23434355, 23435435, 23443355, 23443535, 23443553, 24333335, 24334355, 24335435, 24343355, 24343535, 24353435, 24433355, 24443555, 333333333, 333333435, 333334335, 333343335, 333344355, 333434355, 333435435, 333443355, 334334355, 334343355, 334343535, 334353435, 334443555, 343443555, 344354355]
10 : 186 : [112, 1133, 1222, 1313, 1415, 12233, 12323, 12332, 12425, 13223, 14225, 22222, 123333, 123435, 124335, 124353, 132333, 132435, 133233, 133425, 134235, 134253, 134325, 142335, 143235, 144255, 222233, 222323, 222425, 223223, 224225, 1333333, 1333435, 1334335, 1334353, 1343335, 1343353, 1344355, 1433335, 1434355, 1435435, 1443355, 2223333, 2223435, 2224335, 2232333, 2232435, 2233233, 2233425, 2234235, 2234253, 2234325, 2242335, 2243235, 2244255, 2323233, 2323425, 2324235, 2332425, 2424255, 22333333, 22333435, 22334335, 22334353, 22343335, 22343353, 22344355, 22433335, 22434355, 22435435, 22443355, 23233333, 23233435, 23234335, 23234353, 23243335, 23244355, 23323333, 23323435, 23324335, 23332333, 23332435, 23333425, 23334235, 23334253, 23334325, 23342335, 23342353, 23342533, 23343235, 23343253, 23343325, 23344255, 23424355, 23425435, 23432353, 23434255, 23435425, 23442355, 23442535, 23442553, 23443255, 23443525, 24243355, 24243535, 24334255, 24342535, 24352435, 24442555, 233333333, 233333435, 233334335, 233334353, 233343335, 233343353, 233343533, 233344355, 233433335, 233433353, 233433533, 233434355, 233435435, 233443355, 233443535, 233443553, 234333335, 234333353, 234334355, 234335435, 234343355, 234343535, 234343553, 234353435, 234354335, 234354353, 234433355, 234433535, 234433553, 234435335, 234443555, 243333335, 243334355, 243335435, 243343355, 243343535, 243353435, 243354335, 243433355, 243433535, 243443555, 243533435, 243544355, 244333355, 244343555, 244354355, 244433555, 3333333333, 3333333435, 3333334335, 3333343335, 3333344355, 3333433335, 3333434355, 3333435435, 3333443355, 3334334355, 3334335435, 3334343355, 3334343535, 3334353435, 3334433355, 3334443555, 3343343355, 3343343535, 3343433535, 3343443555, 3343533435, 3343544355, 3344343555, 3344354355, 3344433555, 3434343555, 3434354355, 3434435355, 3434435535, 3444435555]
11 : 388 : [1123, 1213, 11333, 11435, 12223, 12232, 13133, 13415, 122333, 122435, 123233, 123323, 123332, 123425, 124235, 124253, 124325, 124352, 132233, 132323, 132425, 134225, 142235, 142325, 222223, 1233333, 1233435, 1234335, 1234353, 1243335, 1243353, 1243533, 1244355, 1323333, 1323435, 1324335, 1324353, 1332333, 1332435, 1333425, 1334235, 1334253, 1334325, 1342335, 1342353, 1343235, 1343325, 1344255, 1423335, 1424355, 1425435, 1432335, 1434255, 1442355, 2222333, 2222435, 2223233, 2223425, 2224235, 2232233, 2232323, 2232425, 2234225, 2242325, 13333333, 13333435, 13334335, 13334353, 13343335, 13343353, 13343533, 13344355, 13433335, 13433353, 13434355, 13435435, 13443355, 13443535, 13443553, 14333335, 14334355, 14335435, 14343355, 14343535, 14353435, 14433355, 14443555, 22233333, 22233435, 22234335, 22234353, 22243335, 22244355, 22323333, 22323435, 22324335, 22324353, 22332333, 22332435, 22333425, 22334235, 22334253, 22334325, 22342335, 22342353, 22343235, 22343325, 22344255, 22423335, 22424355, 22425435, 22432335, 22434255, 22442355, 23232333, 23232435, 23233233, 23233425, 23234235, 23234253, 23234325, 23242335, 23243235, 23244255, 23323425, 23324235, 23332425, 23424255, 23425425, 23442525, 24242535, 24243525, 223333333, 223333435, 223334335, 223334353, 223343335, 223343353, 223343533, 223344355, 223433335, 223433353, 223434355, 223435435, 223443355, 223443535, 223443553, 224333335, 224334355, 224335435, 224343355, 224343535, 224353435, 224433355, 224443555, 232333333, 232333435, 232334335, 232334353, 232343335, 232343353, 232344355, 232433335, 232434355, 232435435, 232443355, 233233333, 233233435, 233234335, 233234353, 233243335, 233244355, 233323333, 233323435, 233324335, 233332435, 233333425, 233334235, 233334253, 233334325, 233342335, 233342353, 233342533, 233343235, 233343253, 233343325, 233344255, 233423353, 233423533, 233424355, 233425435, 233432353, 233433235, 233434255, 233435425, 233442355, 233442535, 233442553, 233443255, 233443525, 234234355, 234235435, 234243355, 234243535, 234243553, 234253435, 234254335, 234254353, 234324355, 234334255, 234335425, 234342355, 234342535, 234342553, 234343255, 234343525, 234352435, 234353425, 234354325, 234423553, 234425335, 234432535, 234433525, 234442555, 242433355, 242433535, 242435335, 242443555, 243334255, 243342535, 243352435, 243442555, 243544255, 244254355, 2333333333, 2333333435, 2333334335, 2333334353, 2333343335, 2333343353, 2333343533, 2333344355, 2333433335, 2333433353, 2333433533, 2333434355, 2333435333, 2333435435, 2333443355, 2333443535, 2333443553, 2334333335, 2334333353, 2334333533, 2334334355, 2334335435, 2334343355, 2334343535, 2334343553, 2334353435, 2334354335, 2334354353, 2334433355, 2334433535, 2334433553, 2334435335, 2334435353, 2334435533, 2334443555, 2343333335, 2343333353, 2343334355, 2343335435, 2343343355, 2343343535, 2343343553, 2343353435, 2343354335, 2343354353, 2343433355, 2343433535, 2343433553, 2343435335, 2343435353, 2343443555, 2343533435, 2343534335, 2343534353, 2343543335, 2343544355, 2344333355, 2344333535, 2344333553, 2344335335, 2344343555, 2344353335, 2344354355, 2344355435, 2344433555, 2344435355, 2344435535, 2344435553, 2433333335, 2433334355, 2433335435, 2433343355, 2433343535, 2433353435, 2433354335, 2433433355, 2433433535, 2433435335, 2433443555, 2433533435, 2433534335, 2433544355, 2434333355, 2434333535, 2434343555, 2434354355, 2434355435, 2434433555, 2434435355, 2434435535, 2435333435, 2435344355, 2435434355, 2435435435, 2435443355, 2443333355, 2443343555, 2443354355, 2443433555, 2443435355, 2443534355, 2444333555, 2444435555, 33333333333, 33333333435, 33333334335, 33333343335, 33333344355, 33333433335, 33333434355, 33333435435, 33333443355, 33334334355, 33334335435, 33334343355, 33334343535, 33334353435, 33334433355, 33334443555, 33343334355, 33343343355, 33343343535, 33343353435, 33343354335, 33343433355, 33343433535, 33343443555, 33343533435, 33343544355, 33344343555, 33344354355, 33344433555, 33433433535, 33433435335, 33433443555, 33434343555, 33434354355, 33434355435, 33434433555, 33434435355, 33434435535, 33435344355, 33435434355, 33435435435, 33435443355, 33443354355, 33443435355, 33443534355, 33444435555, 34343435355, 34343534355, 34344435555, 34434435555, 34435443555]
12 : 955 : [111, 1122, 1212, 11233, 11323, 11425, 12133, 12222, 12313, 12415, 113333, 113435, 114335, 122233, 122323, 122332, 122425, 123223, 123232, 124225, 124252, 131333, 131435, 132223, 133133, 133415, 134135, 134153, 134315, 142225, 144155, 222222, 1223333, 1223435, 1224335, 1224353, 1232333, 1232435, 1233233, 1233323, 1233332, 1233425, 1234235, 1234253, 1234325, 1234352, 1242335, 1242353, 1242533, 1243235, 1243253, 1243325, 1243352, 1243523, 1244255, 1322333, 1322435, 1323233, 1323323, 1323425, 1324235, 1324253, 1324325, 1332233, 1332425, 1334225, 1342235, 1342253, 1342325, 1343225, 1422335, 1423235, 1423325, 1424255, 1425425, 1432235, 1442255, 2222233, 2222323, 2222425, 2223223, 2224225, 12333333, 12333435, 12334335, 12334353, 12343335, 12343353, 12343533, 12344355, 12433335, 12433353, 12433533, 12434355, 12435333, 12435435, 12443355, 12443535, 12443553, 13233333, 13233435, 13234335, 13234353, 13243335, 13243353, 13243533, 13244355, 13323333, 13323435, 13324335, 13324353, 13332333, 13332435, 13333425, 13334235, 13334253, 13334325, 13342335, 13342353, 13342533, 13343235, 13343253, 13343325, 13344255, 13423335, 13423353, 13424355, 13425435, 13432335, 13432353, 13433235, 13433325, 13434255, 13435425, 13442355, 13442535, 13442553, 13443255, 13443525, 14233335, 14234355, 14235435, 14243355, 14243535, 14253435, 14254335, 14323335, 14324355, 14325435, 14332335, 14334255, 14342355, 14342535, 14343255, 14352435, 14423355, 14432355, 14442555, 22223333, 22223435, 22224335, 22232333, 22232435, 22233233, 22233425, 22234235, 22234253, 22234325, 22242335, 22243235, 22244255, 22322333, 22322435, 22323233, 22323323, 22323425, 22324235, 22324253, 22324325, 22332233, 22332425, 22334225, 22342235, 22342253, 22342325, 22343225, 22423235, 22423325, 22424255, 22425425, 22442255, 23232323, 23232425, 23242325, 24242525, 133333333, 133333435, 133334335, 133334353, 133343335, 133343353, 133343533, 133344355, 133433335, 133433353, 133433533, 133434355, 133435435, 133443355, 133443535, 133443553, 134333335, 134333353, 134334355, 134335435, 134343355, 134343535, 134343553, 134353435, 134354335, 134354353, 134433355, 134433535, 134433553, 134435335, 134443555, 143333335, 143334355, 143335435, 143343355, 143343535, 143353435, 143354335, 143433355, 143433535, 143443555, 143533435, 143544355, 144333355, 144343555, 144354355, 144433555, 222333333, 222333435, 222334335, 222334353, 222343335, 222343353, 222344355, 222433335, 222434355, 222435435, 222443355, 223233333, 223233435, 223234335, 223234353, 223243335, 223243353, 223243533, 223244355, 223323333, 223323435, 223324335, 223324353, 223332333, 223332435, 223333425, 223334235, 223334253, 223334325, 223342335, 223342353, 223342533, 223343235, 223343253, 223343325, 223344255, 223423335, 223423353, 223424355, 223425435, 223432335, 223432353, 223433235, 223433325, 223434255, 223435425, 223442355, 223442535, 223442553, 223443255, 223443525, 224233335, 224234355, 224235435, 224243355, 224243535, 224253435, 224254335, 224323335, 224324355, 224325435, 224332335, 224334255, 224342355, 224342535, 224343255, 224352435, 224423355, 224432355, 224442555, 232323333, 232323435, 232324335, 232332333, 232332435, 232333425, 232334235, 232334253, 232334325, 232342335, 232342353, 232343235, 232343325, 232344255, 232423335, 232424355, 232425435, 232432335, 232434255, 232442355, 233233233, 233233425, 233234235, 233234253, 233234325, 233242335, 233243235, 233244255, 233323425, 233324235, 233332425, 233424255, 233425425, 233442525, 234234255, 234235425, 234242355, 234242535, 234242553, 234243255, 234243525, 234252435, 234253425, 234254253, 234254325, 234324255, 234342525, 234352425, 234432525, 242425335, 242433525, 242442555, 244254255, 2233333333, 2233333435, 2233334335, 2233334353, 2233343335, 2233343353, 2233343533, 2233344355, 2233433335, 2233433353, 2233433533, 2233434355, 2233435435, 2233443355, 2233443535, 2233443553, 2234333335, 2234333353, 2234334355, 2234335435, 2234343355, 2234343535, 2234343553, 2234353435, 2234354335, 2234354353, 2234433355, 2234433535, 2234433553, 2234435335, 2234443555, 2243333335, 2243334355, 2243335435, 2243343355, 2243343535, 2243353435, 2243354335, 2243433355, 2243433535, 2243443555, 2243533435, 2243544355, 2244333355, 2244343555, 2244354355, 2244433555, 2323333333, 2323333435, 2323334335, 2323334353, 2323343335, 2323343353, 2323343533, 2323344355, 2323433335, 2323433353, 2323434355, 2323435435, 2323443355, 2323443535, 2323443553, 2324333335, 2324334355, 2324335435, 2324343355, 2324343535, 2324353435, 2324433355, 2324443555, 2332333333, 2332333435, 2332334335, 2332334353, 2332343335, 2332343353, 2332344355, 2332433335, 2332434355, 2332435435, 2332443355, 2333233333, 2333233435, 2333234335, 2333234353, 2333243335, 2333244355, 2333323333, 2333323435, 2333324335, 2333332435, 2333333425, 2333334235, 2333334253, 2333334325, 2333342335, 2333342353, 2333342533, 2333343235, 2333343253, 2333343325, 2333344255, 2333423335, 2333423353, 2333423533, 2333424355, 2333425333, 2333425435, 2333432335, 2333432353, 2333432533, 2333433235, 2333433253, 2333433325, 2333434255, 2333435425, 2333442355, 2333442535, 2333442553, 2333443255, 2333443525, 2334233533, 2334234355, 2334235435, 2334243355, 2334243535, 2334243553, 2334253435, 2334254335, 2334254353, 2334323353, 2334323533, 2334324355, 2334325435, 2334332353, 2334334255, 2334335425, 2334342355, 2334342535, 2334342553, 2334343255, 2334343525, 2334352435, 2334353425, 2334354235, 2334354253, 2334354325, 2334423355, 2334423535, 2334423553, 2334425335, 2334425353, 2334425533, 2334432355, 2334432535, 2334432553, 2334433255, 2334433525, 2334435235, 2334435253, 2334435325, 2334442555, 2342343355, 2342343535, 2342343553, 2342353435, 2342354335, 2342354353, 2342433355, 2342433535, 2342433553, 2342435335, 2342435353, 2342443555, 2342533435, 2342534335, 2342534353, 2342543335, 2342543353, 2342544355, 2343234355, 2343235435, 2343243355, 2343243535, 2343334255, 2343335425, 2343342355, 2343342535, 2343342553, 2343343255, 2343343525, 2343352435, 2343353425, 2343354325, 2343423535, 2343423553, 2343425335, 2343425353, 2343432355, 2343432535, 2343433525, 2343435325, 2343442555, 2343523435, 2343524335, 2343524353, 2343532435, 2343533425, 2343544255, 2344243555, 2344253335, 2344254355, 2344255435, 2344323553, 2344325335, 2344333525, 2344342555, 2344354255, 2344355425, 2344423555, 2344425355, 2344425535, 2344425553, 2344432555, 2344435255, 2344435525, 2424333355, 2424333535, 2424335335, 2424343555, 2424353335, 2424354355, 2424355435, 2424433555, 2424435355, 2424435535, 2433334255, 2433342535, 2433352435, 2433425335, 2433442555, 2433524335, 2433544255, 2434243555, 2434254355, 2434342555, 2434354255, 2434425355, 2434425535, 2434435255, 2435244355, 2435344255, 2435424355, 2435425435, 2442543355, 2444425555, 23333333333, 23333333435, 23333334335, 23333334353, 23333343335, 23333343353, 23333343533, 23333344355, 23333433335, 23333433353, 23333433533, 23333434355, 23333435333, 23333435435, 23333443355, 23333443535, 23333443553, 23334333335, 23334333353, 23334333533, 23334334355, 23334335333, 23334335435, 23334343355, 23334343535, 23334343553, 23334353435, 23334354335, 23334354353, 23334433355, 23334433535, 23334433553, 23334435335, 23334435353, 23334435533, 23334443555, 23343333335, 23343333353, 23343333533, 23343334355, 23343335435, 23343343355, 23343343535, 23343343553, 23343353435, 23343354335, 23343354353, 23343433355, 23343433535, 23343433553, 23343435335, 23343435353, 23343435533, 23343443555, 23343533435, 23343534335, 23343534353, 23343543335, 23343543353, 23343543533, 23343544355, 23344333355, 23344333535, 23344333553, 23344335335, 23344335353, 23344335533, 23344343555, 23344353335, 23344353353, 23344354355, 23344355435, 23344433555, 23344435355, 23344435535, 23344435553, 23433333335, 23433333353, 23433334355, 23433335435, 23433343355, 23433343535, 23433343553, 23433353435, 23433354335, 23433354353, 23433433355, 23433433535, 23433433553, 23433435335, 23433435353, 23433443555, 23433533435, 23433534335, 23433534353, 23433543335, 23433543353, 23433544355, 23434333355, 23434333535, 23434333553, 23434335335, 23434335353, 23434343555, 23434353335, 23434354355, 23434355435, 23434433555, 23434435355, 23434435535, 23434435553, 23435333435, 23435334335, 23435334353, 23435343335, 23435344355, 23435433335, 23435434355, 23435435435, 23435443355, 23435443535, 23435443553, 23443333355, 23443333535, 23443333553, 23443335335, 23443343555, 23443353335, 23443354355, 23443355435, 23443433555, 23443435355, 23443435535, 23443435553, 23443533335, 23443534355, 23443535435, 23443543355, 23443543535, 23443543553, 23443553435, 23443554335, 23444333555, 23444335355, 23444335535, 23444335553, 23444353355, 23444353535, 23444355335, 23444435555, 24333333335, 24333334355, 24333335435, 24333343355, 24333343535, 24333353435, 24333354335, 24333433355, 24333433535, 24333435335, 24333443555, 24333533435, 24333534335, 24333543335, 24333544355, 24334333355, 24334333535, 24334335335, 24334343555, 24334354355, 24334355435, 24334433555, 24334435355, 24334435535, 24335333435, 24335334335, 24335344355, 24335434355, 24335435435, 24335443355, 24335443535, 24343333355, 24343333535, 24343343555, 24343354355, 24343355435, 24343433555, 24343435355, 24343435535, 24343534355, 24343535435, 24343543355, 24343543535, 24343553435, 24344333555, 24344335355, 24344335535, 24344353355, 24344435555, 24353333435, 24353344355, 24353434355, 24353435435, 24353443355, 24354334355, 24354335435, 24354343355, 24354433355, 24354443555, 24433333355, 24433343555, 24433354355, 24433433555, 24433435355, 24433534355, 24433543355, 24434333555, 24434335355, 24434435555, 24435334355, 24435443555, 24435544355, 24443333555, 24443435555, 24443543555, 24444335555, 333333333333, 333333333435, 333333334335, 333333343335, 333333344355, 333333433335, 333333434355, 333333435435, 333333443355, 333334333335, 333334334355, 333334335435, 333334343355, 333334343535, 333334353435, 333334433355, 333334443555, 333343334355, 333343335435, 333343343355, 333343343535, 333343353435, 333343354335, 333343433355, 333343433535, 333343443555, 333343533435, 333343544355, 333344333355, 333344343555, 333344354355, 333344433555, 333433343355, 333433343535, 333433433355, 333433433535, 333433435335, 333433443555, 333433533435, 333433534335, 333433544355, 333434333535, 333434343555, 333434354355, 333434355435, 333434433555, 333434435355, 333434435535, 333435333435, 333435344355, 333435434355, 333435435435, 333435443355, 333443343555, 333443354355, 333443433555, 333443435355, 333443534355, 333444333555, 333444435555, 334334335335, 334334343555, 334334354355, 334334355435, 334334433555, 334334435355, 334334435535, 334343343555, 334343354355, 334343433555, 334343435355, 334343435535, 334343534355, 334343535435, 334343543355, 334343543535, 334343553435, 334344335355, 334344335535, 334344353355, 334344435555, 334353344355, 334353434355, 334353435435, 334353443355, 334354334355, 334354335435, 334354443555, 334433543355, 334434435555, 334435443555, 334435544355, 334443435555, 334443543555, 334444335555, 343434353535, 343434435555, 343435343535, 343435443555, 343443435555, 343443543555, 343443554355, 343444353555, 343444355355, 343444355535, 344344355355, 344354354355, 344444355555]
13 : 2218 : [1113, 11223, 11232, 12123, 12213, 112333, 112435, 113233, 113425, 114235, 121333, 121435, 122223, 122232, 122322, 123133, 123313, 123415, 124135, 124153, 124315, 131323, 131425, 132415, 1133333, 1133435, 1134335, 1134353, 1143335, 1144355, 1222333, 1222435, 1223233, 1223323, 1223332, 1223425, 1224235, 1224253, 1224325, 1224352, 1232233, 1232323, 1232332, 1232425, 1233223, 1234225, 1234252, 1242235, 1242253, 1242325, 1242352, 1242523, 1243225, 1313333, 1313435, 1314335, 1322233, 1322323, 1322425, 1324225, 1331333, 1331435, 1333415, 1334135, 1334153, 1334315, 1341353, 1342225, 1344155, 1414355, 1422235, 1422325, 1434155, 2222223, 12233333, 12233435, 12234335, 12234353, 12243335, 12243353, 12243533, 12244355, 12323333, 12323435, 12324335, 12324353, 12332333, 12332435, 12333233, 12333323, 12333332, 12333425, 12334235, 12334253, 12334325, 12334352, 12342335, 12342353, 12342533, 12343235, 12343253, 12343325, 12343352, 12343523, 12343532, 12344255, 12423335, 12423353, 12423533, 12424355, 12425333, 12425435, 12432335, 12432353, 12432533, 12433235, 12433253, 12433325, 12433352, 12433523, 12434255, 12435233, 12435323, 12435425, 12442355, 12442535, 12442553, 12443255, 12443525, 12443552, 13223333, 13223435, 13224335, 13224353, 13232333, 13232435, 13233233, 13233323, 13233425, 13234235, 13234253, 13234325, 13242335, 13242353, 13242533, 13243235, 13243253, 13243325, 13243523, 13244255, 13322333, 13322435, 13323233, 13323425, 13324235, 13324253, 13324325, 13332425, 13334225, 13342235, 13342253, 13342325, 13343225, 13422335, 13422353, 13423235, 13423253, 13423325, 13424255, 13425425, 13432235, 13432325, 13433225, 13442255, 13442525, 14223335, 14224355, 14225435, 14232335, 14233235, 14233325, 14234255, 14235425, 14242355, 14242535, 14243255, 14243525, 14252435, 14253425, 14254235, 14322335, 14323235, 14324255, 14342255, 14422355, 14423255, 22222333, 22222435, 22223233, 22223425, 22224235, 22232233, 22232323, 22232425, 22234225, 22242235, 22242325, 22322323, 22322425, 22324225, 123333333, 123333435, 123334335, 123334353, 123343335, 123343353, 123343533, 123344355, 123433335, 123433353, 123433533, 123434355, 123435333, 123435435, 123443355, 123443535, 123443553, 124333335, 124333353, 124333533, 124334355, 124335333, 124335435, 124343355, 124343535, 124343553, 124353333, 124353435, 124354335, 124354353, 124433355, 124433535, 124433553, 124435335, 124435353, 124435533, 124443555, 132333333, 132333435, 132334335, 132334353, 132343335, 132343353, 132343533, 132344355, 132433335, 132433353, 132433533, 132434355, 132435333, 132435435, 132443355, 132443535, 132443553, 133233333, 133233435, 133234335, 133234353, 133243335, 133243353, 133243533, 133244355, 133323333, 133323435, 133324335, 133324353, 133332435, 133333425, 133334235, 133334253, 133334325, 133342335, 133342353, 133342533, 133343235, 133343253, 133343325, 133344255, 133423335, 133423353, 133423533, 133424355, 133425435, 133432335, 133432353, 133433235, 133433253, 133433325, 133434255, 133435425, 133442355, 133442535, 133442553, 133443255, 133443525, 134233335, 134233353, 134234355, 134235435, 134243355, 134243535, 134243553, 134253435, 134254335, 134254353, 134323335, 134323353, 134324355, 134325435, 134332335, 134333235, 134333325, 134334255, 134335425, 134342355, 134342535, 134342553, 134343255, 134343525, 134352435, 134353425, 134354235, 134354325, 134423355, 134423535, 134423553, 134425335, 134432355, 134432535, 134433255, 134433525, 134435235, 134435325, 134442555, 142333335, 142334355, 142335435, 142343355, 142343535, 142353435, 142354335, 142433355, 142433535, 142435335, 142443555, 142533435, 142534335, 142543335, 142544355, 143233335, 143234355, 143235435, 143243355, 143243535, 143253435, 143254335, 143323335, 143324355, 143325435, 143334255, 143342355, 143342535, 143343255, 143352435, 143423355, 143423535, 143432355, 143433255, 143442555, 143523435, 143544255, 144233355, 144243555, 144254355, 144323355, 144342555, 144423555, 222233333, 222233435, 222234335, 222234353, 222243335, 222244355, 222323333, 222323435, 222324335, 222324353, 222332333, 222332435, 222333425, 222334235, 222334253, 222334325, 222342335, 222342353, 222343235, 222343325, 222344255, 222423335, 222424355, 222425435, 222432335, 222434255, 222442355, 223223333, 223223435, 223224335, 223232333, 223232435, 223233233, 223233323, 223233425, 223234235, 223234253, 223234325, 223242335, 223242353, 223242533, 223243235, 223243253, 223243325, 223243523, 223244255, 223322333, 223322435, 223323233, 223323425, 223324235, 223324253, 223324325, 223332425, 223334225, 223342235, 223342253, 223342325, 223343225, 223422353, 223423235, 223423253, 223423325, 223424255, 223425425, 223432325, 223442255, 223442525, 224224355, 224232335, 224233235, 224233325, 224234255, 224235425, 224242355, 224242535, 224243255, 224243525, 224252435, 224253425, 224254235, 224323235, 224324255, 224342255, 224423255, 232323233, 232323425, 232324235, 232332425, 232342325, 232423325, 232424255, 232425425, 234242525, 234252425, 1333333333, 1333333435, 1333334335, 1333334353, 1333343335, 1333343353, 1333343533, 1333344355, 1333433335, 1333433353, 1333433533, 1333434355, 1333435333, 1333435435, 1333443355, 1333443535, 1333443553, 1334333335, 1334333353, 1334333533, 1334334355, 1334335435, 1334343355, 1334343535, 1334343553, 1334353435, 1334354335, 1334354353, 1334433355, 1334433535, 1334433553, 1334435335, 1334435353, 1334435533, 1334443555, 1343333335, 1343333353, 1343334355, 1343335435, 1343343355, 1343343535, 1343343553, 1343353435, 1343354335, 1343354353, 1343433355, 1343433535, 1343433553, 1343435335, 1343435353, 1343443555, 1343533435, 1343534335, 1343534353, 1343543335, 1343544355, 1344333355, 1344333535, 1344333553, 1344335335, 1344343555, 1344353335, 1344354355, 1344355435, 1344433555, 1344435355, 1344435535, 1344435553, 1433333335, 1433334355, 1433335435, 1433343355, 1433343535, 1433353435, 1433354335, 1433433355, 1433433535, 1433435335, 1433443555, 1433533435, 1433534335, 1433544355, 1434333355, 1434333535, 1434343555, 1434354355, 1434355435, 1434433555, 1434435355, 1434435535, 1435333435, 1435344355, 1435434355, 1435435435, 1435443355, 1443333355, 1443343555, 1443354355, 1443433555, 1443435355, 1443534355, 1444333555, 1444435555, 2223333333, 2223333435, 2223334335, 2223334353, 2223343335, 2223343353, 2223343533, 2223344355, 2223433335, 2223433353, 2223434355, 2223435435, 2223443355, 2223443535, 2223443553, 2224333335, 2224334355, 2224335435, 2224343355, 2224343535, 2224353435, 2224433355, 2224443555, 2232333333, 2232333435, 2232334335, 2232334353, 2232343335, 2232343353, 2232343533, 2232344355, 2232433335, 2232433353, 2232433533, 2232434355, 2232435333, 2232435435, 2232443355, 2232443535, 2232443553, 2233233333, 2233233435, 2233234335, 2233234353, 2233243335, 2233243353, 2233243533, 2233244355, 2233323333, 2233323435, 2233324335, 2233324353, 2233332435, 2233333425, 2233334235, 2233334253, 2233334325, 2233342335, 2233342353, 2233342533, 2233343235, 2233343253, 2233343325, 2233344255, 2233423335, 2233423353, 2233423533, 2233424355, 2233425435, 2233432335, 2233432353, 2233433235, 2233433253, 2233433325, 2233434255, 2233435425, 2233442355, 2233442535, 2233442553, 2233443255, 2233443525, 2234233335, 2234233353, 2234234355, 2234235435, 2234243355, 2234243535, 2234243553, 2234253435, 2234254335, 2234254353, 2234323335, 2234323353, 2234324355, 2234325435, 2234332335, 2234333235, 2234333325, 2234334255, 2234335425, 2234342355, 2234342535, 2234342553, 2234343255, 2234343525, 2234352435, 2234353425, 2234354235, 2234354325, 2234423355, 2234423535, 2234423553, 2234425335, 2234432355, 2234432535, 2234433255, 2234433525, 2234435235, 2234435325, 2234442555, 2242333335, 2242334355, 2242335435, 2242343355, 2242343535, 2242353435, 2242354335, 2242433355, 2242433535, 2242435335, 2242443555, 2242533435, 2242534335, 2242543335, 2242544355, 2243233335, 2243234355, 2243235435, 2243243355, 2243243535, 2243253435, 2243254335, 2243323335, 2243324355, 2243325435, 2243334255, 2243342355, 2243342535, 2243343255, 2243352435, 2243423355, 2243423535, 2243432355, 2243433255, 2243442555, 2243523435, 2243544255, 2244233355, 2244243555, 2244254355, 2244323355, 2244342555, 2244423555, 2323233333, 2323233435, 2323234335, 2323234353, 2323243335, 2323244355, 2323323333, 2323323435, 2323324335, 2323324353, 2323332333, 2323332435, 2323333425, 2323334235, 2323334253, 2323334325, 2323342335, 2323342353, 2323342533, 2323343235, 2323343253, 2323343325, 2323344255, 2323423335, 2323423353, 2323424355, 2323425435, 2323432335, 2323432353, 2323433235, 2323433325, 2323434255, 2323435425, 2323442355, 2323442535, 2323442553, 2323443255, 2323443525, 2324233335, 2324234355, 2324235435, 2324243355, 2324243535, 2324253435, 2324254335, 2324323335, 2324324355, 2324325435, 2324332335, 2324334255, 2324342355, 2324342535, 2324343255, 2324352435, 2324423355, 2324432355, 2324442555, 2332332333, 2332332435, 2332333425, 2332334235, 2332334253, 2332334325, 2332342335, 2332342353, 2332343235, 2332343325, 2332344255, 2332423335, 2332424355, 2332425435, 2332432335, 2332434255, 2332442355, 2333233425, 2333234235, 2333234253, 2333234325, 2333242335, 2333243235, 2333244255, 2333323425, 2333324235, 2333332425, 2333424255, 2333425425, 2333442525, 2334234255, 2334235425, 2334242355, 2334242535, 2334242553, 2334243255, 2334243525, 2334252435, 2334253425, 2334254235, 2334254253, 2334254325, 2334324255, 2334325425, 2334342525, 2334352425, 2334423525, 2334425235, 2334425253, 2334425325, 2334432525, 2342342355, 2342342535, 2342342553, 2342343255, 2342343525, 2342352435, 2342353425, 2342354253, 2342354325, 2342423535, 2342423553, 2342425335, 2342425353, 2342432355, 2342432535, 2342433525, 2342435253, 2342435325, 2342442555, 2342523435, 2342524335, 2342524353, 2342532435, 2342533425, 2342534253, 2342534325, 2342542353, 2342544255, 2343235425, 2343242355, 2343242535, 2343243525, 2343342525, 2343352425, 2343432525, 2343532425, 2344242555, 2344254255, 2344255425, 2344425255, 2344425525, 2424243555, 2424253335, 2424254355, 2424333525, 2424342555, 2424354255, 2424425355, 2424425535, 2424435255, 2424435525, 2434254255, 2435244255, 22333333333, 22333333435, 22333334335, 22333334353, 22333343335, 22333343353, 22333343533, 22333344355, 22333433335, 22333433353, 22333433533, 22333434355, 22333435333, 22333435435, 22333443355, 22333443535, 22333443553, 22334333335, 22334333353, 22334333533, 22334334355, 22334335435, 22334343355, 22334343535, 22334343553, 22334353435, 22334354335, 22334354353, 22334433355, 22334433535, 22334433553, 22334435335, 22334435353, 22334435533, 22334443555, 22343333335, 22343333353, 22343334355, 22343335435, 22343343355, 22343343535, 22343343553, 22343353435, 22343354335, 22343354353, 22343433355, 22343433535, 22343433553, 22343435335, 22343435353, 22343443555, 22343533435, 22343534335, 22343534353, 22343543335, 22343544355, 22344333355, 22344333535, 22344333553, 22344335335, 22344343555, 22344353335, 22344354355, 22344355435, 22344433555, 22344435355, 22344435535, 22344435553, 22433333335, 22433334355, 22433335435, 22433343355, 22433343535, 22433353435, 22433354335, 22433433355, 22433433535, 22433435335, 22433443555, 22433533435, 22433534335, 22433544355, 22434333355, 22434333535, 22434343555, 22434354355, 22434355435, 22434433555, 22434435355, 22434435535, 22435333435, 22435344355, 22435434355, 22435435435, 22435443355, 22443333355, 22443343555, 22443354355, 22443433555, 22443435355, 22443534355, 22444333555, 22444435555, 23233333333, 23233333435, 23233334335, 23233334353, 23233343335, 23233343353, 23233343533, 23233344355, 23233433335, 23233433353, 23233433533, 23233434355, 23233435435, 23233443355, 23233443535, 23233443553, 23234333335, 23234333353, 23234334355, 23234335435, 23234343355, 23234343535, 23234343553, 23234353435, 23234354335, 23234354353, 23234433355, 23234433535, 23234433553, 23234435335, 23234443555, 23243333335, 23243334355, 23243335435, 23243343355, 23243343535, 23243353435, 23243354335, 23243433355, 23243433535, 23243443555, 23243533435, 23243544355, 23244333355, 23244343555, 23244354355, 23244433555, 23323333333, 23323333435, 23323334335, 23323334353, 23323343335, 23323343353, 23323343533, 23323344355, 23323433335, 23323433353, 23323434355, 23323435435, 23323443355, 23323443535, 23323443553, 23324333335, 23324334355, 23324335435, 23324343355, 23324343535, 23324353435, 23324433355, 23324443555, 23332333333, 23332333435, 23332334335, 23332334353, 23332343335, 23332343353, 23332344355, 23332433335, 23332434355, 23332435435, 23332443355, 23333233333, 23333233435, 23333234335, 23333234353, 23333243335, 23333244355, 23333323435, 23333324335, 23333332435, 23333333425, 23333334235, 23333334253, 23333334325, 23333342335, 23333342353, 23333342533, 23333343235, 23333343253, 23333343325, 23333344255, 23333423335, 23333423353, 23333423533, 23333424355, 23333425333, 23333425435, 23333432335, 23333432353, 23333432533, 23333433235, 23333433253, 23333433325, 23333434255, 23333435425, 23333442355, 23333442535, 23333442553, 23333443255, 23333443525, 23334233353, 23334233533, 23334234355, 23334235333, 23334235435, 23334243355, 23334243535, 23334243553, 23334253435, 23334254335, 23334254353, 23334323353, 23334323533, 23334324355, 23334325435, 23334332335, 23334332353, 23334332533, 23334333235, 23334334255, 23334335425, 23334342355, 23334342535, 23334342553, 23334343255, 23334343525, 23334352435, 23334353425, 23334354235, 23334354253, 23334354325, 23334423355, 23334423535, 23334423553, 23334425335, 23334425353, 23334425533, 23334432355, 23334432535, 23334432553, 23334433255, 23334433525, 23334435235, 23334435253, 23334435325, 23334442555, 23342334355, 23342335435, 23342343355, 23342343535, 23342343553, 23342353435, 23342354335, 23342354353, 23342433355, 23342433535, 23342433553, 23342435335, 23342435353, 23342435533, 23342443555, 23342533435, 23342534335, 23342534353, 23342543335, 23342543353, 23342543533, 23342544355, 23343233533, 23343234355, 23343235435, 23343243355, 23343243535, 23343243553, 23343253435, 23343254335, 23343254353, 23343324355, 23343334255, 23343335425, 23343342355, 23343342535, 23343342553, 23343343255, 23343343525, 23343352435, 23343353425, 23343354235, 23343354253, 23343354325, 23343423355, 23343423535, 23343423553, 23343425335, 23343425353, 23343425533, 23343432355, 23343432535, 23343432553, 23343433255, 23343433525, 23343435235, 23343435253, 23343435325, 23343442555, 23343523435, 23343524335, 23343524353, 23343532435, 23343533425, 23343534235, 23343534253, 23343534325, 23343542353, 23343543235, 23343543253, 23343543325, 23343544255, 23344233553, 23344235335, 23344235353, 23344235533, 23344243555, 23344253335, 23344253353, 23344254355, 23344255435, 23344323535, 23344323553, 23344325335, 23344325353, 23344332355, 23344332535, 23344333525, 23344335235, 23344335253, 23344335325, 23344342555, 23344352353, 23344353235, 23344353253, 23344354255, 23344355425, 23344423555, 23344425355, 23344425535, 23344425553, 23344432555, 23344435255, 23344435525, 23423433355, 23423433535, 23423433553, 23423435335, 23423435353, 23423443555, 23423533435, 23423534335, 23423534353, 23423543335, 23423543353, 23423544355, 23424333355, 23424333535, 23424333553, 23424335335, 23424335353, 23424343555, 23424353335, 23424353353, 23424354355, 23424355435, 23424433555, 23424435355, 23424435535, 23424435553, 23425333435, 23425334335, 23425334353, 23425343335, 23425343353, 23425344355, 23425433335, 23425433353, 23425434355, 23425435435, 23425443355, 23425443535, 23425443553, 23432343355, 23432343535, 23432343553, 23432353435, 23432354335, 23432433355, 23432433535, 23432435335, 23432443555, 23433334255, 23433335425, 23433342355, 23433342535, 23433342553, 23433343255, 23433343525, 23433352435, 23433353425, 23433354325, 23433423535, 23433423553, 23433425335, 23433425353, 23433432355, 23433432535, 23433433525, 23433435325, 23433442555, 23433523435, 23433524335, 23433524353, 23433532435, 23433533425, 23433534325, 23433544255, 23434235353, 23434243555, 23434253335, 23434254355, 23434255435, 23434323553, 23434325335, 23434333525, 23434342555, 23434354255, 23434355425, 23434423555, 23434425355, 23434425535, 23434425553, 23434432555, 23434435255, 23434435525, 23435234353, 23435243335, 23435244355, 23435324335, 23435333425, 23435344255, 23435424355, 23435425435, 23435434255, 23435435425, 23435442355, 23435442535, 23435442553, 23435443255, 23435443525, 23442354355, 23442355435, 23442433555, 23442435355, 23442435535, 23442435553, 23442533335, 23442534355, 23442535435, 23442543355, 23442543535, 23442543553, 23442553435, 23442554335, 23443243555, 23443253335, 23443254355, 23443333525, 23443342555, 23443354255, 23443355425, 23443423555, 23443425355, 23443425535, 23443425553, 23443435255, 23443435525, 23443524355, 23443525435, 23443534255, 23443535425, 23443542535, 23443543255, 23443543525, 23443552435, 23443553425, 23443554325, 23444235553, 23444253355, 23444253535, 23444255335, 23444325355, 23444335255, 23444335525, 23444352535, 23444353525, 23444425555, 24243333355, 24243333535, 24243335335, 24243343555, 24243353335, 24243354355, 24243355435, 24243433555, 24243435355, 24243435535, 24243533335, 24243534355, 24243535435, 24243543355, 24243543535, 24244333555, 24244335355, 24244335535, 24244353355, 24244353535, 24244355335, 24244435555, 24333334255, 24333342535, 24333352435, 24333425335, 24333442555, 24333524335, 24333544255, 24334243555, 24334254355, 24334342555, 24334354255, 24334425355, 24334425535, 24334435255, 24335244355, 24335344255, 24335424355, 24335425435, 24335434255, 24335442535, 24342435355, 24342435535, 24342534355, 24342543355, 24343354255, 24343425355, 24343435255, 24343524355, 24343525435, 24343534255, 24343552435, 24344425555, 24352435435, 24352443355, 24353344255, 24354442555, 24424435555, 24425433355, 24425443555, 24434425555, 24435442555, 233333333333, 233333333435, 233333334335, 233333334353, 233333343335, 233333343353, 233333343533, 233333344355, 233333433335, 233333433353, 233333433533, 233333434355, 233333435333, 233333435435, 233333443355, 233333443535, 233333443553, 233334333335, 233334333353, 233334333533, 233334334355, 233334335333, 233334335435, 233334343355, 233334343535, 233334343553, 233334353333, 233334353435, 233334354335, 233334354353, 233334433355, 233334433535, 233334433553, 233334435335, 233334435353, 233334435533, 233334443555, 233343333335, 233343333353, 233343333533, 233343334355, 233343335333, 233343335435, 233343343355, 233343343535, 233343343553, 233343353435, 233343354335, 233343354353, 233343433355, 233343433535, 233343433553, 233343435335, 233343435353, 233343435533, 233343443555, 233343533435, 233343534335, 233343534353, 233343543335, 233343543353, 233343543533, 233343544355, 233344333355, 233344333535, 233344333553, 233344335335, 233344335353, 233344335533, 233344343555, 233344353335, 233344353353, 233344353533, 233344354355, 233344355333, 233344355435, 233344433555, 233344435355, 233344435535, 233344435553, 233433333335, 233433333353, 233433333533, 233433334355, 233433335435, 233433343355, 233433343535, 233433343553, 233433353435, 233433354335, 233433354353, 233433433355, 233433433535, 233433433553, 233433435335, 233433435353, 233433435533, 233433443555, 233433533435, 233433534335, 233433534353, 233433543335, 233433543353, 233433543533, 233433544355, 233434333355, 233434333535, 233434333553, 233434335335, 233434335353, 233434335533, 233434343555, 233434353335, 233434353353, 233434353533, 233434354355, 233434355435, 233434433555, 233434435355, 233434435535, 233434435553, 233435333435, 233435334335, 233435334353, 233435343335, 233435343353, 233435343533, 233435344355, 233435433335, 233435433353, 233435434355, 233435435435, 233435443355, 233435443535, 233435443553, 233443333355, 233443333535, 233443333553, 233443335335, 233443335353, 233443335533, 233443343555, 233443353335, 233443353353, 233443354355, 233443355435, 233443433555, 233443435355, 233443435535, 233443435553, 233443533335, 233443533353, 233443534355, 233443535435, 233443543355, 233443543535, 233443543553, 233443553435, 233443554335, 233443554353, 233444333555, 233444335355, 233444335535, 233444335553, 233444353355, 233444353535, 233444353553, 233444355335, 233444355353, 233444355533, 233444435555, 234333333335, 234333333353, 234333334355, 234333335435, 234333343355, 234333343535, 234333343553, 234333353435, 234333354335, 234333354353, 234333433355, 234333433535, 234333433553, 234333435335, 234333435353, 234333443555, 234333533435, 234333534335, 234333534353, 234333543335, 234333543353, 234333544355, 234334333355, 234334333535, 234334333553, 234334335335, 234334335353, 234334343555, 234334353335, 234334353353, 234334354355, 234334355435, 234334433555, 234334435355, 234334435535, 234334435553, 234335333435, 234335334335, 234335334353, 234335343335, 234335343353, 234335344355, 234335433335, 234335434355, 234335435435, 234335443355, 234335443535, 234335443553, 234343333355, 234343333535, 234343333553, 234343335335, 234343335353, 234343343555, 234343353335, 234343354355, 234343355435, 234343433555, 234343435355, 234343435535, 234343435553, 234343533335, 234343534355, 234343535435, 234343543355, 234343543535, 234343543553, 234343553435, 234343554335, 234343554353, 234344333555, 234344335355, 234344335535, 234344335553, 234344353355, 234344353535, 234344353553, 234344355335, 234344355353, 234344435555, 234353333435, 234353334335, 234353334353, 234353343335, 234353344355, 234353433335, 234353434355, 234353435435, 234353443355, 234353443535, 234353443553, 234354333335, 234354334355, 234354335435, 234354343355, 234354343535, 234354343553, 234354353435, 234354354335, 234354354353, 234354433355, 234354433535, 234354433553, 234354435335, 234354443555, 234433333355, 234433333535, 234433333553, 234433335335, 234433343555, 234433353335, 234433354355, 234433355435, 234433433555, 234433435355, 234433435535, 234433435553, 234433533335, 234433534355, 234433535435, 234433543355, 234433543535, 234433543553, 234433553435, 234433554335, 234434333555, 234434335355, 234434335535, 234434335553, 234434353355, 234434353535, 234434353553, 234434355335, 234434435555, 234435333335, 234435334355, 234435335435, 234435343355, 234435343535, 234435343553, 234435353435, 234435354335, 234435433355, 234435433535, 234435435335, 234435443555, 234435533435, 234435534335, 234435543335, 234435544355, 234443333555, 234443335355, 234443335535, 234443335553, 234443353355, 234443353535, 234443355335, 234443435555, 234443533355, 234443533535, 234443535335, 234443543555, 234443553335, 234443554355, 234443555435, 234444335555, 234444353555, 234444355355, 234444355535, 234444355553, 243333333335, 243333334355, 243333335435, 243333343355, 243333343535, 243333353435, 243333354335, 243333433355, 243333433535, 243333435335, 243333443555, 243333533435, 243333534335, 243333543335, 243333544355, 243334333355, 243334333535, 243334335335, 243334343555, 243334353335, 243334354355, 243334355435, 243334433555, 243334435355, 243334435535, 243335333435, 243335334335, 243335343335, 243335344355, 243335434355, 243335435435, 243335443355, 243335443535, 243343333355, 243343333535, 243343335335, 243343343555, 243343354355, 243343355435, 243343433555, 243343435355, 243343435535, 243343534355, 243343535435, 243343543355, 243343543535, 243343553435, 243343554335, 243344333555, 243344335355, 243344335535, 243344353355, 243344353535, 243344355335, 243344435555, 243353333435, 243353334335, 243353344355, 243353434355, 243353435435, 243353443355, 243353443535, 243354334355, 243354335435, 243354343355, 243354343535, 243354353435, 243354354335, 243354433355, 243354433535, 243354443555, 243433333355, 243433333535, 243433343555, 243433354355, 243433355435, 243433433555, 243433435355, 243433435535, 243433534355, 243433535435, 243433543355, 243433543535, 243433553435, 243434333555, 243434335355, 243434335535, 243434353355, 243434353535, 243434435555, 243435334355, 243435335435, 243435343355, 243435343535, 243435353435, 243435433355, 243435443555, 243435533435, 243435544355, 243443333555, 243443335355, 243443335535, 243443353355, 243443435555, 243443533355, 243443543555, 243443554355, 243443555435, 243444335555, 243444353555, 243444355355, 243444355535, 243533333435, 243533344355, 243533434355, 243533435435, 243533443355, 243534334355, 243534335435, 243534343355, 243534353435, 243534433355, 243534443555, 243543334355, 243543335435, 243543343355, 243543433355, 243543443555, 243543544355, 243544333355, 243544343555, 243544354355, 243544355435, 243544433555, 243544435355, 244333333355, 244333343555, 244333354355, 244333433555, 244333435355, 244333534355, 244333543355, 244334333555, 244334335355, 244334353355, 244334435555, 244335334355, 244335343355, 244335443555, 244335544355, 244343333555, 244343335355, 244343435555, 244343543555, 244343554355, 244344335555, 244344353555, 244344355355, 244353334355, 244353443555, 244353544355, 244354343555, 244354354355, 244354433555, 244355344355, 244433333555, 244433435555, 244433543555, 244434335555, 244434353555, 244435343555, 244443335555, 244444355555, 3333333333333, 3333333333435, 3333333334335, 3333333343335, 3333333344355, 3333333433335, 3333333434355, 3333333435435, 3333333443355, 3333334333335, 3333334334355, 3333334335435, 3333334343355, 3333334343535, 3333334353435, 3333334433355, 3333334443555, 3333343334355, 3333343335435, 3333343343355, 3333343343535, 3333343353435, 3333343354335, 3333343433355, 3333343433535, 3333343443555, 3333343533435, 3333343544355, 3333344333355, 3333344343555, 3333344354355, 3333344433555, 3333433334355, 3333433343355, 3333433343535, 3333433353435, 3333433354335, 3333433433355, 3333433433535, 3333433435335, 3333433443555, 3333433533435, 3333433534335, 3333433544355, 3333434333355, 3333434333535, 3333434343555, 3333434354355, 3333434355435, 3333434433555, 3333434435355, 3333434435535, 3333435333435, 3333435344355, 3333435434355, 3333435435435, 3333435443355, 3333443343555, 3333443354355, 3333443433555, 3333443435355, 3333443534355, 3333444333555, 3333444435555, 3334333433355, 3334333433535, 3334333435335, 3334333443555, 3334334333535, 3334334335335, 3334334343555, 3334334354355, 3334334355435, 3334334433555, 3334334435355, 3334334435535, 3334335333435, 3334335334335, 3334335344355, 3334335434355, 3334335435435, 3334335443355, 3334335443535, 3334343343555, 3334343354355, 3334343355435, 3334343433555, 3334343435355, 3334343435535, 3334343534355, 3334343535435, 3334343543355, 3334343543535, 3334343553435, 3334344333555, 3334344335355, 3334344335535, 3334344353355, 3334344435555, 3334353344355, 3334353434355, 3334353435435, 3334353443355, 3334354334355, 3334354335435, 3334354343355, 3334354433355, 3334354443555, 3334433354355, 3334433433555, 3334433435355, 3334433534355, 3334433543355, 3334434335355, 3334434435555, 3334435334355, 3334435443555, 3334435544355, 3334443435555, 3334443543555, 3334444335555, 3343343343555, 3343343354355, 3343343433555, 3343343435355, 3343343435535, 3343343534355, 3343343535435, 3343343543355, 3343343543535, 3343344335355, 3343344335535, 3343344353355, 3343344353535, 3343344355335, 3343344435555, 3343433435355, 3343433435535, 3343433534355, 3343433543355, 3343434335355, 3343434353355, 3343434353535, 3343434435555, 3343435334355, 3343435335435, 3343435343355, 3343435343535, 3343435353435, 3343435443555, 3343435533435, 3343435544355, 3343443435555, 3343443543555, 3343443554355, 3343443555435, 3343444335555, 3343444353555, 3343444355355, 3343444355535, 3343533435435, 3343533443355, 3343534353435, 3343534443555, 3343543443555, 3343543544355, 3343544343555, 3343544354355, 3343544355435, 3343544433555, 3343544435355, 3344334435555, 3344335443555, 3344343435555, 3344343543555, 3344343554355, 3344344335555, 3344344353555, 3344344355355, 3344353443555, 3344353544355, 3344354343555, 3344354354355, 3344354433555, 3344355344355, 3344434353555, 3344435343555, 3344444355555, 3434343435555, 3434343543555, 3434344353555, 3434344355355, 3434344355535, 3434353443555, 3434354343555, 3434354354355, 3434434353555, 3434434355355, 3434435354355, 3434435435535, 3434435534355, 3434444355555, 3443444355555, 3443544435555, 3444354435555]